# MOE CNN

## MOE CNN 1

In [ ]:
#!/usr/bin/env python3
"""
MoE CNN Multi-MNIST Trainer with Rich Panels & Rounded Corners
ASCII-only performance indicators (E/G/F/P)
"""

import sys, os, random, math, time
from typing import List, Dict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from rich.console import Console, Group
from rich.panel import Panel
from rich.table import Table
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TaskProgressColumn
from rich import box

console = Console()

# ---------------- Utils ----------------
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# ---------------- Data ----------------
def load_dataset(name: str, root: str, train: bool, transform):
    name = name.lower()
    if name in ["mnist"]:
        ds = datasets.MNIST(root, train=train, download=True, transform=transform)
        n_cls = 10
    elif name in ["fashionmnist", "fashion"]:
        ds = datasets.FashionMNIST(root, train=train, download=True, transform=transform)
        n_cls = 10
    elif name in ["kmnist"]:
        ds = datasets.KMNIST(root, train=train, download=True, transform=transform)
        n_cls = 10
    elif name in ["emnist"]:
        ds = datasets.EMNIST(root, split='balanced', train=train, download=True, transform=transform)
        n_cls = 47
    else:
        raise ValueError(f"Unknown dataset {name}")
    return ds, n_cls

def get_multimnist_loaders(dataset_names: List[str], batch_size: int, root="./data", num_workers=2):
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    loaders = {}
    class_counts = {}
    dataset_info = []

    for name in dataset_names:
        train_ds, n_cls = load_dataset(name, root, True, transform)
        test_ds, _ = load_dataset(name, root, False, transform)
        class_counts[name] = n_cls
        per_bs = max(1, batch_size // len(dataset_names))
        loaders[name] = {
            "train_loader": DataLoader(train_ds, batch_size=per_bs, shuffle=True, num_workers=num_workers),
            "test_loader": DataLoader(test_ds, batch_size=per_bs, shuffle=False)
        }
        dataset_info.append(f"{name}: {len(train_ds)} train, {len(test_ds)} test, {n_cls} classes")

    info_text = "\n".join(dataset_info)
    console.print(Panel(info_text, title="Dataset Information", border_style="blue", box=box.ROUNDED))

    return loaders, class_counts

# ---------------- Model ----------------
class MoECNN(nn.Module):
    def __init__(self, num_experts, feature_dim, hidden_dim, num_classes,
                 router_hidden=64, k=1, gumbel=True, temperature=1.0):
        super().__init__()
        self.num_experts = num_experts
        self.k = k
        self.gumbel = gumbel
        self.temperature = temperature

        self.encoder = nn.Sequential(
            nn.Conv2d(1,32,3,1,1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,1,1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*7*7, feature_dim), nn.ReLU()
        )

        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(feature_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, num_classes))
            for _ in range(num_experts)
        ])

        self.router = nn.Sequential(
            nn.Linear(feature_dim, router_hidden),
            nn.ReLU(),
            nn.Linear(router_hidden, num_experts)
        )

    def forward(self, x, dataset_idx=None):
        feat = self.encoder(x)
        logits_router = self.router(feat)

        if self.gumbel and self.training:
            gumbel_noise = -torch.log(-torch.log(torch.rand_like(logits_router) + 1e-8) + 1e-8)
            logits_router = (logits_router + gumbel_noise) / self.temperature

        if self.k == 1:
            if self.gumbel and self.training:
                gate = F.softmax(logits_router, dim=1)
            else:
                topk_vals, topk_idx = logits_router.max(dim=1)
                gate = F.one_hot(topk_idx, num_classes=self.num_experts).float()
        else:
            topk_vals, topk_idx = torch.topk(logits_router, self.k, dim=1)
            gate = torch.zeros_like(logits_router)
            gate.scatter_(1, topk_idx, 1.0/self.k)

        expert_outs = torch.stack([e(feat) for e in self.experts], dim=1)
        gate_expanded = gate.unsqueeze(-1)
        out = (expert_outs * gate_expanded).sum(dim=1)

        aux_loss = (logits_router.mean(0)**2).mean()
        mean_gate = gate.mean(dim=0)
        return out, aux_loss, mean_gate

# ---------------- Training ----------------
def train_epoch(model, loaders, optimizer, criterion, device, dataset_names, batch_size):
    model.train()
    n_ds = len(dataset_names)
    iters = {name: iter(loaders[name]["train_loader"]) for name in dataset_names}
    steps = min(len(loaders[name]["train_loader"]) for name in dataset_names)

    all_losses = []
    all_gates = []

    with Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        BarColumn(),
        TaskProgressColumn(),
        console=console
    ) as progress:
        task = progress.add_task("Training...", total=steps)

        for step in range(steps):
            optimizer.zero_grad()
            total_loss = 0.0
            batch_gates = []

            for idx, name in enumerate(dataset_names):
                try:
                    x, y = next(iters[name])
                except StopIteration:
                    iters[name] = iter(loaders[name]["train_loader"])
                    x, y = next(iters[name])
                x, y = x.to(device), y.to(device)
                logits, aux, gate = model(x, idx)
                loss = criterion(logits, y) + aux
                total_loss += loss
                batch_gates.append(gate.detach().cpu())

            total_loss.backward()
            optimizer.step()
            all_losses.append(total_loss.item())
            all_gates.append(torch.stack(batch_gates).mean(0))
            progress.update(task, advance=1)

    return all_losses, all_gates

@torch.no_grad()
def eval_model(model, loaders, device, dataset_names):
    model.eval()
    accs = {}

    with Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        BarColumn(),
        TaskProgressColumn(),
        console=console
    ) as progress:
        eval_task = progress.add_task("Evaluating...", total=len(dataset_names))

        for idx, name in enumerate(dataset_names):
            loader = loaders[name]["test_loader"]
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                logits, _, _ = model(x, idx)
                pred = logits.argmax(1)
                correct += (pred==y).sum().item()
                total += y.size(0)
            accs[name] = correct/total
            progress.update(eval_task, advance=1)

    return accs

# ---------------- Metrics & Expert Usage ----------------
def create_expert_usage_table(gates, epoch, num_experts):
    table = Table(title=f"Expert Usage Statistics - Epoch {epoch}", show_header=False, box=box.ROUNDED)
    table.add_column("Expert", style="cyan", no_wrap=True)
    table.add_column("Usage %", style="magenta")
    table.add_column("Bar", style="green")

    mean_gates = torch.stack(gates).mean(0).numpy()
    max_usage = mean_gates.max()

    panel_width = console.width - 10
    bar_max_len = panel_width - 20

    for i in range(num_experts):
        usage = mean_gates[i]
        usage_pct = f"{usage*100:.1f}%"
        bar_length = int((usage / max_usage) * bar_max_len) if max_usage > 0 else 0
        bar = "\n" + "█" * bar_length + "░" * (bar_max_len - bar_length) + "\n"
        table.add_row(f"Expert {i}", usage_pct, bar)

    return table

def create_metrics_table(epoch, total_epochs, loss, accs):
    table = Table(title=f"Training Metrics - Epoch {epoch}/{total_epochs}")
    table.add_column("Dataset", style="cyan")
    table.add_column("Accuracy", style="green")
    table.add_column("Performance", style="yellow")

    for name, acc in accs.items():
        acc_pct = f"{acc*100:.2f}%"
        if acc >= 0.95:
            perf = "[E]"
        elif acc >= 0.90:
            perf = "[G]"
        elif acc >= 0.80:
            perf = "[F]"
        else:
            perf = "[P]"
        table.add_row(name, acc_pct, perf)

    table.add_row("", "", "")
    table.add_row("Loss", f"{loss:.4f}", "")

    return table

# ---------------- Main ----------------
def main(args):
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    startup_info = f"""
Device: {device}
Seed: {args.seed}
Epochs: {args.epochs}
Batch Size: {args.batch_size}
Learning Rate: {args.lr}
Experts: {args.experts}
Top-K: {args.k}
Gumbel: {not args.no_gumbel}
Temperature: {args.router_temp}
    """.strip()

    console.print(Panel(startup_info, title="🚀 Training Configuration", border_style="green", box=box.ROUNDED))

    loaders, class_counts = get_multimnist_loaders(
        args.dataset_names, args.batch_size,
        root=args.data_dir, num_workers=args.num_workers
    )

    num_classes = max(class_counts.values())
    model = MoECNN(
        args.experts, args.feature_dim, args.hidden_dim, num_classes,
        router_hidden=args.router_hidden, k=args.k,
        gumbel=not args.no_gumbel, temperature=args.router_temp
    )

    if args.load_path and os.path.exists(args.load_path):
        model.load_state_dict(torch.load(args.load_path, map_location=device))
        console.print(f"[bold blue]Model loaded from {args.load_path}[/]")

    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    criterion = nn.CrossEntropyLoss()

    total_params, trainable_params = count_parameters(model)
    model_info = f"""
Total Parameters: {total_params:,}
Trainable Parameters: {trainable_params:,}
Model Size: ~{total_params * 4 / 1024 / 1024:.1f} MB (FP32)
    """.strip()

    console.print(Panel(model_info, title="📊 Model Information", border_style="blue", box=box.ROUNDED))

    console.print(Panel("Starting Training...", title="🎯 Training Status", border_style="yellow", box=box.ROUNDED))

    for epoch in range(1, args.epochs + 1):
        start_time = time.time()
        losses, gates = train_epoch(model, loaders, optimizer, criterion, device, args.dataset_names, args.batch_size)
        epoch_time = time.time() - start_time

        if epoch % args.epochs_between_eval == 0:
            accs = eval_model(model, loaders, device, args.dataset_names)
            avg_loss = sum(losses) / len(losses)

            metrics_table = create_metrics_table(epoch, args.epochs, avg_loss, accs)
            expert_table = create_expert_usage_table(gates, epoch, args.experts)

            # Nested panel using Group instead of +
            nested_panel = Panel(
                Group(
                    Panel(metrics_table, title="Metrics", border_style="yellow", box=box.ROUNDED),
                    Panel(expert_table, title="Expert Usage", border_style="green", box=box.ROUNDED)
                ),
                title=f"Epoch {epoch} Summary",
                border_style="blue",
                box=box.ROUNDED
            )
            console.print(nested_panel)

            est_remaining = epoch_time * (args.epochs - epoch)
            est_str = f"{est_remaining/60:.1f} min" if est_remaining > 60 else f"{est_remaining:.0f}s"
            timing_info = f"Estimated remaining: {est_str}"
            console.print(Panel(timing_info, title="⏱️ Timing", border_style="cyan", box=box.ROUNDED))

    if args.save_path:
        os.makedirs(os.path.dirname(args.save_path) if os.path.dirname(args.save_path) else ".", exist_ok=True)
        torch.save(model.state_dict(), args.save_path)
        console.print(Panel(f"Model saved to: {args.save_path}", title="Model Saved", border_style="green", box=box.ROUNDED))

    console.print(Panel("Training Complete!", title="Success", border_style="green", box=box.ROUNDED))

# ---------------- CLI ----------------
if __name__=="__main__":
    import argparse
    parser = argparse.ArgumentParser(description="MoE CNN Multi-MNIST Trainer with Rich Output")

    parser.add_argument("--data_dir", type=str, default="./data", help="Data directory")
    parser.add_argument("--dataset_names", nargs="+", default=["MNIST","FashionMNIST","KMNIST","EMNIST"],
                       help="Datasets to train on")
    parser.add_argument("--num_workers", type=int, default=2, help="DataLoader workers")

    parser.add_argument("--epochs", type=int, default=5, help="Number of epochs")
    parser.add_argument("--batch_size", type=int, default=128, help="Batch size")
    parser.add_argument("--lr", type=float, default=1e-3, help="Learning rate")
    parser.add_argument("--epochs_between_eval", type=int, default=1, help="Epochs between evaluations")

    parser.add_argument("--experts", type=int, default=7, help="Number of experts")
    parser.add_argument("--feature_dim", type=int, default=128, help="Feature dimension")
    parser.add_argument("--hidden_dim", type=int, default=256, help="Hidden dimension")
    parser.add_argument("--k", type=int, default=1, help="Top-k experts to use")
    parser.add_argument("--router_hidden", type=int, default=64, help="Router hidden dimension")

    parser.add_argument("--no_gumbel", action="store_true", help="Disable Gumbel softmax")
    parser.add_argument("--router_temp", type=float, default=1.0, help="Router temperature")

    parser.add_argument("--save_path", type=str, default=None, help="Path to save model")
    parser.add_argument("--load_path", type=str, default=None, help="Path to load model")

    parser.add_argument("--seed", type=int, default=42)

    if "ipykernel" in sys.modules:
        args, _ = parser.parse_known_args()
    else:
        args = parser.parse_args()

    main(args)


╭─────────────────────────────────────────── 🚀 Training Configuration ───────────────────────────────────────────╮
│ Device: cuda                                                                                                    │
│ Seed: 42                                                                                                        │
│ Epochs: 5                                                                                                       │
│ Batch Size: 128                                                                                                 │
│ Learning Rate: 0.001                                                                                            │
│ Experts: 7                                                                                                      │
│ Top-K: 1                                                                                                        │
│ Gumbel: True                                                                                                    │
│ Temperature: 1.0                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Dataset Information ──────────────────────────────────────────────╮
│ MNIST: 60000 train, 10000 test, 10 classes                                                                      │
│ FashionMNIST: 60000 train, 10000 test, 10 classes                                                               │
│ KMNIST: 60000 train, 10000 test, 10 classes                                                                     │
│ EMNIST: 112800 train, 18800 test, 47 classes                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 📊 Model Information ──────────────────────────────────────────────╮
│ Total Parameters: 744,784                                                                                       │
│ Trainable Parameters: 744,784                                                                                   │
│ Model Size: ~2.8 MB (FP32)                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🎯 Training Status ───────────────────────────────────────────────╮
│ Starting Training...                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

## MOE CNN 2

In [ ]:
# #!/usr/bin/env python3
# """
# MoE CNN Multi-MNIST Trainer with Rich Panels & Complete Implementation
# Combines rich visual output with full feature set including Gumbel softmax,
# temperature control, flexible evaluation, and model loading.
# """

# import sys, os, random, math, time
# from typing import List, Dict
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import DataLoader, ConcatDataset
# from torchvision import datasets, transforms
# from tqdm import tqdm
# from rich.console import Console
# from rich.panel import Panel
# from rich.table import Table
# from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TaskProgressColumn

# console = Console()

# # ---------------- Utils ----------------
# def set_seed(seed=42):
#     random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)

# def count_parameters(model):
#     total = sum(p.numel() for p in model.parameters())
#     trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
#     return total, trainable

# # ---------------- Data ----------------
# def load_dataset(name: str, root: str, train: bool, transform):
#     # normalize name
#     name = name.lower()
#     if name in ["mnist"]:
#         ds = datasets.MNIST(root, train=train, download=True, transform=transform)
#         n_cls = 10
#     elif name in ["fashionmnist", "fashion"]:
#         ds = datasets.FashionMNIST(root, train=train, download=True, transform=transform)
#         n_cls = 10
#     elif name in ["kmnist"]:
#         ds = datasets.KMNIST(root, train=train, download=True, transform=transform)
#         n_cls = 10
#     elif name in ["emnist"]:
#         ds = datasets.EMNIST(root, split='balanced', train=train, download=True, transform=transform)
#         n_cls = 47
#     else:
#         raise ValueError(f"Unknown dataset {name}")
#     return ds, n_cls

# def get_multimnist_loaders(dataset_names: List[str], batch_size: int, root="./data", num_workers=2):
#     transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
#     loaders = {}
#     class_counts = {}
#     dataset_info = []

#     for name in dataset_names:
#         train_ds, n_cls = load_dataset(name, root, True, transform)
#         test_ds, _ = load_dataset(name, root, False, transform)
#         class_counts[name] = n_cls
#         per_bs = max(1, batch_size // len(dataset_names))
#         loaders[name] = {
#             "train_loader": DataLoader(train_ds, batch_size=per_bs, shuffle=True, num_workers=num_workers),
#             "test_loader": DataLoader(test_ds, batch_size=per_bs, shuffle=False)
#         }
#         dataset_info.append(f"{name}: {len(train_ds)} train, {len(test_ds)} test, {n_cls} classes")

#     # Display dataset info in a nice panel
#     info_text = "\n".join(dataset_info)
#     console.print(Panel(info_text, title="Dataset Information", border_style="blue"))

#     return loaders, class_counts

# # ---------------- Model ----------------
# class MoECNN(nn.Module):
#     def __init__(self, num_experts, feature_dim, hidden_dim, num_classes,
#                  router_hidden=64, k=1, gumbel=True, temperature=1.0):
#         super().__init__()
#         self.num_experts = num_experts
#         self.k = k
#         self.gumbel = gumbel
#         self.temperature = temperature

#         # shared encoder
#         self.encoder = nn.Sequential(
#             nn.Conv2d(1,32,3,1,1), nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Conv2d(32,64,3,1,1), nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Flatten(),
#             nn.Linear(64*7*7, feature_dim), nn.ReLU()
#         )

#         # experts
#         self.experts = nn.ModuleList([
#             nn.Sequential(nn.Linear(feature_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, num_classes))
#             for _ in range(num_experts)
#         ])

#         # router
#         self.router = nn.Sequential(
#             nn.Linear(feature_dim, router_hidden),
#             nn.ReLU(),
#             nn.Linear(router_hidden, num_experts)
#         )

#     def forward(self, x, dataset_idx=None):
#         feat = self.encoder(x)
#         logits_router = self.router(feat)

#         # Apply gumbel softmax if enabled
#         if self.gumbel and self.training:
#             # Gumbel softmax for differentiable routing
#             gumbel_noise = -torch.log(-torch.log(torch.rand_like(logits_router) + 1e-8) + 1e-8)
#             logits_router = (logits_router + gumbel_noise) / self.temperature

#         if self.k == 1:
#             if self.gumbel and self.training:
#                 gate = F.softmax(logits_router, dim=1)
#             else:
#                 topk_vals, topk_idx = logits_router.max(dim=1)
#                 gate = F.one_hot(topk_idx, num_classes=self.num_experts).float()
#         else:
#             topk_vals, topk_idx = torch.topk(logits_router, self.k, dim=1)
#             gate = torch.zeros_like(logits_router)
#             gate.scatter_(1, topk_idx, 1.0/self.k)

#         expert_outs = torch.stack([e(feat) for e in self.experts], dim=1)
#         gate_expanded = gate.unsqueeze(-1)
#         out = (expert_outs * gate_expanded).sum(dim=1)

#         # aux loss: encourage router to be balanced
#         aux_loss = (logits_router.mean(0)**2).mean()

#         # For stats: mean gate per expert
#         mean_gate = gate.mean(dim=0)
#         return out, aux_loss, mean_gate

# # ---------------- Training ----------------
# def train_epoch(model, loaders, optimizer, criterion, device, dataset_names, batch_size):
#     model.train()
#     n_ds = len(dataset_names)
#     per_bs = max(1, batch_size // n_ds)
#     iters = {name: iter(loaders[name]["train_loader"]) for name in dataset_names}
#     steps = min(len(loaders[name]["train_loader"]) for name in dataset_names)

#     all_losses = []
#     all_gates = []

#     with Progress(
#         SpinnerColumn(),
#         TextColumn("[progress.description]{task.description}"),
#         BarColumn(),
#         TaskProgressColumn(),
#         console=console
#     ) as progress:
#         task = progress.add_task("Training...", total=steps)

#         for step in range(steps):
#             optimizer.zero_grad()
#             total_loss = 0.0
#             batch_gates = []

#             for idx, name in enumerate(dataset_names):
#                 try:
#                     x, y = next(iters[name])
#                 except StopIteration:
#                     iters[name] = iter(loaders[name]["train_loader"])
#                     x, y = next(iters[name])
#                 x, y = x.to(device), y.to(device)
#                 logits, aux, gate = model(x, idx)
#                 loss = criterion(logits, y) + aux
#                 total_loss += loss
#                 batch_gates.append(gate.detach().cpu())

#             total_loss.backward()
#             optimizer.step()
#             all_losses.append(total_loss.item())
#             all_gates.append(torch.stack(batch_gates).mean(0))
#             progress.update(task, advance=1)

#     return all_losses, all_gates

# @torch.no_grad()
# def eval_model(model, loaders, device, dataset_names):
#     model.eval()
#     accs = {}

#     with Progress(
#         SpinnerColumn(),
#         TextColumn("[progress.description]{task.description}"),
#         BarColumn(),
#         TaskProgressColumn(),
#         console=console
#     ) as progress:
#         eval_task = progress.add_task("Evaluating...", total=len(dataset_names))

#         for idx, name in enumerate(dataset_names):
#             loader = loaders[name]["test_loader"]
#             correct, total = 0, 0
#             for x, y in loader:
#                 x, y = x.to(device), y.to(device)
#                 logits, _, _ = model(x, idx)
#                 pred = logits.argmax(1)
#                 correct += (pred==y).sum().item()
#                 total += y.size(0)
#             accs[name] = correct/total
#             progress.update(eval_task, advance=1)

#     return accs

# def create_expert_usage_table(gates, epoch, num_experts):
#     """Create a rich table showing expert usage statistics"""
#     table = Table(title=f"Expert Usage Statistics - Epoch {epoch}")
#     table.add_column("Expert", style="cyan", no_wrap=True)
#     table.add_column("Usage %", style="magenta")
#     table.add_column("Bar", style="green")

#     mean_gates = torch.stack(gates).mean(0).numpy()
#     max_usage = mean_gates.max()

#     for i in range(num_experts):
#         usage = mean_gates[i]
#         usage_pct = f"{usage*100:.1f}%"
#         # Create a simple bar visualization
#         bar_length = int((usage / max_usage) * 20) if max_usage > 0 else 0
#         bar = "█" * bar_length + "░" * (20 - bar_length)
#         table.add_row(f"Expert {i}", usage_pct, bar)

#     return table

# def create_metrics_table(epoch, total_epochs, loss, accs):
#     """Create a rich table showing training metrics"""
#     table = Table(title=f"Training Metrics - Epoch {epoch}/{total_epochs}")
#     table.add_column("Dataset", style="cyan")
#     table.add_column("Accuracy", style="green")
#     table.add_column("Performance", style="yellow")

#     for name, acc in accs.items():
#         acc_pct = f"{acc*100:.2f}%"
#         # Performance indicator
#         if acc >= 0.95:
#             perf = "🟢 Excellent"
#         elif acc >= 0.90:
#             perf = "🟡 Good"
#         elif acc >= 0.80:
#             perf = "🟠 Fair"
#         else:
#             perf = "🔴 Poor"
#         table.add_row(name, acc_pct, perf)

#     # Add loss row
#     table.add_row("", "", "")
#     table.add_row("Loss", f"{loss:.4f}", "")

#     return table

# # ---------------- Main ----------------
# def main(args):
#     set_seed(args.seed)
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#     # Display startup info
#     startup_info = f"""
# Device: {device}
# Seed: {args.seed}
# Epochs: {args.epochs}
# Batch Size: {args.batch_size}
# Learning Rate: {args.lr}
# Experts: {args.experts}
# Top-K: {args.k}
# Gumbel: {not args.no_gumbel}
# Temperature: {args.router_temp}
#     """.strip()

#     console.print(Panel(startup_info, title="🚀 Training Configuration", border_style="green"))

#     loaders, class_counts = get_multimnist_loaders(
#         args.dataset_names, args.batch_size,
#         root=args.data_dir, num_workers=args.num_workers
#     )

#     num_classes = max(class_counts.values())
#     model = MoECNN(
#         args.experts, args.feature_dim, args.hidden_dim, num_classes,
#         router_hidden=args.router_hidden, k=args.k,
#         gumbel=not args.no_gumbel, temperature=args.router_temp
#     )

#     # Load pre-trained model if specified
#     if args.load_path and os.path.exists(args.load_path):
#         model.load_state_dict(torch.load(args.load_path, map_location=device))
#         console.print(f"[bold blue]Model loaded from {args.load_path}[/]")

#     model.to(device)

#     optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
#     criterion = nn.CrossEntropyLoss()

#     total_params, trainable_params = count_parameters(model)
#     model_info = f"""
# Total Parameters: {total_params:,}
# Trainable Parameters: {trainable_params:,}
# Model Size: ~{total_params * 4 / 1024 / 1024:.1f} MB (FP32)
#     """.strip()

#     console.print(Panel(model_info, title="📊 Model Information", border_style="blue"))

#     # Training loop
#     console.print(Panel("Starting Training...", title="🎯 Training Status", border_style="yellow"))

#     for epoch in range(1, args.epochs + 1):
#         start_time = time.time()
#         losses, gates = train_epoch(model, loaders, optimizer, criterion, device, args.dataset_names, args.batch_size)
#         epoch_time = time.time() - start_time

#         if epoch % args.epochs_between_eval == 0:
#             accs = eval_model(model, loaders, device, args.dataset_names)
#             avg_loss = sum(losses) / len(losses)

#             # Create and display tables
#             metrics_table = create_metrics_table(epoch, args.epochs, avg_loss, accs)
#             expert_table = create_expert_usage_table(gates, epoch, args.experts)

#             console.print(metrics_table)
#             console.print(expert_table)

#             # Time info
#             time_info = f"Epoch Time: {epoch_time:.2f}s | Est. Remaining: {epoch_time * (args.epochs - epoch):.1f}s"
#             console.print(Panel(time_info, title="⏱️ Timing", border_style="cyan"))

#     # Save model if specified
#     if args.save_path:
#         os.makedirs(os.path.dirname(args.save_path) if os.path.dirname(args.save_path) else ".", exist_ok=True)
#         torch.save(model.state_dict(), args.save_path)
#         console.print(Panel(f"Model saved to: {args.save_path}", title="💾 Model Saved", border_style="green"))

#     console.print(Panel("Training Complete! 🎉", title="✅ Success", border_style="green"))

# # ---------------- CLI ----------------
# if __name__=="__main__":
#     import argparse
#     parser = argparse.ArgumentParser(description="MoE CNN Multi-MNIST Trainer with Rich Output")

#     # Data arguments
#     parser.add_argument("--data_dir", type=str, default="./data", help="Data directory")
#     parser.add_argument("--dataset_names", nargs="+", default=["MNIST","FashionMNIST","KMNIST","EMNIST"],
#                        help="Datasets to train on")
#     parser.add_argument("--num_workers", type=int, default=2, help="DataLoader workers")

#     # Training arguments
#     parser.add_argument("--epochs", type=int, default=5, help="Number of epochs")
#     parser.add_argument("--batch_size", type=int, default=128, help="Batch size")
#     parser.add_argument("--lr", type=float, default=1e-3, help="Learning rate")
#     parser.add_argument("--epochs_between_eval", type=int, default=1, help="Epochs between evaluations")

#     # Model arguments
#     parser.add_argument("--experts", type=int, default=2, help="Number of experts")
#     parser.add_argument("--feature_dim", type=int, default=128, help="Feature dimension")
#     parser.add_argument("--hidden_dim", type=int, default=256, help="Hidden dimension")
#     parser.add_argument("--k", type=int, default=1, help="Top-k experts to use")
#     parser.add_argument("--router_hidden", type=int, default=64, help="Router hidden dimension")

#     # Router arguments
#     parser.add_argument("--no_gumbel", action="store_true", help="Disable Gumbel softmax")
#     parser.add_argument("--router_temp", type=float, default=1.0, help="Router temperature")

#     # I/O arguments
#     parser.add_argument("--save_path", type=str, default=None, help="Path to save model")
#     parser.add_argument("--load_path", type=str, default=None, help="Path to load model")

#     # Other arguments
#     parser.add_argument("--seed", type=int, default=42, help="Random seed")

#     # Handle Jupyter/Colab environment
#     if "ipykernel" in sys.modules:
#         args, _ = parser.parse_known_args()
#     else:
#         args = parser.parse_args()

#     main(args)

╭─────────────────────────────────────────── 🚀 Training Configuration ───────────────────────────────────────────╮
│ Device: cuda                                                                                                    │
│ Seed: 42                                                                                                        │
│ Epochs: 5                                                                                                       │
│ Batch Size: 128                                                                                                 │
│ Learning Rate: 0.001                                                                                            │
│ Experts: 2                                                                                                      │
│ Top-K: 1                                                                                                        │
│ Gumbel: True                                                                                                    │
│ Temperature: 1.0                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Dataset Information ──────────────────────────────────────────────╮
│ MNIST: 60000 train, 10000 test, 10 classes                                                                      │
│ FashionMNIST: 60000 train, 10000 test, 10 classes                                                               │
│ KMNIST: 60000 train, 10000 test, 10 classes                                                                     │
│ EMNIST: 112800 train, 18800 test, 47 classes                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── 📊 Model Information ──────────────────────────────────────────────╮
│ Total Parameters: 518,944                                                                                       │
│ Trainable Parameters: 518,944                                                                                   │
│ Model Size: ~2.0 MB (FP32)                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🎯 Training Status ───────────────────────────────────────────────╮
│ Starting Training...                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

Output()

       Training Metrics - Epoch 5/5       
┏━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Dataset      ┃ Accuracy ┃ Performance  ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ MNIST        │ 98.32%   │ 🟢 Excellent │
│ FashionMNIST │ 90.02%   │ 🟡 Good      │
│ KMNIST       │ 93.40%   │ 🟡 Good      │
│ EMNIST       │ 84.03%   │ 🟠 Fair      │
│              │          │              │
│ Loss         │ 0.8758   │              │
└──────────────┴──────────┴──────────────┘

      Expert Usage Statistics - Epoch 5      
┏━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Expert   ┃ Usage % ┃ Bar                  ┃
┡━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ Expert 0 │ 50.0%   │ ███████████████████░ │
│ Expert 1 │ 50.0%   │ ████████████████████ │
└──────────┴─────────┴──────────────────────┘

╭─────────────────────────────────────────────────── ⏱️ Timing ────────────────────────────────────────────────────╮
│ Epoch Time: 81.74s | Est. Remaining: 0.0s                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── ✅ Success ───────────────────────────────────────────────────╮
│ Training Complete! 🎉                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## MOE CNN 3

In [ ]:
#!/usr/bin/env python3
"""
Complete MoE CNN Multi-MNIST Trainer with Comprehensive Analysis
Includes performance tracking, visualizations, and automated result packaging
"""

import sys, os, random, math, time, json, shutil, zipfile
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Rich imports for beautiful output
try:
    from rich.console import Console, Group
    from rich.panel import Panel
    from rich.table import Table
    from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TaskProgressColumn
    from rich import box
    RICH_AVAILABLE = True
    console = Console()
except ImportError:
    RICH_AVAILABLE = False
    print("Rich not available, using basic output")

# Set style
plt.style.use('default')
sns.set_palette("husl")

class ExperimentTracker:
    """Tracks all experiment metrics and handles visualization"""

    def __init__(self, save_dir: str):
        self.save_dir = Path(save_dir)
        self.create_directory_structure()

        # Initialize tracking variables
        self.metrics = {
            'epoch': [],
            'train_loss': [],
            'train_acc': [],
            'val_acc': {},
            'expert_usage': [],
            'routing_weights': [],
            'attention_patterns': [],
            'gradient_norms': [],
            'learning_rate': [],
            'batch_losses': [],
            'expert_specialization': [],
            'routing_entropy': [],
            'feature_activations': []
        }

        self.best_metrics = {
            'best_val_acc': 0.0,
            'best_epoch': 0,
            'convergence_epoch': None
        }

    def create_directory_structure(self):
        """Create organized directory structure for results"""
        dirs = [
            'models',
            'plots/training_curves',
            'plots/expert_analysis',
            'plots/attention_weights',
            'plots/routing_patterns',
            'plots/feature_analysis',
            'plots/animations',
            'data/metrics',
            'data/embeddings',
            'data/routing_logs',
            'logs',
            'config'
        ]

        for dir_path in dirs:
            (self.save_dir / dir_path).mkdir(parents=True, exist_ok=True)

    def log_epoch_metrics(self, epoch: int, train_loss: float, train_acc: float,
                         val_accs: Dict[str, float], expert_gates: torch.Tensor,
                         routing_weights: Optional[torch.Tensor] = None,
                         gradient_norms: Optional[Dict[str, float]] = None,
                         lr: float = 0.0):
        """Log metrics for current epoch"""

        self.metrics['epoch'].append(epoch)
        self.metrics['train_loss'].append(train_loss)
        self.metrics['train_acc'].append(train_acc)
        self.metrics['learning_rate'].append(lr)

        # Log validation accuracies
        for dataset, acc in val_accs.items():
            if dataset not in self.metrics['val_acc']:
                self.metrics['val_acc'][dataset] = []
            self.metrics['val_acc'][dataset].append(acc)

        # Log expert usage patterns
        expert_usage = expert_gates.cpu().numpy()
        self.metrics['expert_usage'].append(expert_usage)

        # Calculate routing entropy
        entropy = -np.sum(expert_usage * np.log(expert_usage + 1e-8))
        self.metrics['routing_entropy'].append(entropy)

        # Log routing weights if available
        if routing_weights is not None:
            self.metrics['routing_weights'].append(routing_weights.cpu().numpy())

        # Log gradient norms
        if gradient_norms is not None:
            self.metrics['gradient_norms'].append(gradient_norms)

        # Track best performance
        avg_val_acc = np.mean(list(val_accs.values()))
        if avg_val_acc > self.best_metrics['best_val_acc']:
            self.best_metrics['best_val_acc'] = avg_val_acc
            self.best_metrics['best_epoch'] = epoch

    def log_batch_metrics(self, batch_loss: float, expert_gates: torch.Tensor):
        """Log batch-level metrics"""
        self.metrics['batch_losses'].append(batch_loss)

    def save_metrics(self):
        """Save all metrics to JSON files"""
        metrics_file = self.save_dir / 'data/metrics/training_metrics.json'

        # Convert numpy arrays to lists for JSON serialization
        json_metrics = {}
        for key, value in self.metrics.items():
            if key == 'val_acc':
                json_metrics[key] = {k: [float(v) for v in vs] for k, vs in value.items()}
            elif isinstance(value, list) and len(value) > 0:
                if isinstance(value[0], np.ndarray):
                    json_metrics[key] = [arr.tolist() for arr in value]
                elif isinstance(value[0], dict):
                    json_metrics[key] = [{k: float(v) for k, v in d.items()} for d in value]
                else:
                    json_metrics[key] = [float(v) if isinstance(v, (int, float, np.number)) else v for v in value]
            else:
                json_metrics[key] = value

        with open(metrics_file, 'w') as f:
            json.dump(json_metrics, f, indent=2)

        # Save best metrics
        best_metrics_file = self.save_dir / 'data/metrics/best_metrics.json'
        with open(best_metrics_file, 'w') as f:
            json.dump(self.best_metrics, f, indent=2)

class MoECNN(nn.Module):
    def __init__(self, num_experts, feature_dim, hidden_dim, num_classes,
                 router_hidden=64, k=1, gumbel=True, temperature=1.0):
        super().__init__()
        self.num_experts = num_experts
        self.k = k
        self.gumbel = gumbel
        self.temperature = temperature

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*7*7, feature_dim), nn.ReLU()
        )

        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(feature_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, num_classes)
            )
            for _ in range(num_experts)
        ])

        self.router = nn.Sequential(
            nn.Linear(feature_dim, router_hidden),
            nn.ReLU(),
            nn.Linear(router_hidden, num_experts)
        )

    def forward(self, x, dataset_idx=None, return_attention=False):
        feat = self.encoder(x)
        logits_router = self.router(feat)

        if self.gumbel and self.training:
            gumbel_noise = -torch.log(-torch.log(torch.rand_like(logits_router) + 1e-8) + 1e-8)
            logits_router = (logits_router + gumbel_noise) / self.temperature

        if self.k == 1:
            if self.gumbel and self.training:
                gate = F.softmax(logits_router, dim=1)
            else:
                topk_vals, topk_idx = logits_router.max(dim=1)
                gate = F.one_hot(topk_idx, num_classes=self.num_experts).float()
        else:
            topk_vals, topk_idx = torch.topk(logits_router, self.k, dim=1)
            gate = torch.zeros_like(logits_router)
            gate.scatter_(1, topk_idx, 1.0/self.k)

        expert_outs = torch.stack([e(feat) for e in self.experts], dim=1)
        gate_expanded = gate.unsqueeze(-1)
        out = (expert_outs * gate_expanded).sum(dim=1)

        aux_loss = (logits_router.mean(0)**2).mean()
        mean_gate = gate.mean(dim=0)

        if return_attention:
            return out, aux_loss, mean_gate, gate, expert_outs, feat

        return out, aux_loss, mean_gate

class Visualizer:
    """Creates comprehensive visualizations and animations"""

    def __init__(self, tracker: ExperimentTracker):
        self.tracker = tracker
        self.save_dir = tracker.save_dir

    def create_training_curves(self):
        """Create comprehensive training curve plots"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Training Progress Analysis', fontsize=16, fontweight='bold')

        epochs = self.tracker.metrics['epoch']

        # Loss curve
        axes[0,0].plot(epochs, self.tracker.metrics['train_loss'], 'b-', linewidth=2, label='Training Loss')
        axes[0,0].set_title('Training Loss')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Loss')
        axes[0,0].grid(True, alpha=0.3)
        axes[0,0].legend()

        # Validation accuracies
        for dataset, accs in self.tracker.metrics['val_acc'].items():
            axes[0,1].plot(epochs, accs, linewidth=2, label=f'{dataset}', marker='o', markersize=3)
        axes[0,1].set_title('Validation Accuracy by Dataset')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Accuracy')
        axes[0,1].grid(True, alpha=0.3)
        axes[0,1].legend()
        axes[0,1].set_ylim(0, 1)

        # Learning rate
        if len(self.tracker.metrics['learning_rate']) > 0:
            axes[0,2].plot(epochs, self.tracker.metrics['learning_rate'], 'g-', linewidth=2)
            axes[0,2].set_title('Learning Rate Schedule')
            axes[0,2].set_xlabel('Epoch')
            axes[0,2].set_ylabel('Learning Rate')
            axes[0,2].grid(True, alpha=0.3)
            axes[0,2].set_yscale('log')

        # Routing entropy
        axes[1,0].plot(epochs, self.tracker.metrics['routing_entropy'], 'r-', linewidth=2)
        axes[1,0].set_title('Routing Entropy (Diversity)')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Entropy')
        axes[1,0].grid(True, alpha=0.3)

        # Expert usage over time (heatmap)
        if len(self.tracker.metrics['expert_usage']) > 0:
            expert_usage_matrix = np.array(self.tracker.metrics['expert_usage'])
            im = axes[1,1].imshow(expert_usage_matrix.T, aspect='auto', cmap='viridis', origin='lower')
            axes[1,1].set_title('Expert Usage Over Time')
            axes[1,1].set_xlabel('Epoch')
            axes[1,1].set_ylabel('Expert Index')
            plt.colorbar(im, ax=axes[1,1])

        # Gradient norms (if available)
        if len(self.tracker.metrics['gradient_norms']) > 0 and self.tracker.metrics['gradient_norms'][0]:
            grad_data = {}
            for grad_dict in self.tracker.metrics['gradient_norms']:
                for name, norm in grad_dict.items():
                    if name not in grad_data:
                        grad_data[name] = []
                    grad_data[name].append(norm)

            for name, norms in grad_data.items():
                if len(norms) == len(epochs):
                    axes[1,2].plot(epochs, norms, linewidth=2, label=name, alpha=0.7)

            axes[1,2].set_title('Gradient Norms')
            axes[1,2].set_xlabel('Epoch')
            axes[1,2].set_ylabel('Gradient Norm')
            axes[1,2].grid(True, alpha=0.3)
            axes[1,2].legend()
            axes[1,2].set_yscale('log')

        plt.tight_layout()
        plt.savefig(self.save_dir / 'plots/training_curves/comprehensive_training_curves.png',
                   dpi=300, bbox_inches='tight')
        plt.close()

    def create_expert_analysis(self):
        """Create expert specialization analysis"""
        if len(self.tracker.metrics['expert_usage']) == 0:
            return

        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle('Expert Analysis', fontsize=16, fontweight='bold')

        expert_usage = np.array(self.tracker.metrics['expert_usage'])
        epochs = self.tracker.metrics['epoch']
        num_experts = expert_usage.shape[1]

        # Expert usage evolution
        for i in range(num_experts):
            axes[0,0].plot(epochs, expert_usage[:, i], linewidth=2, label=f'Expert {i}', alpha=0.8)
        axes[0,0].set_title('Expert Usage Evolution')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Usage Probability')
        axes[0,0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[0,0].grid(True, alpha=0.3)

        # Final expert distribution
        final_usage = expert_usage[-1] if len(expert_usage) > 0 else np.ones(num_experts) / num_experts
        colors = plt.cm.Set3(np.linspace(0, 1, num_experts))
        axes[0,1].pie(final_usage, labels=[f'Expert {i}' for i in range(num_experts)],
                     autopct='%1.1f%%', colors=colors, startangle=90)
        axes[0,1].set_title('Final Expert Distribution')

        # Expert specialization heatmap
        im = axes[1,0].imshow(expert_usage.T, aspect='auto', cmap='RdYlBu_r', origin='lower')
        axes[1,0].set_title('Expert Usage Heatmap')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Expert Index')
        plt.colorbar(im, ax=axes[1,0])

        # Usage statistics
        mean_usage = np.mean(expert_usage, axis=0)
        std_usage = np.std(expert_usage, axis=0)
        x_pos = np.arange(num_experts)

        axes[1,1].bar(x_pos, mean_usage, yerr=std_usage, capsize=5, alpha=0.7, color=colors)
        axes[1,1].set_title('Expert Usage Statistics (Mean ± Std)')
        axes[1,1].set_xlabel('Expert Index')
        axes[1,1].set_ylabel('Usage Probability')
        axes[1,1].set_xticks(x_pos)
        axes[1,1].grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plt.savefig(self.save_dir / 'plots/expert_analysis/expert_specialization.png',
                   dpi=300, bbox_inches='tight')
        plt.close()

    def create_routing_analysis(self):
        """Create routing pattern analysis"""
        if len(self.tracker.metrics['routing_weights']) == 0:
            return

        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle('Routing Pattern Analysis', fontsize=16, fontweight='bold')

        # Routing weight distribution over time
        routing_data = np.array(self.tracker.metrics['routing_weights'])
        epochs = self.tracker.metrics['epoch']

        # Mean routing strength
        mean_routing = np.mean(routing_data, axis=(1, 2))  # Average across batch and experts
        axes[0,0].plot(epochs, mean_routing, 'b-', linewidth=2)
        axes[0,0].set_title('Mean Routing Strength')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Mean Weight')
        axes[0,0].grid(True, alpha=0.3)

        # Routing weight variance
        var_routing = np.var(routing_data, axis=(1, 2))
        axes[0,1].plot(epochs, var_routing, 'r-', linewidth=2)
        axes[0,1].set_title('Routing Weight Variance')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Variance')
        axes[0,1].grid(True, alpha=0.3)

        # Final epoch routing distribution
        if len(routing_data) > 0:
            final_routing = routing_data[-1]  # Last epoch
            axes[1,0].hist(final_routing.flatten(), bins=50, alpha=0.7, color='skyblue', edgecolor='black')
            axes[1,0].set_title('Final Routing Weight Distribution')
            axes[1,0].set_xlabel('Routing Weight')
            axes[1,0].set_ylabel('Frequency')
            axes[1,0].grid(True, alpha=0.3)

        # Routing entropy over time
        axes[1,1].plot(epochs, self.tracker.metrics['routing_entropy'], 'g-', linewidth=2)
        axes[1,1].set_title('Routing Entropy (Diversity)')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_ylabel('Entropy')
        axes[1,1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(self.save_dir / 'plots/routing_patterns/routing_analysis.png',
                   dpi=300, bbox_inches='tight')
        plt.close()

    def create_animations(self):
        """Create animated visualizations"""
        self._create_expert_usage_animation()
        self._create_loss_evolution_animation()
        self._create_routing_weights_animation()

    def _create_expert_usage_animation(self):
        """Create animated expert usage evolution"""
        if len(self.tracker.metrics['expert_usage']) == 0:
            return

        expert_usage = np.array(self.tracker.metrics['expert_usage'])
        epochs = self.tracker.metrics['epoch']
        num_experts = expert_usage.shape[1]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle('Expert Usage Evolution', fontsize=14, fontweight='bold')

        # Line plot setup
        lines = []
        colors = plt.cm.tab10(np.linspace(0, 1, num_experts))
        for i in range(num_experts):
            line, = ax1.plot([], [], linewidth=2, label=f'Expert {i}', color=colors[i])
            lines.append(line)

        ax1.set_xlim(0, max(epochs) if epochs else 1)
        ax1.set_ylim(0, 1)
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Usage Probability')
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax1.grid(True, alpha=0.3)

        # Bar plot setup
        bars = ax2.bar(range(num_experts), [0] * num_experts, color=colors, alpha=0.7)
        ax2.set_xlabel('Expert Index')
        ax2.set_ylabel('Usage Probability')
        ax2.set_ylim(0, 1)
        ax2.set_xticks(range(num_experts))
        ax2.grid(True, alpha=0.3, axis='y')

        def animate(frame):
            current_epoch = min(frame, len(epochs) - 1)

            # Update line plots
            for i, line in enumerate(lines):
                line.set_data(epochs[:current_epoch + 1], expert_usage[:current_epoch + 1, i])

            # Update bar plot
            current_usage = expert_usage[current_epoch]
            for bar, usage in zip(bars, current_usage):
                bar.set_height(usage)

            ax1.set_title(f'Expert Usage Evolution - Epoch {epochs[current_epoch] if current_epoch < len(epochs) else 0}')
            ax2.set_title(f'Current Distribution - Epoch {epochs[current_epoch] if current_epoch < len(epochs) else 0}')

            return lines + list(bars)

        anim = animation.FuncAnimation(fig, animate, frames=len(epochs), interval=200, blit=False)
        anim.save(self.save_dir / 'plots/animations/expert_usage_evolution.gif', writer='pillow', fps=5)
        plt.close()

    def _create_loss_evolution_animation(self):
        """Create animated loss evolution"""
        if len(self.tracker.metrics['train_loss']) == 0:
            return

        epochs = self.tracker.metrics['epoch']
        losses = self.tracker.metrics['train_loss']

        fig, ax = plt.subplots(figsize=(10, 6))
        fig.suptitle('Training Loss Evolution', fontsize=14, fontweight='bold')

        ax.set_xlim(0, max(epochs) if epochs else 1)
        ax.set_ylim(0, max(losses) * 1.1 if losses else 1)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Training Loss')
        ax.grid(True, alpha=0.3)

        line, = ax.plot([], [], 'b-', linewidth=2)
        point, = ax.plot([], [], 'ro', markersize=8)
        text = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=12,
                      verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        def animate(frame):
            current_epoch = min(frame, len(epochs) - 1)

            # Update line
            line.set_data(epochs[:current_epoch + 1], losses[:current_epoch + 1])

            # Update current point
            if current_epoch < len(epochs):
                point.set_data([epochs[current_epoch]], [losses[current_epoch]])
                text.set_text(f'Epoch: {epochs[current_epoch]}\nLoss: {losses[current_epoch]:.4f}')

            return line, point, text

        anim = animation.FuncAnimation(fig, animate, frames=len(epochs), interval=200, blit=False)
        anim.save(self.save_dir / 'plots/animations/loss_evolution.gif', writer='pillow', fps=5)
        plt.close()

    def _create_routing_weights_animation(self):
        """Create animated routing weights evolution"""
        if len(self.tracker.metrics['routing_weights']) == 0:
            return

        routing_data = np.array(self.tracker.metrics['routing_weights'])
        epochs = self.tracker.metrics['epoch']

        fig, ax = plt.subplots(figsize=(10, 8))
        fig.suptitle('Routing Weights Evolution', fontsize=14, fontweight='bold')

        # Use the first sample to get dimensions
        first_sample = routing_data[0]
        vmin, vmax = np.min(routing_data), np.max(routing_data)

        im = ax.imshow(first_sample, cmap='RdYlBu_r', aspect='auto', vmin=vmin, vmax=vmax)
        ax.set_xlabel('Expert Index')
        ax.set_ylabel('Sample Index')

        plt.colorbar(im, ax=ax)

        def animate(frame):
            current_epoch = min(frame, len(epochs) - 1)
            im.set_array(routing_data[current_epoch])
            ax.set_title(f'Routing Weights - Epoch {epochs[current_epoch] if current_epoch < len(epochs) else 0}')
            return [im]

        anim = animation.FuncAnimation(fig, animate, frames=len(epochs), interval=300, blit=False)
        anim.save(self.save_dir / 'plots/animations/routing_weights_evolution.gif', writer='pillow', fps=3)
        plt.close()

# Data loading functions
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def load_dataset(name: str, root: str, train: bool, transform):
    name = name.lower()
    if name in ["mnist"]:
        ds = datasets.MNIST(root, train=train, download=True, transform=transform)
        n_cls = 10
    elif name in ["fashionmnist", "fashion"]:
        ds = datasets.FashionMNIST(root, train=train, download=True, transform=transform)
        n_cls = 10
    elif name in ["kmnist"]:
        ds = datasets.KMNIST(root, train=train, download=True, transform=transform)
        n_cls = 10
    elif name in ["emnist"]:
        ds = datasets.EMNIST(root, split='balanced', train=train, download=True, transform=transform)
        n_cls = 47
    else:
        raise ValueError(f"Unknown dataset {name}")
    return ds, n_cls

def get_multimnist_loaders(dataset_names: List[str], batch_size: int, root="./data", num_workers=2):
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    loaders = {}
    class_counts = {}
    dataset_info = []

    for name in dataset_names:
        train_ds, n_cls = load_dataset(name, root, True, transform)
        test_ds, _ = load_dataset(name, root, False, transform)
        class_counts[name] = n_cls
        per_bs = max(1, batch_size // len(dataset_names))
        loaders[name] = {
            "train_loader": DataLoader(train_ds, batch_size=per_bs, shuffle=True, num_workers=num_workers),
            "test_loader": DataLoader(test_ds, batch_size=per_bs, shuffle=False)
        }
        dataset_info.append(f"{name}: {len(train_ds)} train, {len(test_ds)} test, {n_cls} classes")

    info_text = "\n".join(dataset_info)
    if RICH_AVAILABLE:
        console.print(Panel(info_text, title="Dataset Information", border_style="blue", box=box.ROUNDED))
    else:
        print("Dataset Information:")
        print(info_text)

    return loaders, class_counts

# Training functions
def compute_gradient_norms(model):
    """Compute gradient norms for different parts of the model"""
    grad_norms = {}

    for name, param in model.named_parameters():
        if param.grad is not None:
            grad_norm = param.grad.norm().item()
            layer_name = name.split('.')[0]  # Get main layer name
            if layer_name not in grad_norms:
                grad_norms[layer_name] = []
            grad_norms[layer_name].append(grad_norm)

    # Average gradients for each layer type
    for layer_name in grad_norms:
        grad_norms[layer_name] = np.mean(grad_norms[layer_name])

    return grad_norms

def train_epoch(model, loaders, optimizer, criterion, device, dataset_names, tracker, epoch):
    model.train()
    n_ds = len(dataset_names)
    iters = {name: iter(loaders[name]["train_loader"]) for name in dataset_names}
    steps = min(len(loaders[name]["train_loader"]) for name in dataset_names)

    all_losses = []
    all_gates = []
    all_routing_weights = []

    if RICH_AVAILABLE:
        with Progress(
            SpinnerColumn(),
            TextColumn("[progress.description]{task.description}"),
            BarColumn(),
            TaskProgressColumn(),
            console=console
        ) as progress:
            task = progress.add_task("Training...", total=steps)

            for step in range(steps):
                optimizer.zero_grad()
                total_loss = 0.0
                batch_gates = []
                batch_routing = []

                for idx, name in enumerate(dataset_names):
                    try:
                        x, y = next(iters[name])
                    except StopIteration:
                        iters[name] = iter(loaders[name]["train_loader"])
                        x, y = next(iters[name])

                    x, y = x.to(device), y.to(device)
                    logits, aux, gate, routing_weights, expert_outputs, features = model(x, idx, return_attention=True)
                    loss = criterion(logits, y) + 0.01 * aux  # Add auxiliary loss with weight
                    total_loss += loss
                    batch_gates.append(gate.detach().cpu())
                    batch_routing.append(routing_weights.detach().cpu())

                total_loss.backward()

                # Compute gradient norms
                grad_norms = compute_gradient_norms(model)

                optimizer.step()
                all_losses.append(total_loss.item())
                all_gates.append(torch.stack(batch_gates).mean(0))
                all_routing_weights.append(torch.stack(batch_routing).mean(0))

                # Log batch metrics
                tracker.log_batch_metrics(total_loss.item(), torch.stack(batch_gates).mean(0))

                progress.update(task, advance=1)
    else:
        for step in range(steps):
            optimizer.zero_grad()
            total_loss = 0.0
            batch_gates = []
            batch_routing = []

            for idx, name in enumerate(dataset_names):
                try:
                    x, y = next(iters[name])
                except StopIteration:
                    iters[name] = iter(loaders[name]["train_loader"])
                    x, y = next(iters[name])

                x, y = x.to(device), y.to(device)
                logits, aux, gate, routing_weights, expert_outputs, features = model(x, idx, return_attention=True)
                loss = criterion(logits, y) + 0.01 * aux
                total_loss += loss
                batch_gates.append(gate.detach().cpu())
                batch_routing.append(routing_weights.detach().cpu())

            total_loss.backward()
            grad_norms = compute_gradient_norms(model)
            optimizer.step()
            all_losses.append(total_loss.item())
            all_gates.append(torch.stack(batch_gates).mean(0))
            all_routing_weights.append(torch.stack(batch_routing).mean(0))

            tracker.log_batch_metrics(total_loss.item(), torch.stack(batch_gates).mean(0))

            if step % 50 == 0:
                print(f"Step {step}/{steps}, Loss: {total_loss.item():.4f}")

    return all_losses, all_gates, all_routing_weights, grad_norms

@torch.no_grad()
def eval_model(model, loaders, device, dataset_names):
    model.eval()
    accs = {}

    if RICH_AVAILABLE:
        with Progress(
            SpinnerColumn(),
            TextColumn("[progress.description]{task.description}"),
            BarColumn(),
            TaskProgressColumn(),
            console=console
        ) as progress:
            eval_task = progress.add_task("Evaluating...", total=len(dataset_names))

            for idx, name in enumerate(dataset_names):
                loader = loaders[name]["test_loader"]
                correct, total = 0, 0
                for x, y in loader:
                    x, y = x.to(device), y.to(device)
                    logits, _, _ = model(x, idx)
                    pred = logits.argmax(1)
                    correct += (pred==y).sum().item()
                    total += y.size(0)
                accs[name] = correct/total
                progress.update(eval_task, advance=1)
    else:
        for idx, name in enumerate(dataset_names):
            loader = loaders[name]["test_loader"]
            correct, total = 0, 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                logits, _, _ = model(x, idx)
                pred = logits.argmax(1)
                correct += (pred==y).sum().item()
                total += y.size(0)
            accs[name] = correct/total
            print(f"{name} Accuracy: {accs[name]:.4f}")

    return accs

def create_rich_tables(epoch, total_epochs, loss, accs, expert_gates, num_experts):
    """Create Rich tables for display"""
    if not RICH_AVAILABLE:
        return None, None

    # Metrics table
    metrics_table = Table(title=f"Training Metrics - Epoch {epoch}/{total_epochs}")
    metrics_table.add_column("Dataset", style="cyan")
    metrics_table.add_column("Accuracy", style="green")
    metrics_table.add_column("Performance", style="yellow")

    for name, acc in accs.items():
        acc_pct = f"{acc*100:.2f}%"
        if acc >= 0.95:
            perf = "[E]"
        elif acc >= 0.90:
            perf = "[G]"
        elif acc >= 0.80:
            perf = "[F]"
        else:
            perf = "[P]"
        metrics_table.add_row(name, acc_pct, perf)

    metrics_table.add_row("", "", "")
    metrics_table.add_row("Loss", f"{loss:.4f}", "")

    # Expert usage table
    expert_table = Table(title=f"Expert Usage - Epoch {epoch}", show_header=False, box=box.ROUNDED)
    expert_table.add_column("Expert", style="cyan", no_wrap=True)
    expert_table.add_column("Usage %", style="magenta")
    expert_table.add_column("Bar", style="green")

    mean_gates = expert_gates.numpy()
    max_usage = mean_gates.max()

    for i in range(num_experts):
        usage = mean_gates[i]
        usage_pct = f"{usage*100:.1f}%"
        bar_length = int((usage / max_usage) * 20) if max_usage > 0 else 0
        bar = "█" * bar_length + "░" * (20 - bar_length)
        expert_table.add_row(f"Expert {i}", usage_pct, bar)

    return metrics_table, expert_table

def create_download_package(tracker: ExperimentTracker, model: nn.Module, args):
    """Create a comprehensive download package with all results"""

    # Save final model
    model_path = tracker.save_dir / 'models/final_model.pth'
    torch.save(model.state_dict(), model_path)

    # Save model architecture info
    total_params, trainable_params = count_parameters(model)
    model_info = {
        'architecture': str(model),
        'total_parameters': total_params,
        'trainable_parameters': trainable_params,
        'model_size_mb': total_params * 4 / 1024 / 1024,
        'hyperparameters': vars(args)
    }

    with open(tracker.save_dir / 'models/model_info.json', 'w') as f:
        json.dump(model_info, f, indent=2, default=str)

    # Save experiment config
    config = {
        'timestamp': datetime.now().isoformat(),
        'hyperparameters': vars(args),
        'best_metrics': tracker.best_metrics,
        'total_epochs_trained': len(tracker.metrics['epoch']),
        'datasets_used': args.dataset_names
    }

    with open(tracker.save_dir / 'config/experiment_config.json', 'w') as f:
        json.dump(config, f, indent=2)

    # Create comprehensive report
    report = generate_experiment_report(tracker, model_info, config)
    with open(tracker.save_dir / 'experiment_report.md', 'w') as f:
        f.write(report)

    # Create zip package
    zip_path = f"moe_experiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(tracker.save_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arc_name = os.path.relpath(file_path, tracker.save_dir)
                zipf.write(file_path, arc_name)

    file_size = os.path.getsize(zip_path) / (1024 * 1024)  # MB

    if RICH_AVAILABLE:
        success_panel = Panel(
            f"""
Package created successfully!

📦 File: {zip_path}
📊 Size: {file_size:.1f} MB
📁 Contents:
  • Trained model (final_model.pth)
  • Training metrics and logs
  • Comprehensive visualizations
  • Animated GIFs showing training progress
  • Expert analysis and routing patterns
  • Experiment configuration and report

🚀 Ready for download and analysis!
            """.strip(),
            title="Download Package Created",
            border_style="green",
            box=box.ROUNDED
        )
        console.print(success_panel)
    else:
        print(f"\nPackage created: {zip_path} ({file_size:.1f} MB)")
        print("Contents: model, metrics, plots, animations, config, and report")

    return zip_path

def generate_experiment_report(tracker: ExperimentTracker, model_info: dict, config: dict) -> str:
    """Generate a comprehensive experiment report"""

    best_acc = tracker.best_metrics['best_val_acc']
    best_epoch = tracker.best_metrics['best_epoch']

    report = f"""# MoE CNN Experiment Report

## Experiment Overview
- **Timestamp**: {config['timestamp']}
- **Total Epochs**: {config['total_epochs_trained']}
- **Datasets**: {', '.join(config['datasets_used'])}
- **Best Validation Accuracy**: {best_acc:.4f} (Epoch {best_epoch})

## Model Architecture
- **Total Parameters**: {model_info['total_parameters']:,}
- **Trainable Parameters**: {model_info['trainable_parameters']:,}
- **Model Size**: {model_info['model_size_mb']:.1f} MB

## Hyperparameters
"""

    for key, value in config['hyperparameters'].items():
        if key not in ['dataset_names']:  # Skip list parameters
            report += f"- **{key}**: {value}\n"

    report += f"""
## Training Results

### Performance Summary
"""

    if tracker.metrics['val_acc']:
        final_epoch = len(tracker.metrics['epoch']) - 1
        for dataset in tracker.metrics['val_acc']:
            if len(tracker.metrics['val_acc'][dataset]) > final_epoch:
                final_acc = tracker.metrics['val_acc'][dataset][final_epoch]
                report += f"- **{dataset}**: {final_acc:.4f}\n"

    if tracker.metrics['train_loss']:
        final_loss = tracker.metrics['train_loss'][-1]
        report += f"- **Final Training Loss**: {final_loss:.4f}\n"

    report += f"""
### Expert Usage Analysis
"""

    if tracker.metrics['expert_usage']:
        final_expert_usage = tracker.metrics['expert_usage'][-1]
        for i, usage in enumerate(final_expert_usage):
            report += f"- **Expert {i}**: {usage:.3f} ({usage*100:.1f}%)\n"

    report += f"""
## Files Included

### Models
- `models/final_model.pth` - Trained model state dict
- `models/model_info.json` - Model architecture and parameter info

### Data & Metrics
- `data/metrics/training_metrics.json` - Complete training metrics
- `data/metrics/best_metrics.json` - Best performance metrics

### Visualizations
- `plots/training_curves/` - Training progress charts
- `plots/expert_analysis/` - Expert specialization analysis
- `plots/routing_patterns/` - Routing behavior analysis
- `plots/animations/` - Animated training visualizations

### Configuration
- `config/experiment_config.json` - Complete experiment configuration
- `experiment_report.md` - This comprehensive report

## Usage Instructions

### Loading the Model
```python
import torch
from your_model_file import MoECNN

model = MoECNN(
    num_experts={config['hyperparameters'].get('experts', 7)},
    feature_dim={config['hyperparameters'].get('feature_dim', 128)},
    hidden_dim={config['hyperparameters'].get('hidden_dim', 256)},
    num_classes=47,  # Adjust based on your dataset
    k={config['hyperparameters'].get('k', 1)}
)

model.load_state_dict(torch.load('models/final_model.pth'))
model.eval()
```

### Analyzing Results
The `data/metrics/training_metrics.json` file contains all training metrics that can be loaded and analyzed:

```python
import json
with open('data/metrics/training_metrics.json', 'r') as f:
    metrics = json.load(f)

# Plot custom analysis
import matplotlib.pyplot as plt
plt.plot(metrics['epoch'], metrics['train_loss'])
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.show()
```

---
*Report generated automatically by MoE CNN Training System*
"""

    return report

def main(args):
    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Create experiment tracker
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_dir = f"moe_experiment_{timestamp}"
    tracker = ExperimentTracker(save_dir)

    startup_info = f"""
Device: {device}
Seed: {args.seed}
Epochs: {args.epochs}
Batch Size: {args.batch_size}
Learning Rate: {args.lr}
Experts: {args.experts}
Top-K: {args.k}
Gumbel: {not args.no_gumbel}
Temperature: {args.router_temp}
Save Directory: {save_dir}
    """.strip()

    if RICH_AVAILABLE:
        console.print(Panel(startup_info, title="🚀 Training Configuration", border_style="green", box=box.ROUNDED))
    else:
        print("Training Configuration:")
        print(startup_info)

    # Load data
    loaders, class_counts = get_multimnist_loaders(
        args.dataset_names, args.batch_size,
        root=args.data_dir, num_workers=args.num_workers
    )

    # Create model
    num_classes = max(class_counts.values())
    model = MoECNN(
        args.experts, args.feature_dim, args.hidden_dim, num_classes,
        router_hidden=args.router_hidden, k=args.k,
        gumbel=not args.no_gumbel, temperature=args.router_temp
    )

    if args.load_path and os.path.exists(args.load_path):
        model.load_state_dict(torch.load(args.load_path, map_location=device))
        if RICH_AVAILABLE:
            console.print(f"[bold blue]Model loaded from {args.load_path}[/]")
        else:
            print(f"Model loaded from {args.load_path}")

    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    criterion = nn.CrossEntropyLoss()

    total_params, trainable_params = count_parameters(model)
    model_info = f"""
Total Parameters: {total_params:,}
Trainable Parameters: {trainable_params:,}
Model Size: ~{total_params * 4 / 1024 / 1024:.1f} MB (FP32)
    """.strip()

    if RICH_AVAILABLE:
        console.print(Panel(model_info, title="📊 Model Information", border_style="blue", box=box.ROUNDED))
        console.print(Panel("Starting Training...", title="🎯 Training Status", border_style="yellow", box=box.ROUNDED))
    else:
        print("Model Information:")
        print(model_info)
        print("\nStarting Training...")

    # Training loop
    for epoch in range(1, args.epochs + 1):
        start_time = time.time()

        # Training
        losses, gates, routing_weights, grad_norms = train_epoch(
            model, loaders, optimizer, criterion, device,
            args.dataset_names, tracker, epoch
        )

        epoch_time = time.time() - start_time

        # Evaluation
        if epoch % args.epochs_between_eval == 0:
            accs = eval_model(model, loaders, device, args.dataset_names)
            avg_loss = sum(losses) / len(losses)
            mean_gates = torch.stack(gates).mean(0)

            # Log metrics
            tracker.log_epoch_metrics(
                epoch, avg_loss, 0.0, accs, mean_gates,
                torch.stack(routing_weights).mean(0) if routing_weights else None,
                grad_norms, args.lr
            )

            # Display results
            if RICH_AVAILABLE:
                metrics_table, expert_table = create_rich_tables(
                    epoch, args.epochs, avg_loss, accs, mean_gates, args.experts
                )

                nested_panel = Panel(
                    Group(
                        Panel(metrics_table, title="Metrics", border_style="yellow", box=box.ROUNDED),
                        Panel(expert_table, title="Expert Usage", border_style="green", box=box.ROUNDED)
                    ),
                    title=f"Epoch {epoch} Summary",
                    border_style="blue",
                    box=box.ROUNDED
                )
                console.print(nested_panel)

                est_remaining = epoch_time * (args.epochs - epoch)
                est_str = f"{est_remaining/60:.1f} min" if est_remaining > 60 else f"{est_remaining:.0f}s"
                timing_info = f"Epoch time: {epoch_time:.1f}s | Estimated remaining: {est_str}"
                console.print(Panel(timing_info, title="⏱️ Timing", border_style="cyan", box=box.ROUNDED))
            else:
                print(f"\nEpoch {epoch}/{args.epochs}")
                print(f"Loss: {avg_loss:.4f}")
                for name, acc in accs.items():
                    print(f"{name} Accuracy: {acc:.4f}")
                print(f"Time: {epoch_time:.1f}s")

        # Save intermediate model
        if epoch % 50 == 0:
            model_path = tracker.save_dir / f'models/model_epoch_{epoch}.pth'
            torch.save(model.state_dict(), model_path)

    # Save final metrics
    tracker.save_metrics()

    # Create visualizations
    if RICH_AVAILABLE:
        console.print(Panel("Creating visualizations...", title="📊 Analysis", border_style="blue", box=box.ROUNDED))
    else:
        print("Creating visualizations...")

    visualizer = Visualizer(tracker)
    visualizer.create_training_curves()
    visualizer.create_expert_analysis()
    visualizer.create_routing_analysis()
    visualizer.create_animations()

    # Create download package
    if RICH_AVAILABLE:
        console.print(Panel("Packaging results...", title="📦 Finalizing", border_style="yellow", box=box.ROUNDED))
    else:
        print("Packaging results...")

    zip_path = create_download_package(tracker, model, args)

    if RICH_AVAILABLE:
        console.print(Panel("Training Complete! 🎉", title="✅ Success", border_style="green", box=box.ROUNDED))
    else:
        print(f"\nTraining Complete!")
        print(f"Results package: {zip_path}")

    return zip_path

if __name__=="__main__":
    import argparse
    parser = argparse.ArgumentParser(description="Complete MoE CNN Multi-MNIST Trainer with Analysis")

    # Data arguments
    parser.add_argument("--data_dir", type=str, default="./data", help="Data directory")
    parser.add_argument("--dataset_names", nargs="+", default=["MNIST","FashionMNIST","KMNIST","EMNIST"],
                       help="Datasets to train on")
    parser.add_argument("--num_workers", type=int, default=2, help="DataLoader workers")

    # Training arguments
    parser.add_argument("--epochs", type=int, default=5, help="Number of epochs") # 250
    parser.add_argument("--batch_size", type=int, default=128, help="Batch size")
    parser.add_argument("--lr", type=float, default=1e-3, help="Learning rate")
    parser.add_argument("--epochs_between_eval", type=int, default=5, help="Epochs between evaluations")

    # Model arguments
    parser.add_argument("--experts", type=int, default=7, help="Number of experts")
    parser.add_argument("--feature_dim", type=int, default=128, help="Feature dimension")
    parser.add_argument("--hidden_dim", type=int, default=256, help="Hidden dimension")
    parser.add_argument("--k", type=int, default=1, help="Top-k experts to use")
    parser.add_argument("--router_hidden", type=int, default=64, help="Router hidden dimension")

    # Router arguments
    parser.add_argument("--no_gumbel", action="store_true", help="Disable Gumbel softmax")
    parser.add_argument("--router_temp", type=float, default=1.0, help="Router temperature")

    # I/O arguments
    parser.add_argument("--save_path", type=str, default=None, help="Path to save model")
    parser.add_argument("--load_path", type=str, default=None, help="Path to load model")

    # Other arguments
    parser.add_argument("--seed", type=int, default=42)

    # Handle Colab/Jupyter environments
    if "ipykernel" in sys.modules or "google.colab" in sys.modules:
        args, _ = parser.parse_known_args()
    else:
        args = parser.parse_args()

    main(args)

# NANOGPT MOE

In [ ]:
import os
os.environ["MOE_ARGS"] = "--mode train_gpt --epochs 3 --gpt_steps_per_epoch 50 --batch_size 64 --save_path artifacts/gpt_moe.pt"


In [ ]:
#!/usr/bin/env python3
# moe_suite.py (all-in-one)
# - Dense MoE classifier (toy blobs)
# - Conv MoE classifier (MNIST)
# - NanoGPT-style Transformer with optional MoE-FFN (Tiny Shakespeare)
#
# Router specialization aids:
#   * learnable shared_scale (weakens shared at start)
#   * entropy bonus to sharpen routing
#   * temperature annealing (explore -> commit)
#   * optional Gumbel noise
#
# Colab-friendly: If no --mode provided, uses MOE_ARGS env or a safe default.

import os, math, json, argparse, random, time
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Any, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# Optional torchvision for MNIST
try:
    from torchvision import datasets, transforms
    _HAS_TORCHVISION = True
except Exception:
    _HAS_TORCHVISION = False

# Pretty terminal
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box

console = Console()

# ======================================================================
# Utils
# ======================================================================

def set_seed(seed: int = 1337):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(module: nn.Module) -> Tuple[int, int]:
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

def kl_to_uniform(probs: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    """KL(mean(probs) || uniform) to globally balance usage."""
    E = probs.size(-1)
    m = probs.mean(dim=0)  # [E]
    return torch.sum(m * (m.add(eps).log() - math.log(1.0 / E)))

def entropy_mean(probs: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    """Mean entropy of per-item routing distributions (maximize sharpness)."""
    # probs: [N,E]
    return -(probs.clamp_min(eps) * probs.clamp_min(eps).log()).sum(dim=-1).mean()

def rich_params_panel(title: str, total: int, trainable: int, rows: List[Tuple[str, str]]) -> Panel:
    t = Table(box=box.SIMPLE_HEAVY)
    t.add_column("Metric", style="bold")
    t.add_column("Value", justify="right")
    t.add_row("Total params", f"{total:,}")
    t.add_row("Trainable params", f"{trainable:,}")
    for k, v in rows:
        t.add_row(k, v)
    return Panel(t, title=title, border_style="yellow", box=box.ROUNDED, expand=True)

def rich_routing_panel(stats: Dict[str, torch.Tensor], title: str) -> Panel:
    counts = stats["expert_selection_counts"].detach().cpu().to(torch.int64).tolist()
    mean_probs = stats["mean_routing_probs"].detach().cpu().tolist()
    E = len(counts)
    t = Table(box=box.SIMPLE_HEAVY)
    t.add_column("Expert", style="bold")
    t.add_column("Batch Picks", justify="right")
    t.add_column("Mean Prob.", justify="right")
    for e in range(E):
        t.add_row(f"E{e}", f"{counts[e]}", f"{mean_probs[e]:.3f}")
    ent = entropy_mean(torch.tensor(mean_probs)[None, :]).item()
    return Panel(t, title=f"{title}  |  entropy={ent:.3f}", border_style="green", box=box.ROUNDED, expand=True)

# ======================================================================
# Data: blobs (dense), MNIST (conv), Tiny Shakespeare (text)
# ======================================================================

def make_toy_blobs(n_per_class=1200, dim=2, n_classes=3, spread=1.2, seed=0, device="cpu"):
    torch.manual_seed(seed)
    centers = torch.randn(n_classes, dim) * 5.0
    xs, ys = [], []
    for c in range(n_classes):
        xs.append(centers[c] + torch.randn(n_per_class, dim) * spread)
        ys.append(torch.full((n_per_class,), c, dtype=torch.long))
    x = torch.cat(xs, 0).to(device)
    y = torch.cat(ys, 0).to(device)
    perm = torch.randperm(x.size(0), device=device)
    return x[perm], y[perm]

def get_mnist_loaders(data_dir: str, batch_size: int):
    if not _HAS_TORCHVISION:
        raise RuntimeError("torchvision not available. Install it to use MNIST.")
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    train_ds = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)
    test_ds  = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    test_dl  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_dl, test_dl

import urllib.request

def _repeat_to_len(s: str, target_len: int) -> str:
    if not s:
        s = " \n"
    reps = (target_len // len(s)) + 1
    return (s * reps)[:target_len]

def load_tiny_shakespeare(data_dir: str = "./data", target_len: int = 100_000) -> Tuple[str, str]:
    """
    Tries to download full Tiny Shakespeare; if offline, repeats a short sample.
    Returns (train_text, val_text).
    """
    os.makedirs(data_dir, exist_ok=True)
    full_path = os.path.join(data_dir, "tinyshakespeare_input.txt")
    if not os.path.exists(full_path):
        try:
            url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url, full_path)
        except Exception:
            sample = (
                "From fairest creatures we desire increase,\n"
                "That thereby beauty's rose might never die,\n"
                "But as the riper should by time decease,\n"
                "His tender heir might bear his memory:\n"
            )
            sample = _repeat_to_len(sample, target_len)
            with open(full_path, "w", encoding="utf-8") as f:
                f.write(sample)
    with open(full_path, "r", encoding="utf-8") as f:
        text = f.read()
    split = int(0.9 * len(text))
    return text[:split], text[split:]

def build_charset(train: str, val: str):
    text = train + val
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

def encode(s: str, stoi: Dict[str,int]) -> List[int]:
    return [stoi[c] for c in s]

def get_batch_text(data: List[int], block_size: int, batch_size: int, device: str):
    if len(data) <= block_size + 1:
        raise RuntimeError(
            f"Text length {len(data)} is too small for block_size {block_size}. "
            f"Use a smaller --gpt_block_size or ensure dataset is larger."
        )
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([torch.tensor(data[i:i+block_size]) for i in ix]).long()
    y = torch.stack([torch.tensor(data[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# ======================================================================
# MoE core: Router + Shared + Experts (dense); CNN; token-level for GPT
# ======================================================================

class TopKRouter(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=1,
                 add_gumbel_noise=True, temperature=1.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, num_experts)
        )
        self.k = k
        self.add_gumbel_noise = add_gumbel_noise
        self.temperature = float(temperature)

    @staticmethod
    def _gumbel(shape, device):
        u = torch.rand(shape, device=device).clamp_(1e-9, 1-1e-9)
        return -torch.log(-torch.log(u))

    def forward(self, x):
        logits = self.net(x)
        if self.add_gumbel_noise and self.training:
            logits = logits + self._gumbel(logits.shape, logits.device)
        probs = F.softmax(logits / self.temperature, dim=-1)
        if self.k >= probs.size(-1):
            sparse_probs = probs
        else:
            topk_vals, topk_idx = torch.topk(probs, k=self.k, dim=-1)
            mask = torch.zeros_like(probs)
            mask.scatter_(dim=-1, index=topk_idx, src=torch.ones_like(topk_vals))
            sparse_probs = probs * mask
            sparse_probs = sparse_probs / (sparse_probs.sum(dim=-1, keepdim=True) + 1e-9)
        return sparse_probs, probs

# ----- Dense MoE -----
class MLPExpert(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, out_dim), nn.ReLU()
        )
    def forward(self, x): return self.net(x)

class MoESharedTopK(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=1, router_hidden=64,
                 balance_loss_weight=0.02, temperature=1.0, add_gumbel_noise=True,
                 shared_init_scale=0.3, entropy_bonus_weight=0.001):
        super().__init__()
        self.shared = MLPExpert(in_dim, hidden_dim, hidden_dim)
        self.experts = nn.ModuleList([MLPExpert(in_dim, hidden_dim, hidden_dim) for _ in range(num_experts)])
        self.router = TopKRouter(in_dim, router_hidden, num_experts, k, add_gumbel_noise, temperature)
        self.balance_loss_weight = balance_loss_weight
        self.entropy_bonus_weight = entropy_bonus_weight
        self.shared_scale = nn.Parameter(torch.tensor(float(shared_init_scale)))
        self.route_norm = nn.LayerNorm(in_dim)

    def forward(self, x):
        shared_out = self.shared(x) * self.shared_scale
        # route on normalized inputs
        sparse_probs, dense_probs = self.router(self.route_norm(x))
        expert_outs = [e(x) for e in self.experts]                 # list of [B,H]
        stack = torch.stack(expert_outs, dim=0)                    # [E,B,H]
        gates = sparse_probs.transpose(0,1).unsqueeze(-1)          # [E,B,1]
        routed = (gates * stack).sum(dim=0)                        # [B,H]
        y = shared_out + routed

        # aux: global balance - lambda * mean entropy (sharp routing)
        ent = entropy_mean(dense_probs)
        aux = self.balance_loss_weight * kl_to_uniform(dense_probs) - self.entropy_bonus_weight * ent

        with torch.no_grad():
            counts = (sparse_probs > 0).float().sum(0)             # [E]
        stats = {"expert_selection_counts": counts, "mean_routing_probs": dense_probs.mean(0)}
        return y, aux, stats

class TinyMoEClassifier(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=64, num_experts=3, k=1, n_classes=3,
                 router_hidden=64, balance_loss_weight=0.02,
                 shared_init_scale=0.3, entropy_bonus_weight=0.001,
                 router_temperature=1.0, add_gumbel_noise=True):
        super().__init__()
        self.pre = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU())
        self.moe = MoESharedTopK(
            hidden_dim, hidden_dim, num_experts, k,
            router_hidden, balance_loss_weight,
            temperature=router_temperature, add_gumbel_noise=add_gumbel_noise,
            shared_init_scale=shared_init_scale, entropy_bonus_weight=entropy_bonus_weight
        )
        self.post = nn.Sequential(nn.ReLU(), nn.Linear(hidden_dim, n_classes))
    def forward(self, x):
        if x.dim() != 2:
            raise RuntimeError(f"Dense classifier expects [B, {self.pre[0].in_features}], got {tuple(x.shape)}")
        h = self.pre(x)
        h, aux, stats = self.moe(h)
        logits = self.post(h)
        return logits, aux, stats

# ----- Conv MoE -----
class ConvExpert(nn.Module):
    def __init__(self, in_ch=1, mid_ch=16, out_ch=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 3, padding=1), nn.ReLU(),
            nn.Conv2d(mid_ch, out_ch, 3, padding=1), nn.ReLU()
        )
    def forward(self, x): return self.net(x)

class ConvMoESharedTopK(nn.Module):
    def __init__(self, in_ch=1, out_ch=32, num_experts=3, k=1, router_hidden=64,
                 balance_loss_weight=0.02, temperature=1.0, add_gumbel_noise=True,
                 shared_init_scale=0.3, entropy_bonus_weight=0.001):
        super().__init__()
        self.shared = ConvExpert(in_ch, out_ch//2, out_ch)
        self.experts = nn.ModuleList([ConvExpert(in_ch, out_ch//2, out_ch) for _ in range(num_experts)])
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        # route on channel descriptor (mean-pooled input)
        self.router = TopKRouter(in_dim=in_ch, hidden_dim=router_hidden,
                                 num_experts=num_experts, k=k,
                                 add_gumbel_noise=add_gumbel_noise, temperature=temperature)
        self.balance_loss_weight = balance_loss_weight
        self.entropy_bonus_weight = entropy_bonus_weight
        self.shared_scale = nn.Parameter(torch.tensor(float(shared_init_scale)))
        self.route_norm = nn.LayerNorm(in_ch)

    def forward(self, x):
        if x.dim() != 4:
            raise RuntimeError(f"Conv MoE expects [B,C,H,W], got {tuple(x.shape)}")
        shared_out = self.shared(x) * self.shared_scale
        desc = self.pool(x).flatten(1)                             # [B,in_ch]
        sparse_probs, dense_probs = self.router(self.route_norm(desc))
        expert_outs = [e(x) for e in self.experts]                 # list of [B,C,H,W]
        stack = torch.stack(expert_outs, dim=0)                    # [E,B,C,H,W]
        gates = sparse_probs.transpose(0,1)[:, :, None, None, None]
        routed = (gates * stack).sum(dim=0)
        y = shared_out + routed

        ent = entropy_mean(dense_probs)
        aux = self.balance_loss_weight * kl_to_uniform(dense_probs) - self.entropy_bonus_weight * ent

        with torch.no_grad():
            counts = (sparse_probs > 0).float().sum(0)
        stats = {"expert_selection_counts": counts, "mean_routing_probs": dense_probs.mean(0)}
        return y, aux, stats

class ConvMoEClassifier(nn.Module):
    def __init__(self, in_ch=1, num_experts=3, k=1, feat_ch=32, num_classes=10,
                 router_hidden=64, balance_loss_weight=0.02,
                 shared_init_scale=0.3, entropy_bonus_weight=0.001,
                 router_temperature=1.0, add_gumbel_noise=True):
        super().__init__()
        self.moe = ConvMoESharedTopK(
            in_ch, feat_ch, num_experts, k, router_hidden, balance_loss_weight,
            temperature=router_temperature, add_gumbel_noise=add_gumbel_noise,
            shared_init_scale=shared_init_scale, entropy_bonus_weight=entropy_bonus_weight
        )
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(), nn.Linear(feat_ch, num_classes))
    def forward(self, x):
        h, aux, stats = self.moe(x)
        logits = self.head(h)
        return logits, aux, stats

# ======================================================================
# NanoGPT-style Transformer with optional MoE-FFN
# ======================================================================

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout, bias):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout
        self.c_attn = nn.Linear(n_embd, 3*n_embd, bias=bias)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=bias)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.flash = hasattr(F, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                 .view(1,1,block_size,block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1,2)
        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=None,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=True
            )
        else:
            att = (q @ k.transpose(-2,-1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        y = self.resid_dropout(self.c_proj(y))
        return y

# MoE-FFN replacing the standard MLP in Transformer block
class MoEFFN(nn.Module):
    def __init__(self, n_embd, expansion=4, num_experts=3, k=1, router_hidden=128,
                 bias=True, dropout=0.0, balance_loss_weight=0.02,
                 add_gumbel_noise=True, temperature=1.0,
                 shared_init_scale=0.3, entropy_bonus_weight=0.001):
        super().__init__()
        hidden = expansion * n_embd
        self.shared_fc = nn.Linear(n_embd, hidden, bias=bias)
        self.shared_proj = nn.Linear(hidden, n_embd, bias=bias)
        self.expert_fc = nn.ModuleList([nn.Linear(n_embd, hidden, bias=bias) for _ in range(num_experts)])
        self.expert_proj = nn.ModuleList([nn.Linear(hidden, n_embd, bias=bias) for _ in range(num_experts)])
        self.router = TopKRouter(n_embd, router_hidden, num_experts, k, add_gumbel_noise, temperature)
        self.dropout = nn.Dropout(dropout)
        self.balance_loss_weight = balance_loss_weight
        self.entropy_bonus_weight = entropy_bonus_weight
        self.act = nn.GELU()
        self.num_experts = num_experts
        self.k = k
        # learnable shared scale (starts < 1)
        self.shared_scale = nn.Parameter(torch.tensor(float(shared_init_scale)))
        self.route_norm = nn.LayerNorm(n_embd)

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.reshape(B*T, C)

        # Shared path (scaled)
        shared = self.act(self.shared_fc(x_flat))
        shared = self.shared_proj(shared) * self.shared_scale  # [BT, C]

        # Router on normalized token states
        sparse_probs, dense_probs = self.router(self.route_norm(x_flat))  # [BT,E]

        # Experts
        routed = 0.0
        for e in range(self.num_experts):
            h = self.act(self.expert_fc[e](x_flat))
            h = self.expert_proj[e](h)
            gate = sparse_probs[:, e].unsqueeze(-1)
            routed = routed + gate * h

        y = shared + routed
        y = self.dropout(y).view(B, T, C)

        # Aux losses
        ent = entropy_mean(dense_probs)
        aux = self.balance_loss_weight * kl_to_uniform(dense_probs) - self.entropy_bonus_weight * ent

        with torch.no_grad():
            counts = (sparse_probs > 0).float().sum(0)
        stats = {"expert_selection_counts": counts, "mean_routing_probs": dense_probs.mean(0)}
        return y, aux, stats

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout, bias,
                 moe_ffn: Optional[MoEFFN] = None):
        super().__init__()
        self.ln_1 = LayerNorm(n_embd, bias=bias)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout, bias)
        self.ln_2 = LayerNorm(n_embd, bias=bias)
        self.moe_ffn = moe_ffn
        if moe_ffn is None:
            hidden = 4 * n_embd
            self.ffn = nn.Sequential(
                nn.Linear(n_embd, hidden, bias=bias), nn.GELU(),
                nn.Linear(hidden, n_embd, bias=bias), nn.Dropout(dropout),
            )

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        if self.moe_ffn is None:
            x = x + self.ffn(self.ln_2(x))
            aux = torch.tensor(0.0, device=x.device)
            stats = {"expert_selection_counts": torch.zeros(1, device=x.device),
                     "mean_routing_probs": torch.zeros(1, device=x.device)}
        else:
            y, aux, stats = self.moe_ffn(self.ln_2(x))
            x = x + y
        return x, aux, stats

@dataclass
class GPTCfg:
    block_size: int = 256
    vocab_size: int = 256
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 192
    dropout: float = 0.1
    bias: bool = True
    # MoE options
    use_moe_ffn: bool = True
    num_experts: int = 3
    k: int = 1
    router_hidden: int = 128
    balance_loss_weight: float = 0.02
    # Specialization aids
    shared_init_scale: float = 0.3
    entropy_bonus_weight: float = 0.001
    router_temperature: float = 1.0
    router_temp_start: float = 1.5
    router_temp_end: float = 0.5
    add_gumbel_noise: bool = True

class GPTMoE(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        blocks = []
        for _ in range(cfg.n_layer):
            moe_ffn = None
            if cfg.use_moe_ffn:
                moe_ffn = MoEFFN(
                    cfg.n_embd, expansion=4, num_experts=cfg.num_experts, k=cfg.k,
                    router_hidden=cfg.router_hidden, bias=cfg.bias, dropout=cfg.dropout,
                    balance_loss_weight=cfg.balance_loss_weight,
                    add_gumbel_noise=cfg.add_gumbel_noise, temperature=cfg.router_temperature,
                    shared_init_scale=cfg.shared_init_scale, entropy_bonus_weight=cfg.entropy_bonus_weight
                )
            blocks.append(TransformerBlock(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout, cfg.bias, moe_ffn))
        self.h = nn.ModuleList(blocks)
        self.ln_f = LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight tie

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2*cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, f"seq len {T} > block_size {self.cfg.block_size}"
        pos = torch.arange(0, T, device=idx.device).long()
        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        aux_total = torch.tensor(0.0, device=idx.device)
        last_stats = None
        for blk in self.h:
            x, aux, stats = blk(x)
            aux_total = aux_total + aux
            last_stats = stats
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            loss = loss + aux_total
        return logits, loss, last_stats

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Active params accounting (approx per-token)
def activated_params_gptmoe(model: GPTMoE) -> Dict[str, int]:
    total, trainable = count_parameters(model)
    attn_total = 0
    shared_ffn = 0
    router = 0
    per_expert = 0
    for blk in model.h:
        if isinstance(blk.attn.c_attn, nn.Linear):
            attn_total += blk.attn.c_attn.weight.numel()
            if blk.attn.c_attn.bias is not None: attn_total += blk.attn.c_attn.bias.numel()
            attn_total += blk.attn.c_proj.weight.numel()
            if blk.attn.c_proj.bias is not None: attn_total += blk.attn.c_proj.bias.numel()
        if blk.moe_ffn is not None:
            m = blk.moe_ffn
            shared_ffn += m.shared_fc.weight.numel() + (m.shared_fc.bias.numel() if m.shared_fc.bias is not None else 0)
            shared_ffn += m.shared_proj.weight.numel() + (m.shared_proj.bias.numel() if m.shared_proj.bias is not None else 0)
            e0_fc = m.expert_fc[0]; e0_proj = m.expert_proj[0]
            per_expert += e0_fc.weight.numel() + (e0_fc.bias.numel() if e0_fc.bias is not None else 0)
            per_expert += e0_proj.weight.numel() + (e0_proj.bias.numel() if e0_proj.bias is not None else 0)
            for p in m.router.parameters():
                router += p.numel()
    k = model.cfg.k
    active_total = attn_total + shared_ffn + router + k * per_expert
    return {
        "total": total, "trainable": trainable,
        "attn_total": attn_total, "shared_ffn": shared_ffn,
        "router": router, "per_expert": per_expert, "k": k,
        "active_total": active_total
    }

# ======================================================================
# Train / Eval for Dense, Conv, GPT(-MoE)
# ======================================================================

def train_dense(epochs: int, batch_size: int, lr: float, device: str,
                num_experts=3, k=1, router_hidden=64, blw=0.02,
                shared_init_scale=0.3, entropy_bonus=0.001,
                router_temp=1.0, add_gumbel_noise=True):
    x, y = make_toy_blobs(device=device, n_per_class=1200, dim=2, n_classes=3)
    dl = DataLoader(TensorDataset(x, y), batch_size=batch_size, shuffle=True)
    model = TinyMoEClassifier(
        in_dim=2, hidden_dim=64, num_experts=num_experts, k=k, n_classes=3,
        router_hidden=router_hidden, balance_loss_weight=blw,
        shared_init_scale=shared_init_scale, entropy_bonus_weight=entropy_bonus,
        router_temperature=router_temp, add_gumbel_noise=add_gumbel_noise
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    for epoch in range(1, epochs+1):
        total, correct, aux_accum = 0, 0, 0.0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            logits, aux, _ = model(xb)
            loss = F.cross_entropy(logits, yb) + aux
            opt.zero_grad(); loss.backward(); opt.step()
            pred = logits.argmax(-1)
            correct += (pred == yb).sum().item()
            total += yb.numel()
            aux_accum += aux.item()
        acc = correct / total
        console.print(Panel(
            f"Epoch {epoch}/{epochs}  acc=[bold green]{acc:.3f}[/]  aux={aux_accum/len(dl):.4f}",
            title="Dense MoE Train", border_style="cyan", box=box.ROUNDED, expand=True))
    return model

def eval_dense(model: TinyMoEClassifier, device: str, batch_size: int = 256):
    x, y = make_toy_blobs(device=device, n_per_class=400, dim=2, n_classes=3, seed=42)
    dl = DataLoader(TensorDataset(x, y), batch_size=batch_size)
    model.eval()
    total, correct = 0, 0
    last_stats = None
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            logits, _, stats = model(xb)
            total += yb.numel()
            correct += (logits.argmax(-1) == yb).sum().item()
            last_stats = stats
    tot, trn = count_parameters(model)
    shared = sum(p.numel() for p in model.moe.shared.parameters())
    router = sum(p.numel() for p in model.moe.router.parameters())
    per_expert = sum(p.numel() for p in model.moe.experts[0].parameters())
    rows = [
        ("Shared", f"{shared:,}"),
        ("Router", f"{router:,}"),
        ("Per expert", f"{per_expert:,}"),
        ("k", f"{model.moe.router.k}"),
        ("Active per forward", f"[bold]{shared + router + model.moe.router.k*per_expert:,}[/]"),
    ]
    console.print(rich_params_panel("Dense Params", tot, trn, rows))
    if last_stats: console.print(rich_routing_panel(last_stats, "Dense Routing (last batch)"))
    console.print(Panel(
        f"Eval acc: [bold cyan]{correct/total:.3f}[/] on {total} samples",
        border_style="cyan", title="Dense Eval", box=box.ROUNDED, expand=True))

def train_conv_mnist(epochs: int, batch_size: int, lr: float, device: str, data_dir: str,
                     num_experts=3, k=1, router_hidden=64, blw=0.02,
                     shared_init_scale=0.3, entropy_bonus=0.001,
                     router_temp=1.0, add_gumbel_noise=True):
    train_dl, test_dl = get_mnist_loaders(data_dir, batch_size)
    model = ConvMoEClassifier(
        in_ch=1, num_experts=num_experts, k=k, feat_ch=32, num_classes=10,
        router_hidden=router_hidden, balance_loss_weight=blw,
        shared_init_scale=shared_init_scale, entropy_bonus_weight=entropy_bonus,
        router_temperature=router_temp, add_gumbel_noise=add_gumbel_noise
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    for epoch in range(1, epochs+1):
        model.train()
        total, correct, aux_accum = 0, 0, 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits, aux, _ = model(xb)
            loss = F.cross_entropy(logits, yb) + aux
            opt.zero_grad(); loss.backward(); opt.step()
            total += yb.numel()
            correct += (logits.argmax(-1) == yb).sum().item()
            aux_accum += aux.item()
        train_acc = correct / total
        # quick test
        model.eval()
        total, correct = 0, 0
        with torch.no_grad():
            for xb, yb in test_dl:
                xb, yb = xb.to(device), yb.to(device)
                logits, _aux, _ = model(xb)
                total += yb.numel()
                correct += (logits.argmax(-1) == yb).sum().item()
        test_acc = correct / total
        console.print(Panel(
            f"Epoch {epoch}/{epochs}  train={train_acc:.3f}  test=[bold]{test_acc:.3f}[/]  aux={aux_accum/len(train_dl):.4f}",
            title="Conv MoE Train (MNIST)", border_style="cyan", box=box.ROUNDED, expand=True))
    return model

def eval_conv_mnist(model: ConvMoEClassifier, device: str, batch_size: int, data_dir: str):
    _train, test_dl = get_mnist_loaders(data_dir, batch_size)
    model.eval()
    total, correct = 0, 0
    last_stats = None
    with torch.no_grad():
        for xb, yb in test_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits, _aux, stats = model(xb)
            total += yb.numel()
            correct += (logits.argmax(-1) == yb).sum().item()
            last_stats = stats
    tot, trn = count_parameters(model)
    shared = sum(p.numel() for p in model.moe.shared.parameters())
    router = sum(p.numel() for p in model.moe.router.parameters())
    per_expert = sum(p.numel() for p in model.moe.experts[0].parameters())
    rows = [
        ("Shared", f"{shared:,}"),
        ("Router", f"{router:,}"),
        ("Per expert", f"{per_expert:,}"),
        ("k", f"{model.moe.router.k}"),
        ("Active per forward", f"[bold]{shared + router + model.moe.router.k*per_expert:,}[/]"),
    ]
    console.print(rich_params_panel("Conv Params (MNIST)", tot, trn, rows))
    if last_stats: console.print(rich_routing_panel(last_stats, "Conv Routing (last batch)"))
    console.print(Panel(
        f"MNIST test acc: [bold cyan]{correct/total:.3f}[/] on {total} samples",
        border_style="cyan", title="Conv Eval", box=box.ROUNDED, expand=True))

# ---- GPT(-MoE) train/eval ----

def train_gpt_moe(epochs: int, steps_per_epoch: int, batch_size: int, lr: float,
                  device: str, data_dir: str, cfg: GPTCfg):
    train_txt, val_txt = load_tiny_shakespeare(data_dir)
    stoi, itos = build_charset(train_txt, val_txt)
    train_ids = encode(train_txt, stoi)
    val_ids   = encode(val_txt, stoi)

    # auto-shrink block_size if text is tiny
    if len(train_ids) <= cfg.block_size + 1:
        new_bs = max(16, min(cfg.block_size, len(train_ids) - 2))
        console.print(Panel(
            f"Auto-adjusting block_size from {cfg.block_size} -> {new_bs} (text too short)",
            border_style="yellow", title="GPT Safety", box=box.ROUNDED, expand=True))
        cfg.block_size = new_bs

    cfg.vocab_size = max(stoi.values()) + 1

    model = GPTMoE(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    console.print(Panel(json.dumps(asdict(cfg), indent=2), title="GPT Config", border_style="blue", box=box.ROUNDED, expand=True))

    # Temperature annealing schedule (per epoch): start -> end
    def set_router_temp(temp: float):
        for blk in model.h:
            if blk.moe_ffn is not None:
                blk.moe_ffn.router.temperature = float(temp)

    for epoch in range(1, epochs+1):
        # cosine anneal temp across epochs
        if epochs > 1:
            alpha = (epoch - 1) / (epochs - 1)
        else:
            alpha = 1.0
        temp = cfg.router_temp_end + (cfg.router_temp_start - cfg.router_temp_end) * (1.0 - alpha)
        set_router_temp(temp)

        model.train()
        t0 = time.time()
        losses = []
        stats_last = None
        for _ in range(steps_per_epoch):
            xb, yb = get_batch_text(train_ids, cfg.block_size, batch_size, device)
            logits, loss, stats = model(xb, yb)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            losses.append(loss.item())
            stats_last = stats
        dt = time.time() - t0
        val_xb, val_yb = get_batch_text(val_ids, cfg.block_size, batch_size, device)
        with torch.no_grad():
            _, val_loss, _ = model(val_xb, val_yb)

        console.print(Panel(
            f"Epoch {epoch}/{epochs}  temp={temp:.2f}  "
            f"train_loss=[bold]{sum(losses)/len(losses):.3f}[/]  "
            f"val_loss=[bold magenta]{val_loss.item():.3f}[/]  time={dt:.1f}s",
            title="GPT(-MoE) Train", border_style="cyan", box=box.ROUNDED, expand=True))

        # Params panel
        ap = activated_params_gptmoe(model)
        rows = [
            ("Attention total", f"{ap['attn_total']:,}"),
            ("Shared FFN (sum L)", f"{ap['shared_ffn']:,}"),
            ("Router (sum L)", f"{ap['router']:,}"),
            ("Per expert (one, sum L)", f"{ap['per_expert']:,}"),
            ("k", f"{ap['k']}"),
            ("Active per forward (approx)", f"[bold]{ap['active_total']:,}[/]"),
        ]
        console.print(rich_params_panel("GPT(-MoE) Parameters", ap["total"], ap["trainable"], rows))
        if stats_last: console.print(rich_routing_panel(stats_last, "GPT-MoE Routing (last batch)"))

    return model, stoi, itos

@torch.no_grad()
def sample_gpt(model: GPTMoE, stoi: Dict[str,int], itos: Dict[int,str], device: str,
               start: str = "\n", max_new_tokens: int = 200, temperature: float = 0.8, top_k: int = 200):
    model.eval().to(device)
    x = torch.tensor([encode(start, stoi)], dtype=torch.long, device=device)
    y = model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    txt = "".join(itos[i] for i in y[0].tolist())
    console.print(Panel(txt, title="GPT Sample", border_style="magenta", box=box.ROUNDED, expand=True))

# ======================================================================
# Save / Load
# ======================================================================

def save_ckpt(path: str, payload: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload, path)
    console.print(Panel(f"Saved checkpoint to: [bold]{path}[/]", border_style="magenta", box=box.ROUNDED, expand=True))

def load_ckpt(path: str) -> Dict[str, Any]:
    payload = torch.load(path, map_location="cpu")
    console.print(Panel(f"Loaded checkpoint from: [bold]{path}[/]", border_style="magenta", box=box.ROUNDED, expand=True))
    return payload

# ======================================================================
# CLI
# ======================================================================

def main(argv=None):
    p = argparse.ArgumentParser(description="MoE Suite: Dense/Conv/GPT with shared + top-k experts")
    p.add_argument("--mode", required=True, choices=[
        "train_dense","infer_dense",
        "train_conv","infer_conv",
        "train_gpt","infer_gpt"
    ])

    # common
    p.add_argument("--device", default="auto", choices=["auto","cpu","cuda"])
    p.add_argument("--batch_size", type=int, default=128)
    p.add_argument("--epochs", type=int, default=5)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--data_dir", type=str, default="./data")
    p.add_argument("--save_path", type=str, default="artifacts/ckpt.pt")
    p.add_argument("--load_path", type=str, default="artifacts/ckpt.pt")
    p.add_argument("--seed", type=int, default=1337)

    # MoE knobs (dense/conv)
    p.add_argument("--num_experts", type=int, default=3)
    p.add_argument("--k", type=int, default=1)
    p.add_argument("--router_hidden", type=int, default=64)
    p.add_argument("--blw", type=float, default=0.0)  # balance loss weight
    p.add_argument("--shared_init_scale", type=float, default=0.3)
    p.add_argument("--moe_entropy", type=float, default=0.001)
    p.add_argument("--router_temp", type=float, default=1.0)
    p.add_argument("--no_gumbel", action="store_true", help="Disable Gumbel noise in router")

    # GPT config
    p.add_argument("--gpt_block_size", type=int, default=256)
    p.add_argument("--gpt_n_layer", type=int, default=4)
    p.add_argument("--gpt_n_head", type=int, default=4)
    p.add_argument("--gpt_n_embd", type=int, default=192)
    p.add_argument("--gpt_dropout", type=float, default=0.1)
    p.add_argument("--gpt_use_moe", action="store_true", default=True)
    p.add_argument("--gpt_steps_per_epoch", type=int, default=100)
    p.add_argument("--sample_start", type=str, default="\n")
    p.add_argument("--sample_tokens", type=int, default=200)
    # GPT router schedule
    p.add_argument("--router_temp_start", type=float, default=1.5)
    p.add_argument("--router_temp_end", type=float, default=0.5)

    args = p.parse_args(argv)
    set_seed(args.seed)

    device = args.device
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"

    add_gumbel = not args.no_gumbel

    if args.mode == "train_dense":
        model = train_dense(args.epochs, args.batch_size, args.lr, device,
                            args.num_experts, args.k, args.router_hidden, args.blw,
                            shared_init_scale=args.shared_init_scale,
                            entropy_bonus=args.moe_entropy,
                            router_temp=args.router_temp,
                            add_gumbel_noise=add_gumbel)
        payload = {"kind":"dense", "state_dict": model.state_dict()}
        save_ckpt(args.save_path, payload)

    elif args.mode == "infer_dense":
        payload = load_ckpt(args.load_path)
        assert payload["kind"] == "dense", "Checkpoint is not dense kind"
        model = TinyMoEClassifier(
            num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden,
            balance_loss_weight=args.blw, shared_init_scale=args.shared_init_scale,
            entropy_bonus_weight=args.moe_entropy, router_temperature=args.router_temp,
            add_gumbel_noise=add_gumbel
        ).to(device)
        model.load_state_dict(payload["state_dict"])
        eval_dense(model, device, args.batch_size)

    elif args.mode == "train_conv":
        model = train_conv_mnist(args.epochs, args.batch_size, args.lr, device, args.data_dir,
                                 args.num_experts, args.k, args.router_hidden, args.blw,
                                 shared_init_scale=args.shared_init_scale,
                                 entropy_bonus=args.moe_entropy,
                                 router_temp=args.router_temp,
                                 add_gumbel_noise=add_gumbel)
        payload = {"kind":"conv", "state_dict": model.state_dict()}
        save_ckpt(args.save_path, payload)

    elif args.mode == "infer_conv":
        payload = load_ckpt(args.load_path)
        assert payload["kind"] == "conv", "Checkpoint is not conv kind"
        model = ConvMoEClassifier(
            num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden,
            balance_loss_weight=args.blw, shared_init_scale=args.shared_init_scale,
            entropy_bonus_weight=args.moe_entropy, router_temperature=args.router_temp,
            add_gumbel_noise=add_gumbel
        ).to(device)
        model.load_state_dict(payload["state_dict"])
        eval_conv_mnist(model, device, args.batch_size, args.data_dir)

    elif args.mode == "train_gpt":
        cfg = GPTCfg(
            block_size=args.gpt_block_size, n_layer=args.gpt_n_layer,
            n_head=args.gpt_n_head, n_embd=args.gpt_n_embd, dropout=args.gpt_dropout,
            use_moe_ffn=args.gpt_use_moe, num_experts=args.num_experts, k=args.k,
            router_hidden=args.router_hidden, balance_loss_weight=args.blw,
            shared_init_scale=args.shared_init_scale,
            entropy_bonus_weight=args.moe_entropy,
            router_temperature=args.router_temp,
            router_temp_start=args.router_temp_start,
            router_temp_end=args.router_temp_end,
            add_gumbel_noise=add_gumbel
        )
        model, stoi, itos = train_gpt_moe(args.epochs, args.gpt_steps_per_epoch,
                                          args.batch_size, args.lr, device, args.data_dir, cfg)
        payload = {"kind":"gpt_moe", "cfg": asdict(cfg), "stoi": stoi, "itos": itos, "state_dict": model.state_dict()}
        save_ckpt(args.save_path, payload)

    elif args.mode == "infer_gpt":
        payload = load_ckpt(args.load_path)
        assert payload["kind"] == "gpt_moe", "Checkpoint is not gpt_moe kind"
        cfg = GPTCfg(**payload["cfg"])
        model = GPTMoE(cfg)
        model.load_state_dict(payload["state_dict"])
        sample_gpt(model, payload["stoi"], payload["itos"], device,
                   start=args.sample_start, max_new_tokens=args.sample_tokens)

if __name__ == "__main__":
    import sys, shlex, os
    argv = sys.argv[1:]
    has_mode = any(a == "--mode" or a.startswith("--mode=") for a in argv)
    if not has_mode:
        env = os.environ.get("MOE_ARGS", "").strip()
        if env:
            argv = shlex.split(env)
        else:
            # safe default demo
            argv = ["--mode","train_dense","--epochs","3","--batch_size","256","--save_path","artifacts/dense.pt"]
    main(argv)


# BPE NANOGPT MOE

In [ ]:
import os
os.environ["MOE_ARGS"] = "--mode train_gpt_bpe --tok_kind basic --vocab_size 512 --tok_prefix artifacts/tokenizers/tiny_bpe --use_moe --epochs 3 --steps_per_epoch 50 --batch_size 64 --save_path artifacts/gpt_moe_bpe.pt"


In [ ]:
#!/usr/bin/env python3
# gpt_moe_bpe.py
# NanoGPT-style LM with optional MoE-FFN, now using minbpe (Basic/Regex) BPE tokens.

import os, math, json, argparse, random, time, urllib.request, shlex
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Any, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box

# --- minbpe (pip install minbpe) ---
# try:
#     from minbpe import BasicTokenizer, RegexTokenizer
# except Exception as e:
#     raise RuntimeError(
#         "minbpe is required. Install with: pip install minbpe\n"
#         f"Underlying import error: {e}"
#     )

"""
Minimal (byte-level) Byte Pair Encoding tokenizer.

Algorithmically follows along the GPT tokenizer:
https://github.com/openai/gpt-2/blob/master/src/encoder.py

But:
- Does not handle the regular expression splitting pattern.
- Does not handle any special tokens.
"""

"""
Contains the base Tokenizer class and a few common helper functions.
The base class also contains the (common) save/load functionality.
It would be possible to be a lot more strict about the interface and
e.g. isolating all regex/pattern parts to the RegexTokenizer, but
some concessions are made for simplicity.
"""
import unicodedata

# -----------------------------------------------------------------------------
# a few helper functions useful for both BasicTokenizer and RegexTokenizer

def get_stats(ids, counts=None):
    """
    Given a list of integers, return a dictionary of counts of consecutive pairs
    Example: [1, 2, 3, 1, 2] -> {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    Optionally allows to update an existing dictionary of counts
    """
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]): # iterate consecutive elements
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge(ids, pair, idx):
    """
    In the list of integers (ids), replace all consecutive occurrences
    of pair with the new integer token idx
    Example: ids=[1, 2, 3, 1, 2], pair=(1, 2), idx=4 -> [4, 3, 4]
    """
    newids = []
    i = 0
    while i < len(ids):
        # if not at the very last position AND the pair matches, replace it
        if ids[i] == pair[0] and i < len(ids) - 1 and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

# first two helper functions...
def replace_control_characters(s: str) -> str:
    # we don't want to print control characters
    # which distort the output (e.g. \n or much worse)
    # https://stackoverflow.com/questions/4324790/removing-control-characters-from-a-string-in-python/19016117#19016117
    # http://www.unicode.org/reports/tr44/#GC_Values_Table
    chars = []
    for ch in s:
        if unicodedata.category(ch)[0] != "C":
            chars.append(ch) # this character is ok
        else:
            chars.append(f"\\u{ord(ch):04x}") # escape
    return "".join(chars)

def render_token(t: bytes) -> str:
    # pretty print a token, escaping control characters
    s = t.decode('utf-8', errors='replace')
    s = replace_control_characters(s)
    return s

# -----------------------------------------------------------------------------
# the base Tokenizer class

class Tokenizer:
    """Base class for Tokenizers"""

    def __init__(self):
        # default: vocab size of 256 (all bytes), no merges, no patterns
        self.merges = {} # (int, int) -> int
        self.pattern = "" # str
        self.special_tokens = {} # str -> int, e.g. {'<|endoftext|>': 100257}
        self.vocab = self._build_vocab() # int -> bytes

    def train(self, text, vocab_size, verbose=False):
        # Tokenizer can train a vocabulary of size vocab_size from text
        raise NotImplementedError

    def encode(self, text):
        # Tokenizer can encode a string into a list of integers
        raise NotImplementedError

    def decode(self, ids):
        # Tokenizer can decode a list of integers into a string
        raise NotImplementedError

    def _build_vocab(self):
        # vocab is simply and deterministically derived from merges
        vocab = {idx: bytes([idx]) for idx in range(256)}
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]
        for special, idx in self.special_tokens.items():
            vocab[idx] = special.encode("utf-8")
        return vocab

    def save(self, file_prefix):
        """
        Saves two files: file_prefix.vocab and file_prefix.model
        This is inspired (but not equivalent to!) sentencepiece's model saving:
        - model file is the critical one, intended for load()
        - vocab file is just a pretty printed version for human inspection only
        """
        # write the model: to be used in load() later
        model_file = file_prefix + ".model"
        with open(model_file, 'w') as f:
            # write the version, pattern and merges, that's all that's needed
            f.write("minbpe v1\n")
            f.write(f"{self.pattern}\n")
            # write the special tokens, first the number of them, then each one
            f.write(f"{len(self.special_tokens)}\n")
            for special, idx in self.special_tokens.items():
                f.write(f"{special} {idx}\n")
            # the merges dict
            for idx1, idx2 in self.merges:
                f.write(f"{idx1} {idx2}\n")
        # write the vocab: for the human to look at
        vocab_file = file_prefix + ".vocab"
        inverted_merges = {idx: pair for pair, idx in self.merges.items()}
        with open(vocab_file, "w", encoding="utf-8") as f:
            for idx, token in self.vocab.items():
                # note: many tokens may be partial utf-8 sequences
                # and cannot be decoded into valid strings. Here we're using
                # errors='replace' to replace them with the replacement char �.
                # this also means that we couldn't possibly use .vocab in load()
                # because decoding in this way is a lossy operation!
                s = render_token(token)
                # find the children of this token, if any
                if idx in inverted_merges:
                    # if this token has children, render it nicely as a merge
                    idx0, idx1 = inverted_merges[idx]
                    s0 = render_token(self.vocab[idx0])
                    s1 = render_token(self.vocab[idx1])
                    f.write(f"[{s0}][{s1}] -> [{s}] {idx}\n")
                else:
                    # otherwise this is leaf token, just print it
                    # (this should just be the first 256 tokens, the bytes)
                    f.write(f"[{s}] {idx}\n")

    def load(self, model_file):
        """Inverse of save() but only for the model file"""
        assert model_file.endswith(".model")
        # read the model file
        merges = {}
        special_tokens = {}
        idx = 256
        with open(model_file, 'r', encoding="utf-8") as f:
            # read the version
            version = f.readline().strip()
            assert version == "minbpe v1"
            # read the pattern
            self.pattern = f.readline().strip()
            # read the special tokens
            num_special = int(f.readline().strip())
            for _ in range(num_special):
                special, special_idx = f.readline().strip().split()
                special_tokens[special] = int(special_idx)
            # read the merges
            for line in f:
                idx1, idx2 = map(int, line.split())
                merges[(idx1, idx2)] = idx
                idx += 1
        self.merges = merges
        self.special_tokens = special_tokens
        self.vocab = self._build_vocab()

class BasicTokenizer(Tokenizer):

    def __init__(self):
        super().__init__()

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        # input text preprocessing
        text_bytes = text.encode("utf-8") # raw bytes
        ids = list(text_bytes) # list of integers in range 0..255

        # iteratively merge the most common pairs to create new tokens
        merges = {} # (int, int) -> int
        vocab = {idx: bytes([idx]) for idx in range(256)} # int -> bytes
        for i in range(num_merges):
            # count up the number of times every consecutive pair appears
            stats = get_stats(ids)
            # find the pair with the highest count
            pair = max(stats, key=stats.get)
            # mint a new token: assign it the next available id
            idx = 256 + i
            # replace all occurrences of pair in ids with idx
            ids = merge(ids, pair, idx)
            # save the merge
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            # prints
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")

        # save class variables
        self.merges = merges # used in encode()
        self.vocab = vocab   # used in decode()

    def decode(self, ids):
        # given ids (list of integers), return Python string
        text_bytes = b"".join(self.vocab[idx] for idx in ids)
        text = text_bytes.decode("utf-8", errors="replace")
        return text

    def encode(self, text):
        # given a string text, return the token ids
        text_bytes = text.encode("utf-8") # raw bytes
        ids = list(text_bytes) # list of integers in range 0..255
        while len(ids) >= 2:
            # find the pair with the lowest merge index
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            # subtle: if there are no more merges available, the key will
            # result in an inf for every single pair, and the min will be
            # just the first pair in the list, arbitrarily
            # we can detect this terminating case by a membership check
            if pair not in self.merges:
                break # nothing else can be merged anymore
            # otherwise let's merge the best pair (lowest merge index)
            idx = self.merges[pair]
            ids = merge(ids, pair, idx)
        return ids

"""
Minimal (byte-level) Byte Pair Encoding tokenizer.

Algorithmically follows along the GPT tokenizer:
https://github.com/openai/gpt-2/blob/master/src/encoder.py

Unlike BasicTokenizer:
- RegexTokenizer handles an optional regex splitting pattern.
- RegexTokenizer handles optional special tokens.
"""

import regex as re

# the main GPT text split patterns, see
# https://github.com/openai/tiktoken/blob/main/tiktoken_ext/openai_public.py
GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""


class RegexTokenizer(Tokenizer):

    def __init__(self, pattern=None):
        """
        - pattern: optional string to override the default (GPT-4 split pattern)
        - special_tokens: str -> int dictionary of special tokens
          example: {'<|endoftext|>': 100257}
        """
        super().__init__()
        self.pattern = GPT4_SPLIT_PATTERN if pattern is None else pattern
        self.compiled_pattern = re.compile(self.pattern)
        self.special_tokens = {}
        self.inverse_special_tokens = {}

    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        # split the text up into text chunks
        text_chunks = re.findall(self.compiled_pattern, text)

        # input text preprocessing
        ids = [list(ch.encode("utf-8")) for ch in text_chunks]

        # iteratively merge the most common pairs to create new tokens
        merges = {} # (int, int) -> int
        vocab = {idx: bytes([idx]) for idx in range(256)} # idx -> bytes
        for i in range(num_merges):
            # count the number of times every consecutive pair appears
            stats = {}
            for chunk_ids in ids:
                # passing in stats will update it in place, adding up counts
                get_stats(chunk_ids, stats)
            # find the pair with the highest count
            pair = max(stats, key=stats.get)
            # mint a new token: assign it the next available id
            idx = 256 + i
            # replace all occurrences of pair in ids with idx
            ids = [merge(chunk_ids, pair, idx) for chunk_ids in ids]
            # save the merge
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            # prints
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")

        # save class variables
        self.merges = merges # used in encode()
        self.vocab = vocab   # used in decode()

    def register_special_tokens(self, special_tokens):
        # special_tokens is a dictionary of str -> int
        # example: {"<|endoftext|>": 100257}
        self.special_tokens = special_tokens
        self.inverse_special_tokens = {v: k for k, v in special_tokens.items()}

    def decode(self, ids):
        # given ids (list of integers), return Python string
        part_bytes = []
        for idx in ids:
            if idx in self.vocab:
                part_bytes.append(self.vocab[idx])
            elif idx in self.inverse_special_tokens:
                part_bytes.append(self.inverse_special_tokens[idx].encode("utf-8"))
            else:
                raise ValueError(f"invalid token id: {idx}")
        text_bytes = b"".join(part_bytes)
        text = text_bytes.decode("utf-8", errors="replace")
        return text

    def _encode_chunk(self, text_bytes):
        # return the token ids
        # let's begin. first, convert all bytes to integers in range 0..255
        ids = list(text_bytes)
        while len(ids) >= 2:
            # find the pair with the lowest merge index
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            # subtle: if there are no more merges available, the key will
            # result in an inf for every single pair, and the min will be
            # just the first pair in the list, arbitrarily
            # we can detect this terminating case by a membership check
            if pair not in self.merges:
                break # nothing else can be merged anymore
            # otherwise let's merge the best pair (lowest merge index)
            idx = self.merges[pair]
            ids = merge(ids, pair, idx)
        return ids

    def encode_ordinary(self, text):
        """Encoding that ignores any special tokens."""
        # split text into chunks of text by categories defined in regex pattern
        text_chunks = re.findall(self.compiled_pattern, text)
        # all chunks of text are encoded separately, then results are joined
        ids = []
        for chunk in text_chunks:
            chunk_bytes = chunk.encode("utf-8") # raw bytes
            chunk_ids = self._encode_chunk(chunk_bytes)
            ids.extend(chunk_ids)
        return ids

    def encode(self, text, allowed_special="none_raise"):
        """
        Unlike encode_ordinary, this function handles special tokens.
        allowed_special: can be "all"|"none"|"none_raise" or a custom set of special tokens
        if none_raise, then an error is raised if any special token is encountered in text
        this is the default tiktoken behavior right now as well
        any other behavior is either annoying, or a major footgun
        """
        # decode the user desire w.r.t. handling of special tokens
        special = None
        if allowed_special == "all":
            special = self.special_tokens
        elif allowed_special == "none":
            special = {}
        elif allowed_special == "none_raise":
            special = {}
            assert all(token not in text for token in self.special_tokens)
        elif isinstance(allowed_special, set):
            special = {k: v for k, v in self.special_tokens.items() if k in allowed_special}
        else:
            raise ValueError(f"allowed_special={allowed_special} not understood")
        if not special:
            # shortcut: if no special tokens, just use the ordinary encoding
            return self.encode_ordinary(text)
        # otherwise, we have to be careful with potential special tokens in text
        # we handle special tokens by splitting the text
        # based on the occurrence of any exact match with any of the special tokens
        # we can use re.split for this. note that surrounding the pattern with ()
        # makes it into a capturing group, so the special tokens will be included
        special_pattern = "(" + "|".join(re.escape(k) for k in special) + ")"
        special_chunks = re.split(special_pattern, text)
        # now all the special characters are separated from the rest of the text
        # all chunks of text are encoded separately, then results are joined
        ids = []
        for part in special_chunks:
            if part in special:
                # this is a special token, encode it separately as a special case
                ids.append(special[part])
            else:
                # this is an ordinary sequence, encode it normally
                ids.extend(self.encode_ordinary(part))
        return ids

console = Console()

# ============================== Utils ==============================

def set_seed(seed=1337):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(m: nn.Module) -> Tuple[int,int]:
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

def kl_to_uniform(probs: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    E = probs.size(-1)
    m = probs.mean(dim=0)
    return torch.sum(m * (m.add(eps).log() - math.log(1.0 / E)))

def entropy_mean(probs: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    return -(probs.clamp_min(eps) * probs.clamp_min(eps).log()).sum(dim=-1).mean()

def params_panel(title: str, totals: Dict[str,int], extras: List[Tuple[str,str]] = None):
    t = Table(box=box.SIMPLE_HEAVY)
    t.add_column("Metric", style="bold"); t.add_column("Value", justify="right")
    for k in ["Total params","Trainable params"]:
        t.add_row(k, f"{totals[k]:,}")
    for k in ["Attention params","Shared FFN","Router","Per expert","k","Active per forward"]:
        if k in totals:
            v = totals[k]
            t.add_row(k, f"{v:,}" if isinstance(v,int) else f"{v}")
    if extras:
        for k,v in extras: t.add_row(k,v)
    console.print(Panel(t, title=title, border_style="yellow", box=box.ROUNDED, expand=True))

def routing_panel(stats: Dict[str, torch.Tensor], title: str):
    counts = stats["expert_selection_counts"].detach().cpu().to(torch.int64).tolist()
    mean_probs = stats["mean_routing_probs"].detach().cpu().tolist()
    t = Table(box=box.SIMPLE_HEAVY)
    t.add_column("Expert"); t.add_column("Batch Picks", justify="right"); t.add_column("Mean Prob.", justify="right")
    for i,(c,p) in enumerate(zip(counts,mean_probs)): t.add_row(f"E{i}", f"{c}", f"{p:.3f}")
    ent = entropy_mean(torch.tensor(mean_probs)[None,:]).item()
    console.print(Panel(t, title=f"{title} | entropy={ent:.3f}", border_style="green", box=box.ROUNDED, expand=True))

# ============================ Data (Tiny Shakespeare) ============================

def _repeat_to_len(s: str, target_len: int) -> str:
    if not s: s=" \n"
    return (s * ((target_len // len(s))+1))[:target_len]

def load_tiny_shakespeare(data_dir="./data", target_len=100_000) -> Tuple[str,str]:
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, "tinyshakespeare_input.txt")
    if not os.path.exists(path):
        try:
            url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url, path)
        except Exception:
            sample=("From fairest creatures we desire increase,\n"
                    "That thereby beauty's rose might never die,\n"
                    "But as the riper should by time decease,\n"
                    "His tender heir might bear his memory:\n")
            with open(path,"w",encoding="utf-8") as f: f.write(_repeat_to_len(sample, target_len))
    with open(path,"r",encoding="utf-8") as f: text=f.read()
    split=int(0.9*len(text)); return text[:split], text[split:]

# ============================ Tokenizer (minbpe) ============================

def train_or_load_tokenizer(tok_kind:str, vocab_size:int, text:str, out_prefix:str, verbose:bool=True):
    """
    If out_prefix.model exists, loads it. Else trains and saves.
    Returns (tokenizer, model_path, vocab_size_actual)
    """
    os.makedirs(os.path.dirname(out_prefix), exist_ok=True) if os.path.dirname(out_prefix) else None
    model_path = out_prefix + ".model"
    if os.path.exists(model_path):
        tok = _new_tok(tok_kind)
        tok.load(model_path)
        vocab_size_actual = max(tok.vocab.keys()) + 1
        if verbose:
            console.print(Panel(f"Loaded tokenizer: [bold]{model_path}[/] (size={vocab_size_actual})",
                                border_style="magenta", title="Tokenizer"))
        return tok, model_path, vocab_size_actual

    tok = _new_tok(tok_kind)
    # NOTE: minbpe requires vocab_size >= 256 for byte level + merges
    vocab_size = max(256, vocab_size)
    if verbose:
        console.print(Panel(f"Training {tok_kind} BPE (vocab={vocab_size})", border_style="blue", title="Tokenizer"))
    tok.train(text, vocab_size, verbose=verbose)
    tok.save(out_prefix)  # writes .model and .vocab
    vocab_size_actual = max(tok.vocab.keys()) + 1
    if verbose:
        console.print(Panel(f"Saved tokenizer to: [bold]{out_prefix}.model[/] (size={vocab_size_actual})",
                            border_style="green", title="Tokenizer"))
    return tok, model_path, vocab_size_actual

def _new_tok(kind:str):
    kind = kind.lower()
    if kind == "basic": return BasicTokenizer()
    if kind == "regex": return RegexTokenizer()
    raise ValueError(f"Unknown tokenizer kind: {kind}")

def encode_text(tok, s:str) -> List[int]:
    return tok.encode(s)

def decode_ids(tok, ids:List[int]) -> str:
    return tok.decode(ids)

def get_batch_tokens(ids: List[int], block_size: int, batch_size: int, device: str):
    if len(ids) <= block_size + 1:
        raise RuntimeError(f"Token length {len(ids)} too small for block_size {block_size}. Lower block size.")
    ix=torch.randint(len(ids)-block_size-1,(batch_size,))
    x=torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y=torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================== GPT + MoE ===============================

class LayerNorm(nn.Module):
    def __init__(self, n, bias): super().__init__(); self.weight=nn.Parameter(torch.ones(n)); self.bias=nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self, x): return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout, bias, backend="sdpa"):
        super().__init__(); assert n_embd % n_head == 0
        self.n_head=n_head; self.n_embd=n_embd; self.dropout=dropout; self.backend=backend
        self.c_attn=nn.Linear(n_embd,3*n_embd,bias=bias); self.c_proj=nn.Linear(n_embd,n_embd,bias=bias)
        self.attn_dropout=nn.Dropout(dropout); self.resid_dropout=nn.Dropout(dropout)
        if backend=="vanilla" or not hasattr(F,"scaled_dot_product_attention"):
            self.register_buffer("bias", torch.tril(torch.ones(block,block)).view(1,1,block,block))
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        if self.backend=="sdpa" and hasattr(F,"scaled_dot_product_attention"):
            y=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=self.dropout if self.training else 0.,is_causal=True)
        else:
            att=(q@k.transpose(-2,-1))*(1.0/math.sqrt(k.size(-1)))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0, float("-inf"))
            att=F.softmax(att,dim=-1); att=self.attn_dropout(att); y=att@v
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.c_proj(y))

class TopKRouter(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=1, add_gumbel_noise=True, temperature=1.0):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden_dim), nn.ReLU(), nn.Linear(hidden_dim,num_experts))
        self.k=k; self.add_gumbel=add_gumbel_noise; self.temperature=float(temperature)
    @staticmethod
    def _gumbel(shape, device):
        u=torch.rand(shape,device=device).clamp_(1e-9,1-1e-9); return -torch.log(-torch.log(u))
    def forward(self, x):
        logits=self.net(x)
        if self.add_gumbel and self.training: logits=logits+self._gumbel(logits.shape,logits.device)
        probs=F.softmax(logits/self.temperature, dim=-1)
        if self.k>=probs.size(-1): return probs, probs
        topk_vals, topk_idx = torch.topk(probs, k=self.k, dim=-1)
        mask = torch.zeros_like(probs); mask.scatter_(dim=-1, index=topk_idx, src=torch.ones_like(topk_vals))
        sp = probs*mask; sp = sp/(sp.sum(dim=-1,keepdim=True)+1e-9)
        return sp, probs

class MoEFFN(nn.Module):
    def __init__(self, in_dim, expansion=4, num_experts=3, k=1, router_hidden=128, bias=True, dropout=0.1,
                 balance_loss_weight=0.02, shared_init_scale=0.3, entropy_bonus_weight=0.001,
                 add_gumbel_noise=True, temperature=1.0):
        super().__init__()
        hidden=expansion*in_dim
        self.shared_fc=nn.Linear(in_dim,hidden,bias=bias); self.shared_proj=nn.Linear(hidden,in_dim,bias=bias)
        self.expert_fc=nn.ModuleList([nn.Linear(in_dim,hidden,bias=bias) for _ in range(num_experts)])
        self.expert_proj=nn.ModuleList([nn.Linear(hidden,in_dim,bias=bias) for _ in range(num_experts)])
        self.router=TopKRouter(in_dim, router_hidden, num_experts, k, add_gumbel_noise, temperature)
        self.route_norm=nn.LayerNorm(in_dim); self.act=nn.GELU(); self.drop=nn.Dropout(dropout)
        self.shared_scale=nn.Parameter(torch.tensor(float(shared_init_scale)))
        self.blw=balance_loss_weight; self.entw=entropy_bonus_weight; self.k=k; self.num_experts=num_experts
    def forward(self, x):  # [B,T,C]
        B,T,C=x.shape; xf=x.view(B*T,C)
        shared=self.shared_proj(self.act(self.shared_fc(xf)))*self.shared_scale
        sp,dp=self.router(self.route_norm(xf))
        routed=0.0
        for e in range(self.num_experts):
            h=self.expert_proj[e](self.act(self.expert_fc[e](xf)))
            routed=routed + sp[:,e].unsqueeze(-1)*h
        y=(shared+routed).view(B,T,C); y=self.drop(y)
        ent=entropy_mean(dp); aux=self.blw*kl_to_uniform(dp) - self.entw*ent
        with torch.no_grad():
            counts=(sp>0).float().sum(0)
        stats={"expert_selection_counts":counts,"mean_routing_probs":dp.mean(0)}
        return y, aux, stats

class GPTBlock(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout, bias, backend="sdpa", moe: Optional[MoEFFN]=None):
        super().__init__(); self.ln1=LayerNorm(n_embd,bias); self.attn=CausalSelfAttention(n_embd,n_head,block,dropout,bias,backend)
        self.ln2=LayerNorm(n_embd,bias); self.moe=moe
        if moe is None:
            hidden=4*n_embd
            self.ffn=nn.Sequential(nn.Linear(n_embd,hidden,bias=bias), nn.GELU(), nn.Linear(hidden,n_embd,bias=bias), nn.Dropout(dropout))
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        if self.moe is None:
            x=x+self.ffn(self.ln2(x)); aux=torch.tensor(0.0,device=x.device)
            stats={"expert_selection_counts":torch.zeros(1,device=x.device),"mean_routing_probs":torch.zeros(1,device=x.device)}
        else:
            y,aux,stats=self.moe(self.ln2(x)); x=x+y
        return x, aux, stats

@dataclass
class GPTCfg:
    block_size:int=256; vocab_size:int=256; n_layer:int=6; n_head:int=6; n_embd:int=384
    dropout:float=0.1; bias:bool=True; attention_backend:str="sdpa"
    use_moe:bool=False; num_experts:int=3; k:int=1; router_hidden:int=128; blw:float=0.02
    shared_init_scale:float=0.3; entropy_bonus:float=0.001
    router_temp:float=1.0; router_temp_start:float=1.5; router_temp_end:float=0.5; add_gumbel:bool=True

class NanoGPT(nn.Module):
    def __init__(self, cfg:GPTCfg):
        super().__init__(); self.cfg=cfg
        self.wte=nn.Embedding(cfg.vocab_size,cfg.n_embd); self.wpe=nn.Embedding(cfg.block_size,cfg.n_embd)
        blocks=[]
        for _ in range(cfg.n_layer):
            moe=None
            if cfg.use_moe:
                moe=MoEFFN(cfg.n_embd, expansion=4, num_experts=cfg.num_experts, k=cfg.k, router_hidden=cfg.router_hidden,
                           bias=cfg.bias, dropout=cfg.dropout, balance_loss_weight=cfg.blw,
                           shared_init_scale=cfg.shared_init_scale, entropy_bonus_weight=cfg.entropy_bonus,
                           add_gumbel_noise=cfg.add_gumbel, temperature=cfg.router_temp)
            blocks.append(GPTBlock(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout, cfg.bias, cfg.attention_backend, moe))
        self.blocks=nn.ModuleList(blocks); self.ln_f=LayerNorm(cfg.n_embd,cfg.bias); self.lm_head=nn.Linear(cfg.n_embd,cfg.vocab_size,bias=False)
        self.lm_head.weight=self.wte.weight
        self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02);
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward(self, idx, targets=None):
        B,T=idx.shape; assert T<=self.cfg.block_size
        pos=torch.arange(0,T,device=idx.device).long(); x=self.wte(idx)+self.wpe(pos)[None,:,:]
        aux_total=torch.tensor(0.0,device=idx.device); last=None
        for blk in self.blocks:
            x,aux,stats=blk(x); aux_total=aux_total+aux; last=stats
        x=self.ln_f(x); logits=self.lm_head(x); loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1)); loss=loss+aux_total
        return logits, loss, last
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.cfg.block_size else idx[:,-self.cfg.block_size:]
            logits,_,_=self(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); idx_next=torch.multinomial(probs,1); idx=torch.cat((idx,idx_next),dim=1)
        return idx

def activated_params_gpt(model:NanoGPT)->Dict[str,int]:
    total,train=count_parameters(model)
    attn=0; shared=0; router=0; per_exp=0
    for blk in model.blocks:
        c=blk.attn
        attn+=c.c_attn.weight.numel(); attn+=c.c_proj.weight.numel()
        if c.c_attn.bias is not None: attn+=c.c_attn.bias.numel()
        if c.c_proj.bias is not None: attn+=c.c_proj.bias.numel()
        if blk.moe is not None:
            m=blk.moe
            shared+=m.shared_fc.weight.numel()+(m.shared_fc.bias.numel() if m.shared_fc.bias is not None else 0)
            shared+=m.shared_proj.weight.numel()+(m.shared_proj.bias.numel() if m.shared_proj.bias is not None else 0)
            ef,ep=m.expert_fc[0], m.expert_proj[0]
            per_exp+=ef.weight.numel()+(ef.bias.numel() if ef.bias is not None else 0)
            per_exp+=ep.weight.numel()+(ep.bias.numel() if ep.bias is not None else 0)
            for p in m.router.parameters(): router+=p.numel()
    k=model.cfg.k; active=attn+shared+router+k*per_exp
    return {"Total params":total,"Trainable params":train,"Attention params":attn,"Shared FFN":shared,"Router":router,"Per expert":per_exp,"k":k,"Active per forward":active}

# =========================== Training / Sampling ===========================

def train_lm(model, train_ids, val_ids, block_size, epochs, steps_per_epoch, batch_size, lr, device, title, temp_sched=None):
    model=model.to(device); opt=torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.95), weight_decay=0.1)
    for ep in range(1,epochs+1):
        if temp_sched is not None:
            temp=temp_sched(ep,epochs)
            for m in model.modules():
                if isinstance(m, MoEFFN): m.router.temperature=float(temp)
        t0=time.time(); model.train(); losses=[]; last=None
        for _ in range(steps_per_epoch):
            xb,yb=get_batch_tokens(train_ids, block_size, batch_size, device)
            _,loss,stats=model(xb,yb); opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            losses.append(loss.item()); last=stats
        dt=time.time()-t0
        with torch.no_grad():
            model.eval(); xb,yb=get_batch_tokens(val_ids, block_size, batch_size, device); _,vl,_=model(xb,yb)
        console.print(Panel(f"Epoch {ep}/{epochs}  train=[bold]{sum(losses)/len(losses):.3f}[/]  val=[bold magenta]{vl.item():.3f}[/]  time={dt:.1f}s",
                            title=title, border_style="cyan", box=box.ROUNDED, expand=True))
        if last is not None: routing_panel(last, f"{title} Routing (last batch)")

@torch.no_grad()
def sample_lm(model, tok, device, start="\n", tokens=200, title="Sample"):
    model.eval().to(device)
    x=torch.tensor([encode_text(tok, start)], dtype=torch.long, device=device)
    y=model.generate(x, max_new_tokens=tokens, temperature=0.8, top_k=200)[0].tolist()
    txt=decode_ids(tok, y)
    console.print(Panel(txt, title=title, border_style="magenta", box=box.ROUNDED, expand=True))

# ================================ Save / Load ================================

def save_ckpt(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload, path); console.print(Panel(f"Saved to [bold]{path}[/]", border_style="magenta"))

def load_ckpt(path):
    payload=torch.load(path, map_location="cpu"); console.print(Panel(f"Loaded [bold]{path}[/]", border_style="magenta")); return payload

# =================================== CLI ===================================

def main(argv=None):
    ap=argparse.ArgumentParser(description="BPE-based NanoGPT with optional MoE-FFN (minbpe)")
    ap.add_argument("--mode",required=True,choices=["train_gpt_bpe","infer_gpt_bpe"])
    ap.add_argument("--device",default="auto",choices=["auto","cpu","cuda"])
    ap.add_argument("--epochs",type=int,default=2)
    ap.add_argument("--steps_per_epoch",type=int,default=200)
    ap.add_argument("--batch_size",type=int,default=64)
    ap.add_argument("--lr",type=float,default=3e-4)
    ap.add_argument("--block_size",type=int,default=256)
    ap.add_argument("--dropout",type=float,default=0.1)
    ap.add_argument("--n_layer",type=int,default=6)
    ap.add_argument("--n_head",type=int,default=6)
    ap.add_argument("--n_embd",type=int,default=384)
    ap.add_argument("--bias",action="store_true",default=True)
    ap.add_argument("--attention_backend",default="sdpa",choices=["sdpa","vanilla"])
    ap.add_argument("--use_moe",action="store_true")
    ap.add_argument("--num_experts",type=int,default=3)
    ap.add_argument("--k",type=int,default=1)
    ap.add_argument("--router_hidden",type=int,default=128)
    ap.add_argument("--blw",type=float,default=0.02)
    ap.add_argument("--shared_init_scale",type=float,default=0.3)
    ap.add_argument("--entropy_bonus",type=float,default=0.001)
    ap.add_argument("--router_temp",type=float,default=1.0)
    ap.add_argument("--router_temp_start",type=float,default=1.5)
    ap.add_argument("--router_temp_end",type=float,default=0.5)
    ap.add_argument("--no_gumbel",action="store_true")

    # Tokenizer knobs
    ap.add_argument("--tok_kind",default="basic",choices=["basic","regex"])
    ap.add_argument("--vocab_size",type=int,default=2048)
    ap.add_argument("--tok_prefix",default="artifacts/tokenizers/tiny_bpe")  # saves .model/.vocab here
    ap.add_argument("--train_corpus",default="")  # optional path to your own text; default uses Tiny Shakespeare

    ap.add_argument("--save_path",default="artifacts/gpt_moe_bpe.pt")
    ap.add_argument("--load_path",default="artifacts/gpt_moe_bpe.pt")
    ap.add_argument("--seed",type=int,default=1337)
    ap.add_argument("--sample_start",default="\n")
    ap.add_argument("--sample_tokens",type=int,default=200)
    args=ap.parse_args(argv)
    set_seed(args.seed); device=args.device if args.device!="auto" else ("cuda" if torch.cuda.is_available() else "cpu")

    # 1) Load text
    if args.train_corpus and os.path.exists(args.train_corpus):
        with open(args.train_corpus,"r",encoding="utf-8") as f:
            full_text = f.read()
        split=int(0.9*len(full_text)); train_txt, val_txt = full_text[:split], full_text[split:]
    else:
        train_txt, val_txt = load_tiny_shakespeare("./data")

    # 2) Train or load tokenizer
    tok, tok_model_path, vocab_size_actual = train_or_load_tokenizer(
        args.tok_kind, args.vocab_size, train_txt + val_txt, args.tok_prefix, verbose=True
    )

    # 3) Encode to BPE IDs
    train_ids = encode_text(tok, train_txt)
    val_ids   = encode_text(tok, val_txt)

    # 4) Maybe shrink block size if needed
    if len(train_ids) <= args.block_size + 1:
        new_bs=max(16, min(args.block_size, len(train_ids)-2))
        console.print(Panel(f"Auto-adjusting block_size {args.block_size} -> {new_bs}", border_style="yellow", title="Safety"))
        args.block_size=new_bs

    if args.mode == "train_gpt_bpe":
        cfg=GPTCfg(block_size=args.block_size, vocab_size=vocab_size_actual, n_layer=args.n_layer, n_head=args.n_head,
                   n_embd=args.n_embd, dropout=args.dropout, bias=args.bias, attention_backend=args.attention_backend,
                   use_moe=args.use_moe, num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden, blw=args.blw,
                   shared_init_scale=args.shared_init_scale, entropy_bonus=args.entropy_bonus,
                   router_temp=args.router_temp, router_temp_start=args.router_temp_start,
                   router_temp_end=args.router_temp_end, add_gumbel=not args.no_gumbel)
        console.print(Panel(json.dumps(asdict(cfg),indent=2), title="GPT-BPE Config", border_style="blue"))

        model=NanoGPT(cfg)

        # router temp schedule across epochs (optional)
        def temp_sched(ep, E):
            alpha=(ep-1)/max(1,(E-1))
            return args.router_temp_end + (args.router_temp_start-args.router_temp_end)*(1.0-alpha)

        train_lm(model, train_ids, val_ids, cfg.block_size, args.epochs, args.steps_per_epoch, args.batch_size, args.lr, device,
                 title=f"NanoGPT-BPE ({'MoE' if cfg.use_moe else 'no-MoE'}, {cfg.attention_backend})",
                 temp_sched=(temp_sched if cfg.use_moe else None))

        # params panel
        gp=activated_params_gpt(model) if cfg.use_moe else {"Total params":count_parameters(model)[0],"Trainable params":count_parameters(model)[1],"Attention params":sum(p.numel() for n,p in model.named_parameters() if 'c_attn' in n or 'c_proj' in n)}
        params_panel("GPT-BPE Parameters", gp, extras=[("Tokenizer model", tok_model_path), ("Vocab size", f"{vocab_size_actual}")])

        save_ckpt(args.save_path, {
            "kind":"gpt_bpe",
            "cfg":asdict(cfg),
            "state_dict":model.state_dict(),
            "tokenizer": {
                "kind": args.tok_kind,
                "model_path": tok_model_path,
                "vocab_size": vocab_size_actual,
            }
        })

    elif args.mode == "infer_gpt_bpe":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe", "Checkpoint is not gpt_bpe kind."
        tok_info=payload["tokenizer"]; tok=_new_tok(tok_info["kind"]); tok.load(tok_info["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        sample_lm(model, tok, device, start=args.sample_start, tokens=args.sample_tokens,
                  title=f"NanoGPT-BPE Sample ({'MoE' if cfg.use_moe else 'no-MoE'}, {cfg.attention_backend})")

if __name__ == "__main__":
    import sys, os
    argv=sys.argv[1:]; has_mode=any(a=="--mode" or a.startswith("--mode=") for a in argv)
    if not has_mode:
        env=os.environ.get("MOE_ARGS","").strip()
        argv=shlex.split(env) if env else [
            "--mode","train_gpt_bpe",
            "--tok_kind","basic","--vocab_size","512",
            "--use_moe","--epochs","1","--steps_per_epoch","50",
            "--batch_size","64","--attention_backend","sdpa",
            "--save_path","artifacts/gpt_moe_bpe.pt",
            "--tok_prefix","artifacts/tokenizers/tiny_bpe"
        ]
    main(argv)


# NONE UNIFORM EXPERTS USE BPE NANOGPT MOE

In [ ]:
import os

os.environ["MOE_ARGS"] = " ".join([
    "--mode train_gpt_bpe",
    "--tok_kind basic --vocab_size 512",
    "--use_moe",
    "--routing_mode specialize",
    "--blw 0.001",
    "--entropy_penalty 0.015",
    "--router_temp_start 1.5 --router_temp_end 0.7",
    "--gumbel_off_epoch 4",
    "--shared_init_scale 0.1",
    "--epochs 7 --steps_per_epoch 200",
    "--batch_size 64 --attention_backend sdpa",
    "--save_path artifacts/gpt_moe_bpe.pt",
    "--tok_prefix artifacts/tokenizers/tiny_bpe"
])



In [ ]:
#!/usr/bin/env python3
# gpt_moe_bpe.py
# NanoGPT-style LM with optional MoE-FFN, now using minbpe (Basic/Regex) BPE tokens.
# Adds routing modes, entropy penalty, collapse-rescue, and gumbel_off_epoch.

import os, math, json, argparse, random, time, urllib.request, shlex
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Any, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box

# ------------------------- minbpe (embedded minimal impl) -------------------------

import unicodedata
import regex as re

def get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if ids[i] == pair[0] and i < len(ids) - 1 and ids[i+1] == pair[1]:
            newids.append(idx); i += 2
        else:
            newids.append(ids[i]); i += 1
    return newids

def replace_control_characters(s: str) -> str:
    chars = []
    for ch in s:
        if unicodedata.category(ch)[0] != "C":
            chars.append(ch)
        else:
            chars.append(f"\\u{ord(ch):04x}")
    return "".join(chars)

def render_token(t: bytes) -> str:
    s = t.decode('utf-8', errors='replace')
    s = replace_control_characters(s)
    return s

class Tokenizer:
    def __init__(self):
        self.merges = {}
        self.pattern = ""
        self.special_tokens = {}
        self.vocab = self._build_vocab()
    def train(self, text, vocab_size, verbose=False): raise NotImplementedError
    def encode(self, text): raise NotImplementedError
    def decode(self, ids): raise NotImplementedError
    def _build_vocab(self):
        vocab = {idx: bytes([idx]) for idx in range(256)}
        for (p0, p1), idx in self.merges.items():
            vocab[idx] = vocab[p0] + vocab[p1]
        for special, idx in self.special_tokens.items():
            vocab[idx] = special.encode("utf-8")
        return vocab
    def save(self, file_prefix):
        model_file = file_prefix + ".model"
        with open(model_file, 'w') as f:
            f.write("minbpe v1\n")
            f.write(f"{self.pattern}\n")
            f.write(f"{len(self.special_tokens)}\n")
            for special, idx in self.special_tokens.items():
                f.write(f"{special} {idx}\n")
            for idx1, idx2 in self.merges:
                f.write(f"{idx1} {idx2}\n")
        vocab_file = file_prefix + ".vocab"
        inverted_merges = {idx: pair for pair, idx in self.merges.items()}
        with open(vocab_file, "w", encoding="utf-8") as f:
            for idx, token in self.vocab.items():
                s = render_token(token)
                if idx in inverted_merges:
                    idx0, idx1 = inverted_merges[idx]
                    s0 = render_token(self.vocab[idx0]); s1 = render_token(self.vocab[idx1])
                    f.write(f"[{s0}][{s1}] -> [{s}] {idx}\n")
                else:
                    f.write(f"[{s}] {idx}\n")
    def load(self, model_file):
        assert model_file.endswith(".model")
        merges = {}; special_tokens = {}
        idx = 256
        with open(model_file, 'r', encoding="utf-8") as f:
            version = f.readline().strip(); assert version == "minbpe v1"
            self.pattern = f.readline().strip()
            num_special = int(f.readline().strip())
            for _ in range(num_special):
                special, special_idx = f.readline().strip().split()
                special_tokens[special] = int(special_idx)
            for line in f:
                idx1, idx2 = map(int, line.split())
                merges[(idx1, idx2)] = idx; idx += 1
        self.merges = merges; self.special_tokens = special_tokens; self.vocab = self._build_vocab()

class BasicTokenizer(Tokenizer):
    def __init__(self): super().__init__()
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256
        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)
        merges = {}; vocab = {idx: bytes([idx]) for idx in range(256)}
        for i in range(num_merges):
            stats = get_stats(ids)
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = merge(ids, pair, idx)
            merges[pair] = idx; vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")
        self.merges = merges; self.vocab = vocab
    def decode(self, ids):
        text_bytes = b"".join(self.vocab[idx] for idx in ids)
        return text_bytes.decode("utf-8", errors="replace")
    def encode(self, text):
        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges: break
            idx = self.merges[pair]; ids = merge(ids, pair, idx)
        return ids

GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexTokenizer(Tokenizer):
    def __init__(self, pattern=None):
        super().__init__()
        self.pattern = GPT4_SPLIT_PATTERN if pattern is None else pattern
        self.compiled_pattern = re.compile(self.pattern)
        self.special_tokens = {}
        self.inverse_special_tokens = {}
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256
        text_chunks = re.findall(self.compiled_pattern, text)
        ids = [list(ch.encode("utf-8")) for ch in text_chunks]
        merges = {}; vocab = {idx: bytes([idx]) for idx in range(256)}
        for i in range(num_merges):
            stats = {}
            for chunk_ids in ids:
                get_stats(chunk_ids, stats)
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = [merge(chunk_ids, pair, idx) for chunk_ids in ids]
            merges[pair] = idx; vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose:
                print(f"merge {i+1}/{num_merges}: {pair} -> {idx} ({vocab[idx]}) had {stats[pair]} occurrences")
        self.merges = merges; self.vocab = vocab
    def register_special_tokens(self, special_tokens):
        self.special_tokens = special_tokens
        self.inverse_special_tokens = {v: k for k, v in special_tokens.items()}
    def decode(self, ids):
        part_bytes = []
        for idx in ids:
            if idx in self.vocab:
                part_bytes.append(self.vocab[idx])
            elif idx in self.inverse_special_tokens:
                part_bytes.append(self.inverse_special_tokens[idx].encode("utf-8"))
            else:
                raise ValueError(f"invalid token id: {idx}")
        return b"".join(part_bytes).decode("utf-8", errors="replace")
    def _encode_chunk(self, text_bytes):
        ids = list(text_bytes)
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges: break
            idx = self.merges[pair]; ids = merge(ids, pair, idx)
        return ids
    def encode_ordinary(self, text):
        text_chunks = re.findall(self.compiled_pattern, text)
        ids = []
        for chunk in text_chunks:
            ids.extend(self._encode_chunk(chunk.encode("utf-8")))
        return ids
    def encode(self, text, allowed_special="none_raise"):
        special = None
        if allowed_special == "all": special = self.special_tokens
        elif allowed_special == "none": special = {}
        elif allowed_special == "none_raise":
            special = {}; assert all(token not in text for token in self.special_tokens)
        elif isinstance(allowed_special, set):
            special = {k: v for k, v in self.special_tokens.items() if k in allowed_special}
        else:
            raise ValueError(f"allowed_special={allowed_special} not understood")
        if not special: return self.encode_ordinary(text)
        special_pattern = "(" + "|".join(re.escape(k) for k in special) + ")"
        special_chunks = re.split(special_pattern, text)
        ids = []
        for part in special_chunks:
            if part in special: ids.append(special[part])
            else: ids.extend(self.encode_ordinary(part))
        return ids

# ------------------------- Console and utils -------------------------

console = Console()

def set_seed(seed=1337):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(m: nn.Module) -> Tuple[int,int]:
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

def kl_to_uniform(probs: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    E = probs.size(-1)
    m = probs.mean(dim=0)
    return torch.sum(m * (m.add(eps).log() - math.log(1.0 / E)))

def entropy_mean(probs: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    return -(probs.clamp_min(eps) * probs.clamp_min(eps).log()).sum(dim=-1).mean()

def params_panel(title: str, totals: Dict[str,int], extras: List[Tuple[str,str]] = None):
    t = Table(box=box.SIMPLE_HEAVY)
    t.add_column("Metric", style="bold"); t.add_column("Value", justify="right")
    for k in ["Total params","Trainable params"]:
        t.add_row(k, f"{totals[k]:,}")
    for k in ["Attention params","Shared FFN","Router","Per expert","k","Active per forward"]:
        if k in totals:
            v = totals[k]
            t.add_row(k, f"{v:,}" if isinstance(v,int) else f"{v}")
    if extras:
        for k,v in extras: t.add_row(k,v)
    console.print(Panel(t, title=title, border_style="yellow", box=box.ROUNDED, expand=True))

def routing_panel(stats, title: str):
    counts = stats["expert_selection_counts"].detach().cpu().to(torch.int64).tolist()
    mean_dp = stats["mean_routing_probs"].detach().cpu().tolist()
    mean_sp = stats.get("mean_selected_probs", stats["mean_routing_probs"]).detach().cpu().tolist()
    t = Table(box=box.SIMPLE_HEAVY); t.add_column("Expert"); t.add_column("Batch Picks", justify="right")
    t.add_column("Mean Prob.", justify="right"); t.add_column("Mean Post-TopK", justify="right")
    for i, (c, p, q) in enumerate(zip(counts, mean_dp, mean_sp)):
        t.add_row(f"E{i}", f"{c}", f"{p:.3f}", f"{q:.3f}")
    ent = entropy_mean(torch.tensor(mean_dp)[None, :]).item()
    console.print(Panel(t, title=f"{title} | entropy={ent:.3f}", border_style="green", box=box.ROUNDED, expand=True))


# ------------------------- Data -------------------------

def _repeat_to_len(s: str, target_len: int) -> str:
    if not s: s=" \n"
    return (s * ((target_len // len(s))+1))[:target_len]

def load_tiny_shakespeare(data_dir="./data", target_len=100_000) -> Tuple[str,str]:
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, "tinyshakespeare_input.txt")
    if not os.path.exists(path):
        try:
            url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url, path)
        except Exception:
            sample=("From fairest creatures we desire increase,\n"
                    "That thereby beauty's rose might never die,\n"
                    "But as the riper should by time decease,\n"
                    "His tender heir might bear his memory:\n")
            with open(path,"w",encoding="utf-8") as f: f.write(_repeat_to_len(sample, target_len))
    with open(path,"r",encoding="utf-8") as f: text=f.read()
    split=int(0.9*len(text)); return text[:split], text[split:]

# ------------------------- Tokenizer helpers -------------------------

def train_or_load_tokenizer(tok_kind:str, vocab_size:int, text:str, out_prefix:str, verbose:bool=True):
    os.makedirs(os.path.dirname(out_prefix), exist_ok=True) if os.path.dirname(out_prefix) else None
    model_path = out_prefix + ".model"
    if os.path.exists(model_path):
        tok = _new_tok(tok_kind); tok.load(model_path)
        vocab_size_actual = max(tok.vocab.keys()) + 1
        if verbose:
            console.print(Panel(f"Loaded tokenizer: [bold]{model_path}[/] (size={vocab_size_actual})",
                                border_style="magenta", title="Tokenizer"))
        return tok, model_path, vocab_size_actual
    tok = _new_tok(tok_kind)
    vocab_size = max(256, vocab_size)
    if verbose:
        console.print(Panel(f"Training {tok_kind} BPE (vocab={vocab_size})", border_style="blue", title="Tokenizer"))
    tok.train(text, vocab_size, verbose=verbose); tok.save(out_prefix)
    vocab_size_actual = max(tok.vocab.keys()) + 1
    if verbose:
        console.print(Panel(f"Saved tokenizer to: [bold]{out_prefix}.model[/] (size={vocab_size_actual})",
                            border_style="green", title="Tokenizer"))
    return tok, model_path, vocab_size_actual

def _new_tok(kind:str):
    kind = kind.lower()
    if kind == "basic": return BasicTokenizer()
    if kind == "regex": return RegexTokenizer()
    raise ValueError(f"Unknown tokenizer kind: {kind}")

def encode_text(tok, s:str) -> List[int]: return tok.encode(s)
def decode_ids(tok, ids:List[int]) -> str: return tok.decode(ids)

def get_batch_tokens(ids: List[int], block_size: int, batch_size: int, device: str):
    if len(ids) <= block_size + 1:
        raise RuntimeError(f"Token length {len(ids)} too small for block_size {block_size}. Lower block size.")
    ix=torch.randint(len(ids)-block_size-1,(batch_size,))
    x=torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y=torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# ------------------------- GPT + MoE -------------------------

class LayerNorm(nn.Module):
    def __init__(self, n, bias):
        super().__init__()
        self.weight=nn.Parameter(torch.ones(n))
        self.bias=nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self, x): return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout, bias, backend="sdpa"):
        super().__init__(); assert n_embd % n_head == 0
        self.n_head=n_head; self.n_embd=n_embd; self.dropout=dropout; self.backend=backend
        self.c_attn=nn.Linear(n_embd,3*n_embd,bias=bias); self.c_proj=nn.Linear(n_embd,n_embd,bias=bias)
        self.attn_dropout=nn.Dropout(dropout); self.resid_dropout=nn.Dropout(dropout)
        if backend=="vanilla" or not hasattr(F,"scaled_dot_product_attention"):
            self.register_buffer("bias", torch.tril(torch.ones(block,block)).view(1,1,block,block))
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        if self.backend=="sdpa" and hasattr(F,"scaled_dot_product_attention"):
            y=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=self.dropout if self.training else 0.,is_causal=True)
        else:
            att=(q@k.transpose(-2,-1))*(1.0/math.sqrt(k.size(-1)))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0, float("-inf"))
            att=F.softmax(att,dim=-1); att=self.attn_dropout(att); y=att@v
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.c_proj(y))

class TopKRouter(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=1, add_gumbel_noise=True, temperature=1.0):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden_dim), nn.ReLU(), nn.Linear(hidden_dim,num_experts))
        self.k=k; self.add_gumbel=add_gumbel_noise; self.temperature=float(temperature)
        self.register_buffer("logits_bias", torch.zeros(num_experts))  # used for rescue nudging
    @staticmethod
    def _gumbel(shape, device):
        u=torch.rand(shape,device=device).clamp_(1e-9,1-1e-9); return -torch.log(-torch.log(u))
    def forward(self, x):
        logits=self.net(x)
        if self.add_gumbel and self.training:
            logits=logits+self._gumbel(logits.shape,logits.device)
        logits = logits + self.logits_bias
        probs=F.softmax(logits/self.temperature, dim=-1)
        if self.k>=probs.size(-1): return probs, probs
        topk_vals, topk_idx = torch.topk(probs, k=self.k, dim=-1)
        mask = torch.zeros_like(probs); mask.scatter_(dim=-1, index=topk_idx, src=torch.ones_like(topk_vals))
        sp = probs*mask; sp = sp/(sp.sum(dim=-1,keepdim=True)+1e-9)
        return sp, probs

class MoEFFN(nn.Module):
    def __init__(self, in_dim, expansion=4, num_experts=3, k=1, router_hidden=128, bias=True, dropout=0.1,
                 balance_loss_weight=0.02, shared_init_scale=0.3, entropy_penalty_weight=0.001,
                 add_gumbel_noise=True, temperature=1.0,
                 routing_mode="specialize", rescue_lambda=0.0, usage_momentum=0.99):
        super().__init__()
        hidden=expansion*in_dim
        self.shared_fc=nn.Linear(in_dim,hidden,bias=bias); self.shared_proj=nn.Linear(hidden,in_dim,bias=bias)
        self.expert_fc=nn.ModuleList([nn.Linear(in_dim,hidden,bias=bias) for _ in range(num_experts)])
        self.expert_proj=nn.ModuleList([nn.Linear(hidden,in_dim,bias=bias) for _ in range(num_experts)])
        self.router=TopKRouter(in_dim, router_hidden, num_experts, k, add_gumbel_noise, temperature)
        self.route_norm=nn.LayerNorm(in_dim); self.act=nn.GELU(); self.drop=nn.Dropout(dropout)
        self.shared_scale=nn.Parameter(torch.tensor(float(shared_init_scale)))
        self.blw=balance_loss_weight; self.entw=entropy_penalty_weight
        self.k=k; self.num_experts=num_experts
        # rescue
        self.routing_mode = routing_mode
        self.rescue_lambda = float(rescue_lambda)
        self.usage_momentum = float(usage_momentum)
        self.register_buffer("ema_usage", torch.zeros(num_experts))
        self.blw_picks = getattr(self, "blw_picks", 0.01)  # weight for pick-balance (actual selections)

    def forward(self, x):  # [B,T,C]
        B,T,C=x.shape; xf=x.view(B*T,C)
        shared=self.shared_proj(self.act(self.shared_fc(xf)))*self.shared_scale
        sp,dp=self.router(self.route_norm(xf))
        routed=0.0
        for e in range(self.num_experts):
            h=self.expert_proj[e](self.act(self.expert_fc[e](xf)))
            routed=routed + sp[:,e].unsqueeze(-1)*h
        y = (shared + routed).view(B, T, C); y = self.drop(y)

        # --- losses ---
        ent = entropy_mean(dp)  # encourage softer logits when training
        prob_kl = kl_to_uniform(dp)  # uniformity on probabilities (existing)
        # NEW: balance actual selections (post-topk) within this batch
        with torch.no_grad():
            counts = (sp > 0).float().sum(0)                  # [E] assignments
        BT = float(B * T)
        pick_freq = counts / (BT + 1e-9)                       # empirical usage
        # KL(pick_freq || uniform)
        uniform = torch.full_like(pick_freq, 1.0 / self.num_experts)
        pick_kl = torch.sum(pick_freq * (pick_freq.add(1e-9).log() - math.log(1.0 / self.num_experts)))

        aux = self.blw * prob_kl - self.entw * ent + self.blw_picks * pick_kl

        # --- EMA usage + logits_bias rescue (already in your code) ---
        with torch.no_grad():
            mean_dp = dp.mean(0)
            m = self.usage_momentum
            self.ema_usage.mul_(m).add_((1.0 - m) * mean_dp)
            if self.rescue_lambda > 0.0 and self.training:
                target = torch.full_like(self.ema_usage, 1.0 / self.num_experts)
                usage = self.ema_usage / (self.ema_usage.sum() + 1e-9)
                bias = self.rescue_lambda * (target - usage)
                bias = bias - bias.mean()
                self.router.logits_bias.copy_(bias)

        # --- richer stats: show both pre- and post-topk means
        stats = {
            "expert_selection_counts": counts,
            "mean_routing_probs": dp.mean(0),
            "mean_selected_probs": (sp / (sp.sum(0, keepdim=True) + 1e-9)).sum(0)  # ~= sp.mean(0) normalized
        }
        return y, aux, stats

class GPTBlock(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout, bias, backend="sdpa", moe: Optional[MoEFFN]=None):
        super().__init__(); self.ln1=LayerNorm(n_embd,bias); self.attn=CausalSelfAttention(n_embd,n_head,block,dropout,bias,backend)
        self.ln2=LayerNorm(n_embd,bias); self.moe=moe
        if moe is None:
            hidden=4*n_embd
            self.ffn=nn.Sequential(nn.Linear(n_embd,hidden,bias=bias), nn.GELU(), nn.Linear(hidden,n_embd,bias=bias), nn.Dropout(dropout))
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        if self.moe is None:
            x=x+self.ffn(self.ln2(x)); aux=torch.tensor(0.0,device=x.device)
            stats={"expert_selection_counts":torch.zeros(1,device=x.device),"mean_routing_probs":torch.zeros(1,device=x.device)}
        else:
            y,aux,stats=self.moe(self.ln2(x)); x=x+y
        return x, aux, stats

@dataclass
class GPTCfg:
    block_size:int=256; vocab_size:int=256; n_layer:int=6; n_head:int=6; n_embd:int=384
    dropout:float=0.1; bias:bool=True; attention_backend:str="sdpa"
    use_moe:bool=False; num_experts:int=3; k:int=1; router_hidden:int=128; blw:float=0.02
    shared_init_scale:float=0.3
    entropy_penalty:float=0.001
    router_temp:float=1.0; router_temp_start:float=1.5; router_temp_end:float=0.5
    add_gumbel:bool=True; gumbel_off_epoch:int=0
    routing_mode:str="specialize"  # "specialize" or "uniform"
    rescue_lambda:float=0.0
    usage_momentum:float=0.99

class NanoGPT(nn.Module):
    def __init__(self, cfg:GPTCfg):
        super().__init__(); self.cfg=cfg
        self.wte=nn.Embedding(cfg.vocab_size,cfg.n_embd); self.wpe=nn.Embedding(cfg.block_size,cfg.n_embd)
        blocks=[]
        for _ in range(cfg.n_layer):
            moe=None
            if cfg.use_moe:
                moe=MoEFFN(cfg.n_embd, expansion=4, num_experts=cfg.num_experts, k=cfg.k, router_hidden=cfg.router_hidden,
                           bias=cfg.bias, dropout=cfg.dropout, balance_loss_weight=cfg.blw,
                           shared_init_scale=cfg.shared_init_scale, entropy_penalty_weight=cfg.entropy_penalty,
                           add_gumbel_noise=cfg.add_gumbel, temperature=cfg.router_temp,
                           routing_mode=cfg.routing_mode, rescue_lambda=cfg.rescue_lambda, usage_momentum=cfg.usage_momentum)
            blocks.append(GPTBlock(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout, cfg.bias, cfg.attention_backend, moe))
        self.blocks=nn.ModuleList(blocks); self.ln_f=LayerNorm(cfg.n_embd,cfg.bias); self.lm_head=nn.Linear(cfg.n_embd,cfg.vocab_size,bias=False)
        self.lm_head.weight=self.wte.weight
        self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward(self, idx, targets=None):
        B,T=idx.shape; assert T<=self.cfg.block_size
        pos=torch.arange(0,T,device=idx.device).long(); x=self.wte(idx)+self.wpe(pos)[None,:,:]
        aux_total=torch.tensor(0.0,device=idx.device); last=None
        for blk in self.blocks:
            x,aux,stats=blk(x); aux_total=aux_total+aux; last=stats
        x=self.ln_f(x); logits=self.lm_head(x); loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1)); loss=loss+aux_total
        return logits, loss, last
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.cfg.block_size else idx[:,-self.cfg.block_size:]
            logits,_,_=self(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); idx_next=torch.multinomial(probs,1); idx=torch.cat((idx,idx_next),dim=1)
        return idx

def activated_params_gpt(model:NanoGPT)->Dict[str,int]:
    total,train=count_parameters(model)
    attn=0; shared=0; router=0; per_exp=0
    for blk in model.blocks:
        c=blk.attn
        attn+=c.c_attn.weight.numel(); attn+=c.c_proj.weight.numel()
        if c.c_attn.bias is not None: attn+=c.c_attn.bias.numel()
        if c.c_proj.bias is not None: attn+=c.c_proj.bias.numel()
        if blk.moe is not None:
            m=blk.moe
            shared+=m.shared_fc.weight.numel()+(m.shared_fc.bias.numel() if m.shared_fc.bias is not None else 0)
            shared+=m.shared_proj.weight.numel()+(m.shared_proj.bias.numel() if m.shared_proj.bias is not None else 0)
            ef,ep=m.expert_fc[0], m.expert_proj[0]
            per_exp+=ef.weight.numel()+(ef.bias.numel() if ef.bias is not None else 0)
            per_exp+=ep.weight.numel()+(ep.bias.numel() if ep.bias is not None else 0)
            for p in m.router.parameters(): router+=p.numel()
    k=model.cfg.k; active=attn+shared+router+k*per_exp
    return {"Total params":total,"Trainable params":train,"Attention params":attn,"Shared FFN":shared,"Router":router,"Per expert":per_exp,"k":k,"Active per forward":active}

# ------------------------- Training / Sampling -------------------------

def train_lm(model, train_ids, val_ids, block_size, epochs, steps_per_epoch, batch_size, lr, device, title,
             temp_sched=None, gumbel_off_epoch=0):
    model=model.to(device); opt=torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.95), weight_decay=0.1)
    for ep in range(1,epochs+1):
        # router temp schedule
        if temp_sched is not None:
            temp=temp_sched(ep,epochs)
            for m in model.modules():
                if isinstance(m, MoEFFN): m.router.temperature=float(temp)
        # gumbel schedule: disable after gumbel_off_epoch
        for m in model.modules():
            if isinstance(m, MoEFFN):
                m.router.add_gumbel = (gumbel_off_epoch <= 0) or (ep <= gumbel_off_epoch)

        t0=time.time(); model.train(); losses=[]; last=None
        for _ in range(steps_per_epoch):
            xb,yb=get_batch_tokens(train_ids, block_size, batch_size, device)
            _,loss,stats=model(xb,yb); opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            losses.append(loss.item()); last=stats
        dt=time.time()-t0
        with torch.no_grad():
            model.eval(); xb,yb=get_batch_tokens(val_ids, block_size, batch_size, device); _,vl,_=model(xb,yb)
        console.print(Panel(f"Epoch {ep}/{epochs}  train=[bold]{sum(losses)/len(losses):.3f}[/]  val=[bold magenta]{vl.item():.3f}[/]  time={dt:.1f}s",
                            title=title, border_style="cyan", box=box.ROUNDED, expand=True))
        if last is not None: routing_panel(last, f"{title} Routing (last batch)")

@torch.no_grad()
def sample_lm(model, tok, device, start="\n", tokens=200, title="Sample"):
    model.eval().to(device)
    x=torch.tensor([encode_text(tok, start)], dtype=torch.long, device=device)
    y=model.generate(x, max_new_tokens=tokens, temperature=0.8, top_k=200)[0].tolist()
    txt=decode_ids(tok, y)
    console.print(Panel(txt, title=title, border_style="magenta", box=box.ROUNDED, expand=True))

# ------------------------- Save / Load -------------------------

def save_ckpt(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload, path); console.print(Panel(f"Saved to [bold]{path}[/]", border_style="magenta"))

def load_ckpt(path):
    payload=torch.load(path, map_location="cpu"); console.print(Panel(f"Loaded [bold]{path}[/]", border_style="magenta")); return payload

# ------------------------- Environment Detection -------------------------

def is_jupyter_or_colab():
    """Detect if running in Jupyter/Colab environment"""
    try:
        # Check for IPython/Jupyter
        from IPython import get_ipython
        if get_ipython() is not None:
            return True

        # Check for Google Colab specific
        import sys
        if 'google.colab' in sys.modules:
            return True

        # Check for Jupyter kernel
        if any('jupyter' in arg.lower() for arg in sys.argv):
            return True

        return False
    except ImportError:
        return False

# ------------------------- CLI -------------------------

def main(argv=None):
    ap = argparse.ArgumentParser(description="BPE-based NanoGPT with optional MoE-FFN (minbpe)")
    ap.add_argument("--mode", required=True, choices=["train_gpt_bpe", "infer_gpt_bpe"])
    ap.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"])
    ap.add_argument("--epochs", type=int, default=2)
    ap.add_argument("--steps_per_epoch", type=int, default=200)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--block_size", type=int, default=256)
    ap.add_argument("--dropout", type=float, default=0.1)
    ap.add_argument("--n_layer", type=int, default=6)
    ap.add_argument("--n_head", type=int, default=6)
    ap.add_argument("--n_embd", type=int, default=384)
    ap.add_argument("--bias", action="store_true", default=True)
    ap.add_argument("--attention_backend", default="sdpa", choices=["sdpa", "vanilla"])

    # MoE knobs
    ap.add_argument("--use_moe", action="store_true")
    ap.add_argument("--num_experts", type=int, default=3)
    ap.add_argument("--k", type=int, default=1)
    ap.add_argument("--router_hidden", type=int, default=128)
    ap.add_argument("--blw", type=float, default=0.02)
    ap.add_argument("--shared_init_scale", type=float, default=0.3)
    ap.add_argument("--entropy_penalty", type=float, default=0.001)
    ap.add_argument("--router_temp", type=float, default=1.0)
    ap.add_argument("--router_temp_start", type=float, default=1.5)
    ap.add_argument("--router_temp_end", type=float, default=0.5)
    ap.add_argument("--no_gumbel", action="store_true")
    ap.add_argument("--gumbel_off_epoch", type=int, default=0)
    ap.add_argument("--routing_mode", choices=["specialize", "uniform"], default="specialize")
    ap.add_argument("--rescue_lambda", type=float, default=0.0)
    ap.add_argument("--usage_momentum", type=float, default=0.99)

    # Tokenizer knobs
    ap.add_argument("--tok_kind", default="basic", choices=["basic", "regex"])
    ap.add_argument("--vocab_size", type=int, default=2048)
    ap.add_argument("--tok_prefix", default="artifacts/tokenizers/tiny_bpe")
    ap.add_argument("--train_corpus", default="")

    # Checkpoints
    ap.add_argument("--save_path", default="artifacts/gpt_moe_bpe.pt")
    ap.add_argument("--load_path", default="artifacts/gpt_moe_bpe.pt")

    # Misc
    ap.add_argument("--seed", type=int, default=1337)
    ap.add_argument("--sample_start", default="\n")
    ap.add_argument("--sample_tokens", type=int, default=200)

    args = ap.parse_args(argv)
    set_seed(args.seed)
    device = args.device if args.device != "auto" else ("cuda" if torch.cuda.is_available() else "cpu")

    # ---------------- Safety setup ----------------
    os.makedirs("data", exist_ok=True)
    os.makedirs("artifacts", exist_ok=True)
    os.makedirs(os.path.dirname(args.tok_prefix), exist_ok=True)

    # Check tokenizer health
    tok_model_path = args.tok_prefix + ".model"
    if os.path.exists(tok_model_path):
        try:
            _tmp_tok = _new_tok(args.tok_kind)
            _tmp_tok.load(tok_model_path)
        except Exception as e:
            console.print(f"[red]Tokenizer load failed ({e}), retraining from scratch[/red]")
            os.remove(tok_model_path)

    # ---------------- Data ----------------
    if args.train_corpus and os.path.exists(args.train_corpus):
        with open(args.train_corpus, "r", encoding="utf-8") as f:
            full_text = f.read()
        split = int(0.9 * len(full_text))
        train_txt, val_txt = full_text[:split], full_text[split:]
    else:
        train_txt, val_txt = load_tiny_shakespeare("./data")

    # ---------------- Tokenizer ----------------
    tok, tok_model_path, vocab_size_actual = train_or_load_tokenizer(
        args.tok_kind, args.vocab_size, train_txt + val_txt, args.tok_prefix, verbose=True
    )

    # Encode
    train_ids = encode_text(tok, train_txt)
    val_ids   = encode_text(tok, val_txt)

    # Shrink block size if dataset is too small
    if len(train_ids) <= args.block_size + 1:
        new_bs = max(16, min(args.block_size, len(train_ids) - 2))
        console.print(Panel(f"Auto-adjusting block_size {args.block_size} -> {new_bs}", border_style="yellow", title="Safety"))
        args.block_size = new_bs

    # ---------------- Train ----------------
    if args.mode == "train_gpt_bpe":
        cfg = GPTCfg(
            block_size=args.block_size, vocab_size=vocab_size_actual, n_layer=args.n_layer,
            n_head=args.n_head, n_embd=args.n_embd, dropout=args.dropout, bias=args.bias,
            attention_backend=args.attention_backend, use_moe=args.use_moe, num_experts=args.num_experts,
            k=args.k, router_hidden=args.router_hidden, blw=args.blw, shared_init_scale=args.shared_init_scale,
            entropy_penalty=args.entropy_penalty, router_temp=args.router_temp,
            router_temp_start=args.router_temp_start, router_temp_end=args.router_temp_end,
            add_gumbel=not args.no_gumbel, gumbel_off_epoch=args.gumbel_off_epoch,
            routing_mode=args.routing_mode, rescue_lambda=args.rescue_lambda, usage_momentum=args.usage_momentum
        )
        console.print(Panel(json.dumps(asdict(cfg), indent=2), title="GPT-BPE Config", border_style="blue"))
        model = NanoGPT(cfg)

        # Router temp schedule
        def temp_sched(ep, E):
            alpha = (ep - 1) / max(1, (E - 1))
            return args.router_temp_end + (args.router_temp_start - args.router_temp_end) * (1.0 - alpha)

        train_lm(
            model, train_ids, val_ids, cfg.block_size, args.epochs, args.steps_per_epoch,
            args.batch_size, args.lr, device,
            title=f"NanoGPT-BPE ({'MoE' if cfg.use_moe else 'no-MoE'}, {cfg.attention_backend})",
            temp_sched=(temp_sched if cfg.use_moe else None),
            gumbel_off_epoch=(cfg.gumbel_off_epoch if cfg.use_moe else 0)
        )

        # Params report
        gp = activated_params_gpt(model) if cfg.use_moe else {
            "Total params": count_parameters(model)[0],
            "Trainable params": count_parameters(model)[1],
            "Attention params": sum(p.numel() for n, p in model.named_parameters() if 'c_attn' in n or 'c_proj' in n)
        }
        params_panel("GPT-BPE Parameters", gp, extras=[("Tokenizer model", tok_model_path), ("Vocab size", f"{vocab_size_actual}")])

        # Save
        save_ckpt(args.save_path, {
            "kind": "gpt_bpe",
            "cfg": asdict(cfg),
            "state_dict": model.state_dict(),
            "tokenizer": {
                "kind": args.tok_kind,
                "model_path": tok_model_path,
                "vocab_size": vocab_size_actual,
            }
        })

    # ---------------- Inference ----------------
    elif args.mode == "infer_gpt_bpe":
        payload = load_ckpt(args.load_path)
        assert payload["kind"] == "gpt_bpe", "Checkpoint is not gpt_bpe kind."
        tok_info = payload["tokenizer"]
        tok = _new_tok(tok_info["kind"])
        tok.load(tok_info["model_path"])
        cfg = GPTCfg(**payload["cfg"])
        model = NanoGPT(cfg)
        model.load_state_dict(payload["state_dict"])
        sample_lm(model, tok, device, start=args.sample_start, tokens=args.sample_tokens,
                  title=f"NanoGPT-BPE Sample ({'MoE' if cfg.use_moe else 'no-MoE'}, {cfg.attention_backend})")

def run_colab_mode():
    """Run with default arguments for Colab/Jupyter environments"""
    console.print("[yellow]Running in Jupyter/Colab mode with default arguments[/yellow]")

    # Default arguments for Colab
    default_args = [
        "--mode", "train_gpt_bpe",
        "--tok_kind", "basic",
        "--vocab_size", "512",
        "--use_moe",
        "--routing_mode", "specialize",
        "--epochs", "1",
        "--steps_per_epoch", "50",
        "--batch_size", "32",
        "--save_path", "artifacts/gpt_moe.pt",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe"
    ]

    main(default_args)

# ------------------------- Entry Point -------------------------

if __name__ == "__main__":
    import sys

    # Check if running in Jupyter/Colab environment
    if is_jupyter_or_colab():
        run_colab_mode()
    else:
        # Normal CLI mode - pass actual command line arguments
        main()

Running in Jupyter/Colab mode with default arguments

╭─────────────────────────────────────────────────── Tokenizer ───────────────────────────────────────────────────╮
│ Training basic BPE (vocab=512)                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

merge 1/256: (101, 32) -> 256 (b'e ') had 27643 occurrences
merge 2/256: (116, 104) -> 257 (b'th') had 22739 occurrences
merge 3/256: (116, 32) -> 258 (b't ') had 16508 occurrences
merge 4/256: (115, 32) -> 259 (b's ') had 15364 occurrences
merge 5/256: (100, 32) -> 260 (b'd ') had 14165 occurrences
merge 6/256: (44, 32) -> 261 (b', ') had 14098 occurrences
merge 7/256: (111, 117) -> 262 (b'ou') had 12730 occurrences
merge 8/256: (101, 114) -> 263 (b'er') had 11771 occurrences
merge 9/256: (105, 110) -> 264 (b'in') had 10606 occurrences
merge 10/256: (121, 32) -> 265 (b'y ') had 10283 occurrences
merge 11/256: (97, 110) -> 266 (b'an') had 10197 occurrences
merge 12/256: (58, 10) -> 267 (b':\n') had 8762 occurrences
merge 13/256: (111, 114) -> 268 (b'or') had 8458 occurrences
merge 14/256: (111, 32) -> 269 (b'o ') had 8134 occurrences
merge 15/256: (101, 110) -> 270 (b'en') had 7568 occurrences
merge 16/256: (10, 10) -> 271 (b'\n\n') had 7098 occurrences
merge 17/256: (97, 114) -> 272 (

╭─────────────────────────────────────────────────── Tokenizer ───────────────────────────────────────────────────╮
│ Saved tokenizer to: artifacts/tokenizers/tiny_bpe.model (size=512)                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── GPT-BPE Config ─────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "block_size": 256,                                                                                            │
│   "vocab_size": 512,                                                                                            │
│   "n_layer": 6,                                                                                                 │
│   "n_head": 6,                                                                                                  │
│   "n_embd": 384,                                                                                                │
│   "dropout": 0.1,                                                                                               │
│   "bias": true,                                                                                                 │
│   "attention_backend": "sdpa",                                                                                  │
│   "use_moe": true,                                                                                              │
│   "num_experts": 3,                                                                                             │
│   "k": 1,                                                                                                       │
│   "router_hidden": 128,                                                                                         │
│   "blw": 0.02,                                                                                                  │
│   "shared_init_scale": 0.3,                                                                                     │
│   "entropy_penalty": 0.001,                                                                                     │
│   "router_temp": 1.0,                                                                                           │
│   "router_temp_start": 1.5,                                                                                     │
│   "router_temp_end": 0.5,                                                                                       │
│   "add_gumbel": true,                                                                                           │
│   "gumbel_off_epoch": 0,                                                                                        │
│   "routing_mode": "specialize",                                                                                 │
│   "rescue_lambda": 0.0,                                                                                         │
│   "usage_momentum": 0.99                                                                                        │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# FLASH MAMBA NANOGPT

In [ ]:
%env MOE_ARGS=--mode train_mamba --use_moe --attention_backend sdpa --epochs 1 --steps_per_epoch 50 --batch_size 64 --save_path artifacts/gpt_moe_sdpa.pt


In [ ]:
#!/usr/bin/env python3
# seq_moe_flash_mamba.py
# One-file, non-MoE and MoE variants for:
#   - NanoGPT (Flash/SDPA or vanilla attention)
#   - NanoMamba (SSM) with fallback SimpleMamba
# With shared+top-k MoE FFN, specialization aids, Rich panels, save/load, Colab-safe runner.

import os, math, json, argparse, random, time, urllib.request, shlex
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Any, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box

console = Console()

# ============================== Utils ==============================

def set_seed(seed=1337):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(m: nn.Module) -> Tuple[int,int]:
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

def kl_to_uniform(probs: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    E = probs.size(-1)
    m = probs.mean(dim=0)
    return torch.sum(m * (m.add(eps).log() - math.log(1.0 / E)))

def entropy_mean(probs: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    return -(probs.clamp_min(eps) * probs.clamp_min(eps).log()).sum(dim=-1).mean()

def params_panel(title: str, totals: Dict[str,int], extras: List[Tuple[str,str]] = None):
    t = Table(box=box.SIMPLE_HEAVY)
    t.add_column("Metric", style="bold"); t.add_column("Value", justify="right")
    for k in ["Total params","Trainable params"]:
        t.add_row(k, f"{totals[k]:,}")
    if "Attention params" in totals: t.add_row("Attention params", f"{totals['Attention params']:,}")
    if "SSM params" in totals:       t.add_row("SSM params", f"{totals['SSM params']:,}")
    if "Shared FFN" in totals:       t.add_row("Shared FFN", f"{totals['Shared FFN']:,}")
    if "Router" in totals:           t.add_row("Router", f"{totals['Router']:,}")
    if "Per expert" in totals:       t.add_row("Per expert", f"{totals['Per expert']:,}")
    if "k" in totals:                t.add_row("k", f"{totals['k']}")
    if "Active per forward" in totals: t.add_row("Active per forward", f"[bold]{totals['Active per forward']:,}[/]")
    if extras:
        for k,v in extras: t.add_row(k,v)
    console.print(Panel(t, title=title, border_style="yellow", box=box.ROUNDED, expand=True))

def routing_panel(stats: Dict[str, torch.Tensor], title: str):
    counts = stats["expert_selection_counts"].detach().cpu().to(torch.int64).tolist()
    mean_probs = stats["mean_routing_probs"].detach().cpu().tolist()
    t = Table(box=box.SIMPLE_HEAVY); t.add_column("Expert"); t.add_column("Batch Picks", justify="right"); t.add_column("Mean Prob.", justify="right")
    for i,(c,p) in enumerate(zip(counts,mean_probs)): t.add_row(f"E{i}", f"{c}", f"{p:.3f}")
    ent = entropy_mean(torch.tensor(mean_probs)[None,:]).item()
    console.print(Panel(t, title=f"{title} | entropy={ent:.3f}", border_style="green", box=box.ROUNDED, expand=True))

# ============================ Data (char LM) ============================

def _repeat_to_len(s: str, target_len: int) -> str:
    if not s: s=" \n"
    return (s * ((target_len // len(s))+1))[:target_len]

def load_tiny_shakespeare(data_dir="./data", target_len=100_000) -> Tuple[str,str]:
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, "tinyshakespeare_input.txt")
    if not os.path.exists(path):
        try:
            url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url, path)
        except Exception:
            sample=("From fairest creatures we desire increase,\n"
                    "That thereby beauty's rose might never die,\n"
                    "But as the riper should by time decease,\n"
                    "His tender heir might bear his memory:\n")
            with open(path,"w",encoding="utf-8") as f: f.write(_repeat_to_len(sample, target_len))
    with open(path,"r",encoding="utf-8") as f: text=f.read()
    split=int(0.9*len(text)); return text[:split], text[split:]

def build_charset(train: str, val: str):
    text=train+val; chars=sorted(list(set(text)))
    stoi={ch:i for i,ch in enumerate(chars)}; itos={i:ch for ch,i in stoi.items()}
    return stoi, itos

def encode(s: str, stoi: Dict[str,int]) -> List[int]: return [stoi[c] for c in s]

def get_batch_text(data: List[int], block_size: int, batch_size: int, device: str):
    if len(data) <= block_size + 1:
        raise RuntimeError(f"Text length {len(data)} too small for block_size {block_size}. Lower block size.")
    ix=torch.randint(len(data)-block_size-1,(batch_size,))
    x=torch.stack([torch.tensor(data[i:i+block_size]) for i in ix]).long()
    y=torch.stack([torch.tensor(data[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================== Router / MoE core ===============================

class TopKRouter(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=1, add_gumbel_noise=True, temperature=1.0):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden_dim), nn.ReLU(), nn.Linear(hidden_dim,num_experts))
        self.k=k; self.add_gumbel=add_gumbel_noise; self.temperature=float(temperature)
    @staticmethod
    def _gumbel(shape, device):
        u=torch.rand(shape,device=device).clamp_(1e-9,1-1e-9); return -torch.log(-torch.log(u))
    def forward(self, x):
        logits=self.net(x)
        if self.add_gumbel and self.training: logits=logits+self._gumbel(logits.shape,logits.device)
        probs=F.softmax(logits/self.temperature, dim=-1)
        if self.k>=probs.size(-1): return probs, probs
        topk_vals, topk_idx = torch.topk(probs, k=self.k, dim=-1)
        mask = torch.zeros_like(probs); mask.scatter_(dim=-1, index=topk_idx, src=torch.ones_like(topk_vals))
        sp = probs*mask; sp = sp/(sp.sum(dim=-1,keepdim=True)+1e-9)
        return sp, probs

# =============================== GPT (Flash/vanilla) ===============================

class LayerNorm(nn.Module):
    def __init__(self, n, bias): super().__init__(); self.weight=nn.Parameter(torch.ones(n)); self.bias=nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self, x): return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout, bias, backend="sdpa"):
        super().__init__(); assert n_embd % n_head == 0
        self.n_head=n_head; self.n_embd=n_embd; self.dropout=dropout; self.backend=backend
        self.c_attn=nn.Linear(n_embd,3*n_embd,bias=bias); self.c_proj=nn.Linear(n_embd,n_embd,bias=bias)
        self.attn_dropout=nn.Dropout(dropout); self.resid_dropout=nn.Dropout(dropout)
        if backend=="vanilla" or not hasattr(F,"scaled_dot_product_attention"):
            self.register_buffer("bias", torch.tril(torch.ones(block,block)).view(1,1,block,block))
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        if self.backend=="sdpa" and hasattr(F,"scaled_dot_product_attention"):
            y=F.scaled_dot_product_attention(q,k,v,attn_mask=None,dropout_p=self.dropout if self.training else 0.,is_causal=True)
        else:
            att=(q@k.transpose(-2,-1))*(1.0/math.sqrt(k.size(-1)))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0, float("-inf"))
            att=F.softmax(att,dim=-1); att=self.attn_dropout(att); y=att@v
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.c_proj(y))

class MoEFFN(nn.Module):
    def __init__(self, in_dim, expansion=4, num_experts=3, k=1, router_hidden=128, bias=True, dropout=0.1,
                 balance_loss_weight=0.02, shared_init_scale=0.3, entropy_bonus_weight=0.001,
                 add_gumbel_noise=True, temperature=1.0):
        super().__init__()
        hidden=expansion*in_dim
        self.shared_fc=nn.Linear(in_dim,hidden,bias=bias); self.shared_proj=nn.Linear(hidden,in_dim,bias=bias)
        self.expert_fc=nn.ModuleList([nn.Linear(in_dim,hidden,bias=bias) for _ in range(num_experts)])
        self.expert_proj=nn.ModuleList([nn.Linear(hidden,in_dim,bias=bias) for _ in range(num_experts)])
        self.router=TopKRouter(in_dim, router_hidden, num_experts, k, add_gumbel_noise, temperature)
        self.route_norm=nn.LayerNorm(in_dim); self.act=nn.GELU(); self.drop=nn.Dropout(dropout)
        self.shared_scale=nn.Parameter(torch.tensor(float(shared_init_scale)))
        self.blw=balance_loss_weight; self.entw=entropy_bonus_weight; self.k=k; self.num_experts=num_experts
    def forward(self, x):  # [B,T,C] or [B,T,E] generic last-dim routing
        B,T,C=x.shape; xf=x.view(B*T,C)
        shared=self.shared_proj(self.act(self.shared_fc(xf)))*self.shared_scale
        sp,dp=self.router(self.route_norm(xf))  # [BT,E],[BT,E]
        routed=0.0
        for e in range(self.num_experts):
            h=self.expert_proj[e](self.act(self.expert_fc[e](xf)))
            routed=routed + sp[:,e].unsqueeze(-1)*h
        y=(shared+routed).view(B,T,C); y=self.drop(y)
        ent=entropy_mean(dp); aux=self.blw*kl_to_uniform(dp) - self.entw*ent
        with torch.no_grad():
            counts=(sp>0).float().sum(0)
        stats={"expert_selection_counts":counts,"mean_routing_probs":dp.mean(0)}
        return y, aux, stats

class GPTBlock(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout, bias, backend="sdpa", moe: Optional[MoEFFN]=None):
        super().__init__(); self.ln1=LayerNorm(n_embd,bias); self.attn=CausalSelfAttention(n_embd,n_head,block,dropout,bias,backend)
        self.ln2=LayerNorm(n_embd,bias); self.moe=moe
        if moe is None:
            hidden=4*n_embd
            self.ffn=nn.Sequential(nn.Linear(n_embd,hidden,bias=bias), nn.GELU(), nn.Linear(hidden,n_embd,bias=bias), nn.Dropout(dropout))
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        if self.moe is None:
            x=x+self.ffn(self.ln2(x)); aux=torch.tensor(0.0,device=x.device)
            stats={"expert_selection_counts":torch.zeros(1,device=x.device),"mean_routing_probs":torch.zeros(1,device=x.device)}
        else:
            y,aux,stats=self.moe(self.ln2(x)); x=x+y
        return x, aux, stats

@dataclass
class GPTCfg:
    block_size:int=256; vocab_size:int=256; n_layer:int=6; n_head:int=6; n_embd:int=384
    dropout:float=0.1; bias:bool=True; attention_backend:str="sdpa"
    use_moe:bool=False; num_experts:int=3; k:int=1; router_hidden:int=128; blw:float=0.02
    shared_init_scale:float=0.3; entropy_bonus:float=0.001
    router_temp:float=1.0; router_temp_start:float=1.5; router_temp_end:float=0.5; add_gumbel:bool=True

class NanoGPT(nn.Module):
    def __init__(self, cfg:GPTCfg):
        super().__init__(); self.cfg=cfg
        self.wte=nn.Embedding(cfg.vocab_size,cfg.n_embd); self.wpe=nn.Embedding(cfg.block_size,cfg.n_embd)
        blocks=[]
        for _ in range(cfg.n_layer):
            moe=None
            if cfg.use_moe:
                moe=MoEFFN(cfg.n_embd, expansion=4, num_experts=cfg.num_experts, k=cfg.k, router_hidden=cfg.router_hidden,
                           bias=cfg.bias, dropout=cfg.dropout, balance_loss_weight=cfg.blw,
                           shared_init_scale=cfg.shared_init_scale, entropy_bonus_weight=cfg.entropy_bonus,
                           add_gumbel_noise=cfg.add_gumbel, temperature=cfg.router_temp)
            blocks.append(GPTBlock(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout, cfg.bias, cfg.attention_backend, moe))
        self.blocks=nn.ModuleList(blocks); self.ln_f=LayerNorm(cfg.n_embd,cfg.bias); self.lm_head=nn.Linear(cfg.n_embd,cfg.vocab_size,bias=False)
        self.lm_head.weight=self.wte.weight
        self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward(self, idx, targets=None):
        B,T=idx.shape; assert T<=self.cfg.block_size
        pos=torch.arange(0,T,device=idx.device).long(); x=self.wte(idx)+self.wpe(pos)[None,:,:]
        aux_total=torch.tensor(0.0,device=idx.device); last=None
        for blk in self.blocks:
            x,aux,stats=blk(x); aux_total=aux_total+aux; last=stats
        x=self.ln_f(x); logits=self.lm_head(x); loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1)); loss=loss+aux_total
        return logits, loss, last
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.cfg.block_size else idx[:,-self.cfg.block_size:]
            logits,_,_=self(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); idx_next=torch.multinomial(probs,1); idx=torch.cat((idx,idx_next),dim=1)
        return idx

def activated_params_gpt(model:NanoGPT)->Dict[str,int]:
    total,train=count_parameters(model)
    attn=0; shared=0; router=0; per_exp=0
    for blk in model.blocks:
        c=blk.attn
        attn+=c.c_attn.weight.numel(); attn+=c.c_proj.weight.numel()
        if c.c_attn.bias is not None: attn+=c.c_attn.bias.numel()
        if c.c_proj.bias is not None: attn+=c.c_proj.bias.numel()
        if blk.moe is not None:
            m=blk.moe
            shared+=m.shared_fc.weight.numel()+ (m.shared_fc.bias.numel() if m.shared_fc.bias is not None else 0)
            shared+=m.shared_proj.weight.numel()+ (m.shared_proj.bias.numel() if m.shared_proj.bias is not None else 0)
            ef,ep=m.expert_fc[0], m.expert_proj[0]
            per_exp+=ef.weight.numel()+(ef.bias.numel() if ef.bias is not None else 0)
            per_exp+=ep.weight.numel()+(ep.bias.numel() if ep.bias is not None else 0)
            for p in m.router.parameters(): router+=p.numel()
    k=model.cfg.k; active=attn+shared+router+k*per_exp
    return {"Total params":total,"Trainable params":train,"Attention params":attn,"Shared FFN":shared,"Router":router,"Per expert":per_exp,"k":k,"Active per forward":active}

# =============================== Mamba (SSM) ===============================

_HAS_MAMBA=False
try:
    from mamba_ssm import Mamba as MambaBlock  # pip install mamba-ssm
    _HAS_MAMBA=True
except Exception:
    _HAS_MAMBA=False

class SimpleMambaBlock(nn.Module):
    """Dependency-free SSM-ish block: depthwise conv + gated exponential decay scan."""
    is_mamba_block = True  # <— tag for duck-typing
    def __init__(self, d_model, d_state=16, d_conv=4, dropout=0.1, bias=True):
        super().__init__(); self.norm=LayerNorm(d_model,bias)
        self.proj_in=nn.Linear(d_model,2*d_model,bias=bias)
        self.dwconv=nn.Conv1d(2*d_model,2*d_model,kernel_size=d_conv,padding=d_conv-1,groups=2*d_model)
        self.act=nn.SiLU(); self.a=nn.Parameter(torch.zeros(d_model)); self.b=nn.Parameter(torch.zeros(d_model))
        self.proj_out=nn.Linear(d_model,d_model,bias=bias); self.drop=nn.Dropout(dropout)
    def forward(self,x): # [B,T,C]
        B,T,C=x.shape; h=self.norm(x); h=self.proj_in(h) # [B,T,2C]
        h=h.transpose(1,2); h=self.dwconv(h)[:,:,:T]; h=h.transpose(1,2)
        u,g=h.chunk(2,dim=-1); g=torch.sigmoid(g); u=self.act(u)
        a=torch.exp(-F.softplus(self.a)).view(1,1,C); b=self.b.view(1,1,C)
        y=[]; s=torch.zeros(B,C,device=x.device)
        for t in range(T):
            s=a.squeeze(0).squeeze(0)*s + b.squeeze(0).squeeze(0)*u[:,t,:]; y.append(s)
        y=torch.stack(y,dim=1); y=g*y; y=self.proj_out(y); y=self.drop(y)
        return x+y

class MambaLayer(nn.Module):
    """Wrap an SSM block (official or simple) then optional MoE FFN."""
    def __init__(self, d_model, dropout, bias, use_official):
        super().__init__(); self.ln=LayerNorm(d_model,bias)
        if use_official:
            self.core = MambaBlock(d_model=d_model, d_state=16, d_conv=4, expand=2)
            # Tag the instance (duck-typing) so param counter finds it
            setattr(self.core, "is_mamba_block", True)
        else:
            self.core = SimpleMambaBlock(d_model, d_state=16, d_conv=4, dropout=dropout, bias=bias)
        # Note: we do NOT tag MambaLayer itself; only the core owns parameters we want to attribute to "SSM"
    def forward(self,x): return self.core(x)

class NanoMamba(nn.Module):
    def __init__(self, vocab_size=256, block_size=256, n_layer=6, n_embd=384, dropout=0.1, bias=True,
                 use_moe=False, num_experts=3, k=1, router_hidden=128, blw=0.02,
                 shared_init_scale=0.3, entropy_bonus=0.001, router_temp=1.0, add_gumbel=True):
        super().__init__(); self.block_size=block_size; self.vocab_size=vocab_size; self.n_embd=n_embd
        self.wte=nn.Embedding(vocab_size,n_embd); self.wpe=nn.Embedding(block_size,n_embd)
        self.layers=nn.ModuleList()
        for _ in range(n_layer):
            self.layers.append(MambaLayer(n_embd, dropout, bias, _HAS_MAMBA))
            # add MoE FFN after each SSM layer if requested
            if use_moe:
                self.layers.append(nn.ModuleDict({
                    "ln": LayerNorm(n_embd,bias),
                    "moe": MoEFFN(n_embd, expansion=4, num_experts=num_experts, k=k, router_hidden=router_hidden,
                                  bias=bias, dropout=dropout, balance_loss_weight=blw,
                                  shared_init_scale=shared_init_scale, entropy_bonus_weight=entropy_bonus,
                                  add_gumbel_noise=add_gumbel, temperature=router_temp)
                }))
        self.ln_f=LayerNorm(n_embd,bias); self.head=nn.Linear(n_embd,vocab_size,bias=False)
        self.head.weight=self.wte.weight
        self.use_moe=use_moe
        self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward(self, idx, targets=None):
        B,T=idx.shape; assert T<=self.block_size
        pos=torch.arange(0,T,device=idx.device).long(); x=self.wte(idx)+self.wpe(pos)[None,:,:]
        aux_total=torch.tensor(0.0,device=idx.device); last=None
        it=iter(self.layers)
        while True:
            try:
                layer=next(it)
                x=layer(x)  # SSM
                if self.use_moe:
                    pack=next(it); x2,aux,stats=pack["moe"](pack["ln"](x))
                    x=x+x2; aux_total=aux_total+aux; last=stats
            except StopIteration:
                break
        x=self.ln_f(x); logits=self.head(x); loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1))
            loss=loss+aux_total
        return logits, loss, last
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.block_size else idx[:,-self.block_size:]
            logits,_,_=self(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1); idx=torch.cat((idx,nxt),dim=1)
        return idx

def activated_params_mamba(model) -> dict:
    total, trainable = count_parameters(model)

    ssm = 0
    shared = 0
    router = 0
    per_expert = 0
    k = 1

    for m in model.modules():
        # Count parameters of SSM cores by duck-typing tag
        if getattr(m, "is_mamba_block", False):
            # Only direct params of the core to avoid double counting submodules elsewhere
            ssm += sum(p.numel() for p in m.parameters(recurse=False))

        # Count MoE bits if present on this module (works for MoEFFN inside ModuleDict)
        has_moe = all(hasattr(m, nm) for nm in ["router", "expert_fc", "expert_proj"])
        if has_moe:
            if hasattr(m, "shared_fc"):
                shared += m.shared_fc.weight.numel()
                if m.shared_fc.bias is not None: shared += m.shared_fc.bias.numel()
            if hasattr(m, "shared_proj"):
                shared += m.shared_proj.weight.numel()
                if m.shared_proj.bias is not None: shared += m.shared_proj.bias.numel()

            ef = m.expert_fc[0]; ep = m.expert_proj[0]
            per_expert += ef.weight.numel() + (ef.bias.numel() if ef.bias is not None else 0)
            per_expert += ep.weight.numel() + (ep.bias.numel() if ep.bias is not None else 0)

            router += sum(p.numel() for p in m.router.parameters())
            k = getattr(m, "k", k)

    active_total = ssm + shared + router + k * per_expert
    return {
        "Total params": total,
        "Trainable params": trainable,
        "SSM params": ssm,
        "Shared FFN": shared,
        "Router": router,
        "Per expert": per_expert,
        "k": k,
        "Active per forward": active_total,
    }

# =========================== Training / Sampling ===========================

def train_lm(model, train_ids, val_ids, block_size, epochs, steps_per_epoch, batch_size, lr, device, title, temp_sched=None):
    model=model.to(device); opt=torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.95), weight_decay=0.1)
    for ep in range(1,epochs+1):
        if temp_sched is not None:
            temp=temp_sched(ep,epochs)
            for m in model.modules():
                if isinstance(m, MoEFFN): m.router.temperature=float(temp)
        t0=time.time(); model.train(); losses=[]; last=None
        for _ in range(steps_per_epoch):
            xb,yb=get_batch_text(train_ids, block_size, batch_size, device)
            _,loss,stats=model(xb,yb); opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            losses.append(loss.item()); last=stats
        dt=time.time()-t0
        with torch.no_grad():
            model.eval(); xb,yb=get_batch_text(val_ids, block_size, batch_size, device); _,vl,_=model(xb,yb)
        console.print(Panel(f"Epoch {ep}/{epochs}  train=[bold]{sum(losses)/len(losses):.3f}[/]  val=[bold magenta]{vl.item():.3f}[/]  time={dt:.1f}s",
                            title=title, border_style="cyan", box=box.ROUNDED, expand=True))
        if last is not None: routing_panel(last, f"{title} Routing (last batch)")

def sample_lm(model, stoi, itos, device, start="\n", tokens=200, title="Sample"):
    model.eval().to(device)
    x=torch.tensor([encode(start, stoi)], dtype=torch.long, device=device)
    y=model.generate(x, max_new_tokens=tokens, temperature=0.8, top_k=200)
    txt="".join(itos[i] for i in y[0].tolist())
    console.print(Panel(txt, title=title, border_style="magenta", box=box.ROUNDED, expand=True))

# ================================ Save / Load ================================

def save_ckpt(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload, path); console.print(Panel(f"Saved to [bold]{path}[/]", border_style="magenta"))

def load_ckpt(path):
    payload=torch.load(path, map_location="cpu"); console.print(Panel(f"Loaded [bold]{path}[/]", border_style="magenta")); return payload

# =================================== CLI ===================================

def main(argv=None):
    ap=argparse.ArgumentParser(description="NanoGPT (Flash/vanilla) & NanoMamba with/without MoE-FFN")
    ap.add_argument("--mode",required=True,choices=["train_gpt","infer_gpt","train_mamba","infer_mamba"])
    ap.add_argument("--device",default="auto",choices=["auto","cpu","cuda"])
    ap.add_argument("--epochs",type=int,default=2)
    ap.add_argument("--steps_per_epoch",type=int,default=200)
    ap.add_argument("--batch_size",type=int,default=64)
    ap.add_argument("--lr",type=float,default=3e-4)
    ap.add_argument("--block_size",type=int,default=256)
    ap.add_argument("--dropout",type=float,default=0.1)
    ap.add_argument("--n_layer",type=int,default=6)
    ap.add_argument("--n_head",type=int,default=6)
    ap.add_argument("--n_embd",type=int,default=384)
    ap.add_argument("--bias",action="store_true",default=True)
    ap.add_argument("--attention_backend",default="sdpa",choices=["sdpa","vanilla"])
    ap.add_argument("--use_moe",action="store_true",help="Turn on MoE FFN")
    ap.add_argument("--num_experts",type=int,default=3)
    ap.add_argument("--k",type=int,default=1)
    ap.add_argument("--router_hidden",type=int,default=128)
    ap.add_argument("--blw",type=float,default=0.02)
    ap.add_argument("--shared_init_scale",type=float,default=0.3)
    ap.add_argument("--entropy_bonus",type=float,default=0.001)
    ap.add_argument("--router_temp",type=float,default=1.0)
    ap.add_argument("--router_temp_start",type=float,default=1.5)
    ap.add_argument("--router_temp_end",type=float,default=0.5)
    ap.add_argument("--no_gumbel",action="store_true")
    ap.add_argument("--save_path",default="artifacts/seq.pt")
    ap.add_argument("--load_path",default="artifacts/seq.pt")
    ap.add_argument("--seed",type=int,default=1337)
    ap.add_argument("--sample_start",default="\n")
    ap.add_argument("--sample_tokens",type=int,default=200)
    args=ap.parse_args(argv)
    set_seed(args.seed); device=args.device if args.device!="auto" else ("cuda" if torch.cuda.is_available() else "cpu")

    # Data
    train_txt, val_txt = load_tiny_shakespeare("./data")
    stoi, itos = build_charset(train_txt, val_txt)
    train_ids = encode(train_txt, stoi); val_ids = encode(val_txt, stoi)
    if len(train_ids) <= args.block_size + 1:
        new_bs=max(16, min(args.block_size, len(train_ids)-2))
        console.print(Panel(f"Auto-adjusting block_size {args.block_size} -> {new_bs}", border_style="yellow", title="Safety"))
        args.block_size=new_bs
    vocab_size=max(stoi.values())+1

    # Router temperature schedule (epochs)
    def temp_sched(ep, E):
        alpha=(ep-1)/max(1,(E-1)); return args.router_temp_end + (args.router_temp_start-args.router_temp_end)*(1.0-alpha)

    if args.mode == "train_gpt":
        cfg=GPTCfg(block_size=args.block_size, vocab_size=vocab_size, n_layer=args.n_layer, n_head=args.n_head,
                   n_embd=args.n_embd, dropout=args.dropout, bias=args.bias, attention_backend=args.attention_backend,
                   use_moe=args.use_moe, num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden, blw=args.blw,
                   shared_init_scale=args.shared_init_scale, entropy_bonus=args.entropy_bonus,
                   router_temp=args.router_temp, router_temp_start=args.router_temp_start,
                   router_temp_end=args.router_temp_end, add_gumbel=not args.no_gumbel)
        console.print(Panel(json.dumps(asdict(cfg),indent=2), title="GPT Config", border_style="blue"))
        model=NanoGPT(cfg)
        train_lm(model, train_ids, val_ids, cfg.block_size, args.epochs, args.steps_per_epoch, args.batch_size, args.lr, device,
                 title=f"NanoGPT ({'MoE' if cfg.use_moe else 'no-MoE'}, {cfg.attention_backend})",
                 temp_sched=(temp_sched if cfg.use_moe else None))
        # params panel
        gp=activated_params_gpt(model) if cfg.use_moe else {"Total params":count_parameters(model)[0],"Trainable params":count_parameters(model)[1],"Attention params":sum(p.numel() for n,p in model.named_parameters() if 'c_attn' in n or 'c_proj' in n)}
        params_panel("GPT Parameters", gp)
        save_ckpt(args.save_path, {"kind":"gpt", "cfg":asdict(cfg), "state_dict":model.state_dict(), "stoi":stoi, "itos":itos})

    elif args.mode == "infer_gpt":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt"
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        sample_lm(model, payload["stoi"], payload["itos"], device, start=args.sample_start, tokens=args.sample_tokens,
                  title=f"NanoGPT Sample ({'MoE' if cfg.use_moe else 'no-MoE'}, {cfg.attention_backend})")

    elif args.mode == "train_mamba":
        model=NanoMamba(vocab_size=vocab_size, block_size=args.block_size, n_layer=args.n_layer, n_embd=args.n_embd,
                        dropout=args.dropout, bias=args.bias, use_moe=args.use_moe, num_experts=args.num_experts, k=args.k,
                        router_hidden=args.router_hidden, blw=args.blw, shared_init_scale=args.shared_init_scale,
                        entropy_bonus=args.entropy_bonus, router_temp=args.router_temp, add_gumbel=not args.no_gumbel)
        title=f"NanoMamba ({'MoE' if args.use_moe else 'no-MoE'}, {'official' if _HAS_MAMBA else 'simple'})"
        console.print(Panel(title, border_style="blue"))
        train_lm(model, train_ids, val_ids, args.block_size, args.epochs, args.steps_per_epoch, args.batch_size, args.lr, device,
                 title=title, temp_sched=(temp_sched if args.use_moe else None))
        mp=activated_params_mamba(model); params_panel("Mamba Parameters", mp)
        save_ckpt(args.save_path, {"kind":"mamba", "state_dict":model.state_dict(), "cfg":{"vocab_size":vocab_size,"block_size":args.block_size,"n_layer":args.n_layer,"n_embd":args.n_embd,"dropout":args.dropout,"bias":args.bias,"use_moe":args.use_moe,"num_experts":args.num_experts,"k":args.k,"router_hidden":args.router_hidden,"blw":args.blw,"shared_init_scale":args.shared_init_scale,"entropy_bonus":args.entropy_bonus,"router_temp":args.router_temp,"add_gumbel":not args.no_gumbel}, "stoi":stoi, "itos":itos})

    elif args.mode == "infer_mamba":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="mamba"
        cfg=payload["cfg"]; model=NanoMamba(**cfg); model.load_state_dict(payload["state_dict"])
        sample_lm(model, payload["stoi"], payload["itos"], device, start=args.sample_start, tokens=args.sample_tokens,
                  title=f"NanoMamba Sample ({'MoE' if cfg['use_moe'] else 'no-MoE'})")

if __name__ == "__main__":
    import sys, os
    argv=sys.argv[1:]; has_mode=any(a=="--mode" or a.startswith("--mode=") for a in argv)
    if not has_mode:
        env=os.environ.get("MOE_ARGS","").strip()
        argv=shlex.split(env) if env else ["--mode","train_gpt","--epochs","1","--steps_per_epoch","50","--batch_size","64","--use_moe","--attention_backend","sdpa","--save_path","artifacts/gpt_moe.pt"]
    main(argv)


# RL REWARD AND GRPO

In [ ]:
# set_env_reward_and_grpo.py
import os

# # ---- Reward model training (pairwise) ----
# os.environ["MOE_ARGS_RM"] = (
#     "--mode train_reward "
#     "--tok_kind basic --vocab_size 512 "
#     "--use_moe --routing_mode specialize "
#     "--blw 0.001 --entropy_penalty 0.015 "
#     "--router_temp_start 1.5 --router_temp_end 0.7 "
#     "--gumbel_off_epoch 4 --shared_init_scale 0.1 "
#     "--epochs 2 --steps_per_epoch 200 --batch_size 32 "
#     "--attention_backend sdpa "
#     "--rm_pairs data/preferences.jsonl "          # JSONL with {"prompt","chosen","rejected"}
#     "--save_path artifacts/rm.pt "
#     "--tok_prefix artifacts/tokenizers/tiny_bpe"
# )

# ---- GRPO policy training (requires a trained reward model) ----
os.environ["MOE_ARGS_GRPO"] = (
    "--mode train_grpo "
    "--tok_kind basic --vocab_size 512 "
    "--use_moe --routing_mode specialize "
    "--blw 0.001 --entropy_penalty 0.015 "
    "--router_temp_start 1.5 --router_temp_end 0.7 "
    "--gumbel_off_epoch 4 --shared_init_scale 0.1 "
    "--epochs 1 --steps_per_epoch 500 --batch_size 8 "
    "--attention_backend sdpa "
    "--grpo_prompts data/prompts.jsonl "          # JSONL with {"prompt": "..."}
    "--rm_ckpt artifacts/rm.pt "                  # trained reward model checkpoint
    "--beta_kl 0.02 --group_size 4 --gen_max_tokens 128 "
    "--save_path artifacts/policy_grpo.pt "
    "--tok_prefix artifacts/tokenizers/tiny_bpe"
)

print("MOE envs set: MOE_ARGS_RM, MOE_ARGS_GRPO")


In [ ]:
import os, json

# Ensure data directory exists
os.makedirs("data", exist_ok=True)

prefs = [
    {
        "prompt": "Explain quantum computing simply.",
        "chosen": "Quantum computing uses special quantum states to solve problems faster.",
        "rejected": "Quantum computing is about computers that are faster than normal."
    },
    {
        "prompt": "Write a haiku about the sea.",
        "chosen": "Ocean whispers soft,\nEndless waves embrace the moon,\nSilent tides return.",
        "rejected": "The sea is big and blue,\nWater splashes here and there,\nIt is very wet."
    },
    {
        "prompt": "Summarize Romeo and Juliet.",
        "chosen": "Two young lovers from rival families fall in love but meet a tragic fate.",
        "rejected": "Romeo and Juliet is a long play about two people who like each other."
    },
    {
        "prompt": "Give a motivational quote.",
        "chosen": "Success is built on small steps taken consistently every day.",
        "rejected": "Motivation is when you feel motivated and want to do things."
    },
    {
        "prompt": "Describe a cat in one sentence.",
        "chosen": "A cat is a graceful, independent animal that often shows affection in subtle ways.",
        "rejected": "A cat is an animal that lives in a house and meows."
    }
]

path = "data/preferences.jsonl"
with open(path, "w", encoding="utf-8") as f:
    for row in prefs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"✅ Created synthetic preference dataset at {path} with {len(prefs)} examples.")


In [ ]:
import os

env=os.environ.get("MOE_ARGS_RM","").strip() or os.environ.get("MOE_ARGS_GRPO","").strip()

In [ ]:
# Create sample preference data first
import os
import json

# Create directories
os.makedirs("data", exist_ok=True)
os.makedirs("artifacts/tokenizers", exist_ok=True)

# Create sample preference pairs for reward model training
sample_preferences = [
    {
        "prompt": "Once upon a time",
        "chosen": " there was a brave knight who saved the kingdom.",
        "rejected": " there was a person who did bad things."
    },
    {
        "prompt": "The weather today is",
        "chosen": " absolutely beautiful with clear blue skies.",
        "rejected": " terrible and makes me feel sad."
    },
    {
        "prompt": "I think that",
        "chosen": " helping others brings joy and meaning to life.",
        "rejected": " nothing really matters in the end."
    },
    {
        "prompt": "When I wake up",
        "chosen": " I feel grateful for a new day of possibilities.",
        "rejected": " I wish I could just stay in bed forever."
    },
    # Add more examples to have sufficient training data
    {
        "prompt": "Learning new things",
        "chosen": " is exciting and helps us grow as individuals.",
        "rejected": " is pointless because we'll forget everything anyway."
    }
]

# Write preference data
with open("data/preferences.jsonl", "w") as f:
    for pref in sample_preferences:
        f.write(json.dumps(pref) + "\n")

print("Created sample preference data at data/preferences.jsonl")

# Now run the reward model training with explicit arguments
import sys
sys.argv = [
    "script.py",  # script name
    "--mode", "train_reward",
    "--tok_kind", "basic",
    "--vocab_size", "512",
    "--epochs", "2",
    "--steps_per_epoch", "50",  # Reduced for faster training
    "--batch_size", "16",      # Reduced for memory
    "--rm_pairs", "data/preferences.jsonl",
    "--save_path", "artifacts/rm.pt",
    "--tok_prefix", "artifacts/tokenizers/tiny_bpe"
]

# Clear any environment variables that might override
if "MOE_ARGS_RM" in os.environ:
    del os.environ["MOE_ARGS_RM"]
if "MOE_ARGS_GRPO" in os.environ:
    del os.environ["MOE_ARGS_GRPO"]

print("Running reward model training...")
# Now your main() function should work

In [ ]:
#!/usr/bin/env python3
# gpt_moe_bpe_rlhf.py
# BPE NanoGPT + MoE with: Reward Model (pairwise BT loss) and GRPO trainer.

import os, math, json, argparse, random, time, urllib.request, shlex
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Any, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box

# -------------------------- Minimal BPE (same as before) --------------------------
import unicodedata, regex as re

def get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]): counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    newids=[]; i=0
    while i<len(ids):
        if ids[i]==pair[0] and i<len(ids)-1 and ids[i+1]==pair[1]: newids.append(idx); i+=2
        else: newids.append(ids[i]); i+=1
    return newids

def render_token(t: bytes) -> str:
    s = t.decode('utf-8', errors='replace')
    out=[]
    for ch in s:
        if unicodedata.category(ch)[0] != "C": out.append(ch)
        else: out.append(f"\\u{ord(ch):04x}")
    return "".join(out)

class Tokenizer:
    def __init__(self):
        self.merges={}; self.pattern=""; self.special_tokens={}; self.vocab=self._build_vocab()
    def _build_vocab(self):
        vocab={idx: bytes([idx]) for idx in range(256)}
        for (p0,p1),idx in self.merges.items(): vocab[idx]=vocab[p0]+vocab[p1]
        for s,idx in self.special_tokens.items(): vocab[idx]=s.encode("utf-8")
        return vocab
    def train(self, text, vocab_size, verbose=False): raise NotImplementedError
    def encode(self, text): raise NotImplementedError
    def decode(self, ids):
        text_bytes=b"".join(self.vocab[idx] for idx in ids)
        return text_bytes.decode("utf-8", errors="replace")
    def save(self, prefix):
        with open(prefix+".model","w") as f:
            f.write("minbpe v1\n"); f.write(f"{self.pattern}\n")
            f.write(f"{len(self.special_tokens)}\n")
            for s,idx in self.special_tokens.items(): f.write(f"{s} {idx}\n")
            for (i,j) in self.merges: f.write(f"{i} {j}\n")
        inv={idx: pair for pair,idx in self.merges.items()}
        with open(prefix+".vocab","w",encoding="utf-8") as f:
            for idx,tok in self.vocab.items():
                s=render_token(tok)
                if idx in inv:
                    i,j=inv[idx]; s0=render_token(self.vocab[i]); s1=render_token(self.vocab[j])
                    f.write(f"[{s0}][{s1}] -> [{s}] {idx}\n")
                else: f.write(f"[{s}] {idx}\n")
    def load(self, model_file):
        merges={}; special={}
        with open(model_file,"r",encoding="utf-8") as f:
            assert f.readline().strip()=="minbpe v1"
            self.pattern=f.readline().strip()
            n=int(f.readline().strip())
            for _ in range(n):
                s,i=f.readline().strip().split(); special[s]=int(i)
            idx=256
            for line in f:
                a,b=map(int,line.split()); merges[(a,b)]=idx; idx+=1
        self.merges=merges; self.special_tokens=special; self.vocab=self._build_vocab()

class BasicTokenizer(Tokenizer):
    def train(self, text, vocab_size, verbose=False):
        vocab_size=max(256,vocab_size); nm=vocab_size-256
        ids=list(text.encode("utf-8")); merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(nm):
            stats=get_stats(ids); pair=max(stats, key=stats.get); idx=256+i
            ids=merge(ids,pair,idx); merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
        self.merges=merges; self.vocab=vocab
    def encode(self, text):
        ids=list(text.encode("utf-8"))
        while len(ids)>=2:
            stats=get_stats(ids); pair=min(stats, key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=merge(ids,pair,self.merges[pair])
        return ids

GPT4_SPLIT_PATTERN=r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexTokenizer(Tokenizer):
    def __init__(self, pattern=None):
        super().__init__(); self.pattern=GPT4_SPLIT_PATTERN if pattern is None else pattern
        self.re=re.compile(self.pattern); self.special_tokens={}; self.inverse={}
    def train(self, text, vocab_size, verbose=False):
        vocab_size=max(256,vocab_size); nm=vocab_size-256
        chunks=self.re.findall(text); ids=[list(c.encode("utf-8")) for c in chunks]
        merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(nm):
            stats={}
            for c in ids: get_stats(c,stats)
            pair=max(stats, key=stats.get); idx=256+i
            ids=[merge(c,pair,idx) for c in ids]; merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
        self.merges=merges; self.vocab=vocab
    def register_special_tokens(self,d): self.special_tokens=d; self.inverse={v:k for k,v in d.items()}
    def _encode_chunk(self, b):
        ids=list(b)
        while len(ids)>=2:
            stats=get_stats(ids); pair=min(stats, key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=merge(ids,pair,self.merges[pair])
        return ids
    def encode_ordinary(self, text):
        chunks=self.re.findall(text); ids=[]
        for c in chunks: ids.extend(self._encode_chunk(c.encode("utf-8")))
        return ids
    def encode(self, text, allowed_special="none_raise"):
        if allowed_special=="all": special=self.special_tokens
        elif allowed_special in ("none","none_raise"): special={}
        elif isinstance(allowed_special,set): special={k:v for k,v in self.special_tokens.items() if k in allowed_special}
        else: raise ValueError("bad allowed_special")
        if not special: return self.encode_ordinary(text)
        pat="("+"|".join(re.escape(k) for k in special)+")"
        parts=re.split(pat,text); ids=[]
        for part in parts:
            if part in special: ids.append(special[part])
            else: ids.extend(self.encode_ordinary(part))
        return ids
    def decode(self, ids):
        out=[]
        for i in ids:
            if i in self.vocab: out.append(self.vocab[i])
            elif i in self.inverse: out.append(self.inverse[i].encode("utf-8"))
            else: raise ValueError("bad id")
        return b"".join(out).decode("utf-8","replace")

# -------------------------- Console & utils --------------------------
console=Console()

def set_seed(seed=1337):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def count_parameters(m: nn.Module) -> Tuple[int,int]:
    total=sum(p.numel() for p in m.parameters())
    train=sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total,train

def kl_to_uniform(probs: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    E=probs.size(-1); m=probs.mean(dim=0)
    return torch.sum(m*(m.add(eps).log()-math.log(1.0/E)))

def entropy_mean(probs: torch.Tensor, eps: float = 1e-9) -> torch.Tensor:
    return -(probs.clamp_min(eps)*probs.clamp_min(eps).log()).sum(dim=-1).mean()

def params_panel(title: str, totals: Dict[str,int], extras: List[Tuple[str,str]] = None):
    t=Table(box=box.SIMPLE_HEAVY); t.add_column("Metric",style="bold"); t.add_column("Value",justify="right")
    for k in ["Total params","Trainable params"]: t.add_row(k,f"{totals[k]:,}")
    for k in ["Attention params","Shared FFN","Router","Per expert","k","Active per forward"]:
        if k in totals: v=totals[k]; t.add_row(k, f"{v:,}" if isinstance(v,int) else f"{v}")
    if extras:
        for k,v in extras: t.add_row(k,v)
    console.print(Panel(t,title=title,border_style="yellow",box=box.ROUNDED,expand=True))

def routing_panel(stats: Dict[str, torch.Tensor], title: str):
    counts=stats["expert_selection_counts"].detach().cpu().to(torch.int64).tolist()
    mean_probs=stats["mean_routing_probs"].detach().cpu().tolist()
    t=Table(box=box.SIMPLE_HEAVY); t.add_column("Expert"); t.add_column("Batch Picks",justify="right"); t.add_column("Mean Prob.",justify="right")
    for i,(c,p) in enumerate(zip(counts,mean_probs)): t.add_row(f"E{i}", f"{c}", f"{p:.3f}")
    ent=entropy_mean(torch.tensor(mean_probs)[None,:]).item()
    console.print(Panel(t,title=f"{title} | entropy={ent:.3f}",border_style="green",box=box.ROUNDED,expand=True))

# -------------------------- Data helpers --------------------------
def _repeat_to_len(s: str, target_len: int) -> str:
    if not s: s=" \n"
    return (s*((target_len//len(s))+1))[:target_len]

def load_tiny_shakespeare(data_dir="./data", target_len=100_000) -> Tuple[str,str]:
    os.makedirs(data_dir, exist_ok=True); path=os.path.join(data_dir,"tinyshakespeare_input.txt")
    if not os.path.exists(path):
        try:
            url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url,path)
        except Exception:
            sample=("From fairest creatures we desire increase,\n"
                    "That thereby beauty's rose might never die,\n"
                    "But as the riper should by time decease,\n"
                    "His tender heir might bear his memory:\n")
            with open(path,"w",encoding="utf-8") as f: f.write(_repeat_to_len(sample,target_len))
    with open(path,"r",encoding="utf-8") as f: text=f.read()
    split=int(0.9*len(text)); return text[:split], text[split:]

def train_or_load_tokenizer(kind:str, vocab_size:int, text:str, prefix:str, verbose=True):
    os.makedirs(os.path.dirname(prefix),exist_ok=True) if os.path.dirname(prefix) else None
    model_path=prefix+".model"
    if os.path.exists(model_path):
        tok=_new_tok(kind); tok.load(model_path)
        vs=max(tok.vocab.keys())+1
        if verbose: console.print(Panel(f"Loaded tokenizer: {model_path} (size={vs})",border_style="magenta",title="Tokenizer"))
        return tok, model_path, vs
    tok=_new_tok(kind); vocab_size=max(256,vocab_size)
    if verbose: console.print(Panel(f"Training {kind} BPE (vocab={vocab_size})",border_style="blue",title="Tokenizer"))
    tok.train(text,vocab_size,verbose=False); tok.save(prefix); vs=max(tok.vocab.keys())+1
    if verbose: console.print(Panel(f"Saved tokenizer -> {prefix}.model (size={vs})",border_style="green",title="Tokenizer"))
    return tok, model_path, vs

def _new_tok(kind:str):
    kind=kind.lower()
    if kind=="basic": return BasicTokenizer()
    if kind=="regex": return RegexTokenizer()
    raise ValueError("unknown tokenizer")

def encode_text(tok, s:str)->List[int]: return tok.encode(s)
def decode_ids(tok, ids:List[int])->str: return tok.decode(ids)

def get_batch_tokens(ids: List[int], block_size: int, batch_size: int, device: str):
    if len(ids)<=block_size+1: raise RuntimeError("text too small for block_size")
    ix=torch.randint(len(ids)-block_size-1,(batch_size,))
    x=torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y=torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# -------------------------- GPT + MoE --------------------------
class LayerNorm(nn.Module):
    def __init__(self,n,bias):
        super().__init__(); self.weight=nn.Parameter(torch.ones(n)); self.bias=nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self,x): return F.layer_norm(x,self.weight.shape,self.weight,self.bias,1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self,n_embd,n_head,block,dropout,bias,backend="sdpa"):
        super().__init__(); assert n_embd%n_head==0
        self.n_head=n_head; self.n_embd=n_embd; self.dropout=dropout; self.backend=backend
        self.c_attn=nn.Linear(n_embd,3*n_embd,bias=bias); self.c_proj=nn.Linear(n_embd,n_embd,bias=bias)
        self.attn_dropout=nn.Dropout(dropout); self.resid_dropout=nn.Dropout(dropout)
        if backend=="vanilla" or not hasattr(F,"scaled_dot_product_attention"):
            self.register_buffer("bias", torch.tril(torch.ones(block,block)).view(1,1,block,block))
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        if self.backend=="sdpa" and hasattr(F,"scaled_dot_product_attention"):
            y=F.scaled_dot_product_attention(q,k,v,None,self.dropout if self.training else 0.,True)
        else:
            att=(q@k.transpose(-2,-1))*(1.0/math.sqrt(k.size(-1)))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0,float("-inf"))
            att=F.softmax(att,dim=-1); att=self.attn_dropout(att); y=att@v
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.c_proj(y))

class TopKRouter(nn.Module):
    def __init__(self,in_dim,hidden_dim,num_experts,k=1,add_gumbel_noise=True,temperature=1.0):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,num_experts))
        self.k=k; self.add_gumbel=add_gumbel_noise; self.temperature=float(temperature)
        self.register_buffer("logits_bias", torch.zeros(num_experts))
    @staticmethod
    def _gumbel(shape, device):
        u=torch.rand(shape,device=device).clamp_(1e-9,1-1e-9); return -torch.log(-torch.log(u))
    def forward(self,x):
        logits=self.net(x)
        if self.add_gumbel and self.training: logits=logits+self._gumbel(logits.shape,logits.device)
        logits=logits+self.logits_bias
        probs=F.softmax(logits/self.temperature,dim=-1)
        if self.k>=probs.size(-1): return probs,probs
        tk,idx=torch.topk(probs,k=self.k,dim=-1)
        mask=torch.zeros_like(probs); mask.scatter_(dim=-1,index=idx,src=torch.ones_like(tk))
        sp=probs*mask; sp=sp/(sp.sum(dim=-1,keepdim=True)+1e-9)
        return sp,probs

class MoEFFN(nn.Module):
    def __init__(self,in_dim,expansion=4,num_experts=3,k=1,router_hidden=128,bias=True,dropout=0.1,
                 balance_loss_weight=0.02, shared_init_scale=0.3, entropy_penalty_weight=0.001,
                 add_gumbel_noise=True, temperature=1.0,
                 routing_mode="specialize", rescue_lambda=0.0, usage_momentum=0.99):
        super().__init__()
        hidden=expansion*in_dim
        self.shared_fc=nn.Linear(in_dim,hidden,bias=bias); self.shared_proj=nn.Linear(hidden,in_dim,bias=bias)
        self.expert_fc=nn.ModuleList([nn.Linear(in_dim,hidden,bias=bias) for _ in range(num_experts)])
        self.expert_proj=nn.ModuleList([nn.Linear(hidden,in_dim,bias=bias) for _ in range(num_experts)])
        self.router=TopKRouter(in_dim,router_hidden,num_experts,k,add_gumbel_noise,temperature)
        self.route_norm=nn.LayerNorm(in_dim); self.act=nn.GELU(); self.drop=nn.Dropout(dropout)
        self.shared_scale=nn.Parameter(torch.tensor(float(shared_init_scale)))
        self.blw=balance_loss_weight; self.entw=entropy_penalty_weight
        self.k=k; self.num_experts=num_experts
        self.routing_mode=routing_mode; self.rescue_lambda=float(rescue_lambda); self.usage_momentum=float(usage_momentum)
        self.register_buffer("ema_usage", torch.zeros(num_experts))
    def forward(self,x):
        B,T,C=x.shape; xf=x.view(B*T,C)
        shared=self.shared_proj(self.act(self.shared_fc(xf)))*self.shared_scale
        sp,dp=self.router(self.route_norm(xf))
        routed=0.0
        for e in range(self.num_experts):
            h=self.expert_proj[e](self.act(self.expert_fc[e](xf)))
            routed=routed+sp[:,e].unsqueeze(-1)*h
        y=(shared+routed).view(B,T,C); y=self.drop(y)
        ent=entropy_mean(dp); aux=self.blw*kl_to_uniform(dp) - self.entw*ent
        with torch.no_grad():
            counts=(sp>0).float().sum(0); mean_dp=dp.mean(0)
            m=self.usage_momentum; self.ema_usage.mul_(m).add_((1.0-m)*mean_dp)
            if self.rescue_lambda>0.0 and self.training:
                target=torch.full_like(self.ema_usage, 1.0/self.num_experts)
                usage=self.ema_usage/(self.ema_usage.sum()+1e-9)
                bias=self.rescue_lambda*(target-usage); bias=bias-bias.mean()
                self.router.logits_bias.copy_(bias)
        return y,aux,{"expert_selection_counts":counts,"mean_routing_probs":dp.mean(0)}

class GPTBlock(nn.Module):
    def __init__(self,n_embd,n_head,block,dropout,bias,backend="sdpa",moe: Optional[MoEFFN]=None):
        super().__init__(); self.ln1=LayerNorm(n_embd,bias); self.attn=CausalSelfAttention(n_embd,n_head,block,dropout,bias,backend)
        self.ln2=LayerNorm(n_embd,bias); self.moe=moe
        if moe is None:
            hidden=4*n_embd
            self.ffn=nn.Sequential(nn.Linear(n_embd,hidden,bias=bias),nn.GELU(),nn.Linear(hidden,n_embd,bias=bias),nn.Dropout(dropout))
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        if self.moe is None:
            x=x+self.ffn(self.ln2(x)); aux=torch.tensor(0.0,device=x.device)
            stats={"expert_selection_counts":torch.zeros(1,device=x.device),"mean_routing_probs":torch.zeros(1,device=x.device)}
        else:
            y,aux,stats=self.moe(self.ln2(x)); x=x+y
        return x,aux,stats

@dataclass
class GPTCfg:
    block_size:int=256; vocab_size:int=256; n_layer:int=6; n_head:int=6; n_embd:int=384
    dropout:float=0.1; bias:bool=True; attention_backend:str="sdpa"
    use_moe:bool=False; num_experts:int=3; k:int=1; router_hidden:int=128; blw:float=0.02
    shared_init_scale:float=0.3
    entropy_penalty:float=0.001
    router_temp:float=1.0; router_temp_start:float=1.5; router_temp_end:float=0.5
    add_gumbel:bool=True; gumbel_off_epoch:int=0
    routing_mode:str="specialize"; rescue_lambda:float=0.0; usage_momentum:float=0.99

class NanoGPT(nn.Module):
    def __init__(self,cfg:GPTCfg):
        super().__init__(); self.cfg=cfg
        self.wte=nn.Embedding(cfg.vocab_size,cfg.n_embd); self.wpe=nn.Embedding(cfg.block_size,cfg.n_embd)
        blocks=[]
        for _ in range(cfg.n_layer):
            moe=None
            if cfg.use_moe:
                moe=MoEFFN(cfg.n_embd,4,cfg.num_experts,cfg.k,cfg.router_hidden,cfg.bias,cfg.dropout,
                           cfg.blw,cfg.shared_init_scale,cfg.entropy_penalty,cfg.add_gumbel,cfg.router_temp,
                           cfg.routing_mode,cfg.rescue_lambda,cfg.usage_momentum)
            blocks.append(GPTBlock(cfg.n_embd,cfg.n_head,cfg.block_size,cfg.dropout,cfg.bias,cfg.attention_backend,moe))
        self.blocks=nn.ModuleList(blocks); self.ln_f=LayerNorm(cfg.n_embd,cfg.bias); self.lm_head=nn.Linear(cfg.n_embd,cfg.vocab_size,bias=False)
        self.lm_head.weight=self.wte.weight; self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward(self, idx, targets=None):
        B,T=idx.shape; assert T<=self.cfg.block_size
        pos=torch.arange(0,T,device=idx.device).long(); x=self.wte(idx)+self.wpe(pos)[None,:,:]
        aux_total=torch.tensor(0.0,device=idx.device); last=None
        for blk in self.blocks:
            x,aux,stats=blk(x); aux_total=aux_total+aux; last=stats
        x=self.ln_f(x); logits=self.lm_head(x); loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1)); loss=loss+aux_total
        return logits, loss, last
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.cfg.block_size else idx[:,-self.cfg.block_size:]
            logits,_,_=self(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1); idx=torch.cat((idx,nxt),dim=1)
        return idx

# -------------------------- Reward Model --------------------------
class RewardHead(nn.Module):
    """Simple scalar reward: mean-pool hidden states + MLP -> scalar."""
    def __init__(self, d_model: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Linear(d_model, 1),
        )
    def forward(self, hidden: torch.Tensor) -> torch.Tensor:
        # hidden: [B,T,C]
        pooled = hidden.mean(dim=1)  # [B,C]
        return self.mlp(pooled).squeeze(-1)  # [B]

class RewardModel(nn.Module):
    """Wraps NanoGPT's encoder to produce a scalar reward."""
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.backbone = NanoGPT(cfg)
        self.reward_head = RewardHead(cfg.n_embd)
    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        # We forward backbone to pre-final hidden states
        B,T = idx.shape
        with torch.no_grad():
            # run blocks to get last hidden (no LM loss)
            pos=torch.arange(0,T,device=idx.device).long()
            x=self.backbone.wte(idx)+self.backbone.wpe(pos)[None,:,:]
            for blk in self.backbone.blocks:
                x,_,_ = blk(x)
            x = self.backbone.ln_f(x)
        # reward head is trained
        r = self.reward_head(x)
        return r

def bt_pairwise_loss(r_chosen: torch.Tensor, r_rejected: torch.Tensor) -> torch.Tensor:
    # Bradley–Terry: maximize log σ(r_c - r_r)
    return -F.logsigmoid(r_chosen - r_rejected).mean()

# -------------------------- RL: GRPO trainer --------------------------
@torch.no_grad()
def compute_logprobs(model: NanoGPT, input_ids: torch.Tensor, target_ids: torch.Tensor) -> torch.Tensor:
    # returns token logprobs per sequence [B]
    logits, _, _ = model(input_ids)
    # shift to next-token prediction
    logprobs = F.log_softmax(logits[:,:-1,:], dim=-1)  # [B,T-1,V]
    tgt = target_ids[:,1:]  # [B,T-1]
    lp = logprobs.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)  # [B,T-1]
    # mask: we assume all positions valid; otherwise pass an attention mask
    return lp.sum(dim=-1)  # sequence logprob

def kl_divergence_to_ref(policy_lp: torch.Tensor, ref_lp: torch.Tensor) -> torch.Tensor:
    # Using reverse KL approx on trajectories: KL ≈ E[-log π_ref + log π_pol]
    return (policy_lp - ref_lp).mean()

# -------------------------- Panels --------------------------
def activated_params_gpt(model:NanoGPT)->Dict[str,int]:
    total,train=count_parameters(model); attn=0; shared=0; router=0; per_exp=0
    for blk in model.blocks:
        c=blk.attn
        attn+=c.c_attn.weight.numel()+c.c_proj.weight.numel()
        if c.c_attn.bias is not None: attn+=c.c_attn.bias.numel()
        if c.c_proj.bias is not None: attn+=c.c_proj.bias.numel()
        if blk.moe is not None:
            m=blk.moe
            shared+=m.shared_fc.weight.numel()+(m.shared_fc.bias.numel() if m.shared_fc.bias is not None else 0)
            shared+=m.shared_proj.weight.numel()+(m.shared_proj.bias.numel() if m.shared_proj.bias is not None else 0)
            ef,ep=m.expert_fc[0],m.expert_proj[0]
            per_exp+=ef.weight.numel()+(ef.bias.numel() if ef.bias is not None else 0)
            per_exp+=ep.weight.numel()+(ep.bias.numel() if ep.bias is not None else 0)
            for p in m.router.parameters(): router+=p.numel()
    k=model.cfg.k; active=attn+shared+router+k*per_exp
    return {"Total params":total,"Trainable params":train,"Attention params":attn,"Shared FFN":shared,"Router":router,"Per expert":per_exp,"k":k,"Active per forward":active}

# -------------------------- IO helpers --------------------------
def save_ckpt(path, payload):
    os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload, path); console.print(Panel(f"Saved: {path}", border_style="magenta"))

def load_ckpt(path):
    payload=torch.load(path, map_location="cpu"); console.print(Panel(f"Loaded: {path}", border_style="magenta")); return payload

# -------------------------- Trainers --------------------------
def train_supervised(model, train_ids, val_ids, block_size, epochs, steps_per_epoch, batch_size, lr, device, title,
                     temp_sched=None, gumbel_off_epoch=0):
    model=model.to(device); opt=torch.optim.AdamW(model.parameters(),lr=lr,betas=(0.9,0.95),weight_decay=0.1)
    for ep in range(1,epochs+1):
        if temp_sched is not None:
            temp=temp_sched(ep,epochs)
            for m in model.modules():
                if isinstance(m, MoEFFN): m.router.temperature=float(temp)
        for m in model.modules():
            if isinstance(m, MoEFFN): m.router.add_gumbel=(gumbel_off_epoch<=0) or (ep<=gumbel_off_epoch)
        t0=time.time(); model.train(); losses=[]; last=None
        for _ in range(steps_per_epoch):
            xb,yb=get_batch_tokens(train_ids, block_size, batch_size, device)
            _,loss,stats=model(xb,yb)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            losses.append(loss.item()); last=stats
        dt=time.time()-t0
        with torch.no_grad():
            model.eval(); xb,yb=get_batch_tokens(val_ids, block_size, batch_size, device); _,vl,_=model(xb,yb)
        console.print(Panel(f"Epoch {ep}/{epochs} train={sum(losses)/len(losses):.3f} val={vl.item():.3f} time={dt:.1f}s",
                            title=title,border_style="cyan",box=box.ROUNDED,expand=True))
        if last is not None: routing_panel(last, f"{title} Routing (last batch)")

# ---------- Reward model training on preference pairs ----------
def load_pairs(path: str) -> List[dict]:
    rows=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if not line: continue
            rows.append(json.loads(line))
    return rows

def rm_batchify(pairs, tok, block_size, batch_size, device):
    # Return (ids_chosen, ids_rejected) batches (tokenized & padded/truncated)
    def tok_seq(text):
        ids=encode_text(tok,text)[:block_size]
        if len(ids)<block_size: ids = ids + [0]*(block_size-len(ids))
        return torch.tensor(ids).long()
    batch=random.sample(pairs, k=min(batch_size,len(pairs)))
    chosen=torch.stack([tok_seq(p["prompt"]+p["chosen"]) for p in batch]).to(device)
    rejected=torch.stack([tok_seq(p["prompt"]+p["rejected"]) for p in batch]).to(device)
    return chosen, rejected

def train_reward_model(rm: RewardModel, tok, pairs_path: str, block_size: int, epochs:int, steps:int, batch:int, lr:float, device:str):
    pairs=load_pairs(pairs_path)
    rm=rm.to(device)
    # Only train reward head (and optionally final layer norm if you want)
    for p in rm.backbone.parameters(): p.requires_grad=False
    opt=torch.optim.AdamW(rm.reward_head.parameters(), lr=lr)
    for ep in range(1,epochs+1):
        t0=time.time(); rm.train(); losses=[]
        for _ in range(steps):
            ch, rj = rm_batchify(pairs,tok,block_size,batch,device)
            r_c = rm(ch); r_r = rm(rj)
            loss = bt_pairwise_loss(r_c, r_r)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        dt=time.time()-t0
        console.print(Panel(f"[RM] Epoch {ep}/{epochs} loss={sum(losses)/len(losses):.4f} time={dt:.1f}s",
                            border_style="cyan",box=box.ROUNDED,expand=True))

# ---------- GRPO policy training ----------
def sample_group(model: NanoGPT, tok, prompts: List[str], max_new_tokens:int, temperature:float, top_k:int, device:str):
    # returns: input_ids, full_ids, stop-at length per sample
    model.eval().to(device)
    B=len(prompts); inputs=[]
    for p in prompts:
        ids=encode_text(tok,p)
        if len(ids)==0: ids=[0]
        inputs.append(torch.tensor(ids).long())
    max_in=max(len(x) for x in inputs)
    input_ids=torch.stack([torch.cat([x, torch.zeros(max_in-len(x),dtype=torch.long)]) for x in inputs]).to(device)
    # generate continuations
    with torch.no_grad():
        out=model.generate(input_ids.clone(), max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return input_ids, out

def read_jsonl_prompts(path: str)->List[str]:
    items=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            obj=json.loads(line); items.append(obj["prompt"])
    return items

def train_grpo(
    policy: NanoGPT,
    ref_policy: NanoGPT,
    reward_model: RewardModel,
    tok,
    prompts_path: str,
    block_size: int,
    epochs: int,
    steps_per_epoch: int,
    batch_prompts: int,
    group_size: int,
    gen_max_tokens: int,
    beta_kl: float,
    lr: float,
    device: str,
    temperature: float = 0.8,
    top_k: Optional[int] = 200,
):
    prompts_all=read_jsonl_prompts(prompts_path)
    opt=torch.optim.AdamW(policy.parameters(), lr=lr, betas=(0.9,0.95), weight_decay=0.1)
    reward_model.eval().to(device); ref_policy.eval().to(device); policy.train().to(device)

    for ep in range(1,epochs+1):
        t0=time.time(); losses=[]
        for step in range(steps_per_epoch):
            # 1) pick a mini-batch of prompts
            batch_prompts_list=random.sample(prompts_all, k=min(batch_prompts,len(prompts_all)))

            # 2) for each prompt, sample group_size responses
            # We’ll do it by replication
            prompts_rep=[]
            for p in batch_prompts_list: prompts_rep.extend([p]*group_size)

            in_ids, full_ids = sample_group(policy, tok, prompts_rep, gen_max_tokens, temperature, top_k, device)
            # align to block_size (truncate)
            full_ids = full_ids[:,:block_size]
            in_ids = in_ids[:,:block_size]
            # teacher-forcing logprobs on generated sequences
            pol_lp = compute_logprobs(policy, full_ids, full_ids).detach()  # [B*G]
            ref_lp = compute_logprobs(ref_policy, full_ids, full_ids).detach()

            # 3) compute rewards from RM
            with torch.no_grad():
                rewards = reward_model(full_ids).detach()  # [B*G]

            # 4) group-relative baseline: advantage = r - mean(r in same group)
            rewards = rewards.view(len(batch_prompts_list), group_size)
            pol_lp = pol_lp.view(len(batch_prompts_list), group_size)
            ref_lp = ref_lp.view(len(batch_prompts_list), group_size)

            group_mean = rewards.mean(dim=1, keepdim=True)
            adv = rewards - group_mean  # [B, G]

            # 5) policy gradient surrogate: - E[ adv * logpi ]
            # we need fresh logprobs with gradient
            pol_lp_grad = compute_logprobs(policy, full_ids, full_ids).view_as(adv)  # re-compute with grad
            loss_pg = -(adv.detach() * pol_lp_grad).mean()

            # 6) KL penalty to reference
            ref_lp_grad = ref_lp.detach()  # no grad
            pol_lp_again = pol_lp_grad.detach()  # only for measuring KL
            kl = (pol_lp_again - ref_lp_grad).mean()
            loss_kl = beta_kl * kl

            loss = loss_pg + loss_kl

            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(policy.parameters(),1.0); opt.step()
            losses.append(loss.item())

        dt=time.time()-t0
        console.print(Panel(f"[GRPO] Epoch {ep}/{epochs} loss={sum(losses)/len(losses):.4f} time={dt:.1f}s",
                            border_style="cyan",box=box.ROUNDED,expand=True))


def create_sample_data_if_missing():
    """Create sample training data files if they don't exist."""

    # Create directories
    os.makedirs("data", exist_ok=True)
    os.makedirs("artifacts", exist_ok=True)
    os.makedirs("artifacts/tokenizers", exist_ok=True)

    # Create sample preference pairs if missing
    prefs_path = "data/preferences.jsonl"
    if not os.path.exists(prefs_path):
        console.print(Panel("Creating sample preference data...", border_style="blue", title="Data Setup"))

        sample_preferences = [
            {"prompt": "Once upon a time", "chosen": " there was a brave knight who saved the kingdom.", "rejected": " there was a person who did bad things."},
            {"prompt": "The weather today is", "chosen": " absolutely beautiful with clear blue skies.", "rejected": " terrible and makes me feel sad."},
            {"prompt": "I think that", "chosen": " helping others brings joy and meaning to life.", "rejected": " nothing really matters in the end."},
            {"prompt": "When I wake up", "chosen": " I feel grateful for a new day of possibilities.", "rejected": " I wish I could just stay in bed forever."},
            {"prompt": "Learning new things", "chosen": " is exciting and helps us grow as individuals.", "rejected": " is pointless because we'll forget everything anyway."},
            {"prompt": "The best part of my day", "chosen": " is when I accomplish something meaningful.", "rejected": " is when it finally ends."},
            {"prompt": "If I could change one thing", "chosen": " I would make the world more peaceful.", "rejected": " I would make everyone disappear."},
            {"prompt": "My favorite memory", "chosen": " is spending time with loved ones laughing.", "rejected": " is being completely alone and isolated."},
        ]

        with open(prefs_path, "w", encoding="utf-8") as f:
            for pref in sample_preferences:
                f.write(json.dumps(pref) + "\n")

        console.print(Panel(f"Created {len(sample_preferences)} preference pairs at {prefs_path}", border_style="green"))

    # Create sample prompts if missing
    prompts_path = "data/prompts.jsonl"
    if not os.path.exists(prompts_path):
        console.print(Panel("Creating sample prompts for GRPO...", border_style="blue", title="Data Setup"))

        sample_prompts = [
            {"prompt": "Once upon a time"},
            {"prompt": "The weather today is"},
            {"prompt": "I think that"},
            {"prompt": "When I wake up"},
            {"prompt": "Learning new things"},
            {"prompt": "The best part of my day"},
            {"prompt": "If I could change one thing"},
            {"prompt": "My favorite memory"},
            {"prompt": "Tomorrow I will"},
            {"prompt": "The most important thing"},
            {"prompt": "In a perfect world"},
            {"prompt": "My greatest hope is"},
        ]

        with open(prompts_path, "w", encoding="utf-8") as f:
            for prompt in sample_prompts:
                f.write(json.dumps(prompt) + "\n")

        console.print(Panel(f"Created {len(sample_prompts)} prompts at {prompts_path}", border_style="green"))


# -------------------------- CLI --------------------------
def main(argv=None):
    ap=argparse.ArgumentParser(description="BPE NanoGPT + MoE + Reward Model + GRPO")
    ap.add_argument("--mode",required=True,choices=[
        "train_gpt_bpe","infer_gpt_bpe",
        "train_reward","score_reward",
        "train_grpo"
    ])
    ap.add_argument("--device",default="auto",choices=["auto","cpu","cuda"])
    ap.add_argument("--epochs",type=int,default=2)
    ap.add_argument("--steps_per_epoch",type=int,default=200)
    ap.add_argument("--batch_size",type=int,default=64)
    ap.add_argument("--lr",type=float,default=3e-4)
    ap.add_argument("--block_size",type=int,default=256)
    ap.add_argument("--dropout",type=float,default=0.1)
    ap.add_argument("--n_layer",type=int,default=6)
    ap.add_argument("--n_head",type=int,default=6)
    ap.add_argument("--n_embd",type=int,default=384)
    ap.add_argument("--bias",action="store_true",default=True)
    ap.add_argument("--attention_backend",default="sdpa",choices=["sdpa","vanilla"])

    # MoE knobs
    ap.add_argument("--use_moe",action="store_true")
    ap.add_argument("--num_experts",type=int,default=3)
    ap.add_argument("--k",type=int,default=1)
    ap.add_argument("--router_hidden",type=int,default=128)
    ap.add_argument("--blw",type=float,default=0.02)
    ap.add_argument("--shared_init_scale",type=float,default=0.3)
    ap.add_argument("--entropy_penalty",type=float,default=0.001)
    ap.add_argument("--router_temp",type=float,default=1.0)
    ap.add_argument("--router_temp_start",type=float,default=1.5)
    ap.add_argument("--router_temp_end",type=float,default=0.5)
    ap.add_argument("--no_gumbel",action="store_true")
    ap.add_argument("--gumbel_off_epoch",type=int,default=0)
    ap.add_argument("--routing_mode",choices=["specialize","uniform"],default="specialize")
    ap.add_argument("--rescue_lambda",type=float,default=0.0)
    ap.add_argument("--usage_momentum",type=float,default=0.99)

    # Tokenizer
    ap.add_argument("--tok_kind",default="basic",choices=["basic","regex"])
    ap.add_argument("--vocab_size",type=int,default=2048)
    ap.add_argument("--tok_prefix",default="artifacts/tokenizers/tiny_bpe")
    ap.add_argument("--train_corpus",default="")

    # IO
    ap.add_argument("--save_path",default="artifacts/gpt_moe_bpe.pt")
    ap.add_argument("--load_path",default="artifacts/gpt_moe_bpe.pt")
    ap.add_argument("--seed",type=int,default=1337)
    ap.add_argument("--sample_start",default="\n")
    ap.add_argument("--sample_tokens",type=int,default=200)

    # RM data
    ap.add_argument("--rm_pairs",default="")         # JSONL with {"prompt","chosen","rejected"}

    # RM scoring
    ap.add_argument("--score_text",default="")       # free text to score with RM
    ap.add_argument("--rm_ckpt",default="")          # reward model checkpoint for GRPO / scoring

    # GRPO data
    ap.add_argument("--grpo_prompts",default="")     # JSONL with {"prompt": "..."}
    ap.add_argument("--group_size",type=int,default=4)
    ap.add_argument("--gen_max_tokens",type=int,default=128)
    ap.add_argument("--beta_kl",type=float,default=0.02)

    args=ap.parse_args(argv)
    set_seed(args.seed); device=args.device if args.device!="auto" else ("cuda" if torch.cuda.is_available() else "cpu")

    create_sample_data_if_missing()

    # 1) Load text (for tokenizer)
    if args.train_corpus and os.path.exists(args.train_corpus):
        with open(args.train_corpus,"r",encoding="utf-8") as f:
            full=f.read()
        split=int(0.9*len(full)); train_txt, val_txt = full[:split], full[split:]
    else:
        train_txt, val_txt = load_tiny_shakespeare("./data")

    # 2) Tokenizer
    tok, tok_model_path, vocab_size_actual = train_or_load_tokenizer(
        args.tok_kind, args.vocab_size, train_txt + val_txt, args.tok_prefix, verbose=True
    )

    # 3) Encode to BPE IDs for supervised LM modes
    train_ids = encode_text(tok, train_txt)
    val_ids   = encode_text(tok, val_txt)

    # 4) Safety shrink
    if len(train_ids) <= args.block_size + 1:
        new_bs=max(16, min(args.block_size, len(train_ids)-2))
        console.print(Panel(f"Auto-adjusting block_size {args.block_size} -> {new_bs}", border_style="yellow", title="Safety"))
        args.block_size=new_bs

    # Config
    cfg=GPTCfg(
        block_size=args.block_size, vocab_size=vocab_size_actual, n_layer=args.n_layer, n_head=args.n_head,
        n_embd=args.n_embd, dropout=args.dropout, bias=args.bias, attention_backend=args.attention_backend,
        use_moe=args.use_moe, num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden, blw=args.blw,
        shared_init_scale=args.shared_init_scale, entropy_penalty=args.entropy_penalty,
        router_temp=args.router_temp, router_temp_start=args.router_temp_start, router_temp_end=args.router_temp_end,
        add_gumbel=not args.no_gumbel, gumbel_off_epoch=args.gumbel_off_epoch,
        routing_mode=args.routing_mode, rescue_lambda=args.rescue_lambda, usage_momentum=args.usage_momentum
    )

    if args.mode == "train_gpt_bpe":
        console.print(Panel(json.dumps(asdict(cfg),indent=2), title="GPT-BPE Config", border_style="blue"))
        model=NanoGPT(cfg)

        def temp_sched(ep,E):
            a=(ep-1)/max(1,(E-1))
            return args.router_temp_end + (args.router_temp_start-args.router_temp_end)*(1.0-a)

        train_supervised(model, train_ids, val_ids, cfg.block_size, args.epochs, args.steps_per_epoch,
                         args.batch_size, args.lr, device,
                         title=f"NanoGPT-BPE ({'MoE' if cfg.use_moe else 'no-MoE'}, {cfg.attention_backend})",
                         temp_sched=(temp_sched if cfg.use_moe else None),
                         gumbel_off_epoch=(cfg.gumbel_off_epoch if cfg.use_moe else 0))
        gp=activated_params_gpt(model) if cfg.use_moe else {
            "Total params":count_parameters(model)[0],
            "Trainable params":count_parameters(model)[1],
            "Attention params":sum(p.numel() for n,p in model.named_parameters() if 'c_attn' in n or 'c_proj' in n)
        }
        params_panel("GPT-BPE Parameters", gp, extras=[("Tokenizer model", tok_model_path), ("Vocab size", f"{vocab_size_actual}")])
        save_ckpt(args.save_path, {"kind":"gpt_bpe","cfg":asdict(cfg),"state_dict":model.state_dict(),
                                   "tokenizer":{"kind":args.tok_kind,"model_path":tok_model_path,"vocab_size":vocab_size_actual}})

    elif args.mode == "infer_gpt_bpe":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tok_info=payload["tokenizer"]; tok=_new_tok(tok_info["kind"]); tok.load(tok_info["model_path"])
        cfg2=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg2); model.load_state_dict(payload["state_dict"])
        model.eval().to(device)
        x=torch.tensor([encode_text(tok, args.sample_start)], dtype=torch.long, device=device)
        y=model.generate(x, max_new_tokens=args.sample_tokens, temperature=0.8, top_k=200)[0].tolist()
        txt=decode_ids(tok, y)
        console.print(Panel(txt, title=f"NanoGPT-BPE Sample ({'MoE' if cfg2.use_moe else 'no-MoE'})", border_style="magenta", expand=True))

    elif args.mode == "train_reward":
        # Train RM on preference pairs; freeze backbone, train reward_head only
        rm=RewardModel(cfg)
        if not args.rm_pairs: raise ValueError("--rm_pairs JSONL required")
        train_reward_model(rm, tok, args.rm_pairs, cfg.block_size, args.epochs, args.steps_per_epoch, args.batch_size, args.lr, device)
        save_ckpt(args.save_path, {"kind":"reward_model","cfg":asdict(cfg),"state_dict":rm.state_dict(),
                                   "tokenizer":{"kind":args.tok_kind,"model_path":tok_model_path,"vocab_size":vocab_size_actual}})

    elif args.mode == "score_reward":
        if not args.rm_ckpt: raise ValueError("--rm_ckpt required")
        payload=load_ckpt(args.rm_ckpt); assert payload["kind"]=="reward_model"
        tok_info=payload["tokenizer"]; tok=_new_tok(tok_info["kind"]); tok.load(tok_info["model_path"])
        cfg2=GPTCfg(**payload["cfg"]); rm=RewardModel(cfg2); rm.load_state_dict(payload["state_dict"]); rm.eval().to(device)
        text=args.score_text or "Hello world."
        ids=encode_text(tok, text)[:cfg2.block_size]
        if len(ids)<cfg2.block_size: ids=ids+[0]*(cfg2.block_size-len(ids))
        score=rm(torch.tensor([ids],device=device)).item()
        console.print(Panel(f"Reward(text) = {score:.4f}", border_style="magenta"))

    elif args.mode == "train_grpo":
        # Load / init policy and reference (clone of initial policy)
        policy=NanoGPT(cfg)
        ref=NanoGPT(cfg); ref.load_state_dict(policy.state_dict())  # ref frozen
        for p in ref.parameters(): p.requires_grad=False
        if not args.rm_ckpt: raise ValueError("--rm_ckpt required for GRPO")
        if not args.grpo_prompts: raise ValueError("--grpo_prompts required")
        rm_payload=load_ckpt(args.rm_ckpt); assert rm_payload["kind"]=="reward_model"
        cfg_rm=GPTCfg(**rm_payload["cfg"]); rm=RewardModel(cfg_rm); rm.load_state_dict(rm_payload["state_dict"]); rm.eval()
        train_grpo(policy, ref, rm, tok,
                   prompts_path=args.grpo_prompts,
                   block_size=cfg.block_size,
                   epochs=args.epochs, steps_per_epoch=args.steps_per_epoch,
                   batch_prompts=args.batch_size, group_size=args.group_size,
                   gen_max_tokens=args.gen_max_tokens, beta_kl=args.beta_kl,
                   lr=args.lr, device=device)
        save_ckpt(args.save_path, {"kind":"policy_grpo","cfg":asdict(cfg),"state_dict":policy.state_dict(),
                                   "tokenizer":{"kind":args.tok_kind,"model_path":tok_model_path,"vocab_size":vocab_size_actual}})

# Replace the main execution block at the bottom of your script with this:
# Add this function after the imports and before the main() function


if __name__=="__main__":
    import sys
    argv = sys.argv[1:]

    # If no mode is explicitly provided, determine the appropriate default
    if not any(a.startswith("--mode") for a in argv):
        # Check what files exist to determine appropriate mode
        rm_exists = os.path.exists("artifacts/rm.pt")
        policy_exists = os.path.exists("artifacts/gpt_moe_bpe.pt")
        prefs_exist = os.path.exists("data/preferences.jsonl")
        prompts_exist = os.path.exists("data/prompts.jsonl")

        # Determine mode based on what's available
        if not prefs_exist and not rm_exists:
            # Start with basic GPT training
            console.print(Panel(
                "No preference data found. Starting with basic GPT training.\n"
                "Create data/preferences.jsonl to train reward model next.",
                title="Auto Mode Selection", border_style="yellow"
            ))
            argv = [
                "--mode", "train_gpt_bpe",
                "--tok_kind", "basic", "--vocab_size", "512",
                "--use_moe", "--routing_mode", "specialize",
                "--epochs", "2", "--steps_per_epoch", "100",
                "--batch_size", "32", "--attention_backend", "sdpa",
                "--save_path", "artifacts/gpt_moe_bpe.pt",
                "--tok_prefix", "artifacts/tokenizers/tiny_bpe"
            ]
        elif prefs_exist and not rm_exists:
            # Train reward model
            console.print(Panel(
                "Preference data found, no reward model. Training reward model.",
                title="Auto Mode Selection", border_style="yellow"
            ))
            argv = [
                "--mode", "train_reward",
                "--tok_kind", "basic", "--vocab_size", "512",
                "--epochs", "2", "--steps_per_epoch", "50",
                "--batch_size", "16", "--lr", "1e-4",
                "--rm_pairs", "data/preferences.jsonl",
                "--save_path", "artifacts/rm.pt",
                "--tok_prefix", "artifacts/tokenizers/tiny_bpe"
            ]
        elif rm_exists and prompts_exist:
            # Train with GRPO
            console.print(Panel(
                "Reward model and prompts found. Training with GRPO.",
                title="Auto Mode Selection", border_style="yellow"
            ))
            argv = [
                "--mode", "train_grpo",
                "--tok_kind", "basic", "--vocab_size", "512",
                "--epochs", "2", "--steps_per_epoch", "20",
                "--batch_size", "8", "--group_size", "4",
                "--gen_max_tokens", "50", "--beta_kl", "0.02",
                "--rm_ckpt", "artifacts/rm.pt",
                "--grpo_prompts", "data/prompts.jsonl",
                "--save_path", "artifacts/policy_grpo.pt",
                "--tok_prefix", "artifacts/tokenizers/tiny_bpe"
            ]
        elif rm_exists and not prompts_exist:
            # Score with reward model
            console.print(Panel(
                "Reward model found, no prompts. Running reward scoring demo.",
                title="Auto Mode Selection", border_style="yellow"
            ))
            argv = [
                "--mode", "score_reward",
                "--rm_ckpt", "artifacts/rm.pt",
                "--score_text", "Once upon a time there was a helpful assistant.",
                "--tok_prefix", "artifacts/tokenizers/tiny_bpe"
            ]
        else:
            # Default inference mode if we have any trained model
            if os.path.exists("artifacts/policy_grpo.pt"):
                model_path = "artifacts/policy_grpo.pt"
            elif os.path.exists("artifacts/gpt_moe_bpe.pt"):
                model_path = "artifacts/gpt_moe_bpe.pt"
            else:
                # Fallback to basic training
                console.print(Panel(
                    "No trained models found. Starting basic GPT training.",
                    title="Auto Mode Selection", border_style="yellow"
                ))
                argv = [
                    "--mode", "train_gpt_bpe",
                    "--tok_kind", "basic", "--vocab_size", "512",
                    "--epochs", "2", "--steps_per_epoch", "100",
                    "--batch_size", "32",
                    "--save_path", "artifacts/gpt_moe_bpe.pt",
                    "--tok_prefix", "artifacts/tokenizers/tiny_bpe"
                ]

            if 'model_path' in locals():
                console.print(Panel(
                    f"Using trained model: {model_path}",
                    title="Auto Mode Selection", border_style="yellow"
                ))
                argv = [
                    "--mode", "infer_gpt_bpe",
                    "--load_path", model_path,
                    "--sample_start", "Once upon a time",
                    "--sample_tokens", "100",
                    "--tok_prefix", "artifacts/tokenizers/tiny_bpe"
                ]

    main(argv)


# UNIFIED ALL

## Full NanoGPT / NanoMamba stack with Tokenizer (Char or BPE), MoE, Reward Model (pairwise), GRPO, and a runnable full pipeline — with a comprehensive TrainingMonitor that organizes plots, logs, models, and bundles everything into a ZIP.




In [ ]:
#!/usr/bin/env python3
# unified_all_in_one_with_monitor.py
# Full NanoGPT / NanoMamba stack with Tokenizer (Char or BPE), MoE, Reward Model (pairwise),
# GRPO, and a runnable full pipeline — with a comprehensive TrainingMonitor
# that organizes plots, logs, models, and bundles everything into a ZIP.
#
# Quick examples:
#   python unified_all_in_one_with_monitor.py --mode train --arch gpt --tok_kind bpe --use_moe --routing_mode specialize
#   python unified_all_in_one_with_monitor.py --mode infer --arch gpt --tok_kind bpe --load_path artifacts/gpt_moe.pt
#   python unified_all_in_one_with_monitor.py --mode train_rm --arch gpt --tok_kind bpe --rm_pairs data/preferences.jsonl
#   python unified_all_in_one_with_monitor.py --mode train_grpo --arch gpt --tok_kind bpe --load_path artifacts/gpt_moe.pt
#
#   # With NO arguments, it runs a full monitored pipeline across all 4 combos and creates ZIPs:
#   python unified_all_in_one_with_monitor.py

import os, json, argparse, random, time, urllib.request, zipfile, shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict
import collections
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

# Optional rich console
try:
    from rich.console import Console
    from rich.panel import Panel
    HAS_RICH = True
except ImportError:
    HAS_RICH = False
    class Console:
        def print(self, *a, **k): print(*a)
        def rule(self, *a, **k): print("=" * 50)
    class Panel:
        def __init__(self, text, title="", style=""): self.text=text; self.title=title
        def __str__(self): return f"[{self.title}] {self.text}"

console = Console()

# -----------------------------------------------------------------------------
# Plotting (matplotlib only; seaborn optional)
# -----------------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")  # non-interactive backends (servers/Colab safe)
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec

try:
    import seaborn as sns
    sns.set_palette("husl")
except Exception:
    pass

plt.style.use('default')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

# =============================================================================
# TRAINING MONITOR (plots, logs, animations, ZIP packaging)
# =============================================================================

class TrainingMonitor:
    """Comprehensive training monitor that saves everything organized."""
    def __init__(self, experiment_name="unified_experiment"):
        self.experiment_name = experiment_name
        self.base_dir = Path("training_outputs") / experiment_name
        self._setup_folders()
        # Tracking
        self.metrics = defaultdict(list)
        self.attention_snapshots = []
        self.embedding_snapshots = []
        self.moe_snapshots = []
        self.step_count = 0
        self.epoch_count = 0

    # ---------------------- FS ----------------------
    def _setup_folders(self):
        folders = [
            self.base_dir,
            self.base_dir / "models",
            self.base_dir / "plots" / "training_curves",
            self.base_dir / "plots" / "attention_analysis",
            self.base_dir / "plots" / "embedding_evolution",
            self.base_dir / "plots" / "moe_analysis",
            self.base_dir / "plots" / "animations",
            self.base_dir / "tokenizers",
            self.base_dir / "logs",
            self.base_dir / "data",
        ]
        for f in folders: f.mkdir(parents=True, exist_ok=True)
        console.print(f"📁 Output structure at: {self.base_dir}")

    # ---------------------- Logging ----------------------
    def log_training_step(self, epoch, step, train_loss, val_loss=None, lr=None, grad_norm=None):
        self.metrics['epoch'].append(epoch)
        self.metrics['step'].append(self.step_count)
        self.metrics['train_loss'].append(float(train_loss))
        if val_loss is not None: self.metrics['val_loss'].append(float(val_loss))
        if lr is not None: self.metrics['learning_rate'].append(float(lr))
        if grad_norm is not None: self.metrics['grad_norm'].append(float(grad_norm))
        self.step_count += 1

    def capture_model_state(self, model):
        # Attention Q/K alignment (if present)
        if hasattr(model, 'blocks'):
            attn_data = []
            for i, block in enumerate(model.blocks):
                if hasattr(block.attn, 'c_attn'):
                    with torch.no_grad():
                        w = block.attn.c_attn.weight.data.detach().cpu().numpy()
                    n_embd = w.shape[1]
                    q_w = w[:n_embd, :]
                    k_w = w[n_embd:2*n_embd, :]
                    corr = np.corrcoef(q_w.flatten(), k_w.flatten())[0, 1]
                    attn_data.append({
                        'layer': i, 'qk_correlation': float(corr)
                    })
            if attn_data:
                self.attention_snapshots.append({
                    'step': self.step_count, 'epoch': self.epoch_count, 'data': attn_data
                })
        # Embedding snapshots
        if hasattr(model, 'wte'):
            with torch.no_grad():
                emb = model.wte.weight.data.detach().cpu().numpy()
            self.embedding_snapshots.append({
                'step': self.step_count,
                'epoch': self.epoch_count,
                'norm': float(np.linalg.norm(emb)),
                'mean': float(np.mean(emb)),
                'std': float(np.std(emb)),
                'weights': emb.copy(),
            })

    def capture_moe_stats(self, stats):
        if stats and 'mean_routing_probs' in stats and stats['mean_routing_probs'] is not None:
            probs = stats['mean_routing_probs']
            if isinstance(probs, torch.Tensor):
                probs = probs.detach().cpu().numpy()
            probs = np.asarray(probs)
            entropy = float(-(probs * np.log(probs + 1e-8)).sum())
            balance = float(1.0 - (np.std(probs) / (np.mean(probs) + 1e-8)))
            self.moe_snapshots.append({
                'step': self.step_count,
                'epoch': self.epoch_count,
                'routing_probs': probs.tolist(),
                'entropy': entropy,
                'balance': balance,
            })

    # ---------------------- Plots ----------------------
    def _plot_training_curves(self):
        if not self.metrics['train_loss']:
            return
        fig = plt.figure(figsize=(20, 12))
        gs = GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)
        epochs = np.array(self.metrics['epoch'])
        train_losses = np.array(self.metrics['train_loss'])

        ax1 = fig.add_subplot(gs[0, :2])
        ax1.plot(epochs, train_losses, linewidth=2, alpha=0.9, label='Training Loss')
        if self.metrics.get('val_loss'):
            vl = np.array(self.metrics['val_loss'])
            ax1.plot(epochs[-len(vl):], vl, linestyle='--', linewidth=2, alpha=0.9, label='Validation Loss')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Progress'); ax1.legend(); ax1.grid(True, alpha=0.3)

        if self.metrics.get('learning_rate'):
            ax2 = fig.add_subplot(gs[0, 2])
            lrs = np.array(self.metrics['learning_rate'])
            ax2.plot(epochs[-len(lrs):], lrs, linewidth=2)
            ax2.set_xlabel('Epoch'); ax2.set_ylabel('LR'); ax2.set_title('LR Schedule'); ax2.set_yscale('log'); ax2.grid(True, alpha=0.3)

        if self.metrics.get('grad_norm'):
            ax3 = fig.add_subplot(gs[1, 0])
            gns = np.array(self.metrics['grad_norm'])
            ax3.plot(epochs[-len(gns):], gns, linewidth=2, alpha=0.9)
            ax3.set_xlabel('Epoch'); ax3.set_ylabel('Grad Norm'); ax3.set_title('Gradient Norms'); ax3.grid(True, alpha=0.3)

        ax4 = fig.add_subplot(gs[1, 1])
        if len(train_losses) > 4:
            w = max(3, min(20, len(train_losses)//10))
            sm = np.convolve(train_losses, np.ones(w)/w, mode='valid')
            ax4.plot(epochs[w-1:], sm, linewidth=3, alpha=0.9)
            ax4.set_xlabel('Epoch'); ax4.set_ylabel('Smoothed Loss'); ax4.set_title(f'Loss (Moving Avg, window={w})'); ax4.grid(True, alpha=0.3)
        else:
            ax4.text(0.5,0.5,'Insufficient steps for smoothing',ha='center',va='center'); ax4.axis('off')

        ax5 = fig.add_subplot(gs[1, 2])
        final_loss = float(train_losses[-1]) if len(train_losses) else 0.0
        min_loss = float(np.min(train_losses)) if len(train_losses) else 0.0
        improvement = float(train_losses[0] - final_loss) if len(train_losses) > 1 else 0.0
        stats_text = f'Final Loss: {final_loss:.4f}\nBest Loss: {min_loss:.4f}\nImprovement: {improvement:.4f}\nTotal Steps: {len(train_losses)}'
        ax5.text(0.1, 0.5, stats_text, transform=ax5.transAxes, fontsize=12, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        ax5.axis('off'); ax5.set_title('Training Stats')

        out = self.base_dir / "plots" / "training_curves" / "comprehensive_training.png"
        plt.suptitle(f'{self.experiment_name} - Training Analysis', fontsize=16)
        plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"📊 Training curves saved to {out}")

    def _create_loss_animation(self):
        if not self.metrics['train_loss']:
            return
        fig, ax = plt.subplots(figsize=(12, 8))
        train_losses = np.array(self.metrics['train_loss'])
        epochs = np.array(self.metrics['epoch'])
        def animate(i):
            ax.clear(); ax.plot(epochs[:i+1], train_losses[:i+1], linewidth=3, alpha=0.9)
            ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title(f'Training Progress - Step {i+1}/{len(train_losses)}'); ax.grid(True, alpha=0.3)
            ax.set_xlim(0, max(epochs) if len(epochs) else 1)
            if len(train_losses):
                ax.set_ylim(min(train_losses)*0.9, max(train_losses)*1.1)
        frames = min(len(train_losses), 100)
        anim = animation.FuncAnimation(fig, animate, frames=frames, interval=150, repeat=True)
        out = self.base_dir / "plots" / "animations" / "loss_evolution.gif"
        anim.save(out, writer='pillow', fps=5); plt.close()
        console.print(f"🎬 Loss animation saved to {out}")

    def _plot_attention(self):
        if not self.attention_snapshots:
            return
        steps = [s['step'] for s in self.attention_snapshots]
        n_layers = len(self.attention_snapshots[0]['data'])
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        # layer curves
        for layer in range(n_layers):
            corrs = []
            for s in self.attention_snapshots:
                d = s['data'][layer]
                corrs.append(d['qk_correlation'])
            ax1.plot(steps[:len(corrs)], corrs, linewidth=2, alpha=0.9, label=f'L{layer}')
        ax1.set_xlabel('Step'); ax1.set_ylabel('Q-K Corr'); ax1.set_title('Query-Key Alignment Evolution'); ax1.legend(); ax1.grid(True, alpha=0.3)

        latest = self.attention_snapshots[-1]['data']
        cm = np.array([[d['qk_correlation'] for d in latest]])
        im = ax2.imshow(cm, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
        ax2.set_title('Latest Q-K Correlations'); fig.colorbar(im, ax=ax2)

        # Distribution for layer 0 not available without raw weights — show histogram of correlations instead
        all_corrs = []
        for s in self.attention_snapshots:
            all_corrs.extend([d['qk_correlation'] for d in s['data']])
        ax3.hist(all_corrs, bins=40, alpha=0.85)
        ax3.set_title('Distribution of Q-K Correlations'); ax3.set_xlabel('Corr'); ax3.set_ylabel('Freq'); ax3.grid(True, alpha=0.3)

        # Stability (std) per layer
        stdevs = []
        for layer in range(n_layers):
            layer_series = [s['data'][layer]['qk_correlation'] for s in self.attention_snapshots]
            stdevs.append(np.std(layer_series))
        ax4.bar(range(n_layers), stdevs, alpha=0.9)
        ax4.set_xlabel('Layer'); ax4.set_ylabel('Std Dev'); ax4.set_title('Q-K Correlation Stability'); ax4.grid(True, alpha=0.3)

        out = self.base_dir / "plots" / "attention_analysis" / "attention_comprehensive.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"🔍 Attention analysis saved to {out}")

    def _plot_embeddings(self):
        if not self.embedding_snapshots:
            return
        steps = [s['step'] for s in self.embedding_snapshots]
        norms = [s['norm'] for s in self.embedding_snapshots]
        means = [s['mean'] for s in self.embedding_snapshots]
        stds  = [s['std']  for s in self.embedding_snapshots]
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        ax1.plot(steps, norms, linewidth=2); ax1.set_title('Embedding Frobenius Norm'); ax1.set_xlabel('Step'); ax1.set_ylabel('Norm'); ax1.grid(True, alpha=0.3)
        ax2.plot(steps, means, linewidth=2, label='Mean'); ax2.plot(steps, stds, linewidth=2, label='Std'); ax2.legend(); ax2.set_title('Embedding Stats'); ax2.grid(True, alpha=0.3)

        # Latest distribution if we kept weights
        w = self.embedding_snapshots[-1].get('weights', None)
        if w is not None:
            ax3.hist(w.flatten(), bins=60, alpha=0.85)
            ax3.set_title('Latest Embedding Distribution'); ax3.grid(True, alpha=0.3)
            token_norms = np.linalg.norm(w, axis=1)
            ax4.plot(token_norms, alpha=0.9)
            ax4.set_title('Per-Token L2 Norms'); ax4.set_xlabel('Token'); ax4.set_ylabel('L2'); ax4.grid(True, alpha=0.3)
        else:
            ax3.axis('off'); ax4.axis('off')

        out = self.base_dir / "plots" / "embedding_evolution" / "embedding_analysis.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"📐 Embedding analysis saved to {out}")

    def _plot_moe(self):
        if not self.moe_snapshots:
            return
        steps = [s['step'] for s in self.moe_snapshots]
        nE = len(self.moe_snapshots[0]['routing_probs'])
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        for e in range(nE):
            usage = [s['routing_probs'][e] for s in self.moe_snapshots]
            ax1.plot(steps, usage, linewidth=2, alpha=0.9, label=f'E{e}')
        ax1.set_title('Expert Usage Evolution'); ax1.set_xlabel('Step'); ax1.set_ylabel('Prob'); ax1.legend(); ax1.grid(True, alpha=0.3)

        ent = [s['entropy'] for s in self.moe_snapshots]
        ax2.plot(steps, ent, linewidth=2, alpha=0.9)
        ax2.set_title('Routing Entropy'); ax2.set_xlabel('Step'); ax2.grid(True, alpha=0.3)

        latest = self.moe_snapshots[-1]['routing_probs']
        ax3.pie(latest, labels=[f'E{i}' for i in range(len(latest))], autopct='%1.1f%%', startangle=90)
        ax3.set_title('Current Expert Distribution')

        bal = [s['balance'] for s in self.moe_snapshots]
        ax4.plot(steps, bal, linewidth=2, alpha=0.9)
        ax4.set_title('Load Balance (1=best)'); ax4.set_xlabel('Step'); ax4.grid(True, alpha=0.3)

        out = self.base_dir / "plots" / "moe_analysis" / "moe_comprehensive.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"🔀 MoE analysis saved to {out}")

    # ---------------------- Persist ----------------------
    def _save_logs(self):
        # Metrics
        with open(self.base_dir / "logs" / "training_metrics.json", 'w') as f:
            json.dump({k: list(v) for k, v in self.metrics.items()}, f, indent=2)
        # Attention
        if self.attention_snapshots:
            with open(self.base_dir / "logs" / "attention_evolution.json", 'w') as f:
                json.dump(self.attention_snapshots, f, indent=2)
        # Embedding (strip weights for logs to keep size small)
        if self.embedding_snapshots:
            light = [{k: v for k, v in s.items() if k != 'weights'} for s in self.embedding_snapshots]
            with open(self.base_dir / "logs" / "embedding_evolution.json", 'w') as f:
                json.dump(light, f, indent=2)
        # MoE
        if self.moe_snapshots:
            with open(self.base_dir / "logs" / "moe_evolution.json", 'w') as f:
                json.dump(self.moe_snapshots, f, indent=2)
        console.print(f"💾 Logs saved to {self.base_dir / 'logs'}")

    def generate_all(self):
        console.print("🎨 Generating visualizations and logs...")
        self._plot_training_curves()
        self._create_loss_animation()
        self._plot_attention()
        self._plot_embeddings()
        self._plot_moe()
        self._save_logs()
        console.print("✅ Visualizations complete")

    def snapshot_tokenizer(self, src_prefix: str):
        # copy tokenizer files into outputs for convenience
        m = Path(src_prefix + ".model")
        if m.exists():
            dst = self.base_dir / "tokenizers" / m.name
            try:
                shutil.copy2(m, dst)
            except Exception:
                pass

    def zip_everything(self):
        # Create summary
        summary = {
            "experiment": self.experiment_name,
            "timestamp": time.strftime("%Y-%m-%d_%H-%M-%S"),
            "total_steps": int(self.step_count),
            "total_epochs": int(self.epoch_count),
            "final_loss": self.metrics['train_loss'][-1] if self.metrics['train_loss'] else None,
        }
        with open(self.base_dir / "experiment_summary.json", 'w') as f:
            json.dump(summary, f, indent=2)

        # README
        readme = f"""# {self.experiment_name} - Results

This folder contains models, plots, logs, tokenizers, and data artifacts.

- Plots in `plots/` (training curves, attention, embeddings, MoE, animations)
- Logs in `logs/` (JSON)
- Models in `models/`
- Tokenizers in `tokenizers/`
- Data in `data/`
"""
        with open(self.base_dir / "README.md", 'w') as f:
            f.write(readme)

        # Zip
        ts = time.strftime("%Y%m%d_%H%M%S")
        zip_name = f"unified_results_{self.experiment_name}_{ts}.zip"
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
            for p in self.base_dir.rglob('*'):
                if p.is_file(): z.write(p, p.relative_to('.'))
        size_mb = os.path.getsize(zip_name) / (1024*1024)
        console.print(Panel(f"Created ZIP: {zip_name} ({size_mb:.1f} MB)", title="Package"))
        return zip_name

# =============================================================================
# TOKENIZERS
# =============================================================================
# Character-level tokenizer utilities (char → id)
def build_charset(train, val):
    chars = sorted(set(train + val))
    stoi = {c:i for i,c in enumerate(chars)}
    itos = {i:c for c,i in stoi.items()}
    return stoi, itos

def encode_chars(s, stoi): return [stoi[c] for c in s]
def decode_chars(ids, itos): return "".join(itos[i] for i in ids)

# Simplified BPE helpers (byte-pair merges)
def _bpe_get_stats(ids):
    counts = collections.defaultdict(int)
    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts

def _bpe_merge_ids(ids, pair, new_id):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

class SimplifiedBPE:
    """
    Byte-level BPE:
    - Train: iteratively merge most frequent adjacent token pairs.
    - Encode: apply learned merges greedily (in fixed order).
    - Decode: join learned byte sequences.
    """
    def __init__(self):
        self.merges = {}          # {(a,b): new_id}
        self.tokens = {i: bytes([i]) for i in range(256)}  # id -> bytes
        self.vocab_size = 256

    def train(self, text, vocab_size):
        assert vocab_size >= 256
        ids = list(text.encode("utf-8"))
        merges = {}
        tokens = {i: bytes([i]) for i in range(256)}
        next_id = 256
        while self.vocab_size < vocab_size:
            stats = _bpe_get_stats(ids)
            if not stats:
                break
            best_pair = max(stats, key=stats.get)
            # allocate new id
            new_id = next_id
            next_id += 1
            tokens[new_id] = tokens[best_pair[0]] + tokens[best_pair[1]]
            merges[best_pair] = new_id
            ids = _bpe_merge_ids(ids, best_pair, new_id)
            self.vocab_size += 1
        self.merges = merges
        self.tokens = tokens

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        for pair, new_id in self.merges.items():
            ids = _bpe_merge_ids(ids, pair, new_id)
        return ids

    def decode(self, ids):
        return b"".join([self.tokens[i] for i in ids]).decode("utf-8", errors="replace")

    def save(self, prefix):
        d = os.path.dirname(prefix)
        if d: os.makedirs(d, exist_ok=True)
        with open(prefix + ".model", "w") as f:
            for (a,b),i in self.merges.items():
                f.write(f"{a} {b} {i}\n")  # include id for determinism

    def load(self, path):
        merges = {}
        tokens = {i: bytes([i]) for i in range(256)}
        max_id = 255
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3:
                    a, b, i = int(parts[0]), int(parts[1]), int(parts[2])
                elif len(parts) == 2:
                    a, b = int(parts[0]), int(parts[1])
                    i = max_id + 1
                else:
                    continue
                merges[(a,b)] = i
                tokens[i] = tokens[a] + tokens[b]
                max_id = max(max_id, i)
        self.merges = merges
        self.tokens = tokens
        self.vocab_size = max_id + 1

def train_or_load_bpe(vocab_size, text, prefix):
    model_path = prefix + ".model"
    tok = SimplifiedBPE()
    if os.path.exists(model_path):
        tok.load(model_path)
        console.print(Panel(f"Loaded BPE tokenizer: {model_path} (size={tok.vocab_size})", title="Tokenizer"))
        return tok, model_path, tok.vocab_size
    tok.train(text, vocab_size)
    tok.save(prefix)
    console.print(Panel(f"Saved BPE tokenizer: {prefix}.model (size={tok.vocab_size})", title="Tokenizer"))
    return tok, model_path, tok.vocab_size

# =============================================================================
# DATA
# =============================================================================

def load_tiny_shakespeare(data_dir="./data"):
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, "tinyshakespeare_input.txt")
    if not os.path.exists(path):
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(url, path)
    text = open(path, "r", encoding="utf-8").read()
    split = int(0.9 * len(text))
    return text[:split], text[split:]

def get_batch_tokens(ids, block_size, batch_size, device):
    ix = torch.randint(len(ids) - block_size - 1, (batch_size,))
    x = torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y = torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================================================================
# MoE BUILDING BLOCKS
# =============================================================================

class TopKRouter(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=1, temp=1.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, num_experts))
        self.k = k; self.temp = temp
    def forward(self, x):
        probs = F.softmax(self.net(x) / self.temp, dim=-1)
        topk_vals, topk_idx = torch.topk(probs, self.k, dim=-1)
        mask = torch.zeros_like(probs)
        mask.scatter_(dim=-1, index=topk_idx, src=torch.ones_like(topk_vals))
        sp = probs * mask
        sp = sp / (sp.sum(dim=-1, keepdim=True) + 1e-9)
        return sp, probs

def entropy_mean(p, eps=1e-9):
    p = p.clamp_min(eps)
    return -(p * p.log()).sum(-1).mean()

class MoEFFN(nn.Module):
    def __init__(self, in_dim, num_experts=4, k=1, router_hidden=128, dropout=0.1,
                 entropy_penalty=0.0, routing_mode="uniform"):
        super().__init__()
        hidden = 4 * in_dim
        self.expert_fc = nn.ModuleList([nn.Linear(in_dim, hidden) for _ in range(num_experts)])
        self.expert_proj = nn.ModuleList([nn.Linear(hidden, in_dim) for _ in range(num_experts)])
        self.router = TopKRouter(in_dim, router_hidden, num_experts, k)
        self.drop = nn.Dropout(dropout)
        self.entropy_penalty = entropy_penalty
        self.routing_mode = routing_mode
        self.num_experts = num_experts
    def forward(self, x):
        B, T, C = x.shape; xf = x.reshape(B*T, C)
        sp, dp = self.router(xf); out = 0.0
        for e in range(self.num_experts):
            h = F.gelu(self.expert_fc[e](xf))
            h = self.expert_proj[e](h)
            out = out + sp[:, e].unsqueeze(-1) * h
        y = self.drop(out.view(B, T, C))
        aux = torch.tensor(0.0, device=x.device)
        if self.routing_mode == "specialize":
            aux = -self.entropy_penalty * entropy_mean(dp)
        return y, aux, {"mean_routing_probs": dp.mean(0)}

# =============================================================================
# MODELS
# =============================================================================

class LayerNorm(nn.Module):
    def __init__(self, n, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n))
        self.bias = nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

@dataclass
class GPTCfg:
    block_size: int = 128
    vocab_size: int = 256
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.1
    # MoE
    use_moe: bool = False
    routing_mode: str = "uniform"  # or "specialize"
    num_experts: int = 4
    topk: int = 1
    router_hidden: int = 128
    moe_dropout: float = 0.1
    entropy_penalty: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.n_head = n_head; self.n_embd = n_embd
        self.c_attn = nn.Linear(n_embd, 3*n_embd, bias=True)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=True)
        self.dropout = dropout; self.block_size = block_size
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class GPTBlock(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.ln1 = LayerNorm(cfg.n_embd, True)
        self.attn = CausalSelfAttention(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout)
        self.ln2 = LayerNorm(cfg.n_embd, True)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd), nn.GELU(), nn.Linear(4 * cfg.n_embd, cfg.n_embd)
        )
        self.moe = (MoEFFN(cfg.n_embd, cfg.num_experts, cfg.topk, cfg.router_hidden,
                           cfg.moe_dropout, cfg.entropy_penalty, cfg.routing_mode)
                    if cfg.use_moe else None)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        if self.moe is None:
            x = x + self.ffn(self.ln2(x))
            return x, torch.tensor(0.0, device=x.device), {}
        else:
            y, aux, stats = self.moe(self.ln2(x))
            return x + y, aux, stats

class NanoGPT(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([GPTBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = LayerNorm(cfg.n_embd, True)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
    def forward(self, idx, targets=None):
        B, T = idx.shape; pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        aux = torch.tensor(0.0, device=idx.device); stats = {}
        for blk in self.blocks:
            x, a, s = blk(x); aux = aux + a; stats = s
        h = self.ln_f(x); logits = self.lm_head(h)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, (loss + aux if (loss is not None) else None), stats
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _, _ = self(idx); logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1); nxt = torch.multinomial(probs, 1)
            idx = torch.cat((idx, nxt), dim=1)
        return idx

class SimpleMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__(); self.ln = LayerNorm(d, True); self.ff = nn.Linear(d, d)
    def forward(self, x): return x + self.ff(self.ln(x))

class NanoMamba(nn.Module):
    def __init__(self, vocab_size=256, block_size=128, n_layer=4, n_embd=128):
        super().__init__()
        self.vocab_size = vocab_size; self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(block_size, n_embd)
        self.layers = nn.ModuleList([SimpleMambaBlock(n_embd) for _ in range(n_layer)])
        self.ln_f = LayerNorm(n_embd, True); self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        B, T = idx.shape; pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        for l in self.layers: x = l(x)
        h = self.ln_f(x); logits = self.head(h)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss, {}
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _, _ = self(idx); logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1); nxt = torch.multinomial(probs, 1)
            idx = torch.cat((idx, nxt), dim=1)
        return idx

# =============================================================================
# TRAIN / GRPO / REWARD MODEL (instrumented with TrainingMonitor)
# =============================================================================

def train_lm(model, train_ids, val_ids, block_size, epochs, steps, batch, lr, device, title, monitor: TrainingMonitor=None):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    best_val = float('inf'); patience = 0

    for ep in range(1, epochs+1):
        model.train(); losses=[]; gnorms=[]; t0=time.time()
        for st in range(steps):
            xb, yb = get_batch_tokens(train_ids, block_size, batch, device)
            opt.zero_grad()
            _, loss, stats = model(xb, yb)
            loss.backward()
            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            gnorms.append(float(total_norm.item()))
            opt.step()
            losses.append(float(loss.item()))
            if monitor and (st % 10 == 0):
                monitor.capture_model_state(model)
                if stats: monitor.capture_moe_stats(stats)
        # validation
        model.eval()
        with torch.no_grad():
            xb, yb = get_batch_tokens(val_ids, block_size, batch, device)
            _, vloss, _ = model(xb, yb)
            v = float(vloss.item())
        tr = float(sum(losses)/len(losses))
        lr_now = float(sched.get_last_lr()[0])
        gn = float(sum(gnorms)/len(gnorms)) if gnorms else 0.0
        if monitor:
            monitor.log_training_step(ep, steps, tr, v, lr_now, gn)
            monitor.epoch_count = ep
        console.print(Panel(f"Epoch {ep}/{epochs} train={tr:.3f} val={v:.3f} grad={gn:.3f} time={time.time()-t0:.1f}s", title=title))
        if v < best_val:
            best_val = v; patience = 0
            save_path = monitor.base_dir / "models" / f"best_{title.replace(' ', '_').lower()}.pt" if monitor else Path("artifacts")/f"best_{title.replace(' ', '_').lower()}.pt"
            # We cannot know tok_kind from here; saved in main after we have args
            torch.save({"state_dict": model.state_dict()}, save_path)
        else:
            patience += 1
        sched.step()
        if patience >= 50:
            console.print("⏹️ Early stopping")
            break
    return model

def grpo_step(model, encode_fn, prompts, gen_tokens, opt, device, monitor: TrainingMonitor=None):
    model.train(); rewards=[]; logps=[]; stats_last=None
    for s in prompts:
        x = torch.tensor([encode_fn(s)], device=device)
        y = model.generate(x, gen_tokens)[0]  # [seq]
        inp, tgt = y[:-1].unsqueeze(0), y[1:].unsqueeze(0)
        logits, _, stats = model(inp, tgt)
        logp = F.log_softmax(logits, dim=-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1).mean()
        rewards.append(logp); logps.append(logp)
        stats_last = stats
    rewards = torch.stack(rewards)
    logps = torch.stack(logps)
    adv = rewards - rewards.mean()
    loss = -(adv.detach() * logps).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if monitor and stats_last:
        monitor.capture_model_state(model)
        monitor.capture_moe_stats(stats_last)
    return float(loss.item()), float(rewards.mean().item())

class RewardModel(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base
        vocab_size = base.cfg.vocab_size if hasattr(base, "cfg") else base.vocab_size
        self.head = nn.Linear(vocab_size, 1)
    def forward(self, idx):
        logits, _, _ = self.base(idx); h = logits[:, -1, :]
        return self.head(h)

def load_pairs(path): return [json.loads(l) for l in open(path, "r", encoding="utf-8")]

def train_reward_model(rm, tok, pairs_path, block_size, epochs, steps, batch, lr, device, monitor: TrainingMonitor=None):
    rm.to(device); pairs = load_pairs(pairs_path)
    opt = torch.optim.AdamW(rm.parameters(), lr=lr)
    for ep in range(1, epochs+1):
        random.shuffle(pairs); losses=[]
        for _ in range(steps):
            b = random.sample(pairs, min(batch, len(pairs)))
            chosen = [tok.encode(p["chosen"]) for p in b]
            rejected = [tok.encode(p["rejected"]) for p in b]
            cl = [torch.tensor(c[:block_size]) for c in chosen]
            rl = [torch.tensor(r[:block_size]) for r in rejected]
            cl = torch.nn.utils.rnn.pad_sequence(cl, batch_first=True).to(device)
            rl = torch.nn.utils.rnn.pad_sequence(rl, batch_first=True).to(device)
            rc = rm(cl); rr = rm(rl)
            loss = -F.logsigmoid(rc - rr).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(float(loss.item()))
        console.print(Panel(f"RM Epoch {ep}/{epochs} loss={sum(losses)/len(losses):.3f}", title="Reward Model"))
        if monitor:
            monitor.log_training_step(ep, steps, sum(losses)/len(losses))
            monitor.epoch_count = ep

# =============================================================================
# MAIN (CLI)
# =============================================================================

def main(argv=None):
    if argv is None: argv = []

    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 100))
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 50))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 64))

    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", required=True, choices=["train", "infer", "train_rm", "train_grpo"])
    ap.add_argument("--arch", required=True, choices=["gpt", "mamba"])
    ap.add_argument("--tok_kind", required=True, choices=["char", "bpe"])
    ap.add_argument("--epochs", type=int, default=DEFAULT_EPOCHS)
    ap.add_argument("--steps_per_epoch", type=int, default=DEFAULT_STEPS_PER_EPOCH)
    ap.add_argument("--batch_size", type:int, default=DEFAULT_BATCH_SIZE)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--block_size", type=int, default=128)
    ap.add_argument("--n_layer", type=int, default=4)
    ap.add_argument("--n_head", type=int, default=4)
    ap.add_argument("--n_embd", type=int, default=128)
    ap.add_argument("--use_moe", action="store_true")
    ap.add_argument("--routing_mode", default="uniform", choices=["uniform", "specialize"])
    ap.add_argument("--num_experts", type=int, default=4)
    ap.add_argument("--topk", type=int, default=1)
    ap.add_argument("--router_hidden", type=int, default=128)
    ap.add_argument("--entropy_penalty", type=float, default=0.0)
    ap.add_argument("--moe_dropout", type=float, default=0.1)
    ap.add_argument("--vocab_size", type=int, default=512)  # for BPE
    ap.add_argument("--tok_prefix", default="artifacts/tokenizers/tiny_bpe")
    ap.add_argument("--save_path", default="artifacts/unified.pt")
    ap.add_argument("--load_path", default="artifacts/unified.pt")
    ap.add_argument("--sample_start", default="\n")
    ap.add_argument("--sample_tokens", type=int, default=100)
    ap.add_argument("--rm_pairs", default="")
    ap.add_argument("--exp_name", default="unified_experiment")
    args = ap.parse_args(argv)

    os.makedirs("artifacts", exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    console.print(Panel(f"Using device: {device}", title="Device"))

    # Monitor
    monitor = TrainingMonitor(args.exp_name)

    # Data
    train_txt, val_txt = load_tiny_shakespeare("./data")

    # Tokenizer / encoding (orthogonal to arch)
    encode_fn, decode_fn = None, None
    vocab_size = None
    tok = None
    if args.mode == "infer":
        # Build tokenizer consistent with checkpoint if possible (handled later when loading)
        pass

    if args.mode in ("train", "train_grpo", "train_rm"):
        if args.tok_kind == "char":
            stoi, itos = build_charset(train_txt, val_txt)
            train_ids, val_ids = encode_chars(train_txt, stoi), encode_chars(val_txt, stoi)
            encode_fn = lambda s: encode_chars(s, stoi)
            decode_fn = lambda ids: decode_chars(ids, itos)
            vocab_size = len(stoi)
        elif args.tok_kind == "bpe":
            tok, _, vocab_size = train_or_load_bpe(args.vocab_size, train_txt + val_txt, args.tok_prefix)
            train_ids, val_ids = tok.encode(train_txt), tok.encode(val_txt)
            encode_fn, decode_fn = tok.encode, tok.decode
            monitor.snapshot_tokenizer(args.tok_prefix)

    # Model (create or load)
    model = None
    payload = None
    if args.mode != "train" and os.path.exists(args.load_path):
        payload = torch.load(args.load_path, map_location=device)
        kind = payload.get("arch", payload.get("kind", args.arch))
        tok_kind = payload.get("tok_kind", args.tok_kind)
        cfg_dict = payload.get("cfg", {})

        # Rebuild tokenizer for infer/train_grpo/train_rm if needed
        if args.tok_kind != tok_kind:
            console.print(Panel(f"Overriding --tok_kind to checkpoint tok_kind='{tok_kind}' for consistency.", title="Tokenizer"))
            args.tok_kind = tok_kind

        if args.tok_kind == "char":
            stoi, itos = build_charset(train_txt, val_txt)
            encode_fn = lambda s: encode_chars(s, stoi)
            decode_fn = lambda ids: decode_chars(ids, itos)
            vocab_size = len(stoi)
        elif args.tok_kind == "bpe":
            tok, _, vocab_size = train_or_load_bpe(args.vocab_size, train_txt + val_txt, args.tok_prefix)
            encode_fn, decode_fn = tok.encode, tok.decode
            monitor.snapshot_tokenizer(args.tok_prefix)

        # Rebuild model
        if kind == "gpt":
            model = NanoGPT(GPTCfg(**cfg_dict))
        elif kind == "mamba":
            model = NanoMamba(cfg_dict.get("vocab_size", vocab_size),
                              cfg_dict.get("block_size", args.block_size),
                              cfg_dict.get("n_layer", args.n_layer),
                              cfg_dict.get("n_embd", args.n_embd))
        else:
            raise ValueError(f"Unknown checkpoint arch/kind: {kind}")
        model.load_state_dict(payload["state_dict"])
    else:
        # Fresh model
        if args.arch == "gpt":
            if vocab_size is None:
                # Build tokenizer if not done (e.g., train mode guaranteed above)
                if args.tok_kind == "char":
                    stoi, itos = build_charset(train_txt, val_txt)
                    vocab_size = len(stoi)
                else:
                    tok, _, vocab_size = train_or_load_bpe(args.vocab_size, train_txt + val_txt, args.tok_prefix)
            cfg = GPTCfg(block_size=args.block_size, vocab_size=vocab_size, n_layer=args.n_layer, n_head=args.n_head, n_embd=args.n_embd,
                         dropout=0.1, use_moe=args.use_moe, routing_mode=args.routing_mode, num_experts=args.num_experts,
                         topk=args.topk, router_hidden=args.router_hidden, moe_dropout=args.moe_dropout, entropy_penalty=args.entropy_penalty)
            model = NanoGPT(cfg)
        else:
            if vocab_size is None:
                if args.tok_kind == "char":
                    stoi, itos = build_charset(train_txt, val_txt)
                    vocab_size = len(stoi)
                else:
                    tok, _, vocab_size = train_or_load_bpe(args.vocab_size, train_txt + val_txt, args.tok_prefix)
            model = NanoMamba(vocab_size, args.block_size, args.n_layer, args.n_embd)

    # Modes
    if args.mode == "train":
        title = f"{args.arch.upper()}-{args.tok_kind.upper()} Train"
        model = train_lm(model, train_ids, val_ids, args.block_size, args.epochs, args.steps_per_epoch, args.batch_size, args.lr, device, title, monitor)
        cfg_to_save = asdict(model.cfg) if hasattr(model, "cfg") else {"vocab_size": getattr(model, "vocab_size", None), "block_size": args.block_size, "n_layer": args.n_layer, "n_embd": args.n_embd}
        ckpt = {"arch": args.arch, "tok_kind": args.tok_kind, "cfg": cfg_to_save, "state_dict": model.state_dict()}
        torch.save(ckpt, args.save_path)
        torch.save(ckpt, monitor.base_dir / "models" / Path(args.save_path).name)
        console.print(Panel(f"Model saved to {args.save_path}", title="Save"))
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        console.print(Panel(f"ZIP ready: {zip_path}", title="Done"))

    elif args.mode == "infer":
        if payload is None:
            payload = torch.load(args.load_path, map_location=device)
            # ensure tokenizer consistent
            tok_kind = payload.get("tok_kind", "char")
            if tok_kind == "char":
                stoi, itos = build_charset(train_txt, val_txt)
                encode_fn = lambda s: encode_chars(s, stoi)
                decode_fn = lambda ids: decode_chars(ids, itos)
            else:
                tok, _, _ = train_or_load_bpe(args.vocab_size, train_txt + val_txt, args.tok_prefix)
                encode_fn, decode_fn = tok.encode, tok.decode

            kind = payload.get("arch", payload.get("kind", "gpt"))
            cfg_dict = payload.get("cfg", {})
            if kind == "gpt":
                model = NanoGPT(GPTCfg(**cfg_dict))
            else:
                model = NanoMamba(cfg_dict.get("vocab_size", 256),
                                  cfg_dict.get("block_size", args.block_size),
                                  cfg_dict.get("n_layer", args.n_layer),
                                  cfg_dict.get("n_embd", args.n_embd))
            model.load_state_dict(payload["state_dict"])
        model.to(device)
        start_ids = encode_fn(args.sample_start)
        idx = torch.tensor([start_ids], device=device)
        sample = model.generate(idx, args.sample_tokens)[0].tolist()
        console.print(Panel(decode_fn(sample), title="Generated Text"))

    elif args.mode == "train_grpo":
        # requires tokenizer prepared above
        model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
        prompts = ["To be, or not to be", "Once upon a time", "The quick brown fox"]
        for step in range(args.steps_per_epoch):
            loss, mr = grpo_step(model, encode_fn, prompts, 32, opt, device, monitor)
            monitor.log_training_step(step+1, step, train_loss=loss)
            console.print(Panel(f"GRPO step {step+1}/{args.steps_per_epoch} loss={loss:.3f} reward={mr:.3f}", title="GRPO"))
        cfg_to_save = asdict(model.cfg) if hasattr(model, "cfg") else {"vocab_size": getattr(model, "vocab_size", None), "block_size": args.block_size, "n_layer": args.n_layer, "n_embd": args.n_embd}
        ckpt = {"arch": args.arch, "tok_kind": args.tok_kind, "cfg": cfg_to_save, "state_dict": model.state_dict()}
        torch.save(ckpt, args.save_path)
        torch.save(ckpt, monitor.base_dir / "models" / Path(args.save_path).name)
        console.print(Panel(f"GRPO-tuned model saved to {args.save_path}", title="Save"))
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        console.print(Panel(f"ZIP ready: {zip_path}", title="Done"))

    elif args.mode == "train_rm":
        if not args.rm_pairs:
            raise ValueError("--rm_pairs is required for train_rm")
        model.to(device)
        rm = RewardModel(model)
        # tok-like object with .encode
        if args.tok_kind == "bpe":
            tok_like = tok
        else:
            class TK:
                def encode(self, s): return encode_fn(s)
            tok_like = TK()
        train_reward_model(rm, tok_like, args.rm_pairs, args.block_size, epochs=args.epochs, steps=args.steps_per_epoch, batch=args.batch_size, lr=1e-5, device=device, monitor=monitor)
        torch.save({"kind": "reward_model", "base_arch": args.arch, "tok_kind": args.tok_kind, "state_dict": rm.state_dict()}, args.save_path)
        torch.save({"kind": "reward_model", "base_arch": args.arch, "tok_kind": args.tok_kind, "state_dict": rm.state_dict()}, monitor.base_dir / "models" / Path(args.save_path).name)
        console.print(Panel(f"Reward model saved to {args.save_path}", title="Save"))
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        console.print(Panel(f"ZIP ready: {zip_path}", title="Done"))

# =============================================================================
# FULL PIPELINE (runs everything and packages results)
# =============================================================================

def run_full_pipeline():
    console.rule("[bold magenta]Unified End-to-End Pipeline (Monitored, 4 combos)")

    # Conservative defaults to keep demo feasible
    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 200))
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 20))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 64))

    # 1) GPT + BPE (MoE, specialize) -> Train
    main([
        "--mode","train",
        "--arch","gpt","--tok_kind","bpe",
        "--use_moe","--routing_mode","specialize",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--save_path","artifacts/gpt_bpe_moe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","gpt_bpe_moe"
    ])

    # 2) Reward Model for GPT+BPE (create toy data if missing)
    rm_file = "data/preferences.jsonl"
    if not os.path.exists(rm_file):
        toy = [
            {"chosen":"I love this model","rejected":"I hate this model"},
            {"chosen":"This is a great answer","rejected":"This is a terrible answer"},
        ]
        os.makedirs("data", exist_ok=True)
        with open(rm_file, "w", encoding="utf-8") as f:
            for row in toy: f.write(json.dumps(row) + "\n")
    main([
        "--mode","train_rm","--arch","gpt","--tok_kind","bpe",
        "--rm_pairs",rm_file,
        "--load_path","artifacts/gpt_bpe_moe.pt",
        "--save_path","artifacts/reward_model_gpt_bpe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--epochs", str(max(5, DEFAULT_EPOCHS//5)),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--exp_name","reward_model_gpt_bpe"
    ])

    # 3) GRPO fine-tuning (GPT+BPE)
    main([
        "--mode","train_grpo","--arch","gpt","--tok_kind","bpe",
        "--load_path","artifacts/gpt_bpe_moe.pt",
        "--save_path","artifacts/gpt_bpe_grpo.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--exp_name","gpt_bpe_grpo"
    ])

    # 4) GPT+BPE inference
    main([
        "--mode","infer",
        "--arch","gpt","--tok_kind","bpe",
        "--load_path","artifacts/gpt_bpe_moe.pt",
        "--sample_start","Hello world,",
        "--sample_tokens","50",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","gpt_bpe_moe"
    ])

    # 5) GPT + Char (baseline fair comparison)
    main([
        "--mode","train",
        "--arch","gpt","--tok_kind","char",
        "--epochs", str(max(50, DEFAULT_EPOCHS//2)),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--save_path","artifacts/gpt_char.pt",
        "--exp_name","gpt_char"
    ])

    # 6) Mamba + BPE (fairness vs GPT+BPE)
    main([
        "--mode","train",
        "--arch","mamba","--tok_kind","bpe",
        "--epochs", str(max(50, DEFAULT_EPOCHS//2)),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH//1),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--save_path","artifacts/mamba_bpe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","mamba_bpe"
    ])

    # 7) Mamba + Char
    main([
        "--mode","train",
        "--arch","mamba","--tok_kind","char",
        "--epochs", str(max(50, DEFAULT_EPOCHS//2)),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH//1),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--save_path","artifacts/mamba_char.pt",
        "--exp_name","mamba_char"
    ])

    # 8) Mamba inference (char)
    main([
        "--mode","infer",
        "--arch","mamba","--tok_kind","char",
        "--load_path","artifacts/mamba_char.pt",
        "--sample_start","ROMEO:",
        "--sample_tokens","80",
        "--exp_name","mamba_char"
    ])

# =============================================================================
# ENTRYPOINT
# =============================================================================

def _sanitize_argv(argv):
    out = []; skip = False
    for a in argv:
        if skip: skip = False; continue
        if a in ("-f", "--f"): skip = True; continue
        if a.startswith("-f=") or a.startswith("--f="): continue
        out.append(a)
    return out

if __name__ == "__main__":
    import sys
    argv = _sanitize_argv(sys.argv[1:])
    # If required args missing, run the full multi-combo pipeline
    if ("--mode" not in argv) or ("--arch" not in argv) or ("--tok_kind" not in argv):
        run_full_pipeline()
    else:
        main(argv)


## Full NanoGPT / NanoMamba stack with Tokenizer, MoE, Reward Model (pairwise), GRPO, and a runnable full pipeline.


In [ ]:
#!/usr/bin/env python3
# unified_all_in_one.py
# Full NanoGPT / NanoMamba stack with Tokenizer, MoE, Reward Model (pairwise),
# GRPO, and a runnable full pipeline.
#
# Examples:
#   python unified_all_in_one.py --mode train --arch gpt_bpe --use_moe --routing_mode specialize
#   python unified_all_in_one.py --mode infer --arch gpt_bpe --load_path artifacts/gpt_moe.pt
#   python unified_all_in_one.py --mode train_rm --arch gpt_bpe --rm_pairs data/preferences.jsonl
#   python unified_all_in_one.py --mode train_grpo --arch gpt_bpe --load_path artifacts/gpt_moe.pt
#
#   # With NO arguments, it runs the full educational pipeline:
#   python unified_all_in_one.py

import os, json, argparse, random, time, urllib.request
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F

# Pretty printing
try:
    from rich.console import Console
    from rich.panel import Panel
    HAS_RICH = True
except ImportError:
    HAS_RICH = False
    class Console:
        def print(self, *a, **k): print(*a)
        def rule(self, *a, **k): print("=" * 50)
    class Panel:
        def __init__(self, text, title="", style=""): self.text=text; self.title=title
        def __str__(self): return f"[{self.title}] {self.text}"

console = Console()

# =============================================================================
# TOKENIZER (simple BPE-ish)
# =============================================================================

def _get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for p in zip(ids, ids[1:]): counts[p] = counts.get(p, 0) + 1
    return counts

def _merge(ids, pair, idx):
    newids = []; i = 0
    while i < len(ids):
        if ids[i] == pair[0] and i < len(ids)-1 and ids[i+1] == pair[1]:
            newids.append(idx); i += 2
        else:
            newids.append(ids[i]); i += 1
    return newids

class Tokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = self._build_vocab()
    def _build_vocab(self):
        vocab = {i: bytes([i]) for i in range(256)}
        for (a,b),i in self.merges.items():
            vocab[i] = vocab[a] + vocab[b]
        return vocab
    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")
    def save(self, prefix):
        d = os.path.dirname(prefix)
        if d: os.makedirs(d, exist_ok=True)
        with open(prefix + ".model", "w") as f:
            for (a,b),i in self.merges.items():
                f.write(f"{a} {b}\n")
    # Replace the existing load() in Tokenizer with this:
    def load(self, path):
        merges = {}
        idx = 256
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                # accept only lines that begin with two ints
                if len(parts) >= 2:
                    try:
                        a = int(parts[0]); b = int(parts[1])
                    except ValueError:
                        continue  # skip headers like "minbpe v1", patterns, etc.
                    merges[(a, b)] = idx
                    idx += 1
        self.merges = merges
        self.vocab = self._build_vocab()


class BasicTokenizer(Tokenizer):
    def train(self, text, vocab_size):
        assert vocab_size >= 256
        ids = list(text.encode("utf-8"))
        merges = {}
        vocab = {i: bytes([i]) for i in range(256)}
        for i in range(vocab_size - 256):
            stats = _get_stats(ids)
            if not stats: break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = _merge(ids, pair, idx)
            merges[pair] = idx
            vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        self.merges = merges
        self.vocab = vocab
    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = _get_stats(ids)
            if not stats: break
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges: break
            ids = _merge(ids, pair, self.merges[pair])
        return ids

def train_or_load_tokenizer(kind, vocab_size, text, prefix):
    model_path = prefix + ".model"
    if os.path.exists(model_path):
        tok = BasicTokenizer(); tok.load(model_path)
        vs = max(tok.vocab.keys()) + 1
        console.print(Panel(f"Loaded tokenizer: {model_path} (size={vs})", title="Tokenizer"))
        return tok, model_path, vs
    tok = BasicTokenizer(); tok.train(text, vocab_size); tok.save(prefix)
    vs = max(tok.vocab.keys()) + 1
    console.print(Panel(f"Saved tokenizer: {prefix}.model (size={vs})", title="Tokenizer"))
    return tok, model_path, vs

# =============================================================================
# DATA
# =============================================================================

def load_tiny_shakespeare(data_dir="./data"):
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, "tinyshakespeare_input.txt")
    if not os.path.exists(path):
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(url, path)
    text = open(path, "r", encoding="utf-8").read()
    split = int(0.9 * len(text))
    return text[:split], text[split:]

def build_charset(train, val):
    chars = sorted(set(train + val))
    stoi = {c:i for i,c in enumerate(chars)}
    itos = {i:c for c,i in stoi.items()}
    return stoi, itos

def encode_chars(s, stoi): return [stoi[c] for c in s]

def get_batch_tokens(ids, block_size, batch_size, device):
    ix = torch.randint(len(ids) - block_size - 1, (batch_size,))
    x = torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y = torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================================================================
# MoE BUILDING BLOCKS
# =============================================================================

class TopKRouter(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=1, temp=1.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, num_experts))
        self.k = k
        self.temp = temp
    def forward(self, x):
        probs = F.softmax(self.net(x) / self.temp, dim=-1)           # [B*T, E]
        topk_vals, topk_idx = torch.topk(probs, self.k, dim=-1)
        mask = torch.zeros_like(probs)
        mask.scatter_(dim=-1, index=topk_idx, src=torch.ones_like(topk_vals))
        sp = probs * mask
        sp = sp / (sp.sum(dim=-1, keepdim=True) + 1e-9)
        return sp, probs

def entropy_mean(p, eps=1e-9):
    p = p.clamp_min(eps)
    return -(p * p.log()).sum(-1).mean()

class MoEFFN(nn.Module):
    def __init__(self, in_dim, num_experts=4, k=1, router_hidden=128, dropout=0.1,
                 entropy_penalty=0.0, routing_mode="uniform"):
        super().__init__()
        hidden = 4 * in_dim
        self.expert_fc = nn.ModuleList([nn.Linear(in_dim, hidden) for _ in range(num_experts)])
        self.expert_proj = nn.ModuleList([nn.Linear(hidden, in_dim) for _ in range(num_experts)])
        self.router = TopKRouter(in_dim, router_hidden, num_experts, k)
        self.drop = nn.Dropout(dropout)
        self.entropy_penalty = entropy_penalty
        self.routing_mode = routing_mode
        self.num_experts = num_experts
    def forward(self, x):
        B, T, C = x.shape
        xf = x.reshape(B*T, C)
        sp, dp = self.router(xf)  # sp: sparse probs used for mixing
        out = 0.0
        for e in range(self.num_experts):
            h = F.gelu(self.expert_fc[e](xf))
            h = self.expert_proj[e](h)
            out = out + sp[:, e].unsqueeze(-1) * h
        y = self.drop(out.view(B, T, C))
        aux = torch.tensor(0.0, device=x.device)
        if self.routing_mode == "specialize":
            aux = -self.entropy_penalty * entropy_mean(dp)
        return y, aux, {"mean_routing_probs": dp.mean(0)}

# =============================================================================
# MODELS
# =============================================================================

class LayerNorm(nn.Module):
    def __init__(self, n, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n))
        self.bias = nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

@dataclass
class GPTCfg:
    block_size: int = 128
    vocab_size: int = 256
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.1
    # MoE
    use_moe: bool = False
    routing_mode: str = "uniform"  # or "specialize"
    num_experts: int = 4
    topk: int = 1
    router_hidden: int = 128
    moe_dropout: float = 0.1
    entropy_penalty: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.n_head = n_head
        self.n_embd = n_embd
        self.c_attn = nn.Linear(n_embd, 3*n_embd, bias=True)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=True)
        self.dropout = dropout
        self.block_size = block_size
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        # PyTorch SDPA
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class GPTBlock(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.ln1 = LayerNorm(cfg.n_embd, True)
        self.attn = CausalSelfAttention(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout)
        self.ln2 = LayerNorm(cfg.n_embd, True)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd)
        )
        self.moe = (MoEFFN(cfg.n_embd, cfg.num_experts, cfg.topk, cfg.router_hidden,
                           cfg.moe_dropout, cfg.entropy_penalty, cfg.routing_mode)
                    if cfg.use_moe else None)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        if self.moe is None:
            x = x + self.ffn(self.ln2(x))
            return x, torch.tensor(0.0, device=x.device), {}
        else:
            y, aux, stats = self.moe(self.ln2(x))
            return x + y, aux, stats

class NanoGPT(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([GPTBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = LayerNorm(cfg.n_embd, True)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        # Tie weights
        self.lm_head.weight = self.wte.weight
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        aux = torch.tensor(0.0, device=idx.device)
        stats = {}
        for blk in self.blocks:
            x, a, s = blk(x)
            aux = aux + a
            stats = s
        h = self.ln_f(x)
        logits = self.lm_head(h)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)), targets.view(-1)
        ) if targets is not None else None
        return logits, (loss + aux if (loss is not None) else None), stats
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _, _ = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat((idx, nxt), dim=1)
        return idx

class SimpleMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.ln = LayerNorm(d, True)
        self.ff = nn.Linear(d, d)
    def forward(self, x): return x + self.ff(self.ln(x))

class NanoMamba(nn.Module):
    def __init__(self, vocab_size=256, block_size=128, n_layer=4, n_embd=128):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)
        self.layers = nn.ModuleList([SimpleMambaBlock(n_embd) for _ in range(n_layer)])
        self.ln_f = LayerNorm(n_embd, True)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        for l in self.layers: x = l(x)
        h = self.ln_f(x)
        logits = self.head(h)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)), targets.view(-1)
        ) if targets is not None else None
        return logits, loss, {}
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _, _ = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1)
            idx = torch.cat((idx, nxt), dim=1)
        return idx

# =============================================================================
# TRAIN / GRPO / REWARD MODEL
# =============================================================================

def train_lm(model, train_ids, val_ids, block_size, epochs, steps, batch, lr, device, title):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for ep in range(1, epochs+1):
        model.train(); losses=[]; t0=time.time()
        for _ in range(steps):
            xb, yb = get_batch_tokens(train_ids, block_size, batch, device)
            _, loss, _ = model(xb, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        model.eval()
        xb, yb = get_batch_tokens(val_ids, block_size, batch, device)
        _, vl, _ = model(xb, yb)
        console.print(Panel(f"Epoch {ep}/{epochs} train={sum(losses)/len(losses):.3f} "
                            f"val={vl.item():.3f} time={time.time()-t0:.1f}s", title=title))

def grpo_step(model, encode_fn, prompts, gen_tokens, opt, device):
    model.train(); rewards=[]; logps=[]
    for s in prompts:
        x = torch.tensor([encode_fn(s)], device=device)
        y = model.generate(x, gen_tokens)[0]  # [seq]
        inp, tgt = y[:-1].unsqueeze(0), y[1:].unsqueeze(0)
        logits, _, _ = model(inp, tgt)
        logp = F.log_softmax(logits, dim=-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1).mean()
        rewards.append(logp); logps.append(logp)
    rewards = torch.stack(rewards)
    logps = torch.stack(logps)
    adv = rewards - rewards.mean()
    loss = -(adv.detach() * logps).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    return float(loss.item()), float(rewards.mean().item())

class RewardModel(nn.Module):
    """
    Simple pairwise reward model on top of base LM.
    Uses last-token logits as features (dim = vocab_size) -> scalar reward.
    Works for both GPT and Mamba as long as base returns logits.
    """
    def __init__(self, base):
        super().__init__()
        self.base = base
        vocab_size = base.cfg.vocab_size if hasattr(base, "cfg") else base.vocab_size
        self.head = nn.Linear(vocab_size, 1)
    def forward(self, idx):
        logits, _, _ = self.base(idx)       # [B,T,V]
        h = logits[:, -1, :]                # last token's logits as features
        return self.head(h)                 # [B,1]

def load_pairs(path): return [json.loads(l) for l in open(path, "r", encoding="utf-8")]

def train_reward_model(rm, tok, pairs_path, block_size, epochs, steps, batch, lr, device):
    rm.to(device)
    pairs = load_pairs(pairs_path)
    opt = torch.optim.AdamW(rm.parameters(), lr=lr)
    for ep in range(1, epochs+1):
        random.shuffle(pairs); losses=[]
        for _ in range(steps):
            b = random.sample(pairs, min(batch, len(pairs)))
            chosen = [tok.encode(p["chosen"]) for p in b]
            rejected = [tok.encode(p["rejected"]) for p in b]
            cl = [torch.tensor(c[:block_size]) for c in chosen]
            rl = [torch.tensor(r[:block_size]) for r in rejected]
            cl = torch.nn.utils.rnn.pad_sequence(cl, batch_first=True).to(device)
            rl = torch.nn.utils.rnn.pad_sequence(rl, batch_first=True).to(device)
            rc = rm(cl); rr = rm(rl)
            loss = -F.logsigmoid(rc - rr).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
        console.print(Panel(f"RM Epoch {ep}/{epochs} loss={sum(losses)/len(losses):.3f}", title="Reward Model"))

# =============================================================================
# MAIN (CLI)
# =============================================================================

def main(argv=None):
    if argv is None: argv = []

    # Training defaults (can be overridden by env vars or CLI)
    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 1000))
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 100))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 64))

    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", required=True, choices=["train", "infer", "train_rm", "train_grpo"])
    ap.add_argument("--arch", required=True, choices=["gpt_bpe", "mamba_char"])
    ap.add_argument("--epochs", type=int, default=DEFAULT_EPOCHS)
    ap.add_argument("--steps_per_epoch", type=int, default=DEFAULT_STEPS_PER_EPOCH)
    ap.add_argument("--batch_size", type=int, default=DEFAULT_BATCH_SIZE)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--block_size", type=int, default=128)
    ap.add_argument("--n_layer", type=int, default=4)
    ap.add_argument("--n_head", type=int, default=4)
    ap.add_argument("--n_embd", type=int, default=128)
    ap.add_argument("--use_moe", action="store_true")
    ap.add_argument("--routing_mode", default="uniform", choices=["uniform", "specialize"])
    ap.add_argument("--num_experts", type=int, default=4)
    ap.add_argument("--topk", type=int, default=1)
    ap.add_argument("--router_hidden", type=int, default=128)
    ap.add_argument("--entropy_penalty", type=float, default=0.0)
    ap.add_argument("--moe_dropout", type=float, default=0.1)
    ap.add_argument("--tok_kind", default="basic")
    ap.add_argument("--vocab_size", type=int, default=512)
    ap.add_argument("--tok_prefix", default="artifacts/tokenizers/tiny_bpe")
    ap.add_argument("--save_path", default="artifacts/unified.pt")
    ap.add_argument("--load_path", default="artifacts/unified.pt")
    ap.add_argument("--sample_start", default="\n")
    ap.add_argument("--sample_tokens", type=int, default=100)
    ap.add_argument("--rm_pairs", default="")
    args = ap.parse_args(argv)

    os.makedirs("artifacts", exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    console.print(Panel(f"Using device: {device}", title="Device"))

    train_txt, val_txt = load_tiny_shakespeare("./data")

    # Tokenizer setup
    if args.arch == "gpt_bpe":
        tok, _, vs = train_or_load_tokenizer(args.tok_kind, args.vocab_size,
                                             train_txt + val_txt, args.tok_prefix)
        train_ids, val_ids = tok.encode(train_txt), tok.encode(val_txt)
        encode_fn, decode_fn = tok.encode, tok.decode
    else:
        stoi, itos = build_charset(train_txt, val_txt)
        train_ids, val_ids = encode_chars(train_txt, stoi), encode_chars(val_txt, stoi)
        encode_fn = lambda s: encode_chars(s, stoi)
        decode_fn = lambda ids: "".join(itos[i] for i in ids)

    # Load checkpoint if available
    model = None
    if args.mode != "train" and os.path.exists(args.load_path):
        payload = torch.load(args.load_path, map_location=device)
        kind = payload.get("kind", args.arch)
        cfg_dict = payload.get("cfg", {})

        if kind == "gpt_bpe":
            cfg = GPTCfg(**cfg_dict)
            model = NanoGPT(cfg)
        elif kind == "mamba_char":
            vocab_size = cfg_dict.get("vocab_size", len(stoi) if "stoi" in locals() else 256)
            model = NanoMamba(vocab_size,
                              cfg_dict.get("block_size", args.block_size),
                              cfg_dict.get("n_layer", args.n_layer),
                              cfg_dict.get("n_embd", args.n_embd))
        else:
            raise ValueError(f"Unknown checkpoint kind: {kind}")

        model.load_state_dict(payload["state_dict"])
    else:
        if args.arch == "gpt_bpe":
            cfg = GPTCfg(
                block_size=args.block_size, vocab_size=vs,
                n_layer=args.n_layer, n_head=args.n_head, n_embd=args.n_embd,
                dropout=0.1,
                use_moe=args.use_moe, routing_mode=args.routing_mode,
                num_experts=args.num_experts, topk=args.topk,
                router_hidden=args.router_hidden, moe_dropout=args.moe_dropout,
                entropy_penalty=args.entropy_penalty
            )
            model = NanoGPT(cfg)
        else:
            model = NanoMamba(len(stoi), args.block_size, args.n_layer, args.n_embd)

    # Run mode
    if args.mode == "train":
        train_lm(model, train_ids, val_ids, args.block_size, args.epochs,
                 args.steps_per_epoch, args.batch_size, args.lr, device,
                 title=f"{args.arch.upper()} Train")
        cfg_to_save = asdict(model.cfg) if hasattr(model, "cfg") else {
            "vocab_size": getattr(model, "vocab_size", None),
            "block_size": args.block_size,
            "n_layer": args.n_layer,
            "n_embd": args.n_embd
        }
        torch.save({"kind": args.arch, "cfg": cfg_to_save,
                    "state_dict": model.state_dict()}, args.save_path)
        console.print(Panel(f"Model saved to {args.save_path}", title="Save"))

    elif args.mode == "infer":
        payload = torch.load(args.load_path, map_location=device)
        model.load_state_dict(payload["state_dict"]); model.to(device)
        start_ids = encode_fn(args.sample_start)
        idx = torch.tensor([start_ids], device=device)
        sample = model.generate(idx, args.sample_tokens)[0].tolist()
        console.print(Panel(decode_fn(sample), title="Generated Text"))

    elif args.mode == "train_grpo":
        model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
        prompts = ["To be, or not to be", "Once upon a time", "The quick brown fox"]
        for step in range(args.steps_per_epoch):
            loss, mr = grpo_step(model, encode_fn, prompts, 32, opt, device)
            console.print(Panel(f"GRPO step {step+1}/{args.steps_per_epoch} "
                                f"loss={loss:.3f} reward={mr:.3f}", title="GRPO"))
        if args.save_path:
            cfg_to_save = asdict(model.cfg) if hasattr(model, "cfg") else {}
            torch.save({"kind": args.arch, "cfg": cfg_to_save,
                        "state_dict": model.state_dict()}, args.save_path)
            console.print(Panel(f"GRPO-tuned model saved to {args.save_path}", title="Save"))

    elif args.mode == "train_rm":
        if not args.rm_pairs:
            raise ValueError("--rm_pairs is required for train_rm")
        model.to(device)
        rm = RewardModel(model)
        tok_like = tok if args.arch == "gpt_bpe" else type("TK", (), {"encode": encode_fn})()
        train_reward_model(rm, tok_like, args.rm_pairs, args.block_size,
                           epochs=args.epochs, steps=args.steps_per_epoch,
                           batch=args.batch_size, lr=1e-5, device=device)
        torch.save({"kind": "reward_model", "base_arch": args.arch,
                    "state_dict": rm.state_dict()}, args.save_path)
        console.print(Panel(f"Reward model saved to {args.save_path}", title="Save"))

# =============================================================================
# FULL PIPELINE
# =============================================================================

def run_full_pipeline():
    """Run the complete end-to-end pipeline with defaults (Colab/terminal)."""
    console.rule("[bold magenta]Unified End-to-End Pipeline")

    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 1000))
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 100))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 64))

    # 1) Train base GPT-BPE with MoE
    main([
        "--mode","train",
        "--arch","gpt_bpe",
        "--tok_kind","basic",
        "--vocab_size","512",
        "--use_moe","--routing_mode","specialize",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--save_path","artifacts/gpt_moe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe"
    ])

    # 2) Reward Model
    rm_file = "data/preferences.jsonl"
    if not os.path.exists(rm_file):
        toy_data = [
            {"chosen":"I love this model","rejected":"I hate this model"},
            {"chosen":"This is a great answer","rejected":"This is a terrible answer"},
        ]
        os.makedirs("data", exist_ok=True)
        with open(rm_file,"w",encoding="utf-8") as f:
            for row in toy_data: f.write(json.dumps(row) + "\n")
    main([
        "--mode","train_rm","--arch","gpt_bpe",
        "--rm_pairs",rm_file,
        "--load_path","artifacts/gpt_moe.pt",
        "--save_path","artifacts/reward_model.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
    ])

    # 3) GRPO
    main([
        "--mode","train_grpo","--arch","gpt_bpe",
        "--load_path","artifacts/gpt_moe.pt",
        "--save_path","artifacts/gpt_grpo.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
    ])

    # 4) Inference (base GPT-MoE)
    main([
        "--mode","infer","--arch","gpt_bpe",
        "--load_path","artifacts/gpt_moe.pt",
        "--sample_start","Hello world,",
        "--sample_tokens","50",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe"
    ])

    # 5) Train small Mamba
    main([
        "--mode","train","--arch","mamba_char",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH//2),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--save_path","artifacts/mamba_model.pt"
    ])

    # 6) Mamba inference
    main([
        "--mode","infer","--arch","mamba_char",
        "--load_path","artifacts/mamba_model.pt",
        "--sample_start","ROMEO:",
        "--sample_tokens","80"
    ])

# =============================================================================
# ENTRYPOINT
# =============================================================================

# if __name__ == "__main__":
#     import sys
#     if len(sys.argv) == 1:
#         run_full_pipeline()
#     else:
#         main(sys.argv[1:])
# --- Colab/Jupyter-friendly entrypoint ---
# def _sanitize_argv(argv):
#     """Remove Jupyter/Colab noise like -f <connection.json>."""
#     out = []
#     skip_next = False
#     for i, a in enumerate(argv):
#         if skip_next:
#             skip_next = False
#             continue
#         if a in ("-f", "--f"):
#             skip_next = True  # skip the following filename
#             continue
#         if a.startswith("-f="):
#             continue
#         out.append(a)
#     return out

# =============================================================================
# ENTRYPOINT
# =============================================================================

def _sanitize_argv(argv):
    """Remove Jupyter/Colab/Kaggle noise like -f <connection.json>."""
    out = []
    skip_next = False
    for a in argv:
        if skip_next:
            skip_next = False
            continue
        if a in ("-f", "--f"):
            skip_next = True  # skip the following filename
            continue
        if a.startswith("-f=") or a.startswith("--f="):
            continue
        out.append(a)
    return out

if __name__ == "__main__":
    import sys
    argv = _sanitize_argv(sys.argv[1:])

    # No arguments → run the full pipeline
    if ("--mode" not in argv) or ("--arch" not in argv):
        run_full_pipeline()
    else:
        main(argv)



## Full NanoGPT / NanoMamba stack with Tokenizer, MoE, Reward Model (pairwise), GRPO, and a runnable full pipeline — now with a comprehensive TrainingMonitor, that organizes plots, logs, models, and bundles everything into a ZIP.



In [ ]:
#!/usr/bin/env python3
# unified_all_in_one_with_monitor.py
# Full NanoGPT / NanoMamba stack with Tokenizer, MoE, Reward Model (pairwise),
# GRPO, and a runnable full pipeline — now with a comprehensive TrainingMonitor
# that organizes plots, logs, models, and bundles everything into a ZIP.
#
# Quick examples:
#   python unified_all_in_one_with_monitor.py --mode train --arch gpt_bpe --use_moe --routing_mode specialize
#   python unified_all_in_one_with_monitor.py --mode infer --arch gpt_bpe --load_path artifacts/gpt_moe.pt
#   python unified_all_in_one_with_monitor.py --mode train_rm --arch gpt_bpe --rm_pairs data/preferences.jsonl
#   python unified_all_in_one_with_monitor.py --mode train_grpo --arch gpt_bpe --load_path artifacts/gpt_moe.pt
#
#   # With NO arguments, it runs the full educational pipeline and creates a ZIP:
#   python unified_all_in_one_with_monitor.py

import os, json, argparse, random, time, urllib.request, zipfile, shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

# Optional rich console
try:
    from rich.console import Console
    from rich.panel import Panel
    HAS_RICH = True
except ImportError:
    HAS_RICH = False
    class Console:
        def print(self, *a, **k): print(*a)
        def rule(self, *a, **k): print("=" * 50)
    class Panel:
        def __init__(self, text, title="", style=""): self.text=text; self.title=title
        def __str__(self): return f"[{self.title}] {self.text}"

console = Console()

# -----------------------------------------------------------------------------
# Plotting (matplotlib only; seaborn optional)
# -----------------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")  # non-interactive backends (servers/Colab safe)
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec

try:
    import seaborn as sns
    sns.set_palette("husl")
except Exception:
    pass

plt.style.use('default')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

# =============================================================================
# TRAINING MONITOR (plots, logs, animations, ZIP packaging)
# =============================================================================

class TrainingMonitor:
    """Comprehensive training monitor that saves everything organized."""
    def __init__(self, experiment_name="unified_experiment"):
        self.experiment_name = experiment_name
        self.base_dir = Path("training_outputs")
        self._setup_folders()
        # Tracking
        self.metrics = defaultdict(list)
        self.attention_snapshots = []
        self.embedding_snapshots = []
        self.moe_snapshots = []
        self.step_count = 0
        self.epoch_count = 0

    # ---------------------- FS ----------------------
    def _setup_folders(self):
        folders = [
            self.base_dir,
            self.base_dir / "models",
            self.base_dir / "plots" / "training_curves",
            self.base_dir / "plots" / "attention_analysis",
            self.base_dir / "plots" / "embedding_evolution",
            self.base_dir / "plots" / "moe_analysis",
            self.base_dir / "plots" / "animations",
            self.base_dir / "tokenizers",
            self.base_dir / "logs",
            self.base_dir / "data",
        ]
        for f in folders: f.mkdir(parents=True, exist_ok=True)
        console.print(f"📁 Output structure at: {self.base_dir}")

    # ---------------------- Logging ----------------------
    def log_training_step(self, epoch, step, train_loss, val_loss=None, lr=None, grad_norm=None):
        self.metrics['epoch'].append(epoch)
        self.metrics['step'].append(self.step_count)
        self.metrics['train_loss'].append(float(train_loss))
        if val_loss is not None: self.metrics['val_loss'].append(float(val_loss))
        if lr is not None: self.metrics['learning_rate'].append(float(lr))
        if grad_norm is not None: self.metrics['grad_norm'].append(float(grad_norm))
        self.step_count += 1

    def capture_model_state(self, model):
        # Attention Q/K alignment (if present)
        if hasattr(model, 'blocks'):
            attn_data = []
            for i, block in enumerate(model.blocks):
                if hasattr(block.attn, 'c_attn'):
                    with torch.no_grad():
                        w = block.attn.c_attn.weight.data.detach().cpu().numpy()
                    n_embd = w.shape[1]
                    q_w = w[:n_embd, :]
                    k_w = w[n_embd:2*n_embd, :]
                    corr = np.corrcoef(q_w.flatten(), k_w.flatten())[0, 1]
                    attn_data.append({
                        'layer': i, 'qk_correlation': float(corr)
                    })
            if attn_data:
                self.attention_snapshots.append({
                    'step': self.step_count, 'epoch': self.epoch_count, 'data': attn_data
                })
        # Embedding snapshots
        if hasattr(model, 'wte'):
            with torch.no_grad():
                emb = model.wte.weight.data.detach().cpu().numpy()
            self.embedding_snapshots.append({
                'step': self.step_count,
                'epoch': self.epoch_count,
                'norm': float(np.linalg.norm(emb)),
                'mean': float(np.mean(emb)),
                'std': float(np.std(emb)),
                'weights': emb.copy(),
            })

    def capture_moe_stats(self, stats):
        if stats and 'mean_routing_probs' in stats and stats['mean_routing_probs'] is not None:
            probs = stats['mean_routing_probs']
            if isinstance(probs, torch.Tensor):
                probs = probs.detach().cpu().numpy()
            probs = np.asarray(probs)
            entropy = float(-(probs * np.log(probs + 1e-8)).sum())
            balance = float(1.0 - (np.std(probs) / (np.mean(probs) + 1e-8)))
            self.moe_snapshots.append({
                'step': self.step_count,
                'epoch': self.epoch_count,
                'routing_probs': probs.tolist(),
                'entropy': entropy,
                'balance': balance,
            })

    # ---------------------- Plots ----------------------
    def _plot_training_curves(self):
        if not self.metrics['train_loss']:
            return
        fig = plt.figure(figsize=(20, 12))
        gs = GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)
        epochs = np.array(self.metrics['epoch'])
        train_losses = np.array(self.metrics['train_loss'])

        ax1 = fig.add_subplot(gs[0, :2])
        ax1.plot(epochs, train_losses, linewidth=2, alpha=0.9, label='Training Loss')
        if self.metrics.get('val_loss'):
            vl = np.array(self.metrics['val_loss'])
            ax1.plot(epochs[-len(vl):], vl, linestyle='--', linewidth=2, alpha=0.9, label='Validation Loss')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Progress'); ax1.legend(); ax1.grid(True, alpha=0.3)

        if self.metrics.get('learning_rate'):
            ax2 = fig.add_subplot(gs[0, 2])
            lrs = np.array(self.metrics['learning_rate'])
            ax2.plot(epochs[-len(lrs):], lrs, linewidth=2)
            ax2.set_xlabel('Epoch'); ax2.set_ylabel('LR'); ax2.set_title('LR Schedule'); ax2.set_yscale('log'); ax2.grid(True, alpha=0.3)

        if self.metrics.get('grad_norm'):
            ax3 = fig.add_subplot(gs[1, 0])
            gns = np.array(self.metrics['grad_norm'])
            ax3.plot(epochs[-len(gns):], gns, linewidth=2, alpha=0.9)
            ax3.set_xlabel('Epoch'); ax3.set_ylabel('Grad Norm'); ax3.set_title('Gradient Norms'); ax3.grid(True, alpha=0.3)

        ax4 = fig.add_subplot(gs[1, 1])
        if len(train_losses) > 4:
            w = max(3, min(20, len(train_losses)//10))
            sm = np.convolve(train_losses, np.ones(w)/w, mode='valid')
            ax4.plot(epochs[w-1:], sm, linewidth=3, alpha=0.9)
            ax4.set_xlabel('Epoch'); ax4.set_ylabel('Smoothed Loss'); ax4.set_title(f'Loss (Moving Avg, window={w})'); ax4.grid(True, alpha=0.3)
        else:
            ax4.text(0.5,0.5,'Insufficient steps for smoothing',ha='center',va='center'); ax4.axis('off')

        ax5 = fig.add_subplot(gs[1, 2])
        final_loss = float(train_losses[-1]) if len(train_losses) else 0.0
        min_loss = float(np.min(train_losses)) if len(train_losses) else 0.0
        improvement = float(train_losses[0] - final_loss) if len(train_losses) > 1 else 0.0
        stats_text = f'Final Loss: {final_loss:.4f}\nBest Loss: {min_loss:.4f}\nImprovement: {improvement:.4f}\nTotal Steps: {len(train_losses)}'
        ax5.text(0.1, 0.5, stats_text, transform=ax5.transAxes, fontsize=12, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        ax5.axis('off'); ax5.set_title('Training Stats')

        out = self.base_dir / "plots" / "training_curves" / "comprehensive_training.png"
        plt.suptitle(f'{self.experiment_name} - Training Analysis', fontsize=16)
        plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"📊 Training curves saved to {out}")

    def _create_loss_animation(self):
        if not self.metrics['train_loss']:
            return
        fig, ax = plt.subplots(figsize=(12, 8))
        train_losses = np.array(self.metrics['train_loss'])
        epochs = np.array(self.metrics['epoch'])
        def animate(i):
            ax.clear(); ax.plot(epochs[:i+1], train_losses[:i+1], linewidth=3, alpha=0.9)
            ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title(f'Training Progress - Step {i+1}/{len(train_losses)}'); ax.grid(True, alpha=0.3)
            ax.set_xlim(0, max(epochs) if len(epochs) else 1)
            if len(train_losses):
                ax.set_ylim(min(train_losses)*0.9, max(train_losses)*1.1)
        frames = min(len(train_losses), 100)
        anim = animation.FuncAnimation(fig, animate, frames=frames, interval=150, repeat=True)
        out = self.base_dir / "plots" / "animations" / "loss_evolution.gif"
        anim.save(out, writer='pillow', fps=5); plt.close()
        console.print(f"🎬 Loss animation saved to {out}")

    def _plot_attention(self):
        if not self.attention_snapshots:
            return
        steps = [s['step'] for s in self.attention_snapshots]
        n_layers = len(self.attention_snapshots[0]['data'])
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        # layer curves
        for layer in range(n_layers):
            corrs = []
            for s in self.attention_snapshots:
                d = s['data'][layer]
                corrs.append(d['qk_correlation'])
            ax1.plot(steps[:len(corrs)], corrs, linewidth=2, alpha=0.9, label=f'L{layer}')
        ax1.set_xlabel('Step'); ax1.set_ylabel('Q-K Corr'); ax1.set_title('Query-Key Alignment Evolution'); ax1.legend(); ax1.grid(True, alpha=0.3)

        latest = self.attention_snapshots[-1]['data']
        cm = np.array([[d['qk_correlation'] for d in latest]])
        im = ax2.imshow(cm, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
        ax2.set_title('Latest Q-K Correlations'); fig.colorbar(im, ax=ax2)

        # Distribution for layer 0 not available without raw weights — show histogram of correlations instead
        all_corrs = []
        for s in self.attention_snapshots:
            all_corrs.extend([d['qk_correlation'] for d in s['data']])
        ax3.hist(all_corrs, bins=40, alpha=0.85)
        ax3.set_title('Distribution of Q-K Correlations'); ax3.set_xlabel('Corr'); ax3.set_ylabel('Freq'); ax3.grid(True, alpha=0.3)

        # Stability (std) per layer
        stdevs = []
        for layer in range(n_layers):
            layer_series = [s['data'][layer]['qk_correlation'] for s in self.attention_snapshots]
            stdevs.append(np.std(layer_series))
        ax4.bar(range(n_layers), stdevs, alpha=0.9)
        ax4.set_xlabel('Layer'); ax4.set_ylabel('Std Dev'); ax4.set_title('Q-K Correlation Stability'); ax4.grid(True, alpha=0.3)

        out = self.base_dir / "plots" / "attention_analysis" / "attention_comprehensive.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"🔍 Attention analysis saved to {out}")

    def _plot_embeddings(self):
        if not self.embedding_snapshots:
            return
        steps = [s['step'] for s in self.embedding_snapshots]
        norms = [s['norm'] for s in self.embedding_snapshots]
        means = [s['mean'] for s in self.embedding_snapshots]
        stds  = [s['std']  for s in self.embedding_snapshots]
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        ax1.plot(steps, norms, linewidth=2); ax1.set_title('Embedding Frobenius Norm'); ax1.set_xlabel('Step'); ax1.set_ylabel('Norm'); ax1.grid(True, alpha=0.3)
        ax2.plot(steps, means, linewidth=2, label='Mean'); ax2.plot(steps, stds, linewidth=2, label='Std'); ax2.legend(); ax2.set_title('Embedding Stats'); ax2.grid(True, alpha=0.3)

        # Latest distribution if we kept weights
        w = self.embedding_snapshots[-1].get('weights', None)
        if w is not None:
            ax3.hist(w.flatten(), bins=60, alpha=0.85)
            ax3.set_title('Latest Embedding Distribution'); ax3.grid(True, alpha=0.3)
            token_norms = np.linalg.norm(w, axis=1)
            ax4.plot(token_norms, alpha=0.9)
            ax4.set_title('Per-Token L2 Norms'); ax4.set_xlabel('Token'); ax4.set_ylabel('L2'); ax4.grid(True, alpha=0.3)
        else:
            ax3.axis('off'); ax4.axis('off')

        out = self.base_dir / "plots" / "embedding_evolution" / "embedding_analysis.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"📐 Embedding analysis saved to {out}")

    def _plot_moe(self):
        if not self.moe_snapshots:
            return
        steps = [s['step'] for s in self.moe_snapshots]
        nE = len(self.moe_snapshots[0]['routing_probs'])
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        for e in range(nE):
            usage = [s['routing_probs'][e] for s in self.moe_snapshots]
            ax1.plot(steps, usage, linewidth=2, alpha=0.9, label=f'E{e}')
        ax1.set_title('Expert Usage Evolution'); ax1.set_xlabel('Step'); ax1.set_ylabel('Prob'); ax1.legend(); ax1.grid(True, alpha=0.3)

        ent = [s['entropy'] for s in self.moe_snapshots]
        ax2.plot(steps, ent, linewidth=2, alpha=0.9)
        ax2.set_title('Routing Entropy'); ax2.set_xlabel('Step'); ax2.grid(True, alpha=0.3)

        latest = self.moe_snapshots[-1]['routing_probs']
        ax3.pie(latest, labels=[f'E{i}' for i in range(len(latest))], autopct='%1.1f%%', startangle=90)
        ax3.set_title('Current Expert Distribution')

        bal = [s['balance'] for s in self.moe_snapshots]
        ax4.plot(steps, bal, linewidth=2, alpha=0.9)
        ax4.set_title('Load Balance (1=best)'); ax4.set_xlabel('Step'); ax4.grid(True, alpha=0.3)

        out = self.base_dir / "plots" / "moe_analysis" / "moe_comprehensive.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"🔀 MoE analysis saved to {out}")

    # ---------------------- Persist ----------------------
    def _save_logs(self):
        # Metrics
        with open(self.base_dir / "logs" / "training_metrics.json", 'w') as f:
            json.dump({k: list(v) for k, v in self.metrics.items()}, f, indent=2)
        # Attention
        if self.attention_snapshots:
            with open(self.base_dir / "logs" / "attention_evolution.json", 'w') as f:
                json.dump(self.attention_snapshots, f, indent=2)
        # Embedding (strip weights for logs to keep size small)
        if self.embedding_snapshots:
            light = [{k: v for k, v in s.items() if k != 'weights'} for s in self.embedding_snapshots]
            with open(self.base_dir / "logs" / "embedding_evolution.json", 'w') as f:
                json.dump(light, f, indent=2)
        # MoE
        if self.moe_snapshots:
            with open(self.base_dir / "logs" / "moe_evolution.json", 'w') as f:
                json.dump(self.moe_snapshots, f, indent=2)
        console.print(f"💾 Logs saved to {self.base_dir / 'logs'}")

    def generate_all(self):
        console.print("🎨 Generating visualizations and logs...")
        self._plot_training_curves()
        self._create_loss_animation()
        self._plot_attention()
        self._plot_embeddings()
        self._plot_moe()
        self._save_logs()
        console.print("✅ Visualizations complete")

    def snapshot_tokenizer(self, src_prefix: str):
        # copy tokenizer files into outputs for convenience
        m = Path(src_prefix + ".model")
        if m.exists():
            dst = self.base_dir / "tokenizers" / m.name
            try:
                shutil.copy2(m, dst)
            except Exception:
                pass

    def zip_everything(self):
        # Create summary
        summary = {
            "experiment": self.experiment_name,
            "timestamp": time.strftime("%Y-%m-%d_%H-%M-%S"),
            "total_steps": int(self.step_count),
            "total_epochs": int(self.epoch_count),
            "final_loss": self.metrics['train_loss'][-1] if self.metrics['train_loss'] else None,
        }
        with open(self.base_dir / "experiment_summary.json", 'w') as f:
            json.dump(summary, f, indent=2)

        # README
        readme = f"""# {self.experiment_name} - Results\n\nThis folder contains models, plots, logs, tokenizers, and data artifacts.\n\n- Plots in `plots/` (training curves, attention, embeddings, MoE, animations)\n- Logs in `logs/` (JSON)\n- Models in `models/`\n- Tokenizers in `tokenizers/`\n- Data in `data/`\n"""
        with open(self.base_dir / "README.md", 'w') as f:
            f.write(readme)

        # Zip
        ts = time.strftime("%Y%m%d_%H%M%S")
        zip_name = f"unified_results_{ts}.zip"
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
            for p in self.base_dir.rglob('*'):
                if p.is_file(): z.write(p, p.relative_to('.'))
        size_mb = os.path.getsize(zip_name) / (1024*1024)
        console.print(Panel(f"Created ZIP: {zip_name} ({size_mb:.1f} MB)", title="Package"))
        return zip_name

# =============================================================================
# TOKENIZER (simple BPE-ish)
# =============================================================================

def _get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for p in zip(ids, ids[1:]): counts[p] = counts.get(p, 0) + 1
    return counts

def _merge(ids, pair, idx):
    newids = []; i = 0
    while i < len(ids):
        if ids[i] == pair[0] and i < len(ids)-1 and ids[i+1] == pair[1]:
            newids.append(idx); i += 2
        else:
            newids.append(ids[i]); i += 1
    return newids

class Tokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = self._build_vocab()
    def _build_vocab(self):
        vocab = {i: bytes([i]) for i in range(256)}
        for (a,b),i in self.merges.items():
            vocab[i] = vocab[a] + vocab[b]
        return vocab
    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")
    def save(self, prefix):
        d = os.path.dirname(prefix)
        if d: os.makedirs(d, exist_ok=True)
        with open(prefix + ".model", "w") as f:
            for (a,b),i in self.merges.items():
                f.write(f"{a} {b}\n")
    def load(self, path):
        merges = {}; idx = 256
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    try:
                        a = int(parts[0]); b = int(parts[1])
                    except ValueError:
                        continue
                    merges[(a, b)] = idx; idx += 1
        self.merges = merges
        self.vocab = self._build_vocab()

class BasicTokenizer(Tokenizer):
    def train(self, text, vocab_size):
        assert vocab_size >= 256
        ids = list(text.encode("utf-8"))
        merges = {}; vocab = {i: bytes([i]) for i in range(256)}
        for i in range(vocab_size - 256):
            stats = _get_stats(ids)
            if not stats: break
            pair = max(stats, key=stats.get); idx = 256 + i
            ids = _merge(ids, pair, idx)
            merges[pair] = idx; vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        self.merges = merges; self.vocab = vocab
    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = _get_stats(ids)
            if not stats: break
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges: break
            ids = _merge(ids, pair, self.merges[pair])
        return ids

def train_or_load_tokenizer(kind, vocab_size, text, prefix):
    model_path = prefix + ".model"
    if os.path.exists(model_path):
        tok = BasicTokenizer(); tok.load(model_path)
        vs = max(tok.vocab.keys()) + 1
        console.print(Panel(f"Loaded tokenizer: {model_path} (size={vs})", title="Tokenizer"))
        return tok, model_path, vs
    tok = BasicTokenizer(); tok.train(text, vocab_size); tok.save(prefix)
    vs = max(tok.vocab.keys()) + 1
    console.print(Panel(f"Saved tokenizer: {prefix}.model (size={vs})", title="Tokenizer"))
    return tok, model_path, vs

# =============================================================================
# DATA
# =============================================================================

def load_tiny_shakespeare(data_dir="./data"):
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, "tinyshakespeare_input.txt")
    if not os.path.exists(path):
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(url, path)
    text = open(path, "r", encoding="utf-8").read()
    split = int(0.9 * len(text))
    return text[:split], text[split:]

def build_charset(train, val):
    chars = sorted(set(train + val))
    stoi = {c:i for i,c in enumerate(chars)}
    itos = {i:c for c,i in stoi.items()}
    return stoi, itos

def encode_chars(s, stoi): return [stoi[c] for c in s]

def get_batch_tokens(ids, block_size, batch_size, device):
    ix = torch.randint(len(ids) - block_size - 1, (batch_size,))
    x = torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y = torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================================================================
# MoE BUILDING BLOCKS
# =============================================================================

class TopKRouter(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=1, temp=1.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, num_experts))
        self.k = k; self.temp = temp
    def forward(self, x):
        probs = F.softmax(self.net(x) / self.temp, dim=-1)
        topk_vals, topk_idx = torch.topk(probs, self.k, dim=-1)
        mask = torch.zeros_like(probs)
        mask.scatter_(dim=-1, index=topk_idx, src=torch.ones_like(topk_vals))
        sp = probs * mask
        sp = sp / (sp.sum(dim=-1, keepdim=True) + 1e-9)
        return sp, probs

def entropy_mean(p, eps=1e-9):
    p = p.clamp_min(eps)
    return -(p * p.log()).sum(-1).mean()

class MoEFFN(nn.Module):
    def __init__(self, in_dim, num_experts=4, k=1, router_hidden=128, dropout=0.1,
                 entropy_penalty=0.0, routing_mode="uniform"):
        super().__init__()
        hidden = 4 * in_dim
        self.expert_fc = nn.ModuleList([nn.Linear(in_dim, hidden) for _ in range(num_experts)])
        self.expert_proj = nn.ModuleList([nn.Linear(hidden, in_dim) for _ in range(num_experts)])
        self.router = TopKRouter(in_dim, router_hidden, num_experts, k)
        self.drop = nn.Dropout(dropout)
        self.entropy_penalty = entropy_penalty
        self.routing_mode = routing_mode
        self.num_experts = num_experts
    def forward(self, x):
        B, T, C = x.shape; xf = x.reshape(B*T, C)
        sp, dp = self.router(xf); out = 0.0
        for e in range(self.num_experts):
            h = F.gelu(self.expert_fc[e](xf))
            h = self.expert_proj[e](h)
            out = out + sp[:, e].unsqueeze(-1) * h
        y = self.drop(out.view(B, T, C))
        aux = torch.tensor(0.0, device=x.device)
        if self.routing_mode == "specialize":
            aux = -self.entropy_penalty * entropy_mean(dp)
        return y, aux, {"mean_routing_probs": dp.mean(0)}

# =============================================================================
# MODELS
# =============================================================================

class LayerNorm(nn.Module):
    def __init__(self, n, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n))
        self.bias = nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

@dataclass
class GPTCfg:
    block_size: int = 128
    vocab_size: int = 256
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.1
    # MoE
    use_moe: bool = False
    routing_mode: str = "uniform"  # or "specialize"
    num_experts: int = 4
    topk: int = 1
    router_hidden: int = 128
    moe_dropout: float = 0.1
    entropy_penalty: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.n_head = n_head; self.n_embd = n_embd
        self.c_attn = nn.Linear(n_embd, 3*n_embd, bias=True)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=True)
        self.dropout = dropout; self.block_size = block_size
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class GPTBlock(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.ln1 = LayerNorm(cfg.n_embd, True)
        self.attn = CausalSelfAttention(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout)
        self.ln2 = LayerNorm(cfg.n_embd, True)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd), nn.GELU(), nn.Linear(4 * cfg.n_embd, cfg.n_embd)
        )
        self.moe = (MoEFFN(cfg.n_embd, cfg.num_experts, cfg.topk, cfg.router_hidden,
                           cfg.moe_dropout, cfg.entropy_penalty, cfg.routing_mode)
                    if cfg.use_moe else None)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        if self.moe is None:
            x = x + self.ffn(self.ln2(x))
            return x, torch.tensor(0.0, device=x.device), {}
        else:
            y, aux, stats = self.moe(self.ln2(x))
            return x + y, aux, stats

class NanoGPT(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([GPTBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = LayerNorm(cfg.n_embd, True)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
    def forward(self, idx, targets=None):
        B, T = idx.shape; pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        aux = torch.tensor(0.0, device=idx.device); stats = {}
        for blk in self.blocks:
            x, a, s = blk(x); aux = aux + a; stats = s
        h = self.ln_f(x); logits = self.lm_head(h)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, (loss + aux if (loss is not None) else None), stats
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _, _ = self(idx); logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1); nxt = torch.multinomial(probs, 1)
            idx = torch.cat((idx, nxt), dim=1)
        return idx

class SimpleMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__(); self.ln = LayerNorm(d, True); self.ff = nn.Linear(d, d)
    def forward(self, x): return x + self.ff(self.ln(x))

class NanoMamba(nn.Module):
    def __init__(self, vocab_size=256, block_size=128, n_layer=4, n_embd=128):
        super().__init__()
        self.vocab_size = vocab_size; self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(block_size, n_embd)
        self.layers = nn.ModuleList([SimpleMambaBlock(n_embd) for _ in range(n_layer)])
        self.ln_f = LayerNorm(n_embd, True); self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        B, T = idx.shape; pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        for l in self.layers: x = l(x)
        h = self.ln_f(x); logits = self.head(h)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss, {}
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _, _ = self(idx); logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1); nxt = torch.multinomial(probs, 1)
            idx = torch.cat((idx, nxt), dim=1)
        return idx

# =============================================================================
# TRAIN / GRPO / REWARD MODEL (instrumented with TrainingMonitor)
# =============================================================================

def train_lm(model, train_ids, val_ids, block_size, epochs, steps, batch, lr, device, title, monitor: TrainingMonitor=None):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    best_val = float('inf'); patience = 0

    for ep in range(1, epochs+1):
        model.train(); losses=[]; gnorms=[]; t0=time.time()
        for st in range(steps):
            xb, yb = get_batch_tokens(train_ids, block_size, batch, device)
            opt.zero_grad()
            _, loss, stats = model(xb, yb)
            loss.backward()
            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            gnorms.append(float(total_norm.item()))
            opt.step()
            losses.append(float(loss.item()))
            if monitor and (st % 10 == 0):
                monitor.capture_model_state(model)
                if stats: monitor.capture_moe_stats(stats)
        # validation
        model.eval()
        with torch.no_grad():
            xb, yb = get_batch_tokens(val_ids, block_size, batch, device)
            _, vloss, _ = model(xb, yb)
            v = float(vloss.item())
        tr = float(sum(losses)/len(losses))
        lr_now = float(sched.get_last_lr()[0])
        gn = float(sum(gnorms)/len(gnorms)) if gnorms else 0.0
        if monitor:
            monitor.log_training_step(ep, steps, tr, v, lr_now, gn)
            monitor.epoch_count = ep
        console.print(Panel(f"Epoch {ep}/{epochs} train={tr:.3f} val={v:.3f} grad={gn:.3f} time={time.time()-t0:.1f}s", title=title))
        if v < best_val:
            best_val = v; patience = 0
            # save best into outputs/models
            save_path = monitor.base_dir / "models" / f"best_{title.replace(' ', '_').lower()}.pt" if monitor else Path("artifacts")/f"best_{title.replace(' ', '_').lower()}.pt"
            payload = {"kind": "gpt_bpe" if hasattr(model, 'cfg') else "mamba_char",
                       "cfg": asdict(model.cfg) if hasattr(model, 'cfg') else {"vocab_size": getattr(model, 'vocab_size', None), "block_size": block_size},
                       "state_dict": model.state_dict()}
            torch.save(payload, save_path)
        else:
            patience += 1
        sched.step()
        if patience >= 50: # very conservative default
            console.print("⏹️ Early stopping")
            break
    return model

def grpo_step(model, encode_fn, prompts, gen_tokens, opt, device, monitor: TrainingMonitor=None):
    model.train(); rewards=[]; logps=[]; stats_last=None
    for s in prompts:
        x = torch.tensor([encode_fn(s)], device=device)
        y = model.generate(x, gen_tokens)[0]  # [seq]
        inp, tgt = y[:-1].unsqueeze(0), y[1:].unsqueeze(0)
        logits, _, stats = model(inp, tgt)
        logp = F.log_softmax(logits, dim=-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1).mean()
        rewards.append(logp); logps.append(logp)
        stats_last = stats
    rewards = torch.stack(rewards)
    logps = torch.stack(logps)
    adv = rewards - rewards.mean()
    loss = -(adv.detach() * logps).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if monitor and stats_last:
        monitor.capture_model_state(model)
        monitor.capture_moe_stats(stats_last)
    return float(loss.item()), float(rewards.mean().item())

class RewardModel(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base
        vocab_size = base.cfg.vocab_size if hasattr(base, "cfg") else base.vocab_size
        self.head = nn.Linear(vocab_size, 1)
    def forward(self, idx):
        logits, _, _ = self.base(idx); h = logits[:, -1, :]
        return self.head(h)

def load_pairs(path): return [json.loads(l) for l in open(path, "r", encoding="utf-8")]

def train_reward_model(rm, tok, pairs_path, block_size, epochs, steps, batch, lr, device, monitor: TrainingMonitor=None):
    rm.to(device); pairs = load_pairs(pairs_path)
    opt = torch.optim.AdamW(rm.parameters(), lr=lr)
    for ep in range(1, epochs+1):
        random.shuffle(pairs); losses=[]
        for _ in range(steps):
            b = random.sample(pairs, min(batch, len(pairs)))
            chosen = [tok.encode(p["chosen"]) for p in b]
            rejected = [tok.encode(p["rejected"]) for p in b]
            cl = [torch.tensor(c[:block_size]) for c in chosen]
            rl = [torch.tensor(r[:block_size]) for r in rejected]
            cl = torch.nn.utils.rnn.pad_sequence(cl, batch_first=True).to(device)
            rl = torch.nn.utils.rnn.pad_sequence(rl, batch_first=True).to(device)
            rc = rm(cl); rr = rm(rl)
            loss = -F.logsigmoid(rc - rr).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(float(loss.item()))
        console.print(Panel(f"RM Epoch {ep}/{epochs} loss={sum(losses)/len(losses):.3f}", title="Reward Model"))
        if monitor:
            # Log as training step for visualization (val not meaningful here)
            monitor.log_training_step(ep, steps, sum(losses)/len(losses))
            monitor.epoch_count = ep

# =============================================================================
# MAIN (CLI)
# =============================================================================

def main(argv=None):
    if argv is None: argv = []

    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 100))
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 50))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 64))

    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", required=True, choices=["train", "infer", "train_rm", "train_grpo"])
    ap.add_argument("--arch", required=True, choices=["gpt_bpe", "mamba_char"])
    ap.add_argument("--epochs", type=int, default=DEFAULT_EPOCHS)
    ap.add_argument("--steps_per_epoch", type=int, default=DEFAULT_STEPS_PER_EPOCH)
    ap.add_argument("--batch_size", type=int, default=DEFAULT_BATCH_SIZE)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--block_size", type=int, default=128)
    ap.add_argument("--n_layer", type=int, default=4)
    ap.add_argument("--n_head", type=int, default=4)
    ap.add_argument("--n_embd", type=int, default=128)
    ap.add_argument("--use_moe", action="store_true")
    ap.add_argument("--routing_mode", default="uniform", choices=["uniform", "specialize"])
    ap.add_argument("--num_experts", type=int, default=4)
    ap.add_argument("--topk", type=int, default=1)
    ap.add_argument("--router_hidden", type=int, default=128)
    ap.add_argument("--entropy_penalty", type=float, default=0.0)
    ap.add_argument("--moe_dropout", type=float, default=0.1)
    ap.add_argument("--tok_kind", default="basic")
    ap.add_argument("--vocab_size", type=int, default=512)
    ap.add_argument("--tok_prefix", default="artifacts/tokenizers/tiny_bpe")
    ap.add_argument("--save_path", default="artifacts/unified.pt")
    ap.add_argument("--load_path", default="artifacts/unified.pt")
    ap.add_argument("--sample_start", default="\n")
    ap.add_argument("--sample_tokens", type=int, default=100)
    ap.add_argument("--rm_pairs", default="")
    ap.add_argument("--exp_name", default="unified_experiment")
    args = ap.parse_args(argv)

    os.makedirs("artifacts", exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    console.print(Panel(f"Using device: {device}", title="Device"))

    # Monitor
    monitor = TrainingMonitor(args.exp_name)

    # Data
    train_txt, val_txt = load_tiny_shakespeare("./data")

    # Tokenizer / encoding
    if args.arch == "gpt_bpe":
        tok, _, vs = train_or_load_tokenizer(args.tok_kind, args.vocab_size, train_txt + val_txt, args.tok_prefix)
        train_ids, val_ids = tok.encode(train_txt), tok.encode(val_txt)
        encode_fn, decode_fn = tok.encode, tok.decode
        monitor.snapshot_tokenizer(args.tok_prefix)
    else:
        stoi, itos = build_charset(train_txt, val_txt)
        train_ids, val_ids = encode_chars(train_txt, stoi), encode_chars(val_txt, stoi)
        encode_fn = lambda s: encode_chars(s, stoi)
        decode_fn = lambda ids: "".join(itos[i] for i in ids)

    # Model (load if needed)
    model = None
    if args.mode != "train" and os.path.exists(args.load_path):
        payload = torch.load(args.load_path, map_location=device)
        kind = payload.get("kind", args.arch)
        cfg_dict = payload.get("cfg", {})
        if kind == "gpt_bpe":
            cfg = GPTCfg(**cfg_dict)
            model = NanoGPT(cfg)
        elif kind == "mamba_char":
            vocab_size = cfg_dict.get("vocab_size", len(stoi) if "stoi" in locals() else 256)
            model = NanoMamba(vocab_size, cfg_dict.get("block_size", args.block_size), cfg_dict.get("n_layer", args.n_layer), cfg_dict.get("n_embd", args.n_embd))
        else:
            raise ValueError(f"Unknown checkpoint kind: {kind}")
        model.load_state_dict(payload["state_dict"])
    else:
        if args.arch == "gpt_bpe":
            cfg = GPTCfg(block_size=args.block_size, vocab_size=vs, n_layer=args.n_layer, n_head=args.n_head, n_embd=args.n_embd,
                         dropout=0.1, use_moe=args.use_moe, routing_mode=args.routing_mode, num_experts=args.num_experts,
                         topk=args.topk, router_hidden=args.router_hidden, moe_dropout=args.moe_dropout, entropy_penalty=args.entropy_penalty)
            model = NanoGPT(cfg)
        else:
            model = NanoMamba(len(stoi), args.block_size, args.n_layer, args.n_embd)

    # Modes
    if args.mode == "train":
        title = f"{args.arch.upper()} Train"
        train_lm(model, train_ids, val_ids, args.block_size, args.epochs, args.steps_per_epoch, args.batch_size, args.lr, device, title, monitor)
        cfg_to_save = asdict(model.cfg) if hasattr(model, "cfg") else {"vocab_size": getattr(model, "vocab_size", None), "block_size": args.block_size, "n_layer": args.n_layer, "n_embd": args.n_embd}
        # save final model both to artifacts and monitor tree
        torch.save({"kind": args.arch, "cfg": cfg_to_save, "state_dict": model.state_dict()}, args.save_path)
        torch.save({"kind": args.arch, "cfg": cfg_to_save, "state_dict": model.state_dict()}, monitor.base_dir / "models" / Path(args.save_path).name)
        console.print(Panel(f"Model saved to {args.save_path}", title="Save"))
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        console.print(Panel(f"ZIP ready: {zip_path}", title="Done"))

    elif args.mode == "infer":
        payload = torch.load(args.load_path, map_location=device)
        model.load_state_dict(payload["state_dict"]); model.to(device)
        start_ids = encode_fn(args.sample_start)
        idx = torch.tensor([start_ids], device=device)
        sample = model.generate(idx, args.sample_tokens)[0].tolist()
        console.print(Panel(decode_fn(sample), title="Generated Text"))

    elif args.mode == "train_grpo":
        model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
        prompts = ["To be, or not to be", "Once upon a time", "The quick brown fox"]
        for step in range(args.steps_per_epoch):
            loss, mr = grpo_step(model, encode_fn, prompts, 32, opt, device, monitor)
            # log as pseudo-epoch (ep=step+1)
            monitor.log_training_step(step+1, step, train_loss=loss)
            console.print(Panel(f"GRPO step {step+1}/{args.steps_per_epoch} loss={loss:.3f} reward={mr:.3f}", title="GRPO"))
        cfg_to_save = asdict(model.cfg) if hasattr(model, "cfg") else {}
        torch.save({"kind": args.arch, "cfg": cfg_to_save, "state_dict": model.state_dict()}, args.save_path)
        torch.save({"kind": args.arch, "cfg": cfg_to_save, "state_dict": model.state_dict()}, monitor.base_dir / "models" / Path(args.save_path).name)
        console.print(Panel(f"GRPO-tuned model saved to {args.save_path}", title="Save"))
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        console.print(Panel(f"ZIP ready: {zip_path}", title="Done"))

    elif args.mode == "train_rm":
        if not args.rm_pairs:
            raise ValueError("--rm_pairs is required for train_rm")
        model.to(device)
        rm = RewardModel(model)
        tok_like = tok if args.arch == "gpt_bpe" else type("TK", (), {"encode": encode_fn})()
        train_reward_model(rm, tok_like, args.rm_pairs, args.block_size, epochs=args.epochs, steps=args.steps_per_epoch, batch=args.batch_size, lr=1e-5, device=device, monitor=monitor)
        torch.save({"kind": "reward_model", "base_arch": args.arch, "state_dict": rm.state_dict()}, args.save_path)
        torch.save({"kind": "reward_model", "base_arch": args.arch, "state_dict": rm.state_dict()}, monitor.base_dir / "models" / Path(args.save_path).name)
        console.print(Panel(f"Reward model saved to {args.save_path}", title="Save"))
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        console.print(Panel(f"ZIP ready: {zip_path}", title="Done"))

# =============================================================================
# FULL PIPELINE (runs everything and packages results)
# =============================================================================

def run_full_pipeline():
    console.rule("[bold magenta]Unified End-to-End Pipeline (Monitored)")

    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 50))
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 20))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 64))

    # 1) Train base GPT-BPE with MoE
    main([
        "--mode","train",
        "--arch","gpt_bpe",
        "--tok_kind","basic",
        "--vocab_size","512",
        "--use_moe","--routing_mode","specialize",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--save_path","artifacts/gpt_moe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","gpt_bpe_moe"
    ])

    # 2) Reward Model (create toy data if missing)
    rm_file = "data/preferences.jsonl"
    if not os.path.exists(rm_file):
        toy = [
            {"chosen":"I love this model","rejected":"I hate this model"},
            {"chosen":"This is a great answer","rejected":"This is a terrible answer"},
        ]
        os.makedirs("data", exist_ok=True)
        with open(rm_file, "w", encoding="utf-8") as f:
            for row in toy: f.write(json.dumps(row) + "\n")
    main([
        "--mode","train_rm","--arch","gpt_bpe",
        "--rm_pairs",rm_file,
        "--load_path","artifacts/gpt_moe.pt",
        "--save_path","artifacts/reward_model.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--epochs", str(max(5, DEFAULT_EPOCHS//5)),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--exp_name","reward_model"
    ])

    # 3) GRPO fine-tuning
    main([
        "--mode","train_grpo","--arch","gpt_bpe",
        "--load_path","artifacts/gpt_moe.pt",
        "--save_path","artifacts/gpt_grpo.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--exp_name","gpt_grpo"
    ])

    # 4) Inference (base GPT-MoE)
    main([
        "--mode","infer","--arch","gpt_bpe",
        "--load_path","artifacts/gpt_moe.pt",
        "--sample_start","Hello world,",
        "--sample_tokens","50",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","gpt_bpe_moe"
    ])

    # 5) Train small Mamba
    main([
        "--mode","train","--arch","mamba_char",
        "--epochs", str(max(10, DEFAULT_EPOCHS//2)),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH//2),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--save_path","artifacts/mamba_model.pt",
        "--exp_name","mamba_char"
    ])

    # 6) Mamba inference
    main([
        "--mode","infer","--arch","mamba_char",
        "--load_path","artifacts/mamba_model.pt",
        "--sample_start","ROMEO:",
        "--sample_tokens","80",
        "--exp_name","mamba_char"
    ])

# =============================================================================
# ENTRYPOINT
# =============================================================================

def _sanitize_argv(argv):
    out = []; skip = False
    for a in argv:
        if skip: skip = False; continue
        if a in ("-f", "--f"): skip = True; continue
        if a.startswith("-f=") or a.startswith("--f="): continue
        out.append(a)
    return out

if __name__ == "__main__":
    import sys
    argv = _sanitize_argv(sys.argv[1:])
    if ("--mode" not in argv) or ("--arch" not in argv):
        run_full_pipeline()
    else:
        main(argv)


## Full NanoGPT / NanoMamba stack with Tokenizer, MoE, Reward Model (pairwise), GRPO, TRAINING MONITOR (plots + logs + ZIP), and a runnable full pipeline.



In [ ]:
#!/usr/bin/env python3
# unified_all_in_one_with_monitor.py
# Full NanoGPT / NanoMamba stack with Tokenizer, MoE, Reward Model (pairwise),
# GRPO, TRAINING MONITOR (plots + logs + ZIP), and a runnable full pipeline.
#
# Includes:
#   • NanoGPT (with optional MoE)
#   • NanoMamba (character-level)
#   • Pairwise Reward Model on top of any base LM (GPT or Mamba)
#   • GRPO loop for either architecture
#   • TrainingMonitor: organized outputs (models/plots/logs/tokenizers) + ZIP bundling
#
# Quick examples:
#   python unified_all_in_one_with_monitor.py --mode train --arch gpt_bpe --use_moe --routing_mode specialize
#   python unified_all_in_one_with_monitor.py --mode infer --arch gpt_bpe --load_path artifacts/gpt_moe.pt
#   python unified_all_in_one_with_monitor.py --mode train_rm --arch gpt_bpe --rm_pairs data/preferences.jsonl
#   python unified_all_in_one_with_monitor.py --mode train_grpo --arch gpt_bpe --load_path artifacts/gpt_moe.pt
#
#   # With NO arguments, it runs the full educational pipeline and creates a ZIP:
#   python unified_all_in_one_with_monitor.py

import os, json, argparse, random, time, urllib.request, zipfile, shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

# Optional rich console
try:
    from rich.console import Console
    from rich.panel import Panel
    HAS_RICH = True
except ImportError:
    HAS_RICH = False
    class Console:
        def print(self, *a, **k): print(*a)
        def rule(self, *a, **k): print("=" * 50)
    class Panel:
        def __init__(self, text, title="", style=""): self.text=text; self.title=title
        def __str__(self): return f"[{self.title}] {self.text}"

console = Console()

# -----------------------------------------------------------------------------
# Plotting (matplotlib only; seaborn optional)
# -----------------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")  # non-interactive backends (servers/Colab safe)
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec

try:
    import seaborn as sns
    sns.set_palette("husl")
except Exception:
    pass

plt.style.use('default')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

# =============================================================================
# TRAINING MONITOR (plots, logs, animations, ZIP packaging)
# =============================================================================

class TrainingMonitor:
    """Comprehensive training monitor that saves everything organized."""
    def __init__(self, experiment_name="unified_experiment"):
        self.experiment_name = experiment_name
        self.base_dir = Path("training_outputs")
        self._setup_folders()
        # Tracking
        self.metrics = defaultdict(list)
        self.attention_snapshots = []
        self.embedding_snapshots = []
        self.moe_snapshots = []
        self.step_count = 0
        self.epoch_count = 0

    # ---------------------- FS ----------------------
    def _setup_folders(self):
        folders = [
            self.base_dir,
            self.base_dir / "models",
            self.base_dir / "plots" / "training_curves",
            self.base_dir / "plots" / "attention_analysis",
            self.base_dir / "plots" / "embedding_evolution",
            self.base_dir / "plots" / "moe_analysis",
            self.base_dir / "plots" / "animations",
            self.base_dir / "tokenizers",
            self.base_dir / "logs",
            self.base_dir / "data",
        ]
        for f in folders: f.mkdir(parents=True, exist_ok=True)
        console.print(f"📁 Output structure at: {self.base_dir}")

    # ---------------------- Logging ----------------------
    def log_training_step(self, epoch, step, train_loss, val_loss=None, lr=None, grad_norm=None):
        self.metrics['epoch'].append(epoch)
        self.metrics['step'].append(self.step_count)
        self.metrics['train_loss'].append(float(train_loss))
        if val_loss is not None: self.metrics['val_loss'].append(float(val_loss))
        if lr is not None: self.metrics['learning_rate'].append(float(lr))
        if grad_norm is not None: self.metrics['grad_norm'].append(float(grad_norm))
        self.step_count += 1

    def capture_model_state(self, model):
        # Attention Q/K alignment (if present)
        if hasattr(model, 'blocks'):
            attn_data = []
            for i, block in enumerate(model.blocks):
                if hasattr(block.attn, 'c_attn'):
                    with torch.no_grad():
                        w = block.attn.c_attn.weight.data.detach().cpu().numpy()
                    n_embd = w.shape[1]
                    q_w = w[:n_embd, :]
                    k_w = w[n_embd:2*n_embd, :]
                    corr = np.corrcoef(q_w.flatten(), k_w.flatten())[0, 1]
                    attn_data.append({
                        'layer': i, 'qk_correlation': float(corr)
                    })
            if attn_data:
                self.attention_snapshots.append({
                    'step': self.step_count, 'epoch': self.epoch_count, 'data': attn_data
                })
        # Embedding snapshots
        if hasattr(model, 'wte'):
            with torch.no_grad():
                emb = model.wte.weight.data.detach().cpu().numpy()
            self.embedding_snapshots.append({
                'step': self.step_count,
                'epoch': self.epoch_count,
                'norm': float(np.linalg.norm(emb)),
                'mean': float(np.mean(emb)),
                'std': float(np.std(emb)),
                'weights': emb.copy(),
            })

    def capture_moe_stats(self, stats):
        if stats and 'mean_routing_probs' in stats and stats['mean_routing_probs'] is not None:
            probs = stats['mean_routing_probs']
            if isinstance(probs, torch.Tensor):
                probs = probs.detach().cpu().numpy()
            probs = np.asarray(probs)
            entropy = float(-(probs * np.log(probs + 1e-8)).sum())
            balance = float(1.0 - (np.std(probs) / (np.mean(probs) + 1e-8)))
            self.moe_snapshots.append({
                'step': self.step_count,
                'epoch': self.epoch_count,
                'routing_probs': probs.tolist(),
                'entropy': entropy,
                'balance': balance,
            })

    # ---------------------- Plots ----------------------
    def _plot_training_curves(self):
        if not self.metrics['train_loss']:
            return
        fig = plt.figure(figsize=(20, 12))
        gs = GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)
        epochs = np.array(self.metrics['epoch'])
        train_losses = np.array(self.metrics['train_loss'])

        ax1 = fig.add_subplot(gs[0, :2])
        ax1.plot(epochs, train_losses, linewidth=2, alpha=0.9, label='Training Loss')
        if self.metrics.get('val_loss'):
            vl = np.array(self.metrics['val_loss'])
            ax1.plot(epochs[-len(vl):], vl, linestyle='--', linewidth=2, alpha=0.9, label='Validation Loss')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Progress'); ax1.legend(); ax1.grid(True, alpha=0.3)

        if self.metrics.get('learning_rate'):
            ax2 = fig.add_subplot(gs[0, 2])
            lrs = np.array(self.metrics['learning_rate'])
            ax2.plot(epochs[-len(lrs):], lrs, linewidth=2)
            ax2.set_xlabel('Epoch'); ax2.set_ylabel('LR'); ax2.set_title('LR Schedule'); ax2.set_yscale('log'); ax2.grid(True, alpha=0.3)

        if self.metrics.get('grad_norm'):
            ax3 = fig.add_subplot(gs[1, 0])
            gns = np.array(self.metrics['grad_norm'])
            ax3.plot(epochs[-len(gns):], gns, linewidth=2, alpha=0.9)
            ax3.set_xlabel('Epoch'); ax3.set_ylabel('Grad Norm'); ax3.set_title('Gradient Norms'); ax3.grid(True, alpha=0.3)

        ax4 = fig.add_subplot(gs[1, 1])
        if len(train_losses) > 4:
            w = max(3, min(20, len(train_losses)//10))
            sm = np.convolve(train_losses, np.ones(w)/w, mode='valid')
            ax4.plot(epochs[w-1:], sm, linewidth=3, alpha=0.9)
            ax4.set_xlabel('Epoch'); ax4.set_ylabel('Smoothed Loss'); ax4.set_title(f'Loss (Moving Avg, window={w})'); ax4.grid(True, alpha=0.3)
        else:
            ax4.text(0.5,0.5,'Insufficient steps for smoothing',ha='center',va='center'); ax4.axis('off')

        ax5 = fig.add_subplot(gs[1, 2])
        final_loss = float(train_losses[-1]) if len(train_losses) else 0.0
        min_loss = float(np.min(train_losses)) if len(train_losses) else 0.0
        improvement = float(train_losses[0] - final_loss) if len(train_losses) > 1 else 0.0
        stats_text = f'Final Loss: {final_loss:.4f}\nBest Loss: {min_loss:.4f}\nImprovement: {improvement:.4f}\nTotal Steps: {len(train_losses)}'
        ax5.text(0.1, 0.5, stats_text, transform=ax5.transAxes, fontsize=12, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        ax5.axis('off'); ax5.set_title('Training Stats')

        out = self.base_dir / "plots" / "training_curves" / "comprehensive_training.png"
        plt.suptitle(f'{self.experiment_name} - Training Analysis', fontsize=16)
        plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"📊 Training curves saved to {out}")

    def _create_loss_animation(self):
        if not self.metrics['train_loss']:
            return
        fig, ax = plt.subplots(figsize=(12, 8))
        train_losses = np.array(self.metrics['train_loss'])
        epochs = np.array(self.metrics['epoch'])
        def animate(i):
            ax.clear(); ax.plot(epochs[:i+1], train_losses[:i+1], linewidth=3, alpha=0.9)
            ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title(f'Training Progress - Step {i+1}/{len(train_losses)}'); ax.grid(True, alpha=0.3)
            ax.set_xlim(0, max(epochs) if len(epochs) else 1)
            if len(train_losses):
                ax.set_ylim(min(train_losses)*0.9, max(train_losses)*1.1)
        frames = min(len(train_losses), 100)
        anim = animation.FuncAnimation(fig, animate, frames=frames, interval=150, repeat=True)
        out = self.base_dir / "plots" / "animations" / "loss_evolution.gif"
        anim.save(out, writer='pillow', fps=5); plt.close()
        console.print(f"🎬 Loss animation saved to {out}")

    def _plot_attention(self):
        if not self.attention_snapshots:
            return
        steps = [s['step'] for s in self.attention_snapshots]
        n_layers = len(self.attention_snapshots[0]['data'])
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        # layer curves
        for layer in range(n_layers):
            corrs = []
            for s in self.attention_snapshots:
                d = s['data'][layer]
                corrs.append(d['qk_correlation'])
            ax1.plot(steps[:len(corrs)], corrs, linewidth=2, alpha=0.9, label=f'L{layer}')
        ax1.set_xlabel('Step'); ax1.set_ylabel('Q-K Corr'); ax1.set_title('Query-Key Alignment Evolution'); ax1.legend(); ax1.grid(True, alpha=0.3)

        latest = self.attention_snapshots[-1]['data']
        cm = np.array([[d['qk_correlation'] for d in latest]])
        im = ax2.imshow(cm, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
        ax2.set_title('Latest Q-K Correlations'); fig.colorbar(im, ax=ax2)

        # Distribution for layer 0 not available without raw weights — show histogram of correlations instead
        all_corrs = []
        for s in self.attention_snapshots:
            all_corrs.extend([d['qk_correlation'] for d in s['data']])
        ax3.hist(all_corrs, bins=40, alpha=0.85)
        ax3.set_title('Distribution of Q-K Correlations'); ax3.set_xlabel('Corr'); ax3.set_ylabel('Freq'); ax3.grid(True, alpha=0.3)

        # Stability (std) per layer
        stdevs = []
        for layer in range(n_layers):
            layer_series = [s['data'][layer]['qk_correlation'] for s in self.attention_snapshots]
            stdevs.append(np.std(layer_series))
        ax4.bar(range(n_layers), stdevs, alpha=0.9)
        ax4.set_xlabel('Layer'); ax4.set_ylabel('Std Dev'); ax4.set_title('Q-K Correlation Stability'); ax4.grid(True, alpha=0.3)

        out = self.base_dir / "plots" / "attention_analysis" / "attention_comprehensive.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"🔍 Attention analysis saved to {out}")

    def _plot_embeddings(self):
        if not self.embedding_snapshots:
            return
        steps = [s['step'] for s in self.embedding_snapshots]
        norms = [s['norm'] for s in self.embedding_snapshots]
        means = [s['mean'] for s in self.embedding_snapshots]
        stds  = [s['std']  for s in self.embedding_snapshots]
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        ax1.plot(steps, norms, linewidth=2); ax1.set_title('Embedding Frobenius Norm'); ax1.set_xlabel('Step'); ax1.set_ylabel('Norm'); ax1.grid(True, alpha=0.3)
        ax2.plot(steps, means, linewidth=2, label='Mean'); ax2.plot(steps, stds, linewidth=2, label='Std'); ax2.legend(); ax2.set_title('Embedding Stats'); ax2.grid(True, alpha=0.3)

        # Latest distribution if we kept weights
        w = self.embedding_snapshots[-1].get('weights', None)
        if w is not None:
            ax3.hist(w.flatten(), bins=60, alpha=0.85)
            ax3.set_title('Latest Embedding Distribution'); ax3.grid(True, alpha=0.3)
            token_norms = np.linalg.norm(w, axis=1)
            ax4.plot(token_norms, alpha=0.9)
            ax4.set_title('Per-Token L2 Norms'); ax4.set_xlabel('Token'); ax4.set_ylabel('L2'); ax4.grid(True, alpha=0.3)
        else:
            ax3.axis('off'); ax4.axis('off')

        out = self.base_dir / "plots" / "embedding_evolution" / "embedding_analysis.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"📐 Embedding analysis saved to {out}")

    def _plot_moe(self):
        if not self.moe_snapshots:
            return
        steps = [s['step'] for s in self.moe_snapshots]
        nE = len(self.moe_snapshots[0]['routing_probs'])
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        for e in range(nE):
            usage = [s['routing_probs'][e] for s in self.moe_snapshots]
            ax1.plot(steps, usage, linewidth=2, alpha=0.9, label=f'E{e}')
        ax1.set_title('Expert Usage Evolution'); ax1.set_xlabel('Step'); ax1.set_ylabel('Prob'); ax1.legend(); ax1.grid(True, alpha=0.3)

        ent = [s['entropy'] for s in self.moe_snapshots]
        ax2.plot(steps, ent, linewidth=2, alpha=0.9)
        ax2.set_title('Routing Entropy'); ax2.set_xlabel('Step'); ax2.grid(True, alpha=0.3)

        latest = self.moe_snapshots[-1]['routing_probs']
        ax3.pie(latest, labels=[f'E{i}' for i in range(len(latest))], autopct='%1.1f%%', startangle=90)
        ax3.set_title('Current Expert Distribution')

        bal = [s['balance'] for s in self.moe_snapshots]
        ax4.plot(steps, bal, linewidth=2, alpha=0.9)
        ax4.set_title('Load Balance (1=best)'); ax4.set_xlabel('Step'); ax4.grid(True, alpha=0.3)

        out = self.base_dir / "plots" / "moe_analysis" / "moe_comprehensive.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        console.print(f"🔀 MoE analysis saved to {out}")

    # ---------------------- Persist ----------------------
    def _save_logs(self):
        # Metrics
        with open(self.base_dir / "logs" / "training_metrics.json", 'w') as f:
            json.dump({k: list(v) for k, v in self.metrics.items()}, f, indent=2)
        # Attention
        if self.attention_snapshots:
            with open(self.base_dir / "logs" / "attention_evolution.json", 'w') as f:
                json.dump(self.attention_snapshots, f, indent=2)
        # Embedding (strip weights for logs to keep size small)
        if self.embedding_snapshots:
            light = [{k: v for k, v in s.items() if k != 'weights'} for s in self.embedding_snapshots]
            with open(self.base_dir / "logs" / "embedding_evolution.json", 'w') as f:
                json.dump(light, f, indent=2)
        # MoE
        if self.moe_snapshots:
            with open(self.base_dir / "logs" / "moe_evolution.json", 'w') as f:
                json.dump(self.moe_snapshots, f, indent=2)
        console.print(f"💾 Logs saved to {self.base_dir / 'logs'}")

    def generate_all(self):
        console.print("🎨 Generating visualizations and logs...")
        self._plot_training_curves()
        self._create_loss_animation()
        self._plot_attention()
        self._plot_embeddings()
        self._plot_moe()
        self._save_logs()
        console.print("✅ Visualizations complete")

    def snapshot_tokenizer(self, src_prefix: str):
        # copy tokenizer files into outputs for convenience
        m = Path(src_prefix + ".model")
        if m.exists():
            dst = self.base_dir / "tokenizers" / m.name
            try:
                shutil.copy2(m, dst)
            except Exception:
                pass

    def zip_everything(self):
        # Create summary
        summary = {
            "experiment": self.experiment_name,
            "timestamp": time.strftime("%Y-%m-%d_%H-%M-%S"),
            "total_steps": int(self.step_count),
            "total_epochs": int(self.epoch_count),
            "final_loss": self.metrics['train_loss'][-1] if self.metrics['train_loss'] else None,
        }
        with open(self.base_dir / "experiment_summary.json", 'w') as f:
            json.dump(summary, f, indent=2)

        # README
        readme = f"""# {self.experiment_name} - Results\n\nThis folder contains models, plots, logs, tokenizers, and data artifacts.\n\n- Plots in `plots/` (training curves, attention, embeddings, MoE, animations)\n- Logs in `logs/` (JSON)\n- Models in `models/`\n- Tokenizers in `tokenizers/`\n- Data in `data/`\n"""
        with open(self.base_dir / "README.md", 'w') as f:
            f.write(readme)

        # Zip
        ts = time.strftime("%Y%m%d_%H%M%S")
        zip_name = f"unified_results_{ts}.zip"
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
            for p in self.base_dir.rglob('*'):
                if p.is_file(): z.write(p, p.relative_to('.'))
        size_mb = os.path.getsize(zip_name) / (1024*1024)
        console.print(Panel(f"Created ZIP: {zip_name} ({size_mb:.1f} MB)", title="Package"))
        return zip_name

# =============================================================================
# TOKENIZER (simple BPE-ish)
# =============================================================================

def _get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for p in zip(ids, ids[1:]): counts[p] = counts.get(p, 0) + 1
    return counts

def _merge(ids, pair, idx):
    newids = []; i = 0
    while i < len(ids):
        if ids[i] == pair[0] and i < len(ids)-1 and ids[i+1] == pair[1]:
            newids.append(idx); i += 2
        else:
            newids.append(ids[i]); i += 1
    return newids

class Tokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = self._build_vocab()
    def _build_vocab(self):
        vocab = {i: bytes([i]) for i in range(256)}
        for (a,b),i in self.merges.items():
            vocab[i] = vocab[a] + vocab[b]
        return vocab
    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")
    def save(self, prefix):
        d = os.path.dirname(prefix)
        if d: os.makedirs(d, exist_ok=True)
        with open(prefix + ".model", "w") as f:
            for (a,b),i in self.merges.items():
                f.write(f"{a} {b}\n")
    def load(self, path):
        merges = {}; idx = 256
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    try:
                        a = int(parts[0]); b = int(parts[1])
                    except ValueError:
                        continue
                    merges[(a, b)] = idx; idx += 1
        self.merges = merges
        self.vocab = self._build_vocab()

class BasicTokenizer(Tokenizer):
    def train(self, text, vocab_size):
        assert vocab_size >= 256
        ids = list(text.encode("utf-8"))
        merges = {}; vocab = {i: bytes([i]) for i in range(256)}
        for i in range(vocab_size - 256):
            stats = _get_stats(ids)
            if not stats: break
            pair = max(stats, key=stats.get); idx = 256 + i
            ids = _merge(ids, pair, idx)
            merges[pair] = idx; vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
        self.merges = merges; self.vocab = vocab
    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = _get_stats(ids)
            if not stats: break
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges: break
            ids = _merge(ids, pair, self.merges[pair])
        return ids

def train_or_load_tokenizer(kind, vocab_size, text, prefix):
    model_path = prefix + ".model"
    if os.path.exists(model_path):
        tok = BasicTokenizer(); tok.load(model_path)
        vs = max(tok.vocab.keys()) + 1
        console.print(Panel(f"Loaded tokenizer: {model_path} (size={vs})", title="Tokenizer"))
        return tok, model_path, vs
    tok = BasicTokenizer(); tok.train(text, vocab_size); tok.save(prefix)
    vs = max(tok.vocab.keys()) + 1
    console.print(Panel(f"Saved tokenizer: {prefix}.model (size={vs})", title="Tokenizer"))
    return tok, model_path, vs

# =============================================================================
# DATA
# =============================================================================

def load_tiny_shakespeare(data_dir="./data"):
    os.makedirs(data_dir, exist_ok=True)
    path = os.path.join(data_dir, "tinyshakespeare_input.txt")
    if not os.path.exists(path):
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(url, path)
    text = open(path, "r", encoding="utf-8").read()
    split = int(0.9 * len(text))
    return text[:split], text[split:]

def build_charset(train, val):
    chars = sorted(set(train + val))
    stoi = {c:i for i,c in enumerate(chars)}
    itos = {i:c for c,i in stoi.items()}
    return stoi, itos

def encode_chars(s, stoi): return [stoi[c] for c in s]

def get_batch_tokens(ids, block_size, batch_size, device):
    ix = torch.randint(len(ids) - block_size - 1, (batch_size,))
    x = torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y = torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================================================================
# MoE BUILDING BLOCKS
# =============================================================================

class TopKRouter(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=1, temp=1.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, num_experts))
        self.k = k; self.temp = temp
    def forward(self, x):
        probs = F.softmax(self.net(x) / self.temp, dim=-1)
        topk_vals, topk_idx = torch.topk(probs, self.k, dim=-1)
        mask = torch.zeros_like(probs)
        mask.scatter_(dim=-1, index=topk_idx, src=torch.ones_like(topk_vals))
        sp = probs * mask
        sp = sp / (sp.sum(dim=-1, keepdim=True) + 1e-9)
        return sp, probs

def entropy_mean(p, eps=1e-9):
    p = p.clamp_min(eps)
    return -(p * p.log()).sum(-1).mean()

class MoEFFN(nn.Module):
    def __init__(self, in_dim, num_experts=4, k=1, router_hidden=128, dropout=0.1,
                 entropy_penalty=0.0, routing_mode="uniform"):
        super().__init__()
        hidden = 4 * in_dim
        self.expert_fc = nn.ModuleList([nn.Linear(in_dim, hidden) for _ in range(num_experts)])
        self.expert_proj = nn.ModuleList([nn.Linear(hidden, in_dim) for _ in range(num_experts)])
        self.router = TopKRouter(in_dim, router_hidden, num_experts, k)
        self.drop = nn.Dropout(dropout)
        self.entropy_penalty = entropy_penalty
        self.routing_mode = routing_mode
        self.num_experts = num_experts
    def forward(self, x):
        B, T, C = x.shape; xf = x.reshape(B*T, C)
        sp, dp = self.router(xf); out = 0.0
        for e in range(self.num_experts):
            h = F.gelu(self.expert_fc[e](xf))
            h = self.expert_proj[e](h)
            out = out + sp[:, e].unsqueeze(-1) * h
        y = self.drop(out.view(B, T, C))
        aux = torch.tensor(0.0, device=x.device)
        if self.routing_mode == "specialize":
            aux = -self.entropy_penalty * entropy_mean(dp)
        return y, aux, {"mean_routing_probs": dp.mean(0)}

# =============================================================================
# MODELS
# =============================================================================

class LayerNorm(nn.Module):
    def __init__(self, n, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n))
        self.bias = nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

@dataclass
class GPTCfg:
    block_size: int = 128
    vocab_size: int = 256
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.1
    # MoE
    use_moe: bool = False
    routing_mode: str = "uniform"  # or "specialize"
    num_experts: int = 4
    topk: int = 1
    router_hidden: int = 128
    moe_dropout: float = 0.1
    entropy_penalty: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.n_head = n_head; self.n_embd = n_embd
        self.c_attn = nn.Linear(n_embd, 3*n_embd, bias=True)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=True)
        self.dropout = dropout; self.block_size = block_size
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class GPTBlock(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.ln1 = LayerNorm(cfg.n_embd, True)
        self.attn = CausalSelfAttention(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout)
        self.ln2 = LayerNorm(cfg.n_embd, True)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd), nn.GELU(), nn.Linear(4 * cfg.n_embd, cfg.n_embd)
        )
        self.moe = (MoEFFN(cfg.n_embd, cfg.num_experts, cfg.topk, cfg.router_hidden,
                           cfg.moe_dropout, cfg.entropy_penalty, cfg.routing_mode)
                    if cfg.use_moe else None)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        if self.moe is None:
            x = x + self.ffn(self.ln2(x))
            return x, torch.tensor(0.0, device=x.device), {}
        else:
            y, aux, stats = self.moe(self.ln2(x))
            return x + y, aux, stats

class NanoGPT(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([GPTBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = LayerNorm(cfg.n_embd, True)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
    def forward(self, idx, targets=None):
        B, T = idx.shape; pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        aux = torch.tensor(0.0, device=idx.device); stats = {}
        for blk in self.blocks:
            x, a, s = blk(x); aux = aux + a; stats = s
        h = self.ln_f(x); logits = self.lm_head(h)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, (loss + aux if (loss is not None) else None), stats
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _, _ = self(idx); logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1); nxt = torch.multinomial(probs, 1)
            idx = torch.cat((idx, nxt), dim=1)
        return idx

class SimpleMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__(); self.ln = LayerNorm(d, True); self.ff = nn.Linear(d, d)
    def forward(self, x): return x + self.ff(self.ln(x))

class NanoMamba(nn.Module):
    def __init__(self, vocab_size=256, block_size=128, n_layer=4, n_embd=128):
        super().__init__()
        self.vocab_size = vocab_size; self.block_size = block_size
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(block_size, n_embd)
        self.layers = nn.ModuleList([SimpleMambaBlock(n_embd) for _ in range(n_layer)])
        self.ln_f = LayerNorm(n_embd, True); self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        B, T = idx.shape; pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        for l in self.layers: x = l(x)
        h = self.ln_f(x); logits = self.head(h)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss, {}
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _, _ = self(idx); logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1); nxt = torch.multinomial(probs, 1)
            idx = torch.cat((idx, nxt), dim=1)
        return idx

# =============================================================================
# TRAIN / GRPO / REWARD MODEL (instrumented with TrainingMonitor)
# =============================================================================

def train_lm(model, train_ids, val_ids, block_size, epochs, steps, batch, lr, device, title, monitor: TrainingMonitor=None):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    best_val = float('inf'); patience = 0

    for ep in range(1, epochs+1):
        model.train(); losses=[]; gnorms=[]; t0=time.time()
        for st in range(steps):
            xb, yb = get_batch_tokens(train_ids, block_size, batch, device)
            opt.zero_grad()
            _, loss, stats = model(xb, yb)
            loss.backward()
            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            gnorms.append(float(total_norm.item()))
            opt.step()
            losses.append(float(loss.item()))
            if monitor and (st % 10 == 0):
                monitor.capture_model_state(model)
                if stats: monitor.capture_moe_stats(stats)
        # validation
        model.eval()
        with torch.no_grad():
            xb, yb = get_batch_tokens(val_ids, block_size, batch, device)
            _, vloss, _ = model(xb, yb)
            v = float(vloss.item())
        tr = float(sum(losses)/len(losses))
        lr_now = float(sched.get_last_lr()[0])
        gn = float(sum(gnorms)/len(gnorms)) if gnorms else 0.0
        if monitor:
            monitor.log_training_step(ep, steps, tr, v, lr_now, gn)
            monitor.epoch_count = ep
        console.print(Panel(f"Epoch {ep}/{epochs} train={tr:.3f} val={v:.3f} grad={gn:.3f} time={time.time()-t0:.1f}s", title=title))
        if v < best_val:
            best_val = v; patience = 0
            # save best into outputs/models
            save_path = monitor.base_dir / "models" / f"best_{title.replace(' ', '_').lower()}.pt" if monitor else Path("artifacts")/f"best_{title.replace(' ', '_').lower()}.pt"
            payload = {"kind": "gpt_bpe" if hasattr(model, 'cfg') else "mamba_char",
                       "cfg": asdict(model.cfg) if hasattr(model, 'cfg') else {"vocab_size": getattr(model, 'vocab_size', None), "block_size": block_size},
                       "state_dict": model.state_dict()}
            torch.save(payload, save_path)
        else:
            patience += 1
        sched.step()
        if patience >= 50: # very conservative default
            console.print("⏹️ Early stopping")
            break
    return model

def grpo_step(model, encode_fn, prompts, gen_tokens, opt, device, monitor: TrainingMonitor=None):
    model.train(); rewards=[]; logps=[]; stats_last=None
    for s in prompts:
        x = torch.tensor([encode_fn(s)], device=device)
        y = model.generate(x, gen_tokens)[0]  # [seq]
        inp, tgt = y[:-1].unsqueeze(0), y[1:].unsqueeze(0)
        logits, _, stats = model(inp, tgt)
        logp = F.log_softmax(logits, dim=-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1).mean()
        rewards.append(logp); logps.append(logp)
        stats_last = stats
    rewards = torch.stack(rewards)
    logps = torch.stack(logps)
    adv = rewards - rewards.mean()
    loss = -(adv.detach() * logps).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if monitor and stats_last:
        monitor.capture_model_state(model)
        monitor.capture_moe_stats(stats_last)
    return float(loss.item()), float(rewards.mean().item())

class RewardModel(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base
        vocab_size = base.cfg.vocab_size if hasattr(base, "cfg") else base.vocab_size
        self.head = nn.Linear(vocab_size, 1)
    def forward(self, idx):
        logits, _, _ = self.base(idx); h = logits[:, -1, :]
        return self.head(h)

def load_pairs(path): return [json.loads(l) for l in open(path, "r", encoding="utf-8")]

def train_reward_model(rm, tok, pairs_path, block_size, epochs, steps, batch, lr, device, monitor: TrainingMonitor=None):
    rm.to(device); pairs = load_pairs(pairs_path)
    opt = torch.optim.AdamW(rm.parameters(), lr=lr)
    for ep in range(1, epochs+1):
        random.shuffle(pairs); losses=[]
        for _ in range(steps):
            b = random.sample(pairs, min(batch, len(pairs)))
            chosen = [tok.encode(p["chosen"]) for p in b]
            rejected = [tok.encode(p["rejected"]) for p in b]
            cl = [torch.tensor(c[:block_size]) for c in chosen]
            rl = [torch.tensor(r[:block_size]) for r in rejected]
            cl = torch.nn.utils.rnn.pad_sequence(cl, batch_first=True).to(device)
            rl = torch.nn.utils.rnn.pad_sequence(rl, batch_first=True).to(device)
            rc = rm(cl); rr = rm(rl)
            loss = -F.logsigmoid(rc - rr).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(float(loss.item()))
        console.print(Panel(f"RM Epoch {ep}/{epochs} loss={sum(losses)/len(losses):.3f}", title="Reward Model"))
        if monitor:
            # Log as training step for visualization (val not meaningful here)
            monitor.log_training_step(ep, steps, sum(losses)/len(losses))
            monitor.epoch_count = ep

# =============================================================================
# MAIN (CLI)
# =============================================================================

def main(argv=None):
    if argv is None: argv = []

    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 100))
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 50))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 64))

    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", required=True, choices=["train", "infer", "train_rm", "train_grpo"])
    ap.add_argument("--arch", required=True, choices=["gpt_bpe", "mamba_char"])
    ap.add_argument("--epochs", type=int, default=DEFAULT_EPOCHS)
    ap.add_argument("--steps_per_epoch", type=int, default=DEFAULT_STEPS_PER_EPOCH)
    ap.add_argument("--batch_size", type=int, default=DEFAULT_BATCH_SIZE)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--block_size", type=int, default=128)
    ap.add_argument("--n_layer", type=int, default=4)
    ap.add_argument("--n_head", type=int, default=4)
    ap.add_argument("--n_embd", type=int, default=128)
    ap.add_argument("--use_moe", action="store_true")
    ap.add_argument("--routing_mode", default="uniform", choices=["uniform", "specialize"])
    ap.add_argument("--num_experts", type=int, default=4)
    ap.add_argument("--topk", type=int, default=1)
    ap.add_argument("--router_hidden", type=int, default=128)
    ap.add_argument("--entropy_penalty", type=float, default=0.0)
    ap.add_argument("--moe_dropout", type=float, default=0.1)
    ap.add_argument("--tok_kind", default="basic")
    ap.add_argument("--vocab_size", type=int, default=512)
    ap.add_argument("--tok_prefix", default="artifacts/tokenizers/tiny_bpe")
    ap.add_argument("--save_path", default="artifacts/unified.pt")
    ap.add_argument("--load_path", default="artifacts/unified.pt")
    ap.add_argument("--sample_start", default="\n")
    ap.add_argument("--sample_tokens", type=int, default=100)
    ap.add_argument("--rm_pairs", default="")
    ap.add_argument("--exp_name", default="unified_experiment")
    args = ap.parse_args(argv)

    os.makedirs("artifacts", exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    console.print(Panel(f"Using device: {device}", title="Device"))

    # Monitor
    monitor = TrainingMonitor(args.exp_name)

    # Data
    train_txt, val_txt = load_tiny_shakespeare("./data")

    # Tokenizer / encoding
    if args.arch == "gpt_bpe":
        tok, _, vs = train_or_load_tokenizer(args.tok_kind, args.vocab_size, train_txt + val_txt, args.tok_prefix)
        train_ids, val_ids = tok.encode(train_txt), tok.encode(val_txt)
        encode_fn, decode_fn = tok.encode, tok.decode
        monitor.snapshot_tokenizer(args.tok_prefix)
    else:
        stoi, itos = build_charset(train_txt, val_txt)
        train_ids, val_ids = encode_chars(train_txt, stoi), encode_chars(val_txt, stoi)
        encode_fn = lambda s: encode_chars(s, stoi)
        decode_fn = lambda ids: "".join(itos[i] for i in ids)

    # Model (load if needed)
    model = None
    if args.mode != "train" and os.path.exists(args.load_path):
        payload = torch.load(args.load_path, map_location=device)
        kind = payload.get("kind", args.arch)
        cfg_dict = payload.get("cfg", {})
        if kind == "gpt_bpe":
            cfg = GPTCfg(**cfg_dict)
            model = NanoGPT(cfg)
        elif kind == "mamba_char":
            vocab_size = cfg_dict.get("vocab_size", len(stoi) if "stoi" in locals() else 256)
            model = NanoMamba(vocab_size, cfg_dict.get("block_size", args.block_size), cfg_dict.get("n_layer", args.n_layer), cfg_dict.get("n_embd", args.n_embd))
        else:
            raise ValueError(f"Unknown checkpoint kind: {kind}")
        model.load_state_dict(payload["state_dict"])
    else:
        if args.arch == "gpt_bpe":
            cfg = GPTCfg(block_size=args.block_size, vocab_size=vs, n_layer=args.n_layer, n_head=args.n_head, n_embd=args.n_embd,
                         dropout=0.1, use_moe=args.use_moe, routing_mode=args.routing_mode, num_experts=args.num_experts,
                         topk=args.topk, router_hidden=args.router_hidden, moe_dropout=args.moe_dropout, entropy_penalty=args.entropy_penalty)
            model = NanoGPT(cfg)
        else:
            model = NanoMamba(len(stoi), args.block_size, args.n_layer, args.n_embd)

    # Modes
    if args.mode == "train":
        title = f"{args.arch.upper()} Train"
        train_lm(model, train_ids, val_ids, args.block_size, args.epochs, args.steps_per_epoch, args.batch_size, args.lr, device, title, monitor)
        cfg_to_save = asdict(model.cfg) if hasattr(model, "cfg") else {"vocab_size": getattr(model, "vocab_size", None), "block_size": args.block_size, "n_layer": args.n_layer, "n_embd": args.n_embd}
        # save final model both to artifacts and monitor tree
        torch.save({"kind": args.arch, "cfg": cfg_to_save, "state_dict": model.state_dict()}, args.save_path)
        torch.save({"kind": args.arch, "cfg": cfg_to_save, "state_dict": model.state_dict()}, monitor.base_dir / "models" / Path(args.save_path).name)
        console.print(Panel(f"Model saved to {args.save_path}", title="Save"))
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        console.print(Panel(f"ZIP ready: {zip_path}", title="Done"))

    elif args.mode == "infer":
        payload = torch.load(args.load_path, map_location=device)
        model.load_state_dict(payload["state_dict"]); model.to(device)
        start_ids = encode_fn(args.sample_start)
        idx = torch.tensor([start_ids], device=device)
        sample = model.generate(idx, args.sample_tokens)[0].tolist()
        console.print(Panel(decode_fn(sample), title="Generated Text"))

    elif args.mode == "train_grpo":
        model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
        prompts = ["To be, or not to be", "Once upon a time", "The quick brown fox"]
        for step in range(args.steps_per_epoch):
            loss, mr = grpo_step(model, encode_fn, prompts, 32, opt, device, monitor)
            # log as pseudo-epoch (ep=step+1)
            monitor.log_training_step(step+1, step, train_loss=loss)
            console.print(Panel(f"GRPO step {step+1}/{args.steps_per_epoch} loss={loss:.3f} reward={mr:.3f}", title="GRPO"))
        cfg_to_save = asdict(model.cfg) if hasattr(model, "cfg") else {}
        torch.save({"kind": args.arch, "cfg": cfg_to_save, "state_dict": model.state_dict()}, args.save_path)
        torch.save({"kind": args.arch, "cfg": cfg_to_save, "state_dict": model.state_dict()}, monitor.base_dir / "models" / Path(args.save_path).name)
        console.print(Panel(f"GRPO-tuned model saved to {args.save_path}", title="Save"))
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        console.print(Panel(f"ZIP ready: {zip_path}", title="Done"))

    elif args.mode == "train_rm":
        if not args.rm_pairs:
            raise ValueError("--rm_pairs is required for train_rm")
        model.to(device)
        rm = RewardModel(model)
        tok_like = tok if args.arch == "gpt_bpe" else type("TK", (), {"encode": encode_fn})()
        train_reward_model(rm, tok_like, args.rm_pairs, args.block_size, epochs=args.epochs, steps=args.steps_per_epoch, batch=args.batch_size, lr=1e-5, device=device, monitor=monitor)
        torch.save({"kind": "reward_model", "base_arch": args.arch, "state_dict": rm.state_dict()}, args.save_path)
        torch.save({"kind": "reward_model", "base_arch": args.arch, "state_dict": rm.state_dict()}, monitor.base_dir / "models" / Path(args.save_path).name)
        console.print(Panel(f"Reward model saved to {args.save_path}", title="Save"))
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        console.print(Panel(f"ZIP ready: {zip_path}", title="Done"))

# =============================================================================
# FULL PIPELINE (runs everything and packages results)
# =============================================================================

def run_full_pipeline():
    console.rule("[bold magenta]Unified End-to-End Pipeline (Monitored)")

    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 50))
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 20))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 64))

    # 1) Train base GPT-BPE with MoE
    main([
        "--mode","train",
        "--arch","gpt_bpe",
        "--tok_kind","basic",
        "--vocab_size","512",
        "--use_moe","--routing_mode","specialize",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--save_path","artifacts/gpt_moe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","gpt_bpe_moe"
    ])

    # 2) Reward Model for GPT
    rm_file = "data/preferences.jsonl"
    if not os.path.exists(rm_file):
        toy = [
            {"chosen":"I love this model","rejected":"I hate this model"},
            {"chosen":"This is a great answer","rejected":"This is a terrible answer"},
        ]
        os.makedirs("data", exist_ok=True)
        with open(rm_file, "w", encoding="utf-8") as f:
            for row in toy: f.write(json.dumps(row) + "\n")
    main([
        "--mode","train_rm","--arch","gpt_bpe",
        "--rm_pairs",rm_file,
        "--load_path","artifacts/gpt_moe.pt",
        "--save_path","artifacts/reward_model.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--epochs", str(max(5, DEFAULT_EPOCHS//5)),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--exp_name","reward_model_gpt"
    ])

    # 3) GRPO fine-tuning for GPT
    main([
        "--mode","train_grpo","--arch","gpt_bpe",
        "--load_path","artifacts/gpt_moe.pt",
        "--save_path","artifacts/gpt_grpo.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--exp_name","gpt_grpo"
    ])

    # 4) Inference (base GPT-MoE)
    main([
        "--mode","infer","--arch","gpt_bpe",
        "--load_path","artifacts/gpt_moe.pt",
        "--sample_start","Hello world,",
        "--sample_tokens","50",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","gpt_bpe_moe"
    ])

    # 5) Train small Mamba
    main([
        "--mode","train","--arch","mamba_char",
        "--epochs", str(max(10, DEFAULT_EPOCHS//2)),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH//2),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--save_path","artifacts/mamba_model.pt",
        "--exp_name","mamba_char"
    ])

    # 6) GRPO fine-tuning for Mamba
    main([
        "--mode","train_grpo","--arch","mamba_char",
        "--load_path","artifacts/mamba_model.pt",
        "--save_path","artifacts/mamba_grpo.pt",
        "--steps_per_epoch", str(max(5, DEFAULT_STEPS_PER_EPOCH//2)),
        "--exp_name","mamba_grpo"
    ])

    # 7) Reward Model for Mamba (reuses the same preference file)
    main([
        "--mode","train_rm","--arch","mamba_char",
        "--rm_pairs", rm_file,
        "--load_path","artifacts/mamba_model.pt",
        "--save_path","artifacts/mamba_reward_model.pt",
        "--epochs", str(max(5, DEFAULT_EPOCHS//5)),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH//2),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--exp_name","reward_model_mamba"
    ])

    # 8) Mamba inference
    main([
        "--mode","infer","--arch","mamba_char",
        "--load_path","artifacts/mamba_model.pt",
        "--sample_start","ROMEO:",
        "--sample_tokens","80",
        "--exp_name","mamba_char"
    ])

# =============================================================================
# ENTRYPOINT
# =============================================================================

def _sanitize_argv(argv):
    out = []; skip = False
    for a in argv:
        if skip: skip = False; continue
        if a in ("-f", "--f"): skip = True; continue
        if a.startswith("-f=") or a.startswith("--f="): continue
        out.append(a)
    return out

if __name__ == "__main__":
    import sys
    argv = _sanitize_argv(sys.argv[1:])
    if ("--mode" not in argv) or ("--arch" not in argv):
        run_full_pipeline()
    else:
        main(argv)


## End-to-end: BPE tokenizer + NanoGPT (+ optional MoE) + training + 3 inference modes.

In [ ]:
#!/usr/bin/env python3
# colab_kernel_launcher.py
# End-to-end: BPE tokenizer + NanoGPT (+ optional MoE) + training + 3 inference modes.

import os, math, json, argparse, random, time, urllib.request, shlex
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Any, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------- pretty console (optional; fallback if missing) -----------
try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.table import Table
    from rich import box
    console = Console()
    def info(msg, **kw): console.print(Panel(msg, **({"border_style":"cyan","title":"Info","box":box.ROUNDED} | kw)))
    def warn(msg, **kw): console.print(Panel(msg, **({"border_style":"yellow","title":"Warning","box":box.ROUNDED} | kw)))
    def ok(msg, **kw):   console.print(Panel(msg, **({"border_style":"green","title":"OK","box":box.ROUNDED} | kw)))
except Exception:
    class _Dummy:
        def print(self, *a, **k): print(*a)
        def rule(self, *a, **k): print("="*60, *(a or ()), "="*60)
    console = _Dummy()
    def info(msg, **kw): print("[INFO]", msg)
    def warn(msg, **kw): print("[WARN]", msg)
    def ok(msg, **kw):   print("[ OK ]", msg)

# =============================================================================
#                               TOKENIZER (minBPE)
# =============================================================================
import unicodedata
import regex as re

def _replace_ctl(s: str) -> str:
    return "".join(ch if unicodedata.category(ch)[0] != "C" else f"\\u{ord(ch):04x}" for ch in s)

def _render_tok(t: bytes) -> str:
    return _replace_ctl(t.decode("utf-8", errors="replace"))

def _get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for p in zip(ids, ids[1:]): counts[p] = counts.get(p, 0) + 1
    return counts

def _merge(ids, pair, idx):
    newids=[]; i=0
    while i < len(ids):
        if ids[i]==pair[0] and i < len(ids)-1 and ids[i+1]==pair[1]:
            newids.append(idx); i+=2
        else:
            newids.append(ids[i]); i+=1
    return newids

class Tokenizer:
    def __init__(self):
        self.merges={}; self.pattern=""; self.special_tokens={}
        self.vocab={i:bytes([i]) for i in range(256)}
    def _rebuild_vocab(self):
        vocab={i:bytes([i]) for i in range(256)}
        for (a,b),i in self.merges.items(): vocab[i]=vocab[a]+vocab[b]
        for s,i in self.special_tokens.items(): vocab[i]=s.encode("utf-8")
        self.vocab=vocab
    def save(self, prefix):
        os.makedirs(os.path.dirname(prefix), exist_ok=True) if os.path.dirname(prefix) else None
        with open(prefix+".model","w",encoding="utf-8") as f:
            f.write("minbpe v1\n"); f.write(f"{self.pattern}\n"); f.write(f"{len(self.special_tokens)}\n")
            for s,i in self.special_tokens.items(): f.write(f"{s} {i}\n")
            for (a,b),_i in self.merges.items(): f.write(f"{a} {b}\n")
        with open(prefix+".vocab","w",encoding="utf-8") as f:
            inv={idx:pair for pair,idx in self.merges.items()}
            for i,tok in self.vocab.items():
                s=_render_tok(tok)
                if i in inv:
                    a,b=inv[i]; f.write(f"[{_render_tok(self.vocab[a])}][{_render_tok(self.vocab[b])}] -> [{s}] {i}\n")
                else:
                    f.write(f"[{s}] {i}\n")
    def load(self, model_file):
        merges={}; specials={}
        idx=256
        with open(model_file,"r",encoding="utf-8") as f:
            version=f.readline().strip(); assert version=="minbpe v1"
            self.pattern=f.readline().strip()
            ns=int(f.readline().strip())
            for _ in range(ns):
                s,si=f.readline().strip().split(); specials[s]=int(si)
            for line in f:
                a,b=map(int,line.split()); merges[(a,b)]=idx; idx+=1
        self.merges=merges; self.special_tokens=specials; self._rebuild_vocab()

class BasicTokenizer(Tokenizer):
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size>=256; num_merges=vocab_size-256
        ids=list(text.encode("utf-8")); merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats=_get_stats(ids); pair=max(stats,key=stats.get)
            idx=256+i; ids=_merge(ids,pair,idx); merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
            if verbose and i<10: info(f"merge {i+1}/{num_merges}: {pair} -> {idx}")
        self.merges=merges; self.vocab=vocab
    def encode(self, text):
        ids=list(text.encode("utf-8"))
        while len(ids)>=2:
            stats=_get_stats(ids); pair=min(stats,key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=_merge(ids,pair,self.merges[pair])
        return ids
    def decode(self, ids): return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

GPT4_SPLIT = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexTokenizer(Tokenizer):
    def __init__(self): super().__init__(); self.pattern=GPT4_SPLIT; self.compiled=re.compile(self.pattern)
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size>=256; num_merges=vocab_size-256
        chunks=re.findall(self.compiled,text); ids=[list(c.encode("utf-8")) for c in chunks]
        merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats={}; [ _get_stats(ci,stats) for ci in ids ]
            pair=max(stats,key=stats.get); idx=256+i; ids=[_merge(ci,pair,idx) for ci in ids]
            merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
            if verbose and i<10: info(f"merge {i+1}/{num_merges}: {pair} -> {idx}")
        self.merges=merges; self.vocab=vocab
    def _encode_chunk(self, bs):
        ids=list(bs)
        while len(ids)>=2:
            stats=_get_stats(ids); pair=min(stats,key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=_merge(ids,pair,self.merges[pair])
        return ids
    def encode(self, text):
        ids=[]; [ids.extend(self._encode_chunk(c.encode("utf-8"))) for c in re.findall(self.compiled,text)]
        return ids
    def decode(self, ids): return BasicTokenizer.decode(self, ids)

def _new_tok(kind): return BasicTokenizer() if kind=="basic" else RegexTokenizer()

def train_or_load_tokenizer(kind,vocab_size,text,prefix,verbose=True):
    os.makedirs(os.path.dirname(prefix),exist_ok=True) if os.path.dirname(prefix) else None
    model=prefix+".model"
    if os.path.exists(model):
        tok=_new_tok(kind); tok.load(model); vs=max(tok.vocab.keys())+1
        if verbose: ok(f"Loaded tokenizer {model} (size={vs})", title="Tokenizer")
        return tok,model,vs
    tok=_new_tok(kind); tok.train(text,max(256,vocab_size),verbose); tok.save(prefix); vs=max(tok.vocab.keys())+1
    if verbose: ok(f"Saved tokenizer {prefix}.model (size={vs})", title="Tokenizer")
    return tok,model,vs

# =============================================================================
#                              DATA (tiny shakespeare)
# =============================================================================
def _repeat_to_len(s: str, target_len: int) -> str:
    if not s: s=" \n"
    return (s * ((target_len // len(s))+1))[:target_len]

def load_tiny_shakespeare(data_dir="./data", target_len=100_000):
    os.makedirs(data_dir,exist_ok=True)
    path=os.path.join(data_dir,"tinyshakespeare_input.txt")
    if not os.path.exists(path):
        try:
            url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url,path)
        except Exception:
            warn("Could not download dataset; writing fallback sample.")
            sample=("From fairest creatures we desire increase,\n"
                    "That thereby beauty's rose might never die,\n"
                    "But as the riper should by time decease,\n"
                    "His tender heir might bear his memory:\n")
            with open(path,"w",encoding="utf-8") as f: f.write(_repeat_to_len(sample, target_len))
    text=open(path,"r",encoding="utf-8").read()
    split=int(0.9*len(text)); return text[:split], text[split:]

def get_batch_tokens(ids, block_size, batch_size, device):
    if len(ids) <= block_size + 1:
        raise RuntimeError(f"Sequence too short ({len(ids)}) for block_size={block_size}.")
    ix=torch.randint(len(ids)-block_size-1,(batch_size,))
    x=torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y=torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================================================================
#                               MODEL (GPT + MoE)
# =============================================================================
class LayerNorm(nn.Module):
    def __init__(self,n,bias): super().__init__(); self.weight=nn.Parameter(torch.ones(n)); self.bias=nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self,x): return F.layer_norm(x,self.weight.shape,self.weight,self.bias,1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self,n_embd,n_head,block,dropout,bias,backend="sdpa"):
        super().__init__(); assert n_embd % n_head == 0
        self.n_head=n_head; self.n_embd=n_embd; self.dropout=dropout; self.backend=backend
        self.c_attn=nn.Linear(n_embd,3*n_embd,bias=bias); self.c_proj=nn.Linear(n_embd,n_embd,bias=bias)
        if backend=="vanilla" or not hasattr(F,"scaled_dot_product_attention"):
            self.register_buffer("bias", torch.tril(torch.ones(block,block)).view(1,1,block,block))
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        if self.backend=="sdpa" and hasattr(F,"scaled_dot_product_attention"):
            y=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=self.dropout if self.training else 0.0)
        else:
            att=(q@k.transpose(-2,-1))/math.sqrt(k.size(-1))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0,float("-inf"))
            att=F.softmax(att,dim=-1); y=att@v
        y=y.transpose(1,2).contiguous().view(B,T,C); return self.c_proj(y)

def entropy_mean(probs,eps=1e-9): return -(probs.clamp_min(eps)*probs.clamp_min(eps).log()).sum(-1).mean()

class TopKRouter(nn.Module):
    def __init__(self,in_dim,hidden_dim,num_experts,k=1,temp=1.0):
        super().__init__(); self.net=nn.Sequential(nn.Linear(in_dim,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,num_experts))
        self.k=k; self.temp=float(temp); self.register_buffer("logits_bias", torch.zeros(num_experts))
    @staticmethod
    def _gumbel(shape, device):
        u=torch.rand(shape,device=device).clamp_(1e-9,1-1e-9); return -torch.log(-torch.log(u))
    def forward(self,x,add_gumbel=False):
        logits=self.net(x)/self.temp + self.logits_bias
        if add_gumbel and self.training: logits = logits + self._gumbel(logits.shape, logits.device)
        probs=F.softmax(logits,dim=-1)
        if self.k>=probs.size(-1): return probs, probs
        topk_vals,topk_idx=torch.topk(probs,self.k,dim=-1)
        mask=torch.zeros_like(probs); mask.scatter_(dim=-1,index=topk_idx,src=torch.ones_like(topk_vals))
        sp=probs*mask; sp=sp/(sp.sum(dim=-1,keepdim=True)+1e-9); return sp,probs

class MoEFFN(nn.Module):
    def __init__(self,in_dim,num_experts=3,k=1,router_hidden=128,dropout=0.1,blw=0.02,entropy_penalty=0.001,
                 routing_mode="specialize", router_temp=1.0):
        super().__init__(); hidden=4*in_dim
        self.expert_fc=nn.ModuleList([nn.Linear(in_dim,hidden) for _ in range(num_experts)])
        self.expert_proj=nn.ModuleList([nn.Linear(hidden,in_dim) for _ in range(num_experts)])
        self.shared_fc=nn.Linear(in_dim,hidden); self.shared_proj=nn.Linear(hidden,in_dim)
        self.router=TopKRouter(in_dim,router_hidden,num_experts,k,temp=router_temp)
        self.drop=nn.Dropout(dropout); self.num_experts=num_experts
        self.blw=blw; self.entw=entropy_penalty; self.routing_mode=routing_mode
    def forward(self,x):
        B,T,C=x.shape; xf=x.view(B*T,C); sp,dp=self.router(xf,add_gumbel=self.training)
        routed=0.0
        for e in range(self.num_experts):
            h=self.expert_proj[e](F.gelu(self.expert_fc[e](xf))); routed+=sp[:,e].unsqueeze(-1)*h
        shared=self.shared_proj(F.gelu(self.shared_fc(xf)))
        y=self.drop((shared+routed).view(B,T,C))
        aux=(-self.entw*entropy_mean(dp)) if self.routing_mode=="specialize" else torch.tensor(0.0,device=x.device)
        stats={"expert_selection_counts":(sp>0).float().sum(0),"mean_routing_probs":dp.mean(0)}
        return y,aux,stats

@dataclass
class GPTCfg:
    block_size:int=256; vocab_size:int=256; n_layer:int=6; n_head:int=6; n_embd:int=384
    dropout:float=0.1; bias:bool=True; attention_backend:str="sdpa"
    use_moe:bool=False; num_experts:int=3; k:int=1; router_hidden:int=128; blw:float=0.02
    entropy_penalty:float=0.001; routing_mode:str="specialize"; router_temp:float=1.0

class GPTBlock(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout, bias, backend="sdpa", moe: Optional[MoEFFN]=None):
        super().__init__(); self.ln1=LayerNorm(n_embd,bias); self.attn=CausalSelfAttention(n_embd,n_head,block,dropout,bias,backend)
        self.ln2=LayerNorm(n_embd,bias); self.moe=moe
        if moe is None:
            hidden=4*n_embd
            self.ffn=nn.Sequential(nn.Linear(n_embd,hidden,bias=bias), nn.GELU(), nn.Linear(hidden,n_embd,bias=bias), nn.Dropout(dropout))
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        if self.moe is None:
            x=x+self.ffn(self.ln2(x)); aux=torch.tensor(0.0,device=x.device); stats={}
        else:
            y,aux,stats=self.moe(self.ln2(x)); x=x+y
        return x, aux, stats

class NanoGPT(nn.Module):
    def __init__(self, cfg:GPTCfg):
        super().__init__(); self.cfg=cfg
        self.wte=nn.Embedding(cfg.vocab_size,cfg.n_embd); self.wpe=nn.Embedding(cfg.block_size,cfg.n_embd)
        blocks=[]
        for _ in range(cfg.n_layer):
            moe=None
            if cfg.use_moe: moe=MoEFFN(cfg.n_embd,cfg.num_experts,cfg.k,cfg.router_hidden,cfg.dropout,cfg.blw,cfg.entropy_penalty,cfg.routing_mode,cfg.router_temp)
            blocks.append(GPTBlock(cfg.n_embd,cfg.n_head,cfg.block_size,cfg.dropout,cfg.bias,cfg.attention_backend,moe))
        self.blocks=nn.ModuleList(blocks); self.ln_f=LayerNorm(cfg.n_embd,cfg.bias); self.lm_head=nn.Linear(cfg.n_embd,cfg.vocab_size,bias=False)
        self.lm_head.weight=self.wte.weight
        self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward(self, idx, targets=None):
        B,T=idx.shape; assert T<=self.cfg.block_size
        pos=torch.arange(0,T,device=idx.device).long(); x=self.wte(idx)+self.wpe(pos)[None,:,:]
        aux_total=torch.tensor(0.0,device=idx.device); last={}
        for blk in self.blocks:
            x,aux,stats=blk(x); aux_total=aux_total+aux; last=stats
        x=self.ln_f(x); logits=self.lm_head(x); loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1)); loss=loss+aux_total
        return logits, loss, last
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.cfg.block_size else idx[:,-self.cfg.block_size:]
            logits,_,_=self(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); idx_next=torch.multinomial(probs,1); idx=torch.cat((idx,idx_next),dim=1)
        return idx

def count_parameters(m: nn.Module) -> Tuple[int,int]:
    total=sum(p.numel() for p in m.parameters())
    train=sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, train

# =============================================================================
#                            TRAIN / SAMPLE / SAVELOAD
# =============================================================================
def train_lm(model,train_ids,val_ids,block_size,epochs,steps,batch,lr,device,title):
    model=model.to(device); opt=torch.optim.AdamW(model.parameters(),lr=lr,betas=(0.9,0.95),weight_decay=0.1)
    for ep in range(1,epochs+1):
        t0=time.time(); model.train(); losses=[]
        for _ in range(steps):
            xb,yb=get_batch_tokens(train_ids,block_size,batch,device)
            _,loss,_=model(xb,yb)
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            losses.append(loss.item())
        dt=time.time()-t0
        with torch.no_grad():
            model.eval(); xb,yb=get_batch_tokens(val_ids,block_size,batch,device); _,vl,_=model(xb,yb)
        info(f"Epoch {ep}/{epochs}  train={sum(losses)/len(losses):.3f}  val={vl.item():.3f}  time={dt:.1f}s", title=title)

@torch.no_grad()
def sample_lm(model,tok,device,start="\n",tokens=200,title="Sample"):
    model.eval().to(device)
    x=torch.tensor([tok.encode(start)],dtype=torch.long,device=device)
    y=model.generate(x,max_new_tokens=tokens,temperature=0.8,top_k=200)[0].tolist()
    text=tok.decode(y); info(text, title=title)

def save_ckpt(path,payload):
    os.makedirs(os.path.dirname(path),exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload,path); ok(f"Saved to {path}")

def load_ckpt(path):
    p=torch.load(path,map_location="cpu"); ok(f"Loaded {path}"); return p

# =============================================================================
#                          INFERENCE STRATEGIES
# =============================================================================
@torch.no_grad()
def _last_token_logprobs(model, input_ids: torch.Tensor):
    logits, _, _ = model(input_ids)
    return F.log_softmax(logits[:, -1, :], dim=-1)

def test_time_adapt_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    adapt_steps: int = 2, adapt_lr: float = 5e-4, adapt_layers: str = "lm_head,ln_f",
    context_reuse: int = 64, temperature: float = 0.8, top_k: int = 200,
):
    """Few gradient steps on a small parameter subset over recent context before each token."""
    model = model.to(device)
    model.eval()
    # select params
    adapt_names = {n.strip() for n in adapt_layers.split(",")} if adapt_layers else set()
    for p in model.parameters(): p.requires_grad=False
    to_adapt=[]
    if "lm_head" in adapt_names:
        for p in model.lm_head.parameters(): p.requires_grad=True; to_adapt.append(p)
    if "ln_f" in adapt_names:
        for p in model.ln_f.parameters(): p.requires_grad=True; to_adapt.append(p)
    if "blocks[-1]" in adapt_names and hasattr(model,"blocks"):
        for p in model.blocks[-1].parameters(): p.requires_grad=True; to_adapt.append(p)
    opt = torch.optim.Adam([p for p in to_adapt if p.requires_grad], lr=adapt_lr) if to_adapt else None

    ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
    for _ in range(max_new_tokens):
        if opt is not None and ids.size(1)>1:
            model.train()
            for _step in range(adapt_steps):
                ctx=ids[:,-min(context_reuse, ids.size(1)-1):]
                inp, tgt = ctx[:,:-1], ctx[:,1:]
                opt.zero_grad(set_to_none=True)
                _, loss, _ = model(inp, tgt)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(to_adapt, 1.0)
                opt.step()
        model.eval()
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        logits,_,_=model(cond); logits=logits[:,-1,:]/temperature
        if top_k is not None:
            v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
        probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1)
        ids=torch.cat([ids,nxt],dim=1)
    return tok.decode(ids[0].tolist())

def _rollout_logprob(model, start_ids: torch.Tensor, horizon: int, temperature: float, top_k: Optional[int]):
    model.eval(); ids=start_ids.clone()
    total_logprob=0.0; total_entropy=0.0
    for _ in range(horizon):
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        logits,_,_=model(cond); logits=logits[:,-1,:]/temperature
        if top_k is not None:
            v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
        logprobs=F.log_softmax(logits,dim=-1); probs=logprobs.exp()
        entropy=-(probs*logprobs).sum(dim=-1).mean()
        nxt=torch.multinomial(probs,1)
        total_logprob+=logprobs.gather(-1,nxt).mean().item()
        total_entropy+=entropy.item()
        ids=torch.cat([ids,nxt],dim=1)
    return total_logprob, total_entropy

@torch.no_grad()
def active_inference_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    particles: int = 8, horizon: int = 4, beta: float = 0.2, temperature: float = 0.9, top_k: Optional[int] = 200,
):
    """Minimize expected free energy proxy = -Σ log p + β·Σ entropy via short rollouts per candidate action."""
    model=model.to(device).eval()
    ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
    for _ in range(max_new_tokens):
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        base_lp=_last_token_logprobs(model,cond)  # (1,V)
        probs=base_lp.exp()
        cand=torch.multinomial(probs,num_samples=particles)  # (1,P)
        best_F=float("inf"); best_token=None
        for j in range(particles):
            a=cand[:,j:j+1]
            start=torch.cat([cond,a],dim=1)
            logp_sum, ent_sum=_rollout_logprob(model,start,horizon,temperature,top_k)
            F=-logp_sum + beta*ent_sum
            if F<best_F: best_F=F; best_token=a
        if best_token is None:
            best_token=torch.argmax(base_lp,dim=-1,keepdim=True)
        ids=torch.cat([ids,best_token],dim=1)
    return tok.decode(ids[0].tolist())

@torch.no_grad()
def system2_reasoning_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    branches: int = 8, temperature: float = 0.9, top_k: Optional[int] = 200, vote: str = "majority",
):
    """Self-consistency: sample multiple full continuations and pick majority or best average logprob."""
    model=model.to(device).eval()
    def _sample_once():
        ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
        for _ in range(max_new_tokens):
            cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
            logits,_,_=model(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1)
            ids=torch.cat([ids,nxt],dim=1)
        # score by avg logprob
        if ids.size(1)>1:
            inp, tgt = ids[:,:-1], ids[:,1:]
            _, loss, _ = model(inp, tgt)
            avg_lp = -loss.item() if loss is not None else 0.0
        else:
            avg_lp = 0.0
        return tok.decode(ids[0].tolist()), avg_lp

    samples=[_sample_once() for _ in range(branches)]
    if vote=="best_logprob":
        return max(samples, key=lambda x:x[1])[0]
    from collections import Counter
    cleaned=[s[0].strip() for s in samples]
    return Counter(cleaned).most_common(1)[0][0]

# =============================================================================
#                                     MAIN
# =============================================================================
def main(argv=None):
    ap=argparse.ArgumentParser(description="NanoGPT + BPE + MoE + TTA/ActiveInference/System2")
    ap.add_argument("--mode",required=True,choices=["train_gpt_bpe","infer_gpt_bpe","test_time","active_inference","system2_reasoning"])
    ap.add_argument("--device",default="auto",choices=["auto","cpu","cuda"])
    ap.add_argument("--epochs",type=int,default=2)
    ap.add_argument("--steps_per_epoch",type=int,default=200)
    ap.add_argument("--batch_size",type=int,default=64)
    ap.add_argument("--lr",type=float,default=3e-4)
    ap.add_argument("--block_size",type=int,default=256)
    ap.add_argument("--dropout",type=float,default=0.1)
    ap.add_argument("--n_layer",type=int,default=6)
    ap.add_argument("--n_head",type=int,default=6)
    ap.add_argument("--n_embd",type=int,default=384)
    ap.add_argument("--bias",action="store_true",default=True)
    ap.add_argument("--attention_backend",default="sdpa",choices=["sdpa","vanilla"])

    # MoE
    ap.add_argument("--use_moe",action="store_true")
    ap.add_argument("--num_experts",type=int,default=3)
    ap.add_argument("--k",type=int,default=1)
    ap.add_argument("--router_hidden",type=int,default=128)
    ap.add_argument("--blw",type=float,default=0.02)
    ap.add_argument("--entropy_penalty",type=float,default=0.001)
    ap.add_argument("--routing_mode",choices=["specialize","uniform"],default="specialize")
    ap.add_argument("--router_temp",type=float,default=1.0)

    # Tokenizer/data/checkpoints
    ap.add_argument("--tok_kind",default="basic",choices=["basic","regex"])
    ap.add_argument("--vocab_size",type=int,default=1024)
    ap.add_argument("--tok_prefix",default="artifacts/tokenizers/tiny_bpe")
    ap.add_argument("--train_corpus",default="")
    ap.add_argument("--save_path",default="artifacts/gpt_moe_bpe.pt")
    ap.add_argument("--load_path",default="artifacts/gpt_moe_bpe.pt")
    ap.add_argument("--seed",type=int,default=1337)

    # common sampling
    ap.add_argument("--sample_start",default="\n")
    ap.add_argument("--sample_tokens",type=int,default=200)
    ap.add_argument("--temperature",type=float,default=0.8)
    ap.add_argument("--top_k",type=int,default=200)

    # TTA
    ap.add_argument("--tta_steps",type=int,default=2)
    ap.add_argument("--tta_lr",type=float,default=5e-4)
    ap.add_argument("--tta_layers",default="lm_head,ln_f")
    ap.add_argument("--tta_context",type=int,default=64)

    # Active inference
    ap.add_argument("--ai_particles",type=int,default=8)
    ap.add_argument("--ai_horizon",type=int,default=4)
    ap.add_argument("--ai_beta",type=float,default=0.2)

    # System-2
    ap.add_argument("--s2_branches",type=int,default=8)
    ap.add_argument("--s2_vote",default="majority",choices=["majority","best_logprob"])

    args=ap.parse_args(argv)
    random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
    device=args.device if args.device!="auto" else ("cuda" if torch.cuda.is_available() else "cpu")

    # Load text
    if args.train_corpus and os.path.exists(args.train_corpus):
        full=open(args.train_corpus,"r",encoding="utf-8").read()
        split=int(0.9*len(full)); train_txt, val_txt = full[:split], full[split:]
    else:
        train_txt, val_txt = load_tiny_shakespeare("./data")

    # Tokenizer
    tok, tok_model_path, vocab_size_actual = train_or_load_tokenizer(args.tok_kind,args.vocab_size,train_txt+val_txt,args.tok_prefix,verbose=True)

    # Encode
    train_ids = tok.encode(train_txt)
    val_ids   = tok.encode(val_txt)

    if len(train_ids) <= args.block_size + 1:
        new_bs=max(16, min(args.block_size, len(train_ids)-2))
        warn(f"Auto-adjusting block_size {args.block_size} -> {new_bs}", title="Safety")
        args.block_size=new_bs

    def build_model():
        cfg=GPTCfg(
            block_size=args.block_size, vocab_size=vocab_size_actual, n_layer=args.n_layer, n_head=args.n_head,
            n_embd=args.n_embd, dropout=args.dropout, bias=args.bias, attention_backend=args.attention_backend,
            use_moe=args.use_moe, num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden,
            blw=args.blw, entropy_penalty=args.entropy_penalty, routing_mode=args.routing_mode, router_temp=args.router_temp
        )
        return cfg, NanoGPT(cfg)

    if args.mode=="train_gpt_bpe":
        cfg, model = build_model()
        train_lm(model,train_ids,val_ids,cfg.block_size,args.epochs,args.steps_per_epoch,args.batch_size,args.lr,device,
                 title=f"NanoGPT-BPE ({'MoE' if cfg.use_moe else 'no-MoE'})")
        save_ckpt(args.save_path, {"kind":"gpt_bpe","cfg":asdict(cfg),"state_dict":model.state_dict(),
                                   "tokenizer":{"kind":args.tok_kind,"model_path":tok_model_path,"vocab_size":vocab_size_actual}})

    elif args.mode=="infer_gpt_bpe":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        sample_lm(model,tok2,device,start=args.sample_start,tokens=args.sample_tokens,title="Greedy Sample")

    elif args.mode=="test_time":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=test_time_adapt_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.tta_steps,args.tta_lr,
                                     args.tta_layers,args.tta_context,args.temperature,args.top_k)
        info(out, title="Test-Time Adaptation Output")

    elif args.mode=="active_inference":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=active_inference_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.ai_particles,
                                      args.ai_horizon,args.ai_beta,args.temperature,args.top_k)
        info(out, title="Active Inference Output")

    elif args.mode=="system2_reasoning":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=system2_reasoning_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.s2_branches,
                                       args.temperature,args.top_k,args.s2_vote)
        info(out, title="System-2 Reasoning Output")

# =============================================================================
# FULL PIPELINE (runs all modes and packages results)
# =============================================================================
def run_full_pipeline():
    console.rule("[bold magenta]NanoGPT-BPE End-to-End Pipeline (Train + All Inference Modes)")

    # Conservative defaults for Colab
    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 50))  # Short training for demo
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 100))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 32))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    console.print(Panel(f"Using device: {device}", title="Device"))

    # 1) Train NanoGPT with BPE (no MoE for simplicity)
    console.rule("[bold cyan]Step 1: Training NanoGPT with BPE")
    train_args = [
        "--mode", "train_gpt_bpe",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--lr", "3e-4",
        "--block_size", "128",  # Smaller context for Colab
        "--n_layer", "4",       # Smaller model
        "--n_head", "4",
        "--n_embd", "256",
        "--vocab_size", "512",  # Smaller vocab
        "--tok_kind", "regex",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--save_path", "artifacts/gpt_bpe.pt",
        "--seed", "42",
        "--device", device
    ]
    try:
        main(train_args)
    except Exception as e:
        console.print(Panel(f"Training failed: {e}\nFalling back to smaller model...", title="Warning"))
        fallback_args = [
            "--mode", "train_gpt_bpe",
            "--epochs", "2",
            "--steps_per_epoch", "50",
            "--batch_size", "16",
            "--lr", "3e-4",
            "--block_size", "64",
            "--n_layer", "2",
            "--n_head", "2",
            "--n_embd", "128",
            "--vocab_size", "256",
            "--tok_kind", "regex",
            "--tok_prefix", "artifacts/tokenizers/tiny_bpe_fallback",
            "--save_path", "artifacts/gpt_bpe.pt",
            "--seed", "42",
            "--device", device
        ]
        main(fallback_args)

    # Common inference parameters
    prompt = "In a world where AI and humans collaborate,"
    sample_tokens = 80

    # 2) Standard Inference
    console.rule("[bold cyan]Step 2: Standard Inference")
    main([
        "--mode", "infer_gpt_bpe",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--temperature", "0.8",
        "--top_k", "50",
        "--tok_kind", "regex",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--device", device
    ])

    # 3) Test-Time Adaptation
    console.rule("[bold cyan]Step 3: Test-Time Adaptation")
    main([
        "--mode", "test_time",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--tta_steps", "3",
        "--tta_lr", "1e-3",
        "--tta_layers", "lm_head,ln_f",
        "--tta_context", "64",
        "--temperature", "0.8",
        "--top_k", "50",
        "--tok_kind", "regex",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--device", device
    ])

    # 4) Active Inference
    console.rule("[bold cyan]Step 4: Active Inference")
    main([
        "--mode", "active_inference",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--ai_particles", "6",
        "--ai_horizon", "3",
        "--ai_beta", "0.3",
        "--temperature", "0.8",
        "--top_k", "50",
        "--tok_kind", "regex",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--device", device
    ])

    # 5) System-2 Reasoning
    console.rule("[bold cyan]Step 5: System-2 Reasoning")
    main([
        "--mode", "system2_reasoning",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--s2_branches", "5",
        "--s2_vote", "best_logprob",
        "--temperature", "0.8",
        "--top_k", "50",
        "--tok_kind", "regex",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--device", device
    ])

    # 6) Package results
    console.rule("[bold magenta]Packaging Results")
    import zipfile
    from pathlib import Path
    zip_path = "artifacts/pipeline_results.zip"
    os.makedirs("artifacts", exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in Path("artifacts").glob("*.pt"):
            zf.write(file, file.relative_to("."))
        for file in Path("artifacts/tokenizers").glob("*.model"):
            zf.write(file, file.relative_to("."))
        for file in Path("artifacts/tokenizers").glob("*.vocab"):
            zf.write(file, file.relative_to("."))
    console.print(Panel(f"Pipeline complete! Results saved to {zip_path}", title="Done"))

# =============================================================================
# ENTRYPOINT
# =============================================================================
def _sanitize_argv(argv):
    out = []; skip = False
    for a in argv:
        if skip: skip = False; continue
        if a in ("-f", "--f"): skip = True; continue
        if a.startswith("-f=") or a.startswith("--f="): continue
        out.append(a)
    return out

if __name__ == "__main__":
    import sys
    argv = _sanitize_argv(sys.argv[1:])
    # If required --mode arg is missing, run the full pipeline
    if "--mode" not in argv:
        run_full_pipeline()
    else:
        main(argv)

────────────────────────── NanoGPT-BPE End-to-End Pipeline (Train + All Inference Modes) ──────────────────────────

╭──────────────────────────────────────────────────── Device ─────────────────────────────────────────────────────╮
│ Using device: cuda                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────────────────────── Step 1: Training NanoGPT with BPE ────────────────────────────────────────

╭─────────────────────────────────────────────────── Tokenizer ───────────────────────────────────────────────────╮
│ Loaded tokenizer artifacts/tokenizers/tiny_bpe.model (size=512)                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── NanoGPT-BPE (no-MoE) ──────────────────────────────────────────────╮
│ Epoch 1/50  train=4.980  val=4.333  time=3.5s                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── Tokenizer ───────────────────────────────────────────────────╮
│ Loaded tokenizer artifacts/tokenizers/tiny_bpe.model (size=512)                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── OK ───────────────────────────────────────────────────────╮
│ Loaded artifacts/gpt_bpe.pt                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── System-2 Reasoning Output ───────────────────────────────────────────╮
│ In a world where AI and humans collaborate,--                                                                   │
│ Is ever well begin to breathed,--                                                                               │
│ As Hermione,--heard, for Bolingbroke,                                                                           │
│ Unless the petition of my power                                                                                 │
│ In manage particular eye. For my time                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────────────────────────────── Packaging Results ────────────────────────────────────────────────

╭───────────────────────────────────────────────────── Done ──────────────────────────────────────────────────────╮
│ Pipeline complete! Results saved to artifacts/pipeline_results.zip                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Longer Training

In [ ]:
def main(argv=None):
    ap=argparse.ArgumentParser(description="NanoGPT + BPE + MoE + TTA/ActiveInference/System2")
    ap.add_argument("--mode",required=True,choices=["train_gpt_bpe","infer_gpt_bpe","test_time","active_inference","system2_reasoning"])
    ap.add_argument("--device",default="auto",choices=["auto","cpu","cuda"])
    ap.add_argument("--epochs",type=int,default=200)  # Increased
    ap.add_argument("--steps_per_epoch",type=int,default=200)
    ap.add_argument("--batch_size",type=int,default=32)
    ap.add_argument("--lr",type=float,default=3e-4)
    ap.add_argument("--block_size",type=int,default=128)
    ap.add_argument("--dropout",type=float,default=0.2)
    ap.add_argument("--n_layer",type=int,default=6)  # Increased
    ap.add_argument("--n_head",type=int,default=8)  # Increased
    ap.add_argument("--n_embd",type=int,default=384)  # Increased
    ap.add_argument("--bias",action="store_true",default=True)
    ap.add_argument("--attention_backend",default="sdpa",choices=["sdpa","vanilla"])
    ap.add_argument("--use_moe",action="store_true")
    ap.add_argument("--num_experts",type=int,default=3)
    ap.add_argument("--k",type=int,default=1)
    ap.add_argument("--router_hidden",type=int,default=128)
    ap.add_argument("--blw",type=float,default=0.02)
    ap.add_argument("--entropy_penalty",type=float,default=0.001)
    ap.add_argument("--routing_mode",choices=["specialize","uniform"],default="specialize")
    ap.add_argument("--router_temp",type=float,default=1.0)
    ap.add_argument("--tok_kind",default="regex",choices=["basic","regex"])
    ap.add_argument("--vocab_size",type=int,default=512)
    ap.add_argument("--tok_prefix",default="artifacts/tokenizers/tiny_bpe")
    ap.add_argument("--train_corpus",default="")
    ap.add_argument("--save_path",default="artifacts/gpt_bpe.pt")
    ap.add_argument("--load_path",default="artifacts/gpt_bpe.pt")
    ap.add_argument("--seed",type=int,default=1337)
    ap.add_argument("--sample_start",default="\n")
    ap.add_argument("--sample_tokens",type=int,default=100)
    ap.add_argument("--temperature",type=float,default=0.7)
    ap.add_argument("--top_k",type=int,default=100)
    ap.add_argument("--tta_steps",type=int,default=5)
    ap.add_argument("--tta_lr",type=float,default=1e-4)
    ap.add_argument("--tta_layers",default="lm_head,ln_f")
    ap.add_argument("--tta_context",type=int,default=128)
    ap.add_argument("--ai_particles",type=int,default=10)
    ap.add_argument("--ai_horizon",type=int,default=5)
    ap.add_argument("--ai_beta",type=float,default=0.1)
    ap.add_argument("--s2_branches",type=int,default=8)
    ap.add_argument("--s2_vote",default="majority",choices=["majority","best_logprob"])

    args=ap.parse_args(argv)
    random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
    device=args.device if args.device!="auto" else ("cuda" if torch.cuda.is_available() else "cpu")

    # Load text
    if args.train_corpus and os.path.exists(args.train_corpus):
        full=open(args.train_corpus,"r",encoding="utf-8").read()
        split=int(0.9*len(full)); train_txt, val_txt = full[:split], full[split:]
    else:
        train_txt, val_txt = load_tiny_shakespeare("./data")

    # Tokenizer
    tok, tok_model_path, vocab_size_actual = train_or_load_tokenizer(args.tok_kind,args.vocab_size,train_txt+val_txt,args.tok_prefix,verbose=True)

    # Encode
    train_ids = tok.encode(train_txt)
    val_ids   = tok.encode(val_txt)

    if len(train_ids) <= args.block_size + 1:
        new_bs=max(16, min(args.block_size, len(train_ids)-2))
        warn(f"Auto-adjusting block_size {args.block_size} -> {new_bs}", title="Safety")
        args.block_size=new_bs

    def build_model():
        cfg=GPTCfg(
            block_size=args.block_size, vocab_size=vocab_size_actual, n_layer=args.n_layer, n_head=args.n_head,
            n_embd=args.n_embd, dropout=args.dropout, bias=args.bias, attention_backend=args.attention_backend,
            use_moe=args.use_moe, num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden,
            blw=args.blw, entropy_penalty=args.entropy_penalty, routing_mode=args.routing_mode, router_temp=args.router_temp
        )
        return cfg, NanoGPT(cfg)

    if args.mode=="train_gpt_bpe":
        cfg, model = build_model()
        model = train_lm(model,train_ids,val_ids,cfg.block_size,args.epochs,args.steps_per_epoch,args.batch_size,args.lr,device,
                         title=f"NanoGPT-BPE ({'MoE' if cfg.use_moe else 'no-MoE'})")
        save_ckpt(args.save_path, {"kind":"gpt_bpe","cfg":asdict(cfg),"state_dict":model.state_dict(),
                                   "tokenizer":{"kind":args.tok_kind,"model_path":tok_model_path,"vocab_size":vocab_size_actual}})

    elif args.mode=="infer_gpt_bpe":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        sample_lm(model,tok2,device,start=args.sample_start,tokens=args.sample_tokens,title="Greedy Sample")

    elif args.mode=="test_time":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=test_time_adapt_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.tta_steps,args.tta_lr,
                                     args.tta_layers,args.tta_context,args.temperature,args.top_k)
        info(out, title="Test-Time Adaptation Output")

    elif args.mode=="active_inference":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=active_inference_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.ai_particles,
                                      args.ai_horizon,args.ai_beta,args.temperature,args.top_k)
        info(out, title="Active Inference Output")

    elif args.mode=="system2_reasoning":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=system2_reasoning_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.s2_branches,
                                       args.temperature,args.top_k,args.s2_vote)
        info(out, title="System-2 Reasoning Output")

# =============================================================================
# ENTRYPOINT
# =============================================================================
def _sanitize_argv(argv):
    out = []; skip = False
    for a in argv:
        if skip: skip = False; continue
        if a in ("-f", "--f"): skip = True; continue
        if a.startswith("-f=") or a.startswith("--f="): continue
        out.append(a)
    return out

if __name__ == "__main__":
    import sys
    argv = _sanitize_argv(sys.argv[1:])
    if "--mode" not in argv:
        run_full_pipeline()
    else:
        main(argv)

────────────────────────── NanoGPT-BPE End-to-End Pipeline (Train + All Inference Modes) ──────────────────────────

╭──────────────────────────────────────────────────── Device ─────────────────────────────────────────────────────╮
│ Using device: cuda                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────────────────────── Step 1: Training NanoGPT with BPE ────────────────────────────────────────

╭─────────────────────────────────────────────────── Tokenizer ───────────────────────────────────────────────────╮
│ Loaded tokenizer artifacts/tokenizers/tiny_bpe.model (size=512)                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── NanoGPT-BPE (no-MoE) ──────────────────────────────────────────────╮
│ Epoch 1/100  train=4.549  val=3.969  lr=0.000300  time=7.5s                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────────── OK ───────────────────────────────────────────────────────╮
│ Loaded artifacts/gpt_bpe.pt                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                               System-2 Branches                                
┏━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Branch ┃ Logprob ┃ Output                                                    ┃
┡━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1      │ -2.074  │ In a world where AI and humans collaborate, sweet scope,  │
│        │         │ And speak with kissing Duke of Norfolk.                   │
│        │         │                                                           │
│        │         │ RI...                                                     │
│ 2      │ -2.790  │ In a world where AI and humans collaborate, that know     │
│        │         │ Of mistress, or thoughts of sorrow crown                  │
│        │         │ Saven...                                                  │
│ 3      │ -3.048  │ In a world where AI and humans collaborate, as it         │
│        │         │ ate the content man of itself; I prove as you clos...     │
│ 4      │ -2.197  │ In a world where AI and humans collaborate, give me some  │
│        │         │ corn old full of the fire, my men royal pou...            │
│ 5      │ -2.600  │ In a world where AI and humans collaborate, words         │
│        │         │ That roar'dly a deadly several house of beseech,          │
│        │         │ T...                                                      │
│ 6      │ -2.827  │ In a world where AI and humans collaborate,-              │
│        │         │ His head to sink!--Play, away, there's no man is a        │
│        │         │ This...                                                   │
│ 7      │ -2.633  │ In a world where AI and humans collaborate, who should be │
│        │         │ not example: the most known, that are stro...             │
│ 8      │ -2.823  │ In a world where AI and humans collaborate, as I may.     │
│        │         │ As I would understand, for I thankf;                      │
│        │         │ From many...                                              │
└────────┴─────────┴───────────────────────────────────────────────────────────┘

╭─────────────────────────────────────────── System-2 Reasoning Output ───────────────────────────────────────────╮
│ In a world where AI and humans collaborate, sweet scope,                                                        │
│ And speak with kissing Duke of Norfolk.                                                                         │
│                                                                                                                 │
│ RICHARD:                                                                                                        │
│ Why, resist thee? who know'st thou? how fares Let Stanley?                                                      │
│                                                                                                                 │
│ ROMEO:                                                                                                          │
│ Ay; for they have have to cheer'd, and strange.                                                                 │
│                                                                                                                 │
│ JOHN                                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────────────────────────────── Packaging Results ────────────────────────────────────────────────

╭───────────────────────────────────────────────────── Done ──────────────────────────────────────────────────────╮
│ Pipeline complete! Results saved to artifacts/pipeline_results.zip                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## End-to-end: BPE tokenizer + NanoGPT (+ optional MoE) + training + 3 inference modes.

In [ ]:
#!/usr/bin/env python3
# colab_kernel_launcher.py
# End-to-end: BPE tokenizer + NanoGPT (+ optional MoE) + training + 3 inference modes.

import os, math, json, argparse, random, time, urllib.request, shlex
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Any, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------- pretty console (optional; fallback if missing) -----------
try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.table import Table
    from rich import box
    console = Console()
    def info(msg, **kw): console.print(Panel(msg, **({"border_style":"cyan","title":"Info","box":box.ROUNDED} | kw)))
    def warn(msg, **kw): console.print(Panel(msg, **({"border_style":"yellow","title":"Warning","box":box.ROUNDED} | kw)))
    def ok(msg, **kw):   console.print(Panel(msg, **({"border_style":"green","title":"OK","box":box.ROUNDED} | kw)))
except Exception:
    class _Dummy:
        def print(self, *a, **k): print(*a)
        def rule(self, *a, **k): print("="*60, *(a or ()), "="*60)
    console = _Dummy()
    def info(msg, **kw): print("[INFO]", msg)
    def warn(msg, **kw): print("[WARN]", msg)
    def ok(msg, **kw):   print("[ OK ]", msg)

# =============================================================================
#                               TOKENIZER (minBPE)
# =============================================================================
import unicodedata
import regex as re

def _replace_ctl(s: str) -> str:
    return "".join(ch if unicodedata.category(ch)[0] != "C" else f"\\u{ord(ch):04x}" for ch in s)

def _render_tok(t: bytes) -> str:
    return _replace_ctl(t.decode("utf-8", errors="replace"))

def _get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for p in zip(ids, ids[1:]): counts[p] = counts.get(p, 0) + 1
    return counts

def _merge(ids, pair, idx):
    newids=[]; i=0
    while i < len(ids):
        if ids[i]==pair[0] and i < len(ids)-1 and ids[i+1]==pair[1]:
            newids.append(idx); i+=2
        else:
            newids.append(ids[i]); i+=1
    return newids

class Tokenizer:
    def __init__(self):
        self.merges={}; self.pattern=""; self.special_tokens={}
        self.vocab={i:bytes([i]) for i in range(256)}
    def _rebuild_vocab(self):
        vocab={i:bytes([i]) for i in range(256)}
        for (a,b),i in self.merges.items(): vocab[i]=vocab[a]+vocab[b]
        for s,i in self.special_tokens.items(): vocab[i]=s.encode("utf-8")
        self.vocab=vocab
    def save(self, prefix):
        os.makedirs(os.path.dirname(prefix), exist_ok=True) if os.path.dirname(prefix) else None
        with open(prefix+".model","w",encoding="utf-8") as f:
            f.write("minbpe v1\n"); f.write(f"{self.pattern}\n"); f.write(f"{len(self.special_tokens)}\n")
            for s,i in self.special_tokens.items(): f.write(f"{s} {i}\n")
            for (a,b),_i in self.merges.items(): f.write(f"{a} {b}\n")
        with open(prefix+".vocab","w",encoding="utf-8") as f:
            inv={idx:pair for pair,idx in self.merges.items()}
            for i,tok in self.vocab.items():
                s=_render_tok(tok)
                if i in inv:
                    a,b=inv[i]; f.write(f"[{_render_tok(self.vocab[a])}][{_render_tok(self.vocab[b])}] -> [{s}] {i}\n")
                else:
                    f.write(f"[{s}] {i}\n")
    def load(self, model_file):
        merges={}; specials={}
        idx=256
        with open(model_file,"r",encoding="utf-8") as f:
            version=f.readline().strip(); assert version=="minbpe v1"
            self.pattern=f.readline().strip()
            ns=int(f.readline().strip())
            for _ in range(ns):
                s,si=f.readline().strip().split(); specials[s]=int(si)
            for line in f:
                a,b=map(int,line.split()); merges[(a,b)]=idx; idx+=1
        self.merges=merges; self.special_tokens=specials; self._rebuild_vocab()

class BasicTokenizer(Tokenizer):
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size>=256; num_merges=vocab_size-256
        ids=list(text.encode("utf-8")); merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats=_get_stats(ids); pair=max(stats,key=stats.get)
            idx=256+i; ids=_merge(ids,pair,idx); merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
            if verbose and i<10: info(f"merge {i+1}/{num_merges}: {pair} -> {idx}")
        self.merges=merges; self.vocab=vocab
    def encode(self, text):
        ids=list(text.encode("utf-8"))
        while len(ids)>=2:
            stats=_get_stats(ids); pair=min(stats,key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=_merge(ids,pair,self.merges[pair])
        return ids
    def decode(self, ids): return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

GPT4_SPLIT = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexTokenizer(Tokenizer):
    def __init__(self): super().__init__(); self.pattern=GPT4_SPLIT; self.compiled=re.compile(self.pattern)
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size>=256; num_merges=vocab_size-256
        chunks=re.findall(self.compiled,text); ids=[list(c.encode("utf-8")) for c in chunks]
        merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats={}; [ _get_stats(ci,stats) for ci in ids ]
            pair=max(stats,key=stats.get); idx=256+i; ids=[_merge(ci,pair,idx) for ci in ids]
            merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
            if verbose and i<10: info(f"merge {i+1}/{num_merges}: {pair} -> {idx}")
        self.merges=merges; self.vocab=vocab
    def _encode_chunk(self, bs):
        ids=list(bs)
        while len(ids)>=2:
            stats=_get_stats(ids); pair=min(stats,key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=_merge(ids,pair,self.merges[pair])
        return ids
    def encode(self, text):
        ids=[]; [ids.extend(self._encode_chunk(c.encode("utf-8"))) for c in re.findall(self.compiled,text)]
        return ids
    def decode(self, ids): return BasicTokenizer.decode(self, ids)

def _new_tok(kind): return BasicTokenizer() if kind=="basic" else RegexTokenizer()

def train_or_load_tokenizer(kind,vocab_size,text,prefix,verbose=True):
    os.makedirs(os.path.dirname(prefix),exist_ok=True) if os.path.dirname(prefix) else None
    model=prefix+".model"
    if os.path.exists(model):
        tok=_new_tok(kind); tok.load(model); vs=max(tok.vocab.keys())+1
        if verbose: ok(f"Loaded tokenizer {model} (size={vs})", title="Tokenizer")
        return tok,model,vs
    tok=_new_tok(kind); tok.train(text,max(256,vocab_size),verbose); tok.save(prefix); vs=max(tok.vocab.keys())+1
    if verbose: ok(f"Saved tokenizer {prefix}.model (size={vs})", title="Tokenizer")
    return tok,model,vs

# =============================================================================
#                              DATA (tiny shakespeare)
# =============================================================================
def _repeat_to_len(s: str, target_len: int) -> str:
    if not s: s=" \n"
    return (s * ((target_len // len(s))+1))[:target_len]

def load_tiny_shakespeare(data_dir="./data", target_len=100_000):
    os.makedirs(data_dir,exist_ok=True)
    path=os.path.join(data_dir,"tinyshakespeare_input.txt")
    if not os.path.exists(path):
        try:
            url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url,path)
        except Exception:
            warn("Could not download dataset; writing fallback sample.")
            sample=("From fairest creatures we desire increase,\n"
                    "That thereby beauty's rose might never die,\n"
                    "But as the riper should by time decease,\n"
                    "His tender heir might bear his memory:\n")
            with open(path,"w",encoding="utf-8") as f: f.write(_repeat_to_len(sample, target_len))
    text=open(path,"r",encoding="utf-8").read()
    split=int(0.9*len(text)); return text[:split], text[split:]

def get_batch_tokens(ids, block_size, batch_size, device):
    if len(ids) <= block_size + 1:
        raise RuntimeError(f"Sequence too short ({len(ids)}) for block_size={block_size}.")
    ix=torch.randint(len(ids)-block_size-1,(batch_size,))
    x=torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y=torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================================================================
#                               MODEL (GPT + MoE)
# =============================================================================
class LayerNorm(nn.Module):
    def __init__(self,n,bias): super().__init__(); self.weight=nn.Parameter(torch.ones(n)); self.bias=nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self,x): return F.layer_norm(x,self.weight.shape,self.weight,self.bias,1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self,n_embd,n_head,block,dropout,bias,backend="sdpa"):
        super().__init__(); assert n_embd % n_head == 0
        self.n_head=n_head; self.n_embd=n_embd; self.dropout=dropout; self.backend=backend
        self.c_attn=nn.Linear(n_embd,3*n_embd,bias=bias); self.c_proj=nn.Linear(n_embd,n_embd,bias=bias)
        if backend=="vanilla" or not hasattr(F,"scaled_dot_product_attention"):
            self.register_buffer("bias", torch.tril(torch.ones(block,block)).view(1,1,block,block))
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        if self.backend=="sdpa" and hasattr(F,"scaled_dot_product_attention"):
            y=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=self.dropout if self.training else 0.0)
        else:
            att=(q@k.transpose(-2,-1))/math.sqrt(k.size(-1))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0,float("-inf"))
            att=F.softmax(att,dim=-1); y=att@v
        y=y.transpose(1,2).contiguous().view(B,T,C); return self.c_proj(y)

def entropy_mean(probs,eps=1e-9): return -(probs.clamp_min(eps)*probs.clamp_min(eps).log()).sum(-1).mean()

class TopKRouter(nn.Module):
    def __init__(self,in_dim,hidden_dim,num_experts,k=1,temp=1.0):
        super().__init__(); self.net=nn.Sequential(nn.Linear(in_dim,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,num_experts))
        self.k=k; self.temp=float(temp); self.register_buffer("logits_bias", torch.zeros(num_experts))
    @staticmethod
    def _gumbel(shape, device):
        u=torch.rand(shape,device=device).clamp_(1e-9,1-1e-9); return -torch.log(-torch.log(u))
    def forward(self,x,add_gumbel=False):
        logits=self.net(x)/self.temp + self.logits_bias
        if add_gumbel and self.training: logits = logits + self._gumbel(logits.shape, logits.device)
        probs=F.softmax(logits,dim=-1)
        if self.k>=probs.size(-1): return probs, probs
        topk_vals,topk_idx=torch.topk(probs,self.k,dim=-1)
        mask=torch.zeros_like(probs); mask.scatter_(dim=-1,index=topk_idx,src=torch.ones_like(topk_vals))
        sp=probs*mask; sp=sp/(sp.sum(dim=-1,keepdim=True)+1e-9); return sp,probs

class MoEFFN(nn.Module):
    def __init__(self,in_dim,num_experts=3,k=1,router_hidden=128,dropout=0.1,blw=0.02,entropy_penalty=0.001,
                 routing_mode="specialize", router_temp=1.0):
        super().__init__(); hidden=4*in_dim
        self.expert_fc=nn.ModuleList([nn.Linear(in_dim,hidden) for _ in range(num_experts)])
        self.expert_proj=nn.ModuleList([nn.Linear(hidden,in_dim) for _ in range(num_experts)])
        self.shared_fc=nn.Linear(in_dim,hidden); self.shared_proj=nn.Linear(hidden,in_dim)
        self.router=TopKRouter(in_dim,router_hidden,num_experts,k,temp=router_temp)
        self.drop=nn.Dropout(dropout); self.num_experts=num_experts
        self.blw=blw; self.entw=entropy_penalty; self.routing_mode=routing_mode
    def forward(self,x):
        B,T,C=x.shape; xf=x.view(B*T,C); sp,dp=self.router(xf,add_gumbel=self.training)
        routed=0.0
        for e in range(self.num_experts):
            h=self.expert_proj[e](F.gelu(self.expert_fc[e](xf))); routed+=sp[:,e].unsqueeze(-1)*h
        shared=self.shared_proj(F.gelu(self.shared_fc(xf)))
        y=self.drop((shared+routed).view(B,T,C))
        aux=(-self.entw*entropy_mean(dp)) if self.routing_mode=="specialize" else torch.tensor(0.0,device=x.device)
        stats={"expert_selection_counts":(sp>0).float().sum(0),"mean_routing_probs":dp.mean(0)}
        return y,aux,stats

@dataclass
class GPTCfg:
    block_size:int=256; vocab_size:int=256; n_layer:int=6; n_head:int=6; n_embd:int=384
    dropout:float=0.1; bias:bool=True; attention_backend:str="sdpa"
    use_moe:bool=False; num_experts:int=3; k:int=1; router_hidden:int=128; blw:float=0.02
    entropy_penalty:float=0.001; routing_mode:str="specialize"; router_temp:float=1.0

class GPTBlock(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout, bias, backend="sdpa", moe: Optional[MoEFFN]=None):
        super().__init__(); self.ln1=LayerNorm(n_embd,bias); self.attn=CausalSelfAttention(n_embd,n_head,block,dropout,bias,backend)
        self.ln2=LayerNorm(n_embd,bias); self.moe=moe
        if moe is None:
            hidden=4*n_embd
            self.ffn=nn.Sequential(nn.Linear(n_embd,hidden,bias=bias), nn.GELU(), nn.Linear(hidden,n_embd,bias=bias), nn.Dropout(dropout))
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        if self.moe is None:
            x=x+self.ffn(self.ln2(x)); aux=torch.tensor(0.0,device=x.device); stats={}
        else:
            y,aux,stats=self.moe(self.ln2(x)); x=x+y
        return x, aux, stats

class NanoGPT(nn.Module):
    def __init__(self, cfg:GPTCfg):
        super().__init__(); self.cfg=cfg
        self.wte=nn.Embedding(cfg.vocab_size,cfg.n_embd); self.wpe=nn.Embedding(cfg.block_size,cfg.n_embd)
        blocks=[]
        for _ in range(cfg.n_layer):
            moe=None
            if cfg.use_moe: moe=MoEFFN(cfg.n_embd,cfg.num_experts,cfg.k,cfg.router_hidden,cfg.dropout,cfg.blw,cfg.entropy_penalty,cfg.routing_mode,cfg.router_temp)
            blocks.append(GPTBlock(cfg.n_embd,cfg.n_head,cfg.block_size,cfg.dropout,cfg.bias,cfg.attention_backend,moe))
        self.blocks=nn.ModuleList(blocks); self.ln_f=LayerNorm(cfg.n_embd,cfg.bias); self.lm_head=nn.Linear(cfg.n_embd,cfg.vocab_size,bias=False)
        self.lm_head.weight=self.wte.weight
        self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward(self, idx, targets=None):
        B,T=idx.shape; assert T<=self.cfg.block_size
        pos=torch.arange(0,T,device=idx.device).long(); x=self.wte(idx)+self.wpe(pos)[None,:,:]
        aux_total=torch.tensor(0.0,device=idx.device); last={}
        for blk in self.blocks:
            x,aux,stats=blk(x); aux_total=aux_total+aux; last=stats
        x=self.ln_f(x); logits=self.lm_head(x); loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1)); loss=loss+aux_total
        return logits, loss, last
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.cfg.block_size else idx[:,-self.cfg.block_size:]
            logits,_,_=self(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); idx_next=torch.multinomial(probs,1); idx=torch.cat((idx,idx_next),dim=1)
        return idx

def count_parameters(m: nn.Module) -> Tuple[int,int]:
    total=sum(p.numel() for p in m.parameters())
    train=sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, train

# =============================================================================
#                            TRAIN / SAMPLE / SAVELOAD
# =============================================================================
def train_lm(model, train_ids, val_ids, block_size, epochs, steps, batch, lr, device, title):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.1)

    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0

    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        losses = []

        for _ in range(steps):
            xb, yb = get_batch_tokens(train_ids, block_size, batch, device)
            _, loss, _ = model(xb, yb)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())

        scheduler.step()
        dt = time.time() - t0

        with torch.no_grad():
            model.eval()
            xb, yb = get_batch_tokens(val_ids, block_size, batch, device)
            _, vl, _ = model(xb, yb)
            val_loss = vl.item()

        current_lr = scheduler.get_last_lr()[0]
        info(f"Epoch {ep}/{epochs}  train={sum(losses)/len(losses):.3f}  val={val_loss:.3f}  lr={current_lr:.6f}  time={dt:.1f}s", title=title)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                info(f"Early stopping at epoch {ep}: no improvement in validation loss for {patience} epochs.", title="Early Stopping")
                break

    return model

@torch.no_grad()
def sample_lm(model, tok, device, start="\n", tokens=200, temperature=0.8, top_k=200, title="Sample"):
    model.eval().to(device)
    x = torch.tensor([tok.encode(start)], dtype=torch.long, device=device)
    y = model.generate(x, max_new_tokens=tokens, temperature=temperature, top_k=top_k)[0].tolist()
    text = tok.decode(y)
    info(text, title=title)

def save_ckpt(path,payload):
    os.makedirs(os.path.dirname(path),exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload,path); ok(f"Saved to {path}")

def load_ckpt(path):
    p=torch.load(path,map_location="cpu"); ok(f"Loaded {path}"); return p

# =============================================================================
#                          INFERENCE STRATEGIES
# =============================================================================
@torch.no_grad()
def _last_token_logprobs(model, input_ids: torch.Tensor):
    logits, _, _ = model(input_ids)
    return F.log_softmax(logits[:, -1, :], dim=-1)

def test_time_adapt_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    adapt_steps: int = 2, adapt_lr: float = 5e-4, adapt_layers: str = "lm_head,ln_f",
    context_reuse: int = 64, temperature: float = 0.8, top_k: int = 200,
):
    """Few gradient steps on a small parameter subset over recent context before each token."""
    model = model.to(device)
    original_state = {name: param.clone() for name, param in model.named_parameters()}

    # select params
    adapt_names = {n.strip() for n in adapt_layers.split(",")} if adapt_layers else set()
    for p in model.parameters(): p.requires_grad = False
    to_adapt = []

    if "lm_head" in adapt_names:
        for p in model.lm_head.parameters():
            p.requires_grad = True
            to_adapt.append(p)
    if "ln_f" in adapt_names:
        for p in model.ln_f.parameters():
            p.requires_grad = True
            to_adapt.append(p)
    if "blocks[-1]" in adapt_names and hasattr(model, "blocks"):
        for p in model.blocks[-1].parameters():
            p.requires_grad = True
            to_adapt.append(p)

    opt = torch.optim.Adam([p for p in to_adapt if p.requires_grad], lr=adapt_lr) if to_adapt else None

    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)

    for token_idx in range(max_new_tokens):
        # Adaptation phase
        if opt is not None and ids.size(1) > 1:
            model.train()
            initial_loss = None

            for step in range(adapt_steps):
                ctx = ids[:, -min(context_reuse, ids.size(1)-1):]
                if ctx.size(1) <= 1:
                    break

                inp, tgt = ctx[:, :-1], ctx[:, 1:]
                opt.zero_grad(set_to_none=True)

                try:
                    _, loss, _ = model(inp, tgt)
                    if loss is None or torch.isnan(loss) or torch.isinf(loss):
                        break
                    if initial_loss is None:
                        initial_loss = loss.item()

                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(to_adapt, 0.5)  # Smaller clip value
                    opt.step()

                    # Safety check: if loss explodes, reset and break
                    if loss.item() > initial_loss * 3:
                        for name, param in model.named_parameters():
                            if name in original_state and param.requires_grad:
                                param.data.copy_(original_state[name])
                        break

                except RuntimeError:
                    break

        # Generation phase
        model.eval()
        with torch.no_grad():
            cond = ids if ids.size(1) <= model.cfg.block_size else ids[:, -model.cfg.block_size:]
            try:
                logits, _, _ = model(cond)
                logits = logits[:, -1, :] / temperature

                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float("Inf")

                probs = F.softmax(logits, dim=-1)
                if torch.isnan(probs).any():
                    # Fallback to greedy
                    nxt = torch.argmax(logits, dim=-1, keepdim=True)
                else:
                    nxt = torch.multinomial(probs, 1)

                ids = torch.cat([ids, nxt], dim=1)
            except RuntimeError:
                break

    return tok.decode(ids[0].tolist())

def _rollout_logprob(model, start_ids: torch.Tensor, horizon: int, temperature: float, top_k: Optional[int]):
    model.eval(); ids=start_ids.clone()
    total_logprob=0.0; total_entropy=0.0
    for _ in range(horizon):
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        logits,_,_=model(cond); logits=logits[:,-1,:]/temperature
        if top_k is not None:
            v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
        logprobs=F.log_softmax(logits,dim=-1); probs=logprobs.exp()
        entropy=-(probs*logprobs).sum(dim=-1).mean()
        nxt=torch.multinomial(probs,1)
        total_logprob+=logprobs.gather(-1,nxt).mean().item()
        total_entropy+=entropy.item()
        ids=torch.cat([ids,nxt],dim=1)
    return total_logprob, total_entropy

@torch.no_grad()
def active_inference_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    particles: int = 8, horizon: int = 4, beta: float = 0.2, temperature: float = 0.9, top_k: Optional[int] = 200,
):
    """Minimize expected free energy proxy = -Σ log p + β·Σ entropy via short rollouts per candidate action."""
    model=model.to(device).eval()
    ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
    for _ in range(max_new_tokens):
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        base_lp=_last_token_logprobs(model,cond)  # (1,V)
        probs=base_lp.exp()
        cand=torch.multinomial(probs,num_samples=particles)  # (1,P)
        best_F=float("inf"); best_token=None
        for j in range(particles):
            a=cand[:,j:j+1]
            start=torch.cat([cond,a],dim=1)
            logp_sum, ent_sum=_rollout_logprob(model,start,horizon,temperature,top_k)
            F=-logp_sum + beta*ent_sum
            if F<best_F: best_F=F; best_token=a
        if best_token is None:
            best_token=torch.argmax(base_lp,dim=-1,keepdim=True)
        ids=torch.cat([ids,best_token],dim=1)
    return tok.decode(ids[0].tolist())

@torch.no_grad()
def system2_reasoning_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    branches: int = 8, temperature: float = 0.9, top_k: Optional[int] = 200, vote: str = "majority",
):
    """Self-consistency: sample multiple full continuations and pick majority or best average logprob."""
    model=model.to(device).eval()
    def _sample_once():
        ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
        for _ in range(max_new_tokens):
            cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
            logits,_,_=model(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1)
            ids=torch.cat([ids,nxt],dim=1)
        # score by avg logprob
        if ids.size(1)>1:
            inp, tgt = ids[:,:-1], ids[:,1:]
            _, loss, _ = model(inp, tgt)
            avg_lp = -loss.item() if loss is not None else 0.0
        else:
            avg_lp = 0.0
        return tok.decode(ids[0].tolist()), avg_lp

    samples=[_sample_once() for _ in range(branches)]
    if vote=="best_logprob":
        return max(samples, key=lambda x:x[1])[0]
    from collections import Counter
    cleaned=[s[0].strip() for s in samples]
    return Counter(cleaned).most_common(1)[0][0]

# =============================================================================
#                                     MAIN
# =============================================================================
def main(argv=None):
    ap=argparse.ArgumentParser(description="NanoGPT + BPE + MoE + TTA/ActiveInference/System2")
    ap.add_argument("--mode",required=True,choices=["train_gpt_bpe","infer_gpt_bpe","test_time","active_inference","system2_reasoning"])
    ap.add_argument("--device",default="auto",choices=["auto","cpu","cuda"])
    ap.add_argument("--epochs",type=int,default=100)
    ap.add_argument("--steps_per_epoch",type=int,default=200)
    ap.add_argument("--batch_size",type=int,default=32)
    ap.add_argument("--lr",type=float,default=3e-4)
    ap.add_argument("--block_size",type=int,default=128)
    ap.add_argument("--dropout",type=float,default=0.2)
    ap.add_argument("--n_layer",type=int,default=6)
    ap.add_argument("--n_head",type=int,default=8)
    ap.add_argument("--n_embd",type=int,default=384)
    ap.add_argument("--bias",action="store_true",default=True)
    ap.add_argument("--attention_backend",default="sdpa",choices=["sdpa","vanilla"])

    # MoE
    ap.add_argument("--use_moe",action="store_true")
    ap.add_argument("--num_experts",type=int,default=3)
    ap.add_argument("--k",type=int,default=1)
    ap.add_argument("--router_hidden",type=int,default=128)
    ap.add_argument("--blw",type=float,default=0.02)
    ap.add_argument("--entropy_penalty",type=float,default=0.001)
    ap.add_argument("--routing_mode",choices=["specialize","uniform"],default="specialize")
    ap.add_argument("--router_temp",type=float,default=1.0)

    # Tokenizer/data/checkpoints
    ap.add_argument("--tok_kind",default="regex",choices=["basic","regex"])
    ap.add_argument("--vocab_size",type=int,default=512)
    ap.add_argument("--tok_prefix",default="artifacts/tokenizers/tiny_bpe")
    ap.add_argument("--train_corpus",default="")
    ap.add_argument("--save_path",default="artifacts/gpt_bpe.pt")
    ap.add_argument("--load_path",default="artifacts/gpt_bpe.pt")
    ap.add_argument("--seed",type=int,default=1337)

    # common sampling
    ap.add_argument("--sample_start",default="\n")
    ap.add_argument("--sample_tokens",type=int,default=100)
    ap.add_argument("--temperature",type=float,default=0.7)
    ap.add_argument("--top_k",type=int,default=100)

    # TTA
    ap.add_argument("--tta_steps",type=int,default=5)
    ap.add_argument("--tta_lr",type=float,default=1e-4)
    ap.add_argument("--tta_layers",default="lm_head,ln_f")
    ap.add_argument("--tta_context",type=int,default=128)

    # Active inference
    ap.add_argument("--ai_particles",type=int,default=10)
    ap.add_argument("--ai_horizon",type=int,default=5)
    ap.add_argument("--ai_beta",type=float,default=0.1)

    # System-2
    ap.add_argument("--s2_branches",type=int,default=8)
    ap.add_argument("--s2_vote",default="majority",choices=["majority","best_logprob"])

    args=ap.parse_args(argv)
    random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
    device=args.device if args.device!="auto" else ("cuda" if torch.cuda.is_available() else "cpu")

    # Load text
    if args.train_corpus and os.path.exists(args.train_corpus):
        full=open(args.train_corpus,"r",encoding="utf-8").read()
        split=int(0.9*len(full)); train_txt, val_txt = full[:split], full[split:]
    else:
        train_txt, val_txt = load_tiny_shakespeare("./data")

    # Tokenizer
    tok, tok_model_path, vocab_size_actual = train_or_load_tokenizer(args.tok_kind,args.vocab_size,train_txt+val_txt,args.tok_prefix,verbose=True)

    # Encode
    train_ids = tok.encode(train_txt)
    val_ids   = tok.encode(val_txt)

    if len(train_ids) <= args.block_size + 1:
        new_bs=max(16, min(args.block_size, len(train_ids)-2))
        warn(f"Auto-adjusting block_size {args.block_size} -> {new_bs}", title="Safety")
        args.block_size=new_bs

    def build_model():
        cfg=GPTCfg(
            block_size=args.block_size, vocab_size=vocab_size_actual, n_layer=args.n_layer, n_head=args.n_head,
            n_embd=args.n_embd, dropout=args.dropout, bias=args.bias, attention_backend=args.attention_backend,
            use_moe=args.use_moe, num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden,
            blw=args.blw, entropy_penalty=args.entropy_penalty, routing_mode=args.routing_mode, router_temp=args.router_temp
        )
        return cfg, NanoGPT(cfg)

    if args.mode=="train_gpt_bpe":
        cfg, model = build_model()
        model = train_lm(model,train_ids,val_ids,cfg.block_size,args.epochs,args.steps_per_epoch,args.batch_size,args.lr,device,
                         title=f"NanoGPT-BPE ({'MoE' if cfg.use_moe else 'no-MoE'})")
        save_ckpt(args.save_path, {"kind":"gpt_bpe","cfg":asdict(cfg),"state_dict":model.state_dict(),
                                   "tokenizer":{"kind":args.tok_kind,"model_path":tok_model_path,"vocab_size":vocab_size_actual}})

    elif args.mode=="infer_gpt_bpe":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        sample_lm(model,tok2,device,start=args.sample_start,tokens=args.sample_tokens,temperature=args.temperature,top_k=args.top_k,title="Standard Sample")

    elif args.mode=="test_time":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=test_time_adapt_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.tta_steps,args.tta_lr,
                                     args.tta_layers,args.tta_context,args.temperature,args.top_k)
        info(out, title="Test-Time Adaptation Output")

    elif args.mode=="active_inference":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=active_inference_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.ai_particles,
                                      args.ai_horizon,args.ai_beta,args.temperature,args.top_k)
        info(out, title="Active Inference Output")

    elif args.mode=="system2_reasoning":
        payload=load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=system2_reasoning_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.s2_branches,
                                       args.temperature,args.top_k,args.s2_vote)
        info(out, title="System-2 Reasoning Output")

# =============================================================================
# FULL PIPELINE (runs all modes and packages results)
# =============================================================================
def run_full_pipeline():
    console.rule("[bold magenta]NanoGPT-BPE End-to-End Pipeline (Train + All Inference Modes)")

    # Conservative defaults for Colab
    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 50))
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 100))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 32))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    console.print(Panel(f"Using device: {device}", title="Device"))

    # 1) Train NanoGPT with BPE
    console.rule("[bold cyan]Step 1: Training NanoGPT with BPE")
    train_args = [
        "--mode", "train_gpt_bpe",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS_PER_EPOCH),
        "--batch_size", str(DEFAULT_BATCH_SIZE),
        "--lr", "3e-4",
        "--block_size", "128",
        "--n_layer", "6",
        "--n_head", "8",
        "--n_embd", "384",
        "--vocab_size", "512",
        "--tok_kind", "regex",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--save_path", "artifacts/gpt_bpe.pt",
        "--seed", "42",
        "--device", device
    ]
    try:
        main(train_args)
    except Exception as e:
        console.print(Panel(f"Training failed: {e}\nFalling back to smaller model...", title="Warning", border_style="yellow"))
        fallback_args = [
            "--mode", "train_gpt_bpe",
            "--epochs", "10",
            "--steps_per_epoch", "50",
            "--batch_size", "16",
            "--lr", "3e-4",
            "--block_size", "64",
            "--n_layer", "4",
            "--n_head", "4",
            "--n_embd", "256",
            "--vocab_size", "256",
            "--tok_kind", "regex",
            "--tok_prefix", "artifacts/tokenizers/tiny_bpe_fallback",
            "--save_path", "artifacts/gpt_bpe.pt",
            "--seed", "42",
            "--device", device
        ]
        main(fallback_args)

    # Common inference parameters
    prompt = "In a world where AI and humans collaborate,"
    sample_tokens = 80

    # 2) Standard Inference
    console.rule("[bold cyan]Step 2: Standard Inference")
    main([
        "--mode", "infer_gpt_bpe",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--temperature", "0.8",
        "--top_k", "50",
        "--device", device
    ])

    # 3) Test-Time Adaptation
    console.rule("[bold cyan]Step 3: Test-Time Adaptation")
    main([
        "--mode", "test_time",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--tta_steps", "3",
        "--tta_lr", "5e-5",  # Much smaller learning rate to prevent divergence
        "--tta_layers", "lm_head,ln_f",
        "--tta_context", "32",  # Smaller context window
        "--temperature", "0.8",
        "--top_k", "50",
        "--device", device
    ])

    # 4) Active Inference
    console.rule("[bold cyan]Step 4: Active Inference")
    main([
        "--mode", "active_inference",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--ai_particles", "6",
        "--ai_horizon", "3",
        "--ai_beta", "0.3",
        "--temperature", "0.8",
        "--top_k", "50",
        "--device", device
    ])

    # 5) System-2 Reasoning
    console.rule("[bold cyan]Step 5: System-2 Reasoning")
    main([
        "--mode", "system2_reasoning",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--s2_branches", "5",
        "--s2_vote", "best_logprob",
        "--temperature", "0.8",
        "--top_k", "50",
        "--device", device
    ])

    # 6) Package results
    console.rule("[bold magenta]Packaging Results")
    import zipfile
    from pathlib import Path
    zip_path = "artifacts/pipeline_results.zip"
    os.makedirs("artifacts", exist_ok=True)

    try:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            # Add model checkpoints
            for file in Path("artifacts").glob("*.pt"):
                if file.exists():
                    zf.write(file, file.relative_to("."))

            # Add tokenizer files
            if Path("artifacts/tokenizers").exists():
                for file in Path("artifacts/tokenizers").glob("*.*"):
                    if file.exists():
                        zf.write(file, file.relative_to("."))

        ok(f"Pipeline complete! Results saved to {zip_path}", title="Success")
    except Exception as e:
        warn(f"Could not create zip file: {e}", title="Packaging Warning")
        ok("Pipeline completed successfully without packaging", title="Success")

# =============================================================================
# ENTRYPOINT
# =============================================================================
def _sanitize_argv(argv):
    out = []; skip = False
    for a in argv:
        if skip: skip = False; continue
        if a in ("-f", "--f"): skip = True; continue
        if a.startswith("-f=") or a.startswith("--f="): continue
        out.append(a)
    return out

if __name__ == "__main__":
    import sys
    argv = _sanitize_argv(sys.argv[1:])
    # If required --mode arg is missing, run the full pipeline
    if "--mode" not in argv:
        run_full_pipeline()
    else:
        main(argv)

────────────────────────── NanoGPT-BPE End-to-End Pipeline (Train + All Inference Modes) ──────────────────────────

╭──────────────────────────────────────────────────── Device ─────────────────────────────────────────────────────╮
│ Using device: cuda                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────────────────────── Step 1: Training NanoGPT with BPE ────────────────────────────────────────

╭─────────────────────────────────────────────────── Tokenizer ───────────────────────────────────────────────────╮
│ Loaded tokenizer artifacts/tokenizers/tiny_bpe.model (size=512)                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── NanoGPT-BPE (no-MoE) ──────────────────────────────────────────────╮
│ Epoch 1/50  train=4.936  val=4.185  lr=0.000300  time=10.0s                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── Tokenizer ───────────────────────────────────────────────────╮
│ Loaded tokenizer artifacts/tokenizers/tiny_bpe.model (size=512)                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── OK ───────────────────────────────────────────────────────╮
│ Loaded artifacts/gpt_bpe.pt                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── System-2 Reasoning Output ───────────────────────────────────────────╮
│ In a world where AI and humans collaborate, yet my commands.                                                    │
│                                                                                                                 │
│ HENRY BOLINGBROKE:                                                                                              │
│ O hoped therefore, and then not glory.                                                                          │
│                                                                                                                 │
│ HENRY BOLINGBROKE:                                                                                              │
│ It deny means have heard it.                                                                                    │
│                                                                                                                 │
│ HENRY BOLINGB                                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────────────────────────────── Packaging Results ────────────────────────────────────────────────

╭──────────────────────────────────────────────────── Success ────────────────────────────────────────────────────╮
│ Pipeline complete! Results saved to artifacts/pipeline_results.zip                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Full NanoGPT / NanoMamba stack with Char/Regex tokenizers, MoE, Reward Model, GRPO, advanced inference strategies (TTA, Active Inference, System-2 Reasoning), and a TrainingMonitor that logs, plots, and packages experiments.


In [ ]:
#!/usr/bin/env python3
# unified_superset.py
# Full NanoGPT / NanoMamba stack with Char/Regex tokenizers, MoE, Reward Model, GRPO,
# advanced inference strategies (TTA, Active Inference, System-2 Reasoning),
# and a TrainingMonitor that logs, plots, and packages experiments.

import os, sys, math, json, argparse, random, time, urllib.request, shutil, zipfile
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict, Counter
from typing import Optional, Tuple, List, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------------------------------------------------------
# Pretty console (rich if available)
# -----------------------------------------------------------------------------
try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.table import Table
    from rich import box
    console = Console()
    def info(msg, **kw): console.print(Panel(msg, **({"border_style":"cyan","title":"Info","box":box.ROUNDED} | kw)))
    def warn(msg, **kw): console.print(Panel(msg, **({"border_style":"yellow","title":"Warning","box":box.ROUNDED} | kw)))
    def ok(msg, **kw):   console.print(Panel(msg, **({"border_style":"green","title":"OK","box":box.ROUNDED} | kw)))
except Exception:
    class _Dummy:
        def print(self, *a, **k): print(*a)
        def rule(self, *a, **k): print("="*60, *(a or ()), "="*60)
    console = _Dummy()
    def info(msg, **kw): print("[INFO]", msg)
    def warn(msg, **kw): print("[WARN]", msg)
    def ok(msg, **kw):   print("[ OK ]", msg)

# -----------------------------------------------------------------------------
# Matplotlib (non-interactive)
# -----------------------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec

plt.style.use('default')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

# =============================================================================
# TRAINING MONITOR (plots, logs, animations, ZIP packaging)
# =============================================================================
class TrainingMonitor:
    """Comprehensive training monitor that saves everything organized."""
    def __init__(self, experiment_name="unified_experiment"):
        self.experiment_name = experiment_name
        self.base_dir = Path("training_outputs") / experiment_name
        self._setup_folders()
        # Tracking
        self.metrics = defaultdict(list)
        self.attention_snapshots = []
        self.embedding_snapshots = []
        self.moe_snapshots = []
        self.step_count = 0
        self.epoch_count = 0

    # ---------------------- FS ----------------------
    def _setup_folders(self):
        folders = [
            self.base_dir,
            self.base_dir / "models",
            self.base_dir / "plots" / "training_curves",
            self.base_dir / "plots" / "attention_analysis",
            self.base_dir / "plots" / "embedding_evolution",
            self.base_dir / "plots" / "moe_analysis",
            self.base_dir / "plots" / "animations",
            self.base_dir / "tokenizers",
            self.base_dir / "logs",
            self.base_dir / "data",
        ]
        for f in folders: f.mkdir(parents=True, exist_ok=True)
        info(f"Output structure at: {self.base_dir}")

    # ---------------------- Logging ----------------------
    def log_training_step(self, epoch, step, train_loss, val_loss=None, lr=None, grad_norm=None):
        self.metrics['epoch'].append(epoch)
        self.metrics['step'].append(self.step_count)
        self.metrics['train_loss'].append(float(train_loss))
        if val_loss is not None: self.metrics['val_loss'].append(float(val_loss))
        if lr is not None: self.metrics['learning_rate'].append(float(lr))
        if grad_norm is not None: self.metrics['grad_norm'].append(float(grad_norm))
        self.step_count += 1

    def capture_model_state(self, model):
        # Attention Q/K alignment (if present)
        if hasattr(model, 'blocks'):
            attn_data = []
            for i, block in enumerate(model.blocks):
                attn = getattr(block, 'attn', None)
                if attn is None: continue
                if hasattr(attn, 'c_attn'):
                    with torch.no_grad():
                        w = attn.c_attn.weight.data.detach().cpu().numpy()
                    n_embd = w.shape[1]
                    q_w = w[:n_embd, :]
                    k_w = w[n_embd:2*n_embd, :]
                    corr = np.corrcoef(q_w.flatten(), k_w.flatten())[0, 1]
                    attn_data.append({'layer': i, 'qk_correlation': float(corr)})
            if attn_data:
                self.attention_snapshots.append({'step': self.step_count, 'epoch': self.epoch_count, 'data': attn_data})
        # Embedding snapshots
        if hasattr(model, 'wte'):
            with torch.no_grad():
                emb = model.wte.weight.data.detach().cpu().numpy()
            self.embedding_snapshots.append({
                'step': self.step_count,
                'epoch': self.epoch_count,
                'norm': float(np.linalg.norm(emb)),
                'mean': float(np.mean(emb)),
                'std': float(np.std(emb)),
                'weights': emb.copy(),
            })

    def capture_moe_stats(self, stats):
        if not stats: return
        if 'mean_routing_probs' in stats and stats['mean_routing_probs'] is not None:
            probs = stats['mean_routing_probs']
            if isinstance(probs, torch.Tensor):
                probs = probs.detach().cpu().numpy()
            probs = np.asarray(probs)
            if probs.size == 0: return
            entropy = float(-(probs * np.log(probs + 1e-8)).sum())
            balance = float(1.0 - (np.std(probs) / (np.mean(probs) + 1e-8)))
            self.moe_snapshots.append({
                'step': self.step_count,
                'epoch': self.epoch_count,
                'routing_probs': probs.tolist(),
                'entropy': entropy,
                'balance': balance,
            })

    # ---------------------- Plots ----------------------
    def _plot_training_curves(self):
        if not self.metrics['train_loss']:
            return
        fig = plt.figure(figsize=(20, 12))
        gs = GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)
        epochs = np.array(self.metrics['epoch'])
        steps = np.array(self.metrics['step'])
        train_losses = np.array(self.metrics['train_loss'])

        ax1 = fig.add_subplot(gs[0, :2])
        ax1.plot(steps, train_losses, linewidth=2, alpha=0.9, label='Training Loss')
        if self.metrics.get('val_loss'):
            vl = np.array(self.metrics['val_loss'])
            # align lengths if needed
            t = np.linspace(steps.min() if len(steps) else 0, steps.max() if len(steps) else 1, num=len(vl))
            ax1.plot(t, vl, linestyle='--', linewidth=2, alpha=0.9, label='Validation Loss')
        ax1.set_xlabel('Step'); ax1.set_ylabel('Loss'); ax1.set_title('Training Progress'); ax1.legend(); ax1.grid(True, alpha=0.3)

        if self.metrics.get('learning_rate'):
            ax2 = fig.add_subplot(gs[0, 2])
            lrs = np.array(self.metrics['learning_rate'])
            t = np.linspace(steps.min() if len(steps) else 0, steps.max() if len(steps) else 1, num=len(lrs))
            ax2.plot(t, lrs, linewidth=2)
            ax2.set_xlabel('Step'); ax2.set_ylabel('LR'); ax2.set_title('LR Schedule'); ax2.set_yscale('log'); ax2.grid(True, alpha=0.3)

        if self.metrics.get('grad_norm'):
            ax3 = fig.add_subplot(gs[1, 0])
            gns = np.array(self.metrics['grad_norm'])
            t = np.linspace(steps.min() if len(steps) else 0, steps.max() if len(steps) else 1, num=len(gns))
            ax3.plot(t, gns, linewidth=2, alpha=0.9)
            ax3.set_xlabel('Step'); ax3.set_ylabel('Grad Norm'); ax3.set_title('Gradient Norms'); ax3.grid(True, alpha=0.3)

        ax4 = fig.add_subplot(gs[1, 1])
        if len(train_losses) > 4:
            w = max(3, min(20, len(train_losses)//10))
            sm = np.convolve(train_losses, np.ones(w)/w, mode='valid')
            t = np.linspace(steps[w-1] if len(steps)>=w else 0, steps[-1] if len(steps) else 1, num=len(sm))
            ax4.plot(t, sm, linewidth=3, alpha=0.9)
            ax4.set_xlabel('Step'); ax4.set_ylabel('Smoothed Loss'); ax4.set_title(f'Loss (Moving Avg, window={w})'); ax4.grid(True, alpha=0.3)
        else:
            ax4.text(0.5,0.5,'Insufficient steps for smoothing',ha='center',va='center'); ax4.axis('off')

        ax5 = fig.add_subplot(gs[1, 2])
        final_loss = float(train_losses[-1]) if len(train_losses) else 0.0
        min_loss = float(np.min(train_losses)) if len(train_losses) else 0.0
        improvement = float(train_losses[0] - final_loss) if len(train_losses) > 1 else 0.0
        stats_text = f'Final Loss: {final_loss:.4f}\nBest Loss: {min_loss:.4f}\nImprovement: {improvement:.4f}\nTotal Steps: {len(train_losses)}'
        ax5.text(0.1, 0.5, stats_text, transform=ax5.transAxes, fontsize=12, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        ax5.axis('off'); ax5.set_title('Training Stats')

        out = self.base_dir / "plots" / "training_curves" / "comprehensive_training.png"
        plt.suptitle(f'{self.experiment_name} - Training Analysis', fontsize=16)
        plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        ok(f"Training curves saved to {out}")

    def _create_loss_animation(self):
        if not self.metrics['train_loss']:
            return
        fig, ax = plt.subplots(figsize=(12, 8))
        train_losses = np.array(self.metrics['train_loss'])
        steps = np.array(self.metrics['step'])

        def animate(i):
            ax.clear(); ax.plot(steps[:i+1], train_losses[:i+1], linewidth=3, alpha=0.9)
            ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.set_title(f'Training Progress - Frame {i+1}/{len(train_losses)}'); ax.grid(True, alpha=0.3)
            if len(train_losses):
                ax.set_xlim(steps.min(), steps.max())
                ax.set_ylim(min(train_losses)*0.9, max(train_losses)*1.1)

        frames = min(len(train_losses), 100)
        anim = animation.FuncAnimation(fig, animate, frames=frames, interval=150, repeat=True)
        out = self.base_dir / "plots" / "animations" / "loss_evolution.gif"
        anim.save(out, writer='pillow', fps=5); plt.close()
        ok(f"Loss animation saved to {out}")

    def _plot_attention(self):
        if not self.attention_snapshots:
            return
        steps = [s['step'] for s in self.attention_snapshots]
        n_layers = len(self.attention_snapshots[0]['data'])
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        # layer curves
        for layer in range(n_layers):
            corrs = []
            for s in self.attention_snapshots:
                d = s['data'][layer]
                corrs.append(d['qk_correlation'])
            ax1.plot(steps[:len(corrs)], corrs, linewidth=2, alpha=0.9, label=f'L{layer}')
        ax1.set_xlabel('Step'); ax1.set_ylabel('Q-K Corr'); ax1.set_title('Query-Key Alignment Evolution'); ax1.legend(); ax1.grid(True, alpha=0.3)

        latest = self.attention_snapshots[-1]['data']
        cm = np.array([[d['qk_correlation'] for d in latest]])
        im = ax2.imshow(cm, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
        ax2.set_title('Latest Q-K Correlations'); fig.colorbar(im, ax=ax2)

        all_corrs = []
        for s in self.attention_snapshots:
            all_corrs.extend([d['qk_correlation'] for s in self.attention_snapshots for d in s['data']])
        ax3.hist(all_corrs, bins=40, alpha=0.85)
        ax3.set_title('Distribution of Q-K Correlations'); ax3.set_xlabel('Corr'); ax3.set_ylabel('Freq'); ax3.grid(True, alpha=0.3)

        stdevs = []
        for layer in range(n_layers):
            layer_series = [s['data'][layer]['qk_correlation'] for s in self.attention_snapshots]
            stdevs.append(np.std(layer_series))
        ax4.bar(range(n_layers), stdevs, alpha=0.9)
        ax4.set_xlabel('Layer'); ax4.set_ylabel('Std Dev'); ax4.set_title('Q-K Correlation Stability'); ax4.grid(True, alpha=0.3)

        out = self.base_dir / "plots" / "attention_analysis" / "attention_comprehensive.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        ok(f"Attention analysis saved to {out}")

    def _plot_embeddings(self):
        if not self.embedding_snapshots:
            return
        steps = [s['step'] for s in self.embedding_snapshots]
        norms = [s['norm'] for s in self.embedding_snapshots]
        means = [s['mean'] for s in self.embedding_snapshots]
        stds  = [s['std']  for s in self.embedding_snapshots]
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        ax1.plot(steps, norms, linewidth=2); ax1.set_title('Embedding Frobenius Norm'); ax1.set_xlabel('Step'); ax1.set_ylabel('Norm'); ax1.grid(True, alpha=0.3)
        ax2.plot(steps, means, linewidth=2, label='Mean'); ax2.plot(steps, stds, linewidth=2, label='Std'); ax2.legend(); ax2.set_title('Embedding Stats'); ax2.grid(True, alpha=0.3)

        # Latest distribution if we kept weights
        w = self.embedding_snapshots[-1].get('weights', None)
        if w is not None:
            ax3.hist(w.flatten(), bins=60, alpha=0.85)
            ax3.set_title('Latest Embedding Distribution'); ax3.grid(True, alpha=0.3)
            token_norms = np.linalg.norm(w, axis=1)
            ax4.plot(token_norms, alpha=0.9)
            ax4.set_title('Per-Token L2 Norms'); ax4.set_xlabel('Token'); ax4.set_ylabel('L2'); ax4.grid(True, alpha=0.3)
        else:
            ax3.axis('off'); ax4.axis('off')

        out = self.base_dir / "plots" / "embedding_evolution" / "embedding_analysis.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        ok(f"Embedding analysis saved to {out}")

    def _plot_moe(self):
        if not self.moe_snapshots:
            return
        steps = [s['step'] for s in self.moe_snapshots]
        nE = len(self.moe_snapshots[0]['routing_probs'])
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        ax1, ax2, ax3, ax4 = axes.ravel()
        for e in range(nE):
            usage = [s['routing_probs'][e] for s in self.moe_snapshots]
            ax1.plot(steps, usage, linewidth=2, alpha=0.9, label=f'E{e}')
        ax1.set_title('Expert Usage Evolution'); ax1.set_xlabel('Step'); ax1.set_ylabel('Prob'); ax1.legend(); ax1.grid(True, alpha=0.3)

        ent = [s['entropy'] for s in self.moe_snapshots]
        ax2.plot(steps, ent, linewidth=2, alpha=0.9)
        ax2.set_title('Routing Entropy'); ax2.set_xlabel('Step'); ax2.grid(True, alpha=0.3)

        latest = self.moe_snapshots[-1]['routing_probs']
        ax3.pie(latest, labels=[f'E{i}' for i in range(len(latest))], autopct='%1.1f%%', startangle=90)
        ax3.set_title('Current Expert Distribution')

        bal = [s['balance'] for s in self.moe_snapshots]
        ax4.plot(steps, bal, linewidth=2, alpha=0.9)
        ax4.set_title('Load Balance (1=best)'); ax4.set_xlabel('Step'); ax4.grid(True, alpha=0.3)

        out = self.base_dir / "plots" / "moe_analysis" / "moe_comprehensive.png"
        plt.tight_layout(); plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
        ok(f"MoE analysis saved to {out}")

    # ---------------------- Persist ----------------------
    def _save_logs(self):
        # Metrics
        with open(self.base_dir / "logs" / "training_metrics.json", 'w') as f:
            json.dump({k: list(v) for k, v in self.metrics.items()}, f, indent=2)
        # Attention
        if self.attention_snapshots:
            with open(self.base_dir / "logs" / "attention_evolution.json", 'w') as f:
                json.dump(self.attention_snapshots, f, indent=2)
        # Embedding (strip weights for logs to keep size small)
        if self.embedding_snapshots:
            light = [{k: v for k, v in s.items() if k != 'weights'} for s in self.embedding_snapshots]
            with open(self.base_dir / "logs" / "embedding_evolution.json", 'w') as f:
                json.dump(light, f, indent=2)
        # MoE
        if self.moe_snapshots:
            with open(self.base_dir / "logs" / "moe_evolution.json", 'w') as f:
                json.dump(self.moe_snapshots, f, indent=2)
        ok(f"Logs saved to {self.base_dir / 'logs'}")

    def generate_all(self):
        info("Generating visualizations and logs...")
        self._plot_training_curves()
        self._create_loss_animation()
        self._plot_attention()
        self._plot_embeddings()
        self._plot_moe()
        self._save_logs()
        ok("Visualizations complete")

    def snapshot_tokenizer(self, src_prefix: str):
        # copy tokenizer files into outputs for convenience
        m = Path(src_prefix + ".model")
        v = Path(src_prefix + ".vocab")
        for p in [m, v]:
            if p.exists():
                dst = self.base_dir / "tokenizers" / p.name
                try:
                    shutil.copy2(p, dst)
                except Exception:
                    pass

    def zip_everything(self):
        # Create summary
        summary = {
            "experiment": self.experiment_name,
            "timestamp": time.strftime("%Y-%m-%d_%H-%M-%S"),
            "total_steps": int(self.step_count),
            "total_epochs": int(self.epoch_count),
            "final_loss": self.metrics['train_loss'][-1] if self.metrics['train_loss'] else None,
        }
        with open(self.base_dir / "experiment_summary.json", 'w') as f:
            json.dump(summary, f, indent=2)

        # README
        readme = f"""# {self.experiment_name} - Results

This folder contains models, plots, logs, tokenizers, and data artifacts.

- Plots in `plots/` (training curves, attention, embeddings, MoE, animations)
- Logs in `logs/` (JSON)
- Models in `models/`
- Tokenizers in `tokenizers/`
- Data in `data/`
"""
        with open(self.base_dir / "README.md", 'w') as f:
            f.write(readme)

        # Zip
        ts = time.strftime("%Y%m%d_%H%M%S")
        zip_name = f"unified_results_{self.experiment_name}_{ts}.zip"
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
            for p in self.base_dir.rglob('*'):
                if p.is_file(): z.write(p, p.relative_to('.'))
        size_mb = os.path.getsize(zip_name) / (1024*1024)
        ok(f"Created ZIP: {zip_name} ({size_mb:.1f} MB)")
        return zip_name

# =============================================================================
# TOKENIZERS — Basic + Regex (GPT-4 split)
# =============================================================================
import unicodedata, regex as re

def _replace_ctl(s: str) -> str:
    return "".join(ch if unicodedata.category(ch)[0] != "C" else f"\\u{ord(ch):04x}" for ch in s)

def _render_tok(t: bytes) -> str:
    return _replace_ctl(t.decode("utf-8", errors="replace"))

def _get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for p in zip(ids, ids[1:]): counts[p] = counts.get(p, 0) + 1
    return counts

def _merge(ids, pair, idx):
    newids=[]; i=0
    while i < len(ids):
        if ids[i]==pair[0] and i < len(ids)-1 and ids[i+1]==pair[1]:
            newids.append(idx); i+=2
        else:
            newids.append(ids[i]); i+=1
    return newids

class TokenizerBase:
    def __init__(self):
        self.merges={}; self.pattern=""; self.special_tokens={}
        self.vocab={i:bytes([i]) for i in range(256)}
    def _rebuild_vocab(self):
        vocab={i:bytes([i]) for i in range(256)}
        for (a,b),i in self.merges.items(): vocab[i]=vocab[a]+vocab[b]
        for s,i in self.special_tokens.items(): vocab[i]=s.encode("utf-8")
        self.vocab=vocab
    def save(self, prefix):
        os.makedirs(os.path.dirname(prefix), exist_ok=True) if os.path.dirname(prefix) else None
        with open(prefix+".model","w",encoding="utf-8") as f:
            f.write("minbpe v1\n"); f.write(f"{self.pattern}\n"); f.write(f"{len(self.special_tokens)}\n")
            for s,i in self.special_tokens.items(): f.write(f"{s} {i}\n")
            for (a,b),_i in self.merges.items(): f.write(f"{a} {b}\n")
        with open(prefix+".vocab","w",encoding="utf-8") as f:
            inv={idx:pair for pair,idx in self.merges.items()}
            for i,tok in self.vocab.items():
                s=_render_tok(tok)
                if i in inv:
                    a,b=inv[i]; f.write(f"[{_render_tok(self.vocab[a])}][{_render_tok(self.vocab[b])}] -> [{s}] {i}\n")
                else:
                    f.write(f"[{s}] {i}\n")
    def load(self, model_file):
        merges={}; specials={}
        idx=256
        with open(model_file,"r",encoding="utf-8") as f:
            version=f.readline().strip(); assert version=="minbpe v1"
            self.pattern=f.readline().strip()
            ns=int(f.readline().strip())
            for _ in range(ns):
                s,si=f.readline().strip().split(); specials[s]=int(si)
            for line in f:
                a,b=map(int,line.split()); merges[(a,b)]=idx; idx+=1
        self.merges=merges; self.special_tokens=specials; self._rebuild_vocab()

class BasicTokenizer(TokenizerBase):
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size>=256; num_merges=vocab_size-256
        ids=list(text.encode("utf-8")); merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats=_get_stats(ids);
            if not stats: break
            pair=max(stats,key=stats.get)
            idx=256+i; ids=_merge(ids,pair,idx); merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
            if verbose and i<10: info(f"merge {i+1}/{num_merges}: {pair} -> {idx}")
        self.merges=merges; self.vocab=vocab
    def encode(self, text):
        ids=list(text.encode("utf-8"))
        while len(ids)>=2:
            stats=_get_stats(ids)
            if not stats: break
            pair=min(stats,key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=_merge(ids,pair,self.merges[pair])
        return ids
    def decode(self, ids): return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

GPT4_SPLIT = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexTokenizer(TokenizerBase):
    def __init__(self):
        super().__init__();
        self.pattern=GPT4_SPLIT;
        self.compiled=re.compile(self.pattern)
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size>=256; num_merges=vocab_size-256
        chunks=re.findall(self.compiled,text); ids=[list(c.encode("utf-8")) for c in chunks]
        merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats={};
            for ci in ids: _get_stats(ci,stats)
            if not stats: break
            pair=max(stats,key=stats.get); idx=256+i; ids=[_merge(ci,pair,idx) for ci in ids]
            merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
            if verbose and i<10: info(f"merge {i+1}/{num_merges}: {pair} -> {idx}")
        self.merges=merges; self.vocab=vocab
    def _encode_chunk(self, bs):
        ids=list(bs)
        while len(ids)>=2:
            stats=_get_stats(ids)
            if not stats: break
            pair=min(stats,key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=_merge(ids,pair,self.merges[pair])
        return ids
    def encode(self, text):
        ids=[];
        for c in re.findall(self.compiled,text):
            ids.extend(self._encode_chunk(c.encode("utf-8")))
        return ids
    def decode(self, ids): return BasicTokenizer.decode(self, ids)

def _new_tok(kind):
    if kind in ("regex","gpt4","gpt4_split"): return RegexTokenizer()
    if kind in ("basic","bpe","minbpe"): return BasicTokenizer()
    raise ValueError(f"Unknown tok_kind: {kind}")

def train_or_load_tokenizer(kind,vocab_size,text,prefix,verbose=True):
    os.makedirs(os.path.dirname(prefix),exist_ok=True) if os.path.dirname(prefix) else None
    model=prefix+".model"
    if os.path.exists(model):
        tok=_new_tok(kind); tok.load(model); vs=max(tok.vocab.keys())+1
        if verbose: ok(f"Loaded tokenizer {model} (size={vs})", title="Tokenizer")
        return tok,model,vs
    tok=_new_tok(kind); tok.train(text,max(256,vocab_size),verbose); tok.save(prefix); vs=max(tok.vocab.keys())+1
    if verbose: ok(f"Saved tokenizer {prefix}.model (size={vs})", title="Tokenizer")
    return tok,model,vs

# =============================================================================
# DATA — tiny Shakespeare
# =============================================================================
def _repeat_to_len(s: str, target_len: int) -> str:
    if not s: s=" \n"
    return (s * ((target_len // len(s))+1))[:target_len]

def load_tiny_shakespeare(data_dir="./data", target_len=100_000):
    os.makedirs(data_dir,exist_ok=True)
    path=os.path.join(data_dir,"tinyshakespeare_input.txt")
    if not os.path.exists(path):
        try:
            url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url,path)
        except Exception:
            warn("Could not download dataset; writing fallback sample.")
            sample=("From fairest creatures we desire increase,\n"
                    "That thereby beauty's rose might never die,\n"
                    "But as the riper should by time decease,\n"
                    "His tender heir might bear his memory:\n")
            with open(path,"w",encoding="utf-8") as f: f.write(_repeat_to_len(sample, target_len))
    text=open(path,"r",encoding="utf-8").read()
    split=int(0.9*len(text)); return text[:split], text[split:]

def get_batch_tokens(ids, block_size, batch_size, device):
    if len(ids) <= block_size + 1:
        raise RuntimeError(f"Sequence too short ({len(ids)}) for block_size={block_size}.")
    ix=torch.randint(len(ids)-block_size-1,(batch_size,))
    x=torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y=torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================================================================
# MoE + MODELS — GPT / Mamba
# =============================================================================
class LayerNorm(nn.Module):
    def __init__(self,n,bias=True): super().__init__(); self.weight=nn.Parameter(torch.ones(n)); self.bias=nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self,x): return F.layer_norm(x,self.weight.shape,self.weight,self.bias,1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self,n_embd,n_head,block,dropout,bias=True,backend="sdpa"):
        super().__init__(); assert n_embd % n_head == 0
        self.n_head=n_head; self.n_embd=n_embd; self.dropout=dropout; self.backend=backend
        self.c_attn=nn.Linear(n_embd,3*n_embd,bias=bias); self.c_proj=nn.Linear(n_embd,n_embd,bias=bias)
        if backend=="vanilla" or not hasattr(F,"scaled_dot_product_attention"):
            self.register_buffer("bias", torch.tril(torch.ones(block,block)).view(1,1,block,block))
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        if self.backend=="sdpa" and hasattr(F,"scaled_dot_product_attention"):
            y=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=self.dropout if self.training else 0.0)
        else:
            att=(q@k.transpose(-2,-1))/math.sqrt(k.size(-1))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0,float("-inf"))
            att=F.softmax(att,dim=-1); y=att@v
        y=y.transpose(1,2).contiguous().view(B,T,C); return self.c_proj(y)

def entropy_mean(probs,eps=1e-9): return -(probs.clamp_min(eps)*probs.clamp_min(eps).log()).sum(-1).mean()

class TopKRouter(nn.Module):
    def __init__(self,in_dim,hidden_dim,num_experts,k=1,temp=1.0):
        super().__init__(); self.net=nn.Sequential(nn.Linear(in_dim,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,num_experts))
        self.k=k; self.temp=float(temp); self.register_buffer("logits_bias", torch.zeros(num_experts))
    @staticmethod
    def _gumbel(shape, device):
        u=torch.rand(shape,device=device).clamp_(1e-9,1-1e-9); return -torch.log(-torch.log(u))
    def forward(self,x,add_gumbel=False):
        logits=self.net(x)/self.temp + self.logits_bias
        if add_gumbel and self.training: logits = logits + self._gumbel(logits.shape, logits.device)
        probs=F.softmax(logits,dim=-1)
        if self.k>=probs.size(-1): return probs, probs
        topk_vals,topk_idx=torch.topk(probs,self.k,dim=-1)
        mask=torch.zeros_like(probs); mask.scatter_(dim=-1,index=topk_idx,src=torch.ones_like(topk_vals))
        sp=probs*mask; sp=sp/(sp.sum(dim=-1,keepdim=True)+1e-9); return sp,probs

class MoEFFN(nn.Module):
    def __init__(self,in_dim,num_experts=3,k=1,router_hidden=128,dropout=0.1,blw=0.02,entropy_penalty=0.001,
                 routing_mode="specialize", router_temp=1.0, shared_ffn=True):
        super().__init__(); hidden=4*in_dim
        self.expert_fc=nn.ModuleList([nn.Linear(in_dim,hidden) for _ in range(num_experts)])
        self.expert_proj=nn.ModuleList([nn.Linear(hidden,in_dim) for _ in range(num_experts)])
        self.shared_ffn=shared_ffn
        if shared_ffn:
            self.shared_fc=nn.Linear(in_dim,hidden); self.shared_proj=nn.Linear(hidden,in_dim)
        self.router=TopKRouter(in_dim,router_hidden,num_experts,k,temp=router_temp)
        self.drop=nn.Dropout(dropout); self.num_experts=num_experts
        self.blw=blw; self.entw=entropy_penalty; self.routing_mode=routing_mode
    def forward(self,x):
        B,T,C=x.shape; xf=x.view(B*T,C); sp,dp=self.router(xf,add_gumbel=self.training)
        routed=0.0
        for e in range(self.num_experts):
            h=self.expert_proj[e](F.gelu(self.expert_fc[e](xf))); routed+=sp[:,e].unsqueeze(-1)*h
        if self.shared_ffn:
            shared=self.shared_proj(F.gelu(self.shared_fc(xf)))
            y=self.drop((shared+routed).view(B,T,C))
        else:
            y=self.drop(routed.view(B,T,C))
        aux=(-self.entw*entropy_mean(dp)) if self.routing_mode=="specialize" else torch.tensor(0.0,device=x.device)
        stats={"expert_selection_counts":(sp>0).float().sum(0).detach(), "mean_routing_probs":dp.mean(0).detach()}
        return y,aux,stats

@dataclass
class GPTCfg:
    block_size:int=256; vocab_size:int=256; n_layer:int=6; n_head:int=6; n_embd:int=384
    dropout:float=0.1; bias:bool=True; attention_backend:str="sdpa"
    use_moe:bool=False; num_experts:int=3; k:int=1; router_hidden:int=128; blw:float=0.02
    entropy_penalty:float=0.001; routing_mode:str="specialize"; router_temp:float=1.0
    shared_ffn:bool=True

class GPTBlock(nn.Module):
    def __init__(self, cfg:GPTCfg):
        super().__init__(); self.ln1=LayerNorm(cfg.n_embd,cfg.bias); self.attn=CausalSelfAttention(cfg.n_embd,cfg.n_head,cfg.block_size,cfg.dropout,cfg.bias,cfg.attention_backend)
        self.ln2=LayerNorm(cfg.n_embd,cfg.bias)
        self.moe = MoEFFN(cfg.n_embd,cfg.num_experts,cfg.k,cfg.router_hidden,cfg.dropout,cfg.blw,cfg.entropy_penalty,cfg.routing_mode,cfg.router_temp,cfg.shared_ffn) if cfg.use_moe else None
        if self.moe is None:
            hidden=4*cfg.n_embd
            self.ffn=nn.Sequential(nn.Linear(cfg.n_embd,hidden,bias=cfg.bias), nn.GELU(), nn.Linear(hidden,cfg.n_embd,bias=cfg.bias), nn.Dropout(cfg.dropout))
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        if self.moe is None:
            x=x+self.ffn(self.ln2(x)); aux=torch.tensor(0.0,device=x.device); stats={}
        else:
            y,aux,stats=self.moe(self.ln2(x)); x=x+y
        return x, aux, stats

class NanoGPT(nn.Module):
    def __init__(self, cfg:GPTCfg):
        super().__init__(); self.cfg=cfg
        self.wte=nn.Embedding(cfg.vocab_size,cfg.n_embd); self.wpe=nn.Embedding(cfg.block_size,cfg.n_embd)
        self.blocks=nn.ModuleList([GPTBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_f=LayerNorm(cfg.n_embd,cfg.bias); self.lm_head=nn.Linear(cfg.n_embd,cfg.vocab_size,bias=False)
        self.lm_head.weight=self.wte.weight
        self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward(self, idx, targets=None):
        B,T=idx.shape; assert T<=self.cfg.block_size
        pos=torch.arange(0,T,device=idx.device).long(); x=self.wte(idx)+self.wpe(pos)[None,:,:]
        aux_total=torch.tensor(0.0,device=idx.device); last={}
        for blk in self.blocks:
            x,aux,stats=blk(x); aux_total=aux_total+aux; last=stats
        x=self.ln_f(x); logits=self.lm_head(x); loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1)); loss=loss+aux_total
        return logits, loss, last
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.cfg.block_size else idx[:,-self.cfg.block_size:]
            logits,_,_=self(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); idx_next=torch.multinomial(probs,1); idx=torch.cat((idx,idx_next),dim=1)
        return idx

# Minimal Mamba-like residual block (toy, works end-to-end)
class SimpleMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__(); self.ln=LayerNorm(d, True); self.ff=nn.Sequential(nn.Linear(d, 4*d), nn.SiLU(), nn.Linear(4*d, d))
    def forward(self, x): return x + self.ff(self.ln(x))

class NanoMamba(nn.Module):
    def __init__(self, vocab_size=256, block_size=256, n_layer=6, n_embd=384):
        super().__init__()
        self.vocab_size=vocab_size; self.block_size=block_size; self.n_layer=n_layer; self.n_embd=n_embd
        self.wte=nn.Embedding(vocab_size,n_embd); self.wpe=nn.Embedding(block_size,n_embd)
        self.layers=nn.ModuleList([SimpleMambaBlock(n_embd) for _ in range(n_layer)])
        self.ln_f=LayerNorm(n_embd, True); self.head=nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight=self.wte.weight
        self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward(self, idx, targets=None):
        B,T=idx.shape; pos=torch.arange(0,T,device=idx.device)
        x=self.wte(idx)+self.wpe(pos)[None,:,:]
        for l in self.layers: x=l(x)
        h=self.ln_f(x); logits=self.head(h)
        loss=None
        if targets is not None:
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1))
        return logits, loss, {}
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.block_size else idx[:,-self.block_size:]
            logits,_,_=self(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1); idx=torch.cat((idx,nxt),dim=1)
        return idx

# =============================================================================
# TRAIN / VALIDATE / SAVELOAD
# =============================================================================
def count_parameters(m: nn.Module) -> Tuple[int,int]:
    total=sum(p.numel() for p in m.parameters())
    train=sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, train

def train_lm(model, train_ids, val_ids, block_size, epochs, steps, batch, lr, device, title, monitor: TrainingMonitor=None, early_stop=50):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
    best_val = float('inf'); patience = 0

    for ep in range(1, epochs+1):
        model.train(); losses=[]; gnorms=[]; t0=time.time()
        for st in range(steps):
            xb, yb = get_batch_tokens(train_ids, block_size, batch, device)
            opt.zero_grad(set_to_none=True)
            _, loss, stats = model(xb, yb)
            loss.backward()
            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            gnorms.append(float(total_norm.item()))
            opt.step()
            losses.append(float(loss.item()))
            if monitor and (st % 10 == 0):
                monitor.capture_model_state(model)
                if stats: monitor.capture_moe_stats(stats)
        # validation
        model.eval()
        with torch.no_grad():
            xb, yb = get_batch_tokens(val_ids, block_size, batch, device)
            _, vloss, _ = model(xb, yb)
            v = float(vloss.item())
        tr = float(sum(losses)/len(losses))
        lr_now = float(sched.get_last_lr()[0])
        gn = float(sum(gnorms)/len(gnorms)) if gnorms else 0.0
        if monitor:
            monitor.log_training_step(ep, steps, tr, v, lr_now, gn)
            monitor.epoch_count = ep
        info(f"{title} — Epoch {ep}/{epochs}  train={tr:.3f}  val={v:.3f}  grad={gn:.3f}  lr={lr_now:.2e}  time={time.time()-t0:.1f}s")

        if v < best_val:
            best_val = v; patience = 0
            save_path = monitor.base_dir / "models" / f"best_{title.replace(' ', '_').lower()}.pt" if monitor else Path("artifacts")/f"best_{title.replace(' ', '_').lower()}.pt"
            ckpt = {"arch": "gpt" if isinstance(model, NanoGPT) else "mamba",
                    "tok_kind": None,  # filled by caller if needed
                    "cfg": asdict(model.cfg) if hasattr(model, "cfg") else {"vocab_size": getattr(model, "vocab_size", None),
                                                                            "block_size": getattr(model, "block_size", None),
                                                                            "n_layer": getattr(model, "n_layer", None),
                                                                            "n_embd": getattr(model, "n_embd", None)},
                    "state_dict": model.state_dict()}
            os.makedirs(save_path.parent, exist_ok=True)
            torch.save(ckpt, save_path)
        else:
            patience += 1
        sched.step()
        if patience >= early_stop:
            warn("Early stopping triggered.")
            break
    return model

def save_ckpt(path,payload):
    os.makedirs(os.path.dirname(path),exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload,path); ok(f"Saved to {path}")

def load_ckpt(path):
    p=torch.load(path,map_location="cpu"); ok(f"Loaded {path}"); return p

# =============================================================================
# Inference strategies (TTA / Active Inference / System-2)
# =============================================================================
@torch.no_grad()
def _last_token_logprobs(model, input_ids: torch.Tensor):
    logits, _, _ = model(input_ids)
    return F.log_softmax(logits[:, -1, :], dim=-1)

def _rollout_logprob(model, start_ids: torch.Tensor, horizon: int, temperature: float, top_k: Optional[int]):
    model.eval(); ids=start_ids.clone()
    total_logprob=0.0; total_entropy=0.0
    for _ in range(horizon):
        cond=ids if ids.size(1)<= (model.cfg.block_size if hasattr(model,"cfg") else model.block_size) else ids[:,-(model.cfg.block_size if hasattr(model,"cfg") else model.block_size):]
        logits,_,_=model(cond); logits=logits[:,-1,:]/temperature
        if top_k is not None:
            v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
        logprobs=F.log_softmax(logits,dim=-1); probs=logprobs.exp()
        entropy=-(probs*logprobs).sum(dim=-1).mean()
        nxt=torch.multinomial(probs,1)
        total_logprob+=logprobs.gather(-1,nxt).mean().item()
        total_entropy+=entropy.item()
        ids=torch.cat([ids,nxt],dim=1)
    return total_logprob, total_entropy

@torch.no_grad()
def test_time_adapt_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    adapt_steps: int = 2, adapt_lr: float = 5e-4, adapt_layers: str = "lm_head,ln_f",
    context_reuse: int = 64, temperature: float = 0.8, top_k: int = 200,
):
    """Few gradient steps on small parameter subset over recent context before each token."""
    model = model.to(device)
    model.eval()
    adapt_names = {n.strip() for n in adapt_layers.split(",")} if adapt_layers else set()

    # Freeze all
    for p in model.parameters():
        p.requires_grad = False

    # Collect adaptation params
    to_adapt = []
    if hasattr(model, "lm_head") and "lm_head" in adapt_names:
        for p in model.lm_head.parameters():
            p.requires_grad = True
            to_adapt.append(p)
    if hasattr(model, "ln_f") and "ln_f" in adapt_names:
        for p in model.ln_f.parameters():
            p.requires_grad = True
            to_adapt.append(p)
    if "blocks[-1]" in adapt_names and hasattr(model, "blocks"):
        for p in model.blocks[-1].parameters():
            p.requires_grad = True
            to_adapt.append(p)

    opt = torch.optim.Adam([p for p in to_adapt if p.requires_grad], lr=adapt_lr) if to_adapt else None

    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
    blk = model.cfg.block_size if hasattr(model, "cfg") else model.block_size

    for _ in range(max_new_tokens):
        # 🔧 Only adapt if we actually have trainable params
        if opt is not None and ids.size(1) > 1 and any(p.requires_grad for p in to_adapt):
            model.train()
            for _step in range(adapt_steps):
                ctx = ids[:, -min(context_reuse, ids.size(1)-1):]
                inp, tgt = ctx[:, :-1], ctx[:, 1:]
                opt.zero_grad(set_to_none=True)
                _, loss, _ = model(inp, tgt)
                if loss.requires_grad:  # ✅ guard against non-differentiable loss
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(to_adapt, 1.0)
                    opt.step()
        else:
            model.eval()  # fallback: no adaptation

        # Standard token generation
        cond = ids if ids.size(1) <= blk else ids[:, -blk:]
        logits, _, _ = model(cond)
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("Inf")
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1)
        ids = torch.cat([ids, nxt], dim=1)

    return tok.decode(ids[0].tolist())

@torch.no_grad()
def active_inference_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    particles: int = 8, horizon: int = 4, beta: float = 0.2, temperature: float = 0.9, top_k: Optional[int] = 200,
):
    """Minimize expected free energy proxy = -Σ log p + β·Σ entropy via short rollouts per candidate action."""
    model=model.to(device).eval()
    ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
    blk = model.cfg.block_size if hasattr(model,"cfg") else model.block_size
    for _ in range(max_new_tokens):
        cond=ids if ids.size(1)<=blk else ids[:,-blk:]
        base_lp=_last_token_logprobs(model,cond)  # (1,V)
        probs=base_lp.exp()
        cand=torch.multinomial(probs,num_samples=min(particles, probs.size(-1)))  # (1,P)
        best_F=float("inf"); best_token=None
        for j in range(cand.size(1)):
            a=cand[:,j:j+1]
            start=torch.cat([cond,a],dim=1)
            logp_sum, ent_sum=_rollout_logprob(model,start,horizon,temperature,top_k)
            F=-logp_sum + beta*ent_sum
            if F<best_F: best_F=F; best_token=a
        if best_token is None:
            best_token=torch.argmax(base_lp,dim=-1,keepdim=True)
        ids=torch.cat([ids,best_token],dim=1)
    return tok.decode(ids[0].tolist())

@torch.no_grad()
def system2_reasoning_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    branches: int = 8, temperature: float = 0.9, top_k: Optional[int] = 200, vote: str = "majority",
):
    """Self-consistency: sample multiple full continuations and pick majority or best average logprob."""
    model=model.to(device).eval()
    blk = model.cfg.block_size if hasattr(model,"cfg") else model.block_size

    def _sample_once():
        ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
        for _ in range(max_new_tokens):
            cond=ids if ids.size(1)<=blk else ids[:,-blk:]
            logits,_,_=model(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1)
            ids=torch.cat([ids,nxt],dim=1)
        # score by avg logprob
        if ids.size(1)>1:
            inp, tgt = ids[:,:-1], ids[:,1:]
            _, loss, _ = model(inp, tgt)
            avg_lp = -loss.item() if loss is not None else 0.0
        else:
            avg_lp = 0.0
        return tok.decode(ids[0].tolist()), avg_lp

    samples=[_sample_once() for _ in range(branches)]
    if vote=="best_logprob":
        return max(samples, key=lambda x:x[1])[0]
    cleaned=[s[0].strip() for s in samples]
    return Counter(cleaned).most_common(1)[0][0]

# =============================================================================
# Reward Model + GRPO
# =============================================================================
class RewardModel(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base
        vocab_size = base.cfg.vocab_size if hasattr(base, "cfg") else base.vocab_size
        self.head = nn.Linear(vocab_size, 1)
    def forward(self, idx):
        logits, _, _ = self.base(idx); h = logits[:, -1, :]
        return self.head(h)

def load_pairs(path): return [json.loads(l) for l in open(path, "r", encoding="utf-8")]

def train_reward_model(rm, tok, pairs_path, block_size, epochs, steps, batch, lr, device, monitor: TrainingMonitor=None):
    rm.to(device); pairs = load_pairs(pairs_path)
    opt = torch.optim.AdamW(rm.parameters(), lr=lr)
    for ep in range(1, epochs+1):
        random.shuffle(pairs); losses=[]
        for _ in range(steps):
            b = random.sample(pairs, min(batch, len(pairs)))
            chosen = [tok.encode(p["chosen"]) for p in b]
            rejected = [tok.encode(p["rejected"]) for p in b]
            cl = [torch.tensor(c[:block_size]) for c in chosen]
            rl = [torch.tensor(r[:block_size]) for r in rejected]
            cl = torch.nn.utils.rnn.pad_sequence(cl, batch_first=True).to(device)
            rl = torch.nn.utils.rnn.pad_sequence(rl, batch_first=True).to(device)
            rc = rm(cl); rr = rm(rl)
            loss = -F.logsigmoid(rc - rr).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(float(loss.item()))
        info(f"RM Epoch {ep}/{epochs} loss={sum(losses)/len(losses):.3f}", title="Reward Model")
        if monitor:
            monitor.log_training_step(ep, steps, sum(losses)/len(losses))
            monitor.epoch_count = ep

def grpo_step(model, encode_fn, prompts, gen_tokens, opt, device, monitor: TrainingMonitor=None):
    model.train(); rewards=[]; logps=[]; stats_last=None
    for s in prompts:
        x = torch.tensor([encode_fn(s)], device=device)
        y = model.generate(x, gen_tokens)[0]  # [seq]
        inp, tgt = y[:-1].unsqueeze(0), y[1:].unsqueeze(0)
        logits, _, stats = model(inp, tgt)
        logp = F.log_softmax(logits, dim=-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1).mean()
        rewards.append(logp); logps.append(logp)
        stats_last = stats
    rewards = torch.stack(rewards)
    logps = torch.stack(logps)
    adv = rewards - rewards.mean()
    loss = -(adv.detach() * logps).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if monitor and stats_last:
        monitor.capture_model_state(model)
        monitor.capture_moe_stats(stats_last)
    return float(loss.item()), float(rewards.mean().item())

# =============================================================================
# MAIN (CLI)
# =============================================================================
def main(argv=None):
    if argv is None: argv = []

    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 50))
    DEFAULT_STEPS_PER_EPOCH = int(os.environ.get("STEPS", 100))
    DEFAULT_BATCH_SIZE = int(os.environ.get("BATCH", 64))

    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", required=True, choices=["train", "infer", "train_rm", "train_grpo", "test_time", "active_inference", "system2_reasoning"])
    ap.add_argument("--arch", required=True, choices=["gpt", "mamba"])
    ap.add_argument("--tok_kind", required=True, choices=["basic", "regex"])
    ap.add_argument("--epochs", type=int, default=DEFAULT_EPOCHS)
    ap.add_argument("--steps_per_epoch", type=int, default=DEFAULT_STEPS_PER_EPOCH)
    ap.add_argument("--batch_size", type=int, default=DEFAULT_BATCH_SIZE)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--block_size", type=int, default=256)
    ap.add_argument("--n_layer", type=int, default=6)
    ap.add_argument("--n_head", type=int, default=6)
    ap.add_argument("--n_embd", type=int, default=384)
    ap.add_argument("--bias", action="store_true", default=True)
    ap.add_argument("--attention_backend", default="sdpa", choices=["sdpa","vanilla"])

    # MoE
    ap.add_argument("--use_moe", action="store_true")
    ap.add_argument("--num_experts", type=int, default=3)
    ap.add_argument("--k", type=int, default=1)
    ap.add_argument("--router_hidden", type=int, default=128)
    ap.add_argument("--blw", type=float, default=0.02)
    ap.add_argument("--entropy_penalty", type=float, default=0.001)
    ap.add_argument("--routing_mode", choices=["specialize","uniform"], default="specialize")
    ap.add_argument("--router_temp", type=float, default=1.0)
    ap.add_argument("--shared_ffn", action="store_true")

    # Tokenizer/data/checkpoints
    ap.add_argument("--vocab_size", type=int, default=1024)
    ap.add_argument("--tok_prefix", default="artifacts/tokenizers/tiny_bpe")
    ap.add_argument("--train_corpus", default="")
    ap.add_argument("--save_path", default="artifacts/unified.pt")
    ap.add_argument("--load_path", default="artifacts/unified.pt")
    ap.add_argument("--seed", type=int, default=1337)

    # common sampling
    ap.add_argument("--sample_start", default="\n")
    ap.add_argument("--sample_tokens", type=int, default=200)
    ap.add_argument("--temperature", type=float, default=0.8)
    ap.add_argument("--top_k", type=int, default=200)

    # TTA
    ap.add_argument("--tta_steps", type=int, default=2)
    ap.add_argument("--tta_lr", type=float, default=5e-4)
    ap.add_argument("--tta_layers", default="lm_head,ln_f")
    ap.add_argument("--tta_context", type=int, default=64)

    # Active inference
    ap.add_argument("--ai_particles", type=int, default=8)
    ap.add_argument("--ai_horizon", type=int, default=4)
    ap.add_argument("--ai_beta", type=float, default=0.2)

    # System-2
    ap.add_argument("--s2_branches", type=int, default=8)
    ap.add_argument("--s2_vote", default="majority", choices=["majority","best_logprob"])

    # Experiment name for monitor
    ap.add_argument("--exp_name", default="unified_experiment")

    args = ap.parse_args(argv)
    random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    info(f"Using device: {device}", title="Device")

    # Monitor
    monitor = TrainingMonitor(args.exp_name)

    # Data
    if args.train_corpus and os.path.exists(args.train_corpus):
        full=open(args.train_corpus,"r",encoding="utf-8").read()
        split=int(0.9*len(full)); train_txt, val_txt = full[:split], full[split:]
    else:
        train_txt, val_txt = load_tiny_shakespeare("./data")

    # Tokenizer / encoding (orthogonal to arch)
    tok, tok_model_path, vocab_size = train_or_load_tokenizer(args.tok_kind,args.vocab_size,train_txt+val_txt,args.tok_prefix,verbose=True)
    encode_fn, decode_fn = tok.encode, tok.decode
    train_ids, val_ids = tok.encode(train_txt), tok.encode(val_txt)
    monitor.snapshot_tokenizer(args.tok_prefix)

    # Adjust block if needed
    if len(train_ids) <= args.block_size + 1:
        new_bs=max(16, min(args.block_size, len(train_ids)-2))
        warn(f"Auto-adjusting block_size {args.block_size} -> {new_bs}", title="Safety")
        args.block_size=new_bs

    # Build model
    if args.arch == "gpt":
        cfg=GPTCfg(
            block_size=args.block_size, vocab_size=vocab_size, n_layer=args.n_layer, n_head=args.n_head,
            n_embd=args.n_embd, dropout=0.1, bias=args.bias, attention_backend=args.attention_backend,
            use_moe=args.use_moe, num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden,
            blw=args.blw, entropy_penalty=args.entropy_penalty, routing_mode=args.routing_mode, router_temp=args.router_temp,
            shared_ffn=args.shared_ffn
        )
        model=NanoGPT(cfg)
    else:
        model=NanoMamba(vocab_size=vocab_size, block_size=args.block_size, n_layer=args.n_layer, n_embd=args.n_embd)

    # Modes
    if args.mode == "train":
        total,trainable=count_parameters(model); info(f"Params total={total/1e6:.2f}M  trainable={trainable/1e6:.2f}M", title="Model")
        title = f"{args.arch.upper()}-{args.tok_kind.upper()} Train"
        model = train_lm(model, train_ids, val_ids, args.block_size, args.epochs, args.steps_per_epoch, args.batch_size, args.lr, device, title, monitor)
        cfg_to_save = asdict(model.cfg) if hasattr(model, "cfg") else {"vocab_size": getattr(model, "vocab_size", None), "block_size": args.block_size, "n_layer": args.n_layer, "n_embd": args.n_embd}
        ckpt = {"arch": args.arch, "tok_kind": args.tok_kind, "cfg": cfg_to_save, "state_dict": model.state_dict(),
                "tokenizer":{"kind":args.tok_kind,"model_path":tok_model_path,"vocab_size":vocab_size}}
        os.makedirs("artifacts", exist_ok=True)
        torch.save(ckpt, args.save_path)
        torch.save(ckpt, monitor.base_dir / "models" / Path(args.save_path).name)
        ok(f"Model saved to {args.save_path}")
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        ok(f"ZIP ready: {zip_path}")

    elif args.mode == "infer":
        payload = load_ckpt(args.load_path)
        tinfo=payload.get("tokenizer", {"kind":args.tok_kind, "model_path": tok_model_path, "vocab_size":vocab_size})
        tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        if payload.get("arch","gpt")=="gpt":
            cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg)
        else:
            cfg=payload["cfg"]; model=NanoMamba(cfg.get("vocab_size",vocab_size), cfg.get("block_size",args.block_size), cfg.get("n_layer",args.n_layer), cfg.get("n_embd",args.n_embd))
        model.load_state_dict(payload["state_dict"]); model.to(device).eval()
        x=torch.tensor([tok2.encode(args.sample_start)],dtype=torch.long,device=device)
        y=model.generate(x,max_new_tokens=args.sample_tokens,temperature=args.temperature,top_k=args.top_k)[0].tolist()
        info(tok2.decode(y), title="Generated Text")

    elif args.mode == "train_rm":
        if not os.path.exists(args.load_path):
            raise ValueError("--load_path must point to a trained base LM checkpoint")
        payload = load_ckpt(args.load_path)
        tinfo=payload.get("tokenizer", {"kind":args.tok_kind, "model_path": tok_model_path, "vocab_size":vocab_size})
        tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        if payload.get("arch","gpt")=="gpt":
            cfg=GPTCfg(**payload["cfg"]); base=NanoGPT(cfg)
        else:
            cfg=payload["cfg"]; base=NanoMamba(cfg.get("vocab_size",vocab_size), cfg.get("block_size",args.block_size), cfg.get("n_layer",args.n_layer), cfg.get("n_embd",args.n_embd))
        base.load_state_dict(payload["state_dict"])
        rm = RewardModel(base)
        rm_pairs = args.train_corpus if args.train_corpus else "data/preferences.jsonl"
        if not os.path.exists(rm_pairs):
            warn(f"{rm_pairs} not found. Creating a tiny toy set.")
            os.makedirs("data", exist_ok=True)
            toy = [
                {"chosen":"I love this model","rejected":"I hate this model"},
                {"chosen":"This is a great answer","rejected":"This is a terrible answer"},
            ]
            with open(rm_pairs, "w", encoding="utf-8") as f:
                for row in toy: f.write(json.dumps(row) + "\n")
        train_reward_model(rm, tok2, rm_pairs, args.block_size, epochs=max(5, args.epochs//5), steps=args.steps_per_epoch, batch=args.batch_size, lr=1e-5, device=device, monitor=monitor)
        torch.save({"kind": "reward_model", "base_arch": args.arch, "tok_kind": tinfo["kind"], "state_dict": rm.state_dict()}, args.save_path)
        torch.save({"kind": "reward_model", "base_arch": args.arch, "tok_kind": tinfo["kind"], "state_dict": rm.state_dict()}, monitor.base_dir / "models" / Path(args.save_path).name)
        ok(f"Reward model saved to {args.save_path}")
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        ok(f"ZIP ready: {zip_path}")

    elif args.mode == "train_grpo":
        payload = load_ckpt(args.load_path)
        tinfo=payload.get("tokenizer", {"kind":args.tok_kind, "model_path": tok_model_path, "vocab_size":vocab_size})
        tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        if payload.get("arch","gpt")=="gpt":
            cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg)
        else:
            cfg=payload["cfg"]; model=NanoMamba(cfg.get("vocab_size",vocab_size), cfg.get("block_size",args.block_size), cfg.get("n_layer",args.n_layer), cfg.get("n_embd",args.n_embd))
        model.load_state_dict(payload["state_dict"]); model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
        prompts = ["To be, or not to be", "Once upon a time", "The quick brown fox"]
        for step in range(args.steps_per_epoch):
            loss, mr = grpo_step(model, tok2.encode, prompts, 32, opt, device, monitor)
            if monitor:
                monitor.log_training_step(step+1, step, train_loss=loss)
            info(f"GRPO step {step+1}/{args.steps_per_epoch} loss={loss:.3f} reward={mr:.3f}", title="GRPO")
        cfg_to_save = asdict(model.cfg) if hasattr(model, "cfg") else {"vocab_size": getattr(model, "vocab_size", None), "block_size": args.block_size, "n_layer": args.n_layer, "n_embd": args.n_embd}
        ckpt = {"arch": args.arch, "tok_kind": tinfo["kind"], "cfg": cfg_to_save, "state_dict": model.state_dict(),
                "tokenizer":{"kind":tinfo["kind"],"model_path":tinfo["model_path"],"vocab_size":tinfo["vocab_size"]}}
        torch.save(ckpt, args.save_path)
        torch.save(ckpt, monitor.base_dir / "models" / Path(args.save_path).name)
        ok(f"GRPO-tuned model saved to {args.save_path}")
        monitor.generate_all()
        zip_path = monitor.zip_everything()
        ok(f"ZIP ready: {zip_path}")

    elif args.mode == "test_time":
        payload=load_ckpt(args.load_path)
        tinfo=payload.get("tokenizer", {"kind":args.tok_kind, "model_path": tok_model_path, "vocab_size":vocab_size})
        tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        if payload.get("arch","gpt")=="gpt":
            cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg)
        else:
            cfg=payload["cfg"]; model=NanoMamba(cfg.get("vocab_size",vocab_size), cfg.get("block_size",args.block_size), cfg.get("n_layer",args.n_layer), cfg.get("n_embd",args.n_embd))
        model.load_state_dict(payload["state_dict"])
        out=test_time_adapt_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.tta_steps,args.tta_lr,
                                     args.tta_layers,args.tta_context,args.temperature,args.top_k)
        info(out, title="Test-Time Adaptation Output")

    elif args.mode == "active_inference":
        payload=load_ckpt(args.load_path)
        tinfo=payload.get("tokenizer", {"kind":args.tok_kind, "model_path": tok_model_path, "vocab_size":vocab_size})
        tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        if payload.get("arch","gpt")=="gpt":
            cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg)
        else:
            cfg=payload["cfg"]; model=NanoMamba(cfg.get("vocab_size",vocab_size), cfg.get("block_size",args.block_size), cfg.get("n_layer",args.n_layer), cfg.get("n_embd",args.n_embd))
        model.load_state_dict(payload["state_dict"])
        out=active_inference_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.ai_particles,
                                      args.ai_horizon,args.ai_beta,args.temperature,args.top_k)
        info(out, title="Active Inference Output")

    elif args.mode == "system2_reasoning":
        payload=load_ckpt(args.load_path)
        tinfo=payload.get("tokenizer", {"kind":args.tok_kind, "model_path": tok_model_path, "vocab_size":vocab_size})
        tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        if payload.get("arch","gpt")=="gpt":
            cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg)
        else:
            cfg=payload["cfg"]; model=NanoMamba(cfg.get("vocab_size",vocab_size), cfg.get("block_size",args.block_size), cfg.get("n_layer",args.n_layer), cfg.get("n_embd",args.n_embd))
        model.load_state_dict(payload["state_dict"])
        out=system2_reasoning_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.s2_branches,
                                       args.temperature,args.top_k,args.s2_vote)
        info(out, title="System-2 Reasoning Output")

# =============================================================================
# FULL PIPELINES
# =============================================================================
def run_full_pipeline():
    console.rule("[bold magenta]Unified End-to-End Pipeline (Monitored, 4+ combos)")
    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 50))
    DEFAULT_STEPS = int(os.environ.get("STEPS", 50))
    DEFAULT_BATCH = int(os.environ.get("BATCH", 64))

    # 1) GPT + BPE (MoE, specialize) -> Train
    main([
        "--mode","train",
        "--arch","gpt","--tok_kind","regex",
        "--use_moe","--routing_mode","specialize",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--save_path","artifacts/gpt_bpe_moe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","gpt_bpe_moe"
    ])

    # 2) Reward Model for GPT+BPE (create toy data if missing)
    rm_file = "data/preferences.jsonl"
    if not os.path.exists(rm_file):
        toy = [
            {"chosen":"I love this model","rejected":"I hate this model"},
            {"chosen":"This is a great answer","rejected":"This is a terrible answer"},
        ]
        os.makedirs("data", exist_ok=True)
        with open(rm_file, "w", encoding="utf-8") as f:
            for row in toy: f.write(json.dumps(row) + "\n")
    main([
        "--mode","train_rm","--arch","gpt","--tok_kind","regex",
        "--train_corpus",rm_file,
        "--load_path","artifacts/gpt_bpe_moe.pt",
        "--save_path","artifacts/reward_model_gpt_bpe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--epochs", str(max(5, DEFAULT_EPOCHS//5)),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--exp_name","reward_model_gpt_bpe"
    ])

    # 3) GRPO fine-tuning (GPT+BPE)
    main([
        "--mode","train_grpo","--arch","gpt","--tok_kind","regex",
        "--load_path","artifacts/gpt_bpe_moe.pt",
        "--save_path","artifacts/gpt_bpe_grpo.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--exp_name","gpt_bpe_grpo"
    ])

    # 4) GPT+BPE inference
    main([
        "--mode","infer",
        "--arch","gpt","--tok_kind","regex",
        "--load_path","artifacts/gpt_bpe_moe.pt",
        "--sample_start","Hello world,",
        "--sample_tokens","50",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","gpt_bpe_moe"
    ])

    # 5) GPT + Char (baseline)
    main([
        "--mode","train",
        "--arch","gpt","--tok_kind","basic",
        "--epochs", str(max(25, DEFAULT_EPOCHS//2)),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--save_path","artifacts/gpt_char.pt",
        "--exp_name","gpt_char"
    ])

    # 6) Mamba + BPE
    main([
        "--mode","train",
        "--arch","mamba","--tok_kind","regex",
        "--epochs", str(max(25, DEFAULT_EPOCHS//2)),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--save_path","artifacts/mamba_bpe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","mamba_bpe"
    ])

    # 7) Mamba + Char
    main([
        "--mode","train",
        "--arch","mamba","--tok_kind","basic",
        "--epochs", str(max(25, DEFAULT_EPOCHS//2)),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--save_path","artifacts/mamba_char.pt",
        "--exp_name","mamba_char"
    ])

    # 8) Mamba inference (char)
    main([
        "--mode","infer",
        "--arch","mamba","--tok_kind","basic",
        "--load_path","artifacts/mamba_char.pt",
        "--sample_start","ROMEO:",
        "--sample_tokens","80",
        "--exp_name","mamba_char"
    ])

def run_colab_quick_pipeline():
    console.rule("[bold magenta]NanoGPT-BPE Quick Pipeline (Train + Inference Modes)")
    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 20))
    DEFAULT_STEPS  = int(os.environ.get("STEPS", 50))
    DEFAULT_BATCH  = int(os.environ.get("BATCH", 32))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    info(f"Using device: {device}", title="Device")

    # Train (small)
    train_args = [
        "--mode", "train",
        "--arch","gpt","--tok_kind","regex",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--lr", "3e-4",
        "--block_size", "128",
        "--n_layer", "4",
        "--n_head", "4",
        "--n_embd", "256",
        "--vocab_size", "512",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--save_path", "artifacts/gpt_bpe.pt",
        "--seed", "42",
        "--exp_name","quick_gpt_bpe"
    ]
    try:
        main(train_args)
    except Exception as e:
        warn(f"Training failed: {e}\nFalling back to smaller model...")
        fallback_args = [
            "--mode", "train",
            "--arch","gpt","--tok_kind","regex",
            "--epochs", "2","--steps_per_epoch", "50","--batch_size", "16","--lr", "3e-4",
            "--block_size", "64","--n_layer", "2","--n_head", "2","--n_embd", "128",
            "--vocab_size", "256","--tok_prefix", "artifacts/tokenizers/tiny_bpe_fallback",
            "--save_path", "artifacts/gpt_bpe.pt","--seed", "42","--exp_name","quick_gpt_bpe_fallback"
        ]
        main(fallback_args)

    prompt = "In a world where AI and humans collaborate,"
    sample_tokens = 80

    # Standard Inference
    main([
        "--mode", "infer",
        "--arch","gpt","--tok_kind","regex",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--exp_name","quick_gpt_bpe"
    ])

    # Test-Time Adaptation
    main([
        "--mode", "test_time",
        "--arch","gpt","--tok_kind","regex",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--tta_steps", "3","--tta_lr", "1e-3","--tta_layers", "lm_head,ln_f","--tta_context", "64",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--exp_name","quick_gpt_bpe"
    ])

    # Active Inference
    main([
        "--mode", "active_inference",
        "--arch","gpt","--tok_kind","regex",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--ai_particles", "6","--ai_horizon", "3","--ai_beta", "0.3",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--exp_name","quick_gpt_bpe"
    ])

    # System-2 Reasoning
    main([
        "--mode", "system2_reasoning",
        "--arch","gpt","--tok_kind","regex",
        "--load_path", "artifacts/gpt_bpe.pt",
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--s2_branches", "5","--s2_vote", "best_logprob",
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--exp_name","quick_gpt_bpe"
    ])

    # Package results
    console.rule("[bold magenta]Packaging Results")
    zip_path = "artifacts/pipeline_results.zip"
    os.makedirs("artifacts", exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file in Path("artifacts").glob("*.pt"): zf.write(file, file.relative_to("."))
        for file in Path("artifacts/tokenizers").glob("*.model"): zf.write(file, file.relative_to("."))
        for file in Path("artifacts/tokenizers").glob("*.vocab"): zf.write(file, file.relative_to("."))
    ok(f"Pipeline complete! Results saved to {zip_path}")

# =============================================================================
# ENTRYPOINT
# =============================================================================
def _sanitize_argv(argv):
    out = []; skip = False
    for a in argv:
        if skip: skip = False; continue
        if a in ("-f", "--f"): skip = True; continue
        if a.startswith("-f=") or a.startswith("--f="): continue
        out.append(a)
    return out

# CHECK IF THIS IS WORKING WITH THE FIX WE NEED FOR THE PREVIOUS ONE
# if __name__ == "__main__":
#     argv = _sanitize_argv(sys.argv[1:])
#     # If required args missing, run the full monitored pipeline
#     if ("--mode" not in argv) or ("--arch" not in argv) or ("--tok_kind" not in argv):
#         # choose the heavier monitored pipeline
#         run_full_pipeline()
#     else:
#         main(argv)


In [ ]:
# =============================================================================
# FULL PIPELINES + INFERENCE SUITE
# =============================================================================
def run_inference_suite(arch, tok_kind, ckpt_path, prompt="Once upon a time", sample_tokens=80):
    console.rule(f"[bold cyan]Inference Suite for {arch.upper()}-{tok_kind.upper()}")
    base_args = [
        "--arch", arch,
        "--tok_kind", tok_kind,
        "--load_path", ckpt_path,
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--exp_name", f"{arch}_{tok_kind}_inference_suite"
    ]

    # Standard Inference
    main(["--mode","infer"] + base_args)

    # Test-Time Adaptation
    main(["--mode","test_time",
          "--tta_steps","3","--tta_lr","1e-3","--tta_layers","lm_head,ln_f","--tta_context","64"] + base_args)

    # Active Inference
    main(["--mode","active_inference",
          "--ai_particles","6","--ai_horizon","3","--ai_beta","0.3"] + base_args)

    # System-2 Reasoning
    main(["--mode","system2_reasoning",
          "--s2_branches","5","--s2_vote","best_logprob"] + base_args)


def run_full_pipeline():
    console.rule("[bold magenta]Unified End-to-End Pipeline (Training + All Inference Strategies)")
    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 20))
    DEFAULT_STEPS  = int(os.environ.get("STEPS", 50))
    DEFAULT_BATCH  = int(os.environ.get("BATCH", 32))

    # === 1) GPT + BPE (MoE) Train ===
    main([
        "--mode","train","--arch","gpt","--tok_kind","regex",
        "--use_moe","--routing_mode","specialize",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--save_path","artifacts/gpt_bpe_moe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","gpt_bpe_moe"
    ])
    run_inference_suite("gpt","regex","artifacts/gpt_bpe_moe.pt", prompt="Hello world,")

    # === 2) GPT + Char Train ===
    main([
        "--mode","train","--arch","gpt","--tok_kind","basic",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--save_path","artifacts/gpt_char.pt",
        "--exp_name","gpt_char"
    ])
    run_inference_suite("gpt","basic","artifacts/gpt_char.pt", prompt="To be, or not to be")

    # === 3) Mamba + BPE Train ===
    main([
        "--mode","train","--arch","mamba","--tok_kind","regex",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--save_path","artifacts/mamba_bpe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","mamba_bpe"
    ])
    run_inference_suite("mamba","regex","artifacts/mamba_bpe.pt", prompt="AI is changing the world,")

    # === 4) Mamba + Char Train ===
    main([
        "--mode","train","--arch","mamba","--tok_kind","basic",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--save_path","artifacts/mamba_char.pt",
        "--exp_name","mamba_char"
    ])
    run_inference_suite("mamba","basic","artifacts/mamba_char.pt", prompt="ROMEO:")


# =============================================================================
# ENTRYPOINT FOR COLAB
# =============================================================================
def _sanitize_argv(argv):
    out = []; skip = False
    for a in argv:
        if skip: skip = False; continue
        if a in ("-f", "--f"): skip = True; continue
        if a.startswith("-f=") or a.startswith("--f="): continue
        out.append(a)
    return out

if __name__ == "__main__":
    argv = _sanitize_argv(sys.argv[1:])
    if ("--mode" not in argv) or ("--arch" not in argv) or ("--tok_kind" not in argv):
        run_full_pipeline()
    else:
        main(argv)


──────────────────────── Unified End-to-End Pipeline (Training + All Inference Strategies) ────────────────────────

╭──────────────────────────────────────────────────── Device ─────────────────────────────────────────────────────╮
│ Using device: cpu                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ Output structure at: training_outputs/gpt_bpe_moe                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ merge 1/768: (32, 116) -> 256                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ merge 2/768: (104, 101) -> 257                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ GPT-REGEX Train — Epoch 15/20  train=3.837  val=4.132  grad=0.674  lr=6.18e-05  time=1247.0s                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ GPT-REGEX Train — Epoch 16/20  train=3.813  val=4.179  grad=0.680  lr=4.39e-05  time=1242.3s                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ GPT-REGEX Train — Epoch 17/20  train=3.800  val=4.079  grad=0.673  lr=2.86e-05  time=1246.6s                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ GPT-REGEX Train — Epoch 18/20  train=3.802  val=4.105  grad=0.676  lr=1.63e-05  time=1252.1s                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ GPT-REGEX Train — Epoch 19/20  train=3.795  val=4.187  grad=0.671  lr=7.34e-06  time=1238.6s                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
# =============================================================================
# FULL PIPELINES + INFERENCE SUITE (with n_layer / n_head / n_embd / block_size)
# =============================================================================
def run_inference_suite(arch, tok_kind, ckpt_path, prompt="Once upon a time", sample_tokens=80,
                        n_layer=4, n_head=4, n_embd=256, block_size=128):
    console.rule(f"[bold cyan]Inference Suite for {arch.upper()}-{tok_kind.upper()}")
    base_args = [
        "--arch", arch,
        "--tok_kind", tok_kind,
        "--load_path", ckpt_path,
        "--sample_start", prompt,
        "--sample_tokens", str(sample_tokens),
        "--tok_prefix", "artifacts/tokenizers/tiny_bpe",
        "--exp_name", f"{arch}_{tok_kind}_inference_suite",
        "--n_layer", str(n_layer),
        "--n_head", str(n_head),
        "--n_embd", str(n_embd),
        "--block_size", str(block_size)
    ]

    # Standard Inference
    main(["--mode","infer"] + base_args)

    # Test-Time Adaptation
    main(["--mode","test_time",
          "--tta_steps","3","--tta_lr","1e-3","--tta_layers","lm_head,ln_f","--tta_context","64"] + base_args)

    # Active Inference
    main(["--mode","active_inference",
          "--ai_particles","6","--ai_horizon","3","--ai_beta","0.3"] + base_args)

    # System-2 Reasoning
    main(["--mode","system2_reasoning",
          "--s2_branches","5","--s2_vote","best_logprob"] + base_args)


def run_full_pipeline():
    console.rule("[bold magenta]Unified End-to-End Pipeline (Training + All Inference Strategies)")
    DEFAULT_EPOCHS = int(os.environ.get("EPOCHS", 20))
    DEFAULT_STEPS  = int(os.environ.get("STEPS", 50))
    DEFAULT_BATCH  = int(os.environ.get("BATCH", 32))

    # Model hyperparams
    n_layer, n_head, n_embd, block_size = 4, 4, 256, 128

    # === 1) GPT + BPE (MoE) Train ===
    main([
        "--mode","train","--arch","gpt","--tok_kind","regex",
        "--use_moe","--routing_mode","specialize",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--n_layer", str(n_layer),
        "--n_head", str(n_head),
        "--n_embd", str(n_embd),
        "--block_size", str(block_size),
        "--save_path","artifacts/gpt_bpe_moe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","gpt_bpe_moe"
    ])
    run_inference_suite("gpt","regex","artifacts/gpt_bpe_moe.pt", prompt="Hello world,",
                        n_layer=n_layer,n_head=n_head,n_embd=n_embd,block_size=block_size)

    # === 2) GPT + Char Train ===
    main([
        "--mode","train","--arch","gpt","--tok_kind","basic",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--n_layer", str(n_layer),
        "--n_head", str(n_head),
        "--n_embd", str(n_embd),
        "--block_size", str(block_size),
        "--save_path","artifacts/gpt_char.pt",
        "--exp_name","gpt_char"
    ])
    run_inference_suite("gpt","basic","artifacts/gpt_char.pt", prompt="To be, or not to be",
                        n_layer=n_layer,n_head=n_head,n_embd=n_embd,block_size=block_size)

    # === 3) Mamba + BPE Train ===
    main([
        "--mode","train","--arch","mamba","--tok_kind","regex",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--n_layer", str(n_layer),
        "--n_embd", str(n_embd),
        "--block_size", str(block_size),
        "--save_path","artifacts/mamba_bpe.pt",
        "--tok_prefix","artifacts/tokenizers/tiny_bpe",
        "--exp_name","mamba_bpe"
    ])
    run_inference_suite("mamba","regex","artifacts/mamba_bpe.pt", prompt="AI is changing the world,",
                        n_layer=n_layer,n_head=n_head,n_embd=n_embd,block_size=block_size)

    # === 4) Mamba + Char Train ===
    main([
        "--mode","train","--arch","mamba","--tok_kind","basic",
        "--epochs", str(DEFAULT_EPOCHS),
        "--steps_per_epoch", str(DEFAULT_STEPS),
        "--batch_size", str(DEFAULT_BATCH),
        "--n_layer", str(n_layer),
        "--n_embd", str(n_embd),
        "--block_size", str(block_size),
        "--save_path","artifacts/mamba_char.pt",
        "--exp_name","mamba_char"
    ])
    run_inference_suite("mamba","basic","artifacts/mamba_char.pt", prompt="ROMEO:",
                        n_layer=n_layer,n_head=n_head,n_embd=n_embd,block_size=block_size)


# =============================================================================
# ENTRYPOINT FOR COLAB
# =============================================================================
def _sanitize_argv(argv):
    out = []; skip = False
    for a in argv:
        if skip: skip = False; continue
        if a in ("-f", "--f"): skip = True; continue
        if a.startswith("-f=") or a.startswith("--f="): continue
        out.append(a)
    return out

if __name__ == "__main__":
    argv = _sanitize_argv(sys.argv[1:])
    if ("--mode" not in argv) or ("--arch" not in argv) or ("--tok_kind" not in argv):
        run_full_pipeline()
    else:
        main(argv)


──────────────────────── Unified End-to-End Pipeline (Training + All Inference Strategies) ────────────────────────

╭──────────────────────────────────────────────────── Device ─────────────────────────────────────────────────────╮
│ Using device: cpu                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ Output structure at: training_outputs/gpt_bpe_moe                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Tokenizer ───────────────────────────────────────────────────╮
│ Loaded tokenizer artifacts/tokenizers/tiny_bpe.model (size=1024)                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Model ─────────────────────────────────────────────────────╮
│ Params total=7.79M  trainable=7.79M                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────────── Device ─────────────────────────────────────────────────────╮
│ Using device: cpu                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── Info ──────────────────────────────────────────────────────╮
│ Output structure at: training_outputs/mamba_basic_inference_suite                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Tokenizer ───────────────────────────────────────────────────╮
│ Loaded tokenizer artifacts/tokenizers/tiny_bpe.model (size=1024)                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── OK ───────────────────────────────────────────────────────╮
│ Loaded artifacts/mamba_char.pt                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── System-2 Reasoning Output ───────────────────────────────────────────╮
│ ROMEO: I have saces, sir, how side                                                                              │
│ I will not, but to holdly for that shall bed in the bany stifcreround,                                          │
│ To his tears as                                                                                                 │
│ Of the dropladiness, I am so.                                                                                   │
│                                                                                                                 │
│ BROKE:                                                                                                          │
│ The ready, be in this ext; I think?                                                                             │
│                                                                                                                 │
│ MARET:                                                                                                          │
│ You                                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## End-to-end: BPE tokenizer + NanoGPT (+ optional MoE) + ENERGY-BASED training (NCE/InfoNCE, Sampled-Softmax) + 3 inference modes.

In [ ]:
#!/usr/bin/env python3
# unified_gpt_moe_bpe_energy.py
# End-to-end: BPE tokenizer + NanoGPT (+ optional MoE) + ENERGY-BASED training (NCE/InfoNCE, Sampled-Softmax) + 3 inference modes.
#
# Highlights
# - Drop-in replacement of cross-entropy with energy-based loss: Noise-Contrastive Estimation (NCE) and sampled softmax.
# - Transformer (with optional MoE) outputs unnormalized scores (energies = -scores) per token.
# - Negative sampling from unigram or uniform proposal; optional temperature.
# - Inference keeps standard autoregressive sampling from scores (no partition function needed).
# - Compatible with previous checkpoints for architecture; training heads/opts differ.

import os, math, json, argparse, random, time, urllib.request
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Any, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------- pretty console (optional; fallback if missing) -----------
try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.table import Table
    from rich import box
    console = Console()
    def info(msg, **kw): console.print(Panel(msg, **({"border_style":"cyan","title":"Info","box":box.ROUNDED} | kw)))
    def warn(msg, **kw): console.print(Panel(msg, **({"border_style":"yellow","title":"Warning","box":box.ROUNDED} | kw)))
    def ok(msg, **kw):   console.print(Panel(msg, **({"border_style":"green","title":"OK","box":box.ROUNDED} | kw)))
except Exception:
    class _Dummy:
        def print(self, *a, **k): print(*a)
        def rule(self, *a, **k): print("="*60, *(a or ()), "="*60)
    console = _Dummy()
    def info(msg, **kw): print("[INFO]", msg)
    def warn(msg, **kw): print("[WARN]", msg)
    def ok(msg, **kw):   print("[ OK ]", msg)

# =============================================================================
#                               TOKENIZER (minBPE)
# =============================================================================
import unicodedata
import regex as re

def _replace_ctl(s: str) -> str:
    return "".join(ch if unicodedata.category(ch)[0] != "C" else f"\\u{ord(ch):04x}" for ch in s)

def _render_tok(t: bytes) -> str:
    return _replace_ctl(t.decode("utf-8", errors="replace"))

def _get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for p in zip(ids, ids[1:]): counts[p] = counts.get(p, 0) + 1
    return counts

def _merge(ids, pair, idx):
    newids=[]; i=0
    while i < len(ids):
        if ids[i]==pair[0] and i < len(ids)-1 and ids[i+1]==pair[1]:
            newids.append(idx); i+=2
        else:
            newids.append(ids[i]); i+=1
    return newids

class Tokenizer:
    def __init__(self):
        self.merges={}; self.pattern=""; self.special_tokens={}
        self.vocab={i:bytes([i]) for i in range(256)}
    def _rebuild_vocab(self):
        vocab={i:bytes([i]) for i in range(256)}
        for (a,b),i in self.merges.items(): vocab[i]=vocab[a]+vocab[b]
        for s,i in self.special_tokens.items(): vocab[i]=s.encode("utf-8")
        self.vocab=vocab
    def save(self, prefix):
        os.makedirs(os.path.dirname(prefix), exist_ok=True) if os.path.dirname(prefix) else None
        with open(prefix+".model","w",encoding="utf-8") as f:
            f.write("minbpe v1\n"); f.write(f"{self.pattern}\n"); f.write(f"{len(self.special_tokens)}\n")
            for s,i in self.special_tokens.items(): f.write(f"{s} {i}\n")
            for (a,b),_i in self.merges.items(): f.write(f"{a} {b}\n")
        with open(prefix+".vocab","w",encoding="utf-8") as f:
            inv={idx:pair for pair,idx in self.merges.items()}
            for i,tok in self.vocab.items():
                s=_render_tok(tok)
                if i in inv:
                    a,b=inv[i]; f.write(f"[{_render_tok(self.vocab[a])}][{_render_tok(self.vocab[b])}] -> [{s}] {i}\n")
                else:
                    f.write(f"[{s}] {i}\n")
    def load(self, model_file):
        merges={}; specials={}
        idx=256
        with open(model_file,"r",encoding="utf-8") as f:
            version=f.readline().strip(); assert version=="minbpe v1"
            self.pattern=f.readline().strip()
            ns=int(f.readline().strip())
            for _ in range(ns):
                s,si=f.readline().strip().split(); specials[s]=int(si)
            for line in f:
                a,b=map(int,line.split()); merges[(a,b)]=idx; idx+=1
        self.merges=merges; self.special_tokens=specials; self._rebuild_vocab()

class BasicTokenizer(Tokenizer):
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size>=256; num_merges=vocab_size-256
        ids=list(text.encode("utf-8")); merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats=_get_stats(ids); pair=max(stats,key=stats.get)
            idx=256+i; ids=_merge(ids,pair,idx); merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
            if verbose and i<10: info(f"merge {i+1}/{num_merges}: {pair} -> {idx}")
        self.merges=merges; self.vocab=vocab
    def encode(self, text):
        ids=list(text.encode("utf-8"))
        while len(ids)>=2:
            stats=_get_stats(ids); pair=min(stats,key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=_merge(ids,pair,self.merges[pair])
        return ids
    def decode(self, ids): return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

GPT4_SPLIT = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexTokenizer(Tokenizer):
    def __init__(self): super().__init__(); self.pattern=GPT4_SPLIT; self.compiled=re.compile(self.pattern)
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size>=256; num_merges=vocab_size-256
        chunks=re.findall(self.compiled,text); ids=[list(c.encode("utf-8")) for c in chunks]
        merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats={}; [ _get_stats(ci,stats) for ci in ids ]
            pair=max(stats,key=stats.get); idx=256+i; ids=[_merge(ci,pair,idx) for ci in ids]
            merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
            if verbose and i<10: info(f"merge {i+1}/{num_merges}: {pair} -> {idx}")
        self.merges=merges; self.vocab=vocab
    def _encode_chunk(self, bs):
        ids=list(bs)
        while len(ids)>=2:
            stats=_get_stats(ids); pair=min(stats,key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=_merge(ids,pair,self.merges[pair])
        return ids
    def encode(self, text):
        ids=[]; [ids.extend(self._encode_chunk(c.encode("utf-8"))) for c in re.findall(self.compiled,text)]
        return ids
    def decode(self, ids): return BasicTokenizer.decode(self, ids)

def _new_tok(kind): return BasicTokenizer() if kind=="basic" else RegexTokenizer()

def train_or_load_tokenizer(kind,vocab_size,text,prefix,verbose=True):
    os.makedirs(os.path.dirname(prefix),exist_ok=True) if os.path.dirname(prefix) else None
    model=prefix+".model"
    if os.path.exists(model):
        tok=_new_tok(kind); tok.load(model); vs=max(tok.vocab.keys())+1
        if verbose: ok(f"Loaded tokenizer {model} (size={vs})", title="Tokenizer")
        return tok,model,vs
    tok=_new_tok(kind); tok.train(text,max(256,vocab_size),verbose); tok.save(prefix); vs=max(tok.vocab.keys())+1
    if verbose: ok(f"Saved tokenizer {prefix}.model (size={vs})", title="Tokenizer")
    return tok,model,vs

# =============================================================================
#                              DATA (tiny shakespeare)
# =============================================================================

def _repeat_to_len(s: str, target_len: int) -> str:
    if not s: s=" \n"
    return (s * ((target_len // len(s))+1))[:target_len]

def load_tiny_shakespeare(data_dir="./data", target_len=100_000):
    os.makedirs(data_dir,exist_ok=True)
    path=os.path.join(data_dir,"tinyshakespeare_input.txt")
    if not os.path.exists(path):
        try:
            url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url,path)
        except Exception:
            warn("Could not download dataset; writing fallback sample.")
            sample=("From fairest creatures we desire increase,\n"
                    "That thereby beauty's rose might never die,\n"
                    "But as the riper should by time decease,\n"
                    "His tender heir might bear his memory:\n")
            with open(path,"w",encoding="utf-8") as f: f.write(_repeat_to_len(sample, target_len))
    text=open(path,"r",encoding="utf-8").read()
    split=int(0.9*len(text)); return text[:split], text[split:]

# utility: minibatches of token ids

def get_batch_tokens(ids, block_size, batch_size, device):
    if len(ids) <= block_size + 1:
        raise RuntimeError(f"Sequence too short ({len(ids)}) for block_size={block_size}.")
    ix=torch.randint(len(ids)-block_size-1,(batch_size,))
    x=torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y=torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================================================================
#                               MODEL (GPT + MoE)
# =============================================================================
class LayerNorm(nn.Module):
    def __init__(self,n,bias): super().__init__(); self.weight=nn.Parameter(torch.ones(n)); self.bias=nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self,x): return F.layer_norm(x,self.weight.shape,self.weight,self.bias,1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self,n_embd,n_head,block,dropout,bias,backend="sdpa"):
        super().__init__(); assert n_embd % n_head == 0
        self.n_head=n_head; self.n_embd=n_embd; self.dropout=dropout; self.backend=backend
        self.c_attn=nn.Linear(n_embd,3*n_embd,bias=bias); self.c_proj=nn.Linear(n_embd,n_embd,bias=bias)
        if backend=="vanilla" or not hasattr(F,"scaled_dot_product_attention"):
            self.register_buffer("bias", torch.tril(torch.ones(block,block)).view(1,1,block,block))
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        if self.backend=="sdpa" and hasattr(F,"scaled_dot_product_attention"):
            y=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=self.dropout if self.training else 0.0)
        else:
            att=(q@k.transpose(-2,-1))/math.sqrt(k.size(-1))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0,float("-inf"))
            att=F.softmax(att,dim=-1); y=att@v
        y=y.transpose(1,2).contiguous().view(B,T,C); return self.c_proj(y)

def entropy_mean(probs,eps=1e-9): return -(probs.clamp_min(eps)*probs.clamp_min(eps).log()).sum(-1).mean()

class TopKRouter(nn.Module):
    def __init__(self,in_dim,hidden_dim,num_experts,k=1,temp=1.0):
        super().__init__(); self.net=nn.Sequential(nn.Linear(in_dim,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,num_experts))
        self.k=k; self.temp=float(temp); self.register_buffer("logits_bias", torch.zeros(num_experts))
    @staticmethod
    def _gumbel(shape, device):
        u=torch.rand(shape,device=device).clamp_(1e-9,1-1e-9); return -torch.log(-torch.log(u))
    def forward(self,x,add_gumbel=False):
        logits=self.net(x)/self.temp + self.logits_bias
        if add_gumbel and self.training: logits = logits + self._gumbel(logits.shape, logits.device)
        probs=F.softmax(logits,dim=-1)
        if self.k>=probs.size(-1): return probs, probs
        topk_vals,topk_idx=torch.topk(probs,self.k,dim=-1)
        mask=torch.zeros_like(probs); mask.scatter_(dim=-1,index=topk_idx,src=torch.ones_like(topk_vals))
        sp=probs*mask; sp=sp/(sp.sum(dim=-1,keepdim=True)+1e-9); return sp,probs

class MoEFFN(nn.Module):
    def __init__(self,in_dim,num_experts=3,k=1,router_hidden=128,dropout=0.1,blw=0.02,entropy_penalty=0.001,
                 routing_mode="specialize", router_temp=1.0):
        super().__init__(); hidden=4*in_dim
        self.expert_fc=nn.ModuleList([nn.Linear(in_dim,hidden) for _ in range(num_experts)])
        self.expert_proj=nn.ModuleList([nn.Linear(hidden,in_dim) for _ in range(num_experts)])
        self.shared_fc=nn.Linear(in_dim,hidden); self.shared_proj=nn.Linear(hidden,in_dim)
        self.router=TopKRouter(in_dim,router_hidden,num_experts,k,temp=router_temp)
        self.drop=nn.Dropout(dropout); self.num_experts=num_experts
        self.blw=blw; self.entw=entropy_penalty; self.routing_mode=routing_mode
    def forward(self,x):
        B,T,C=x.shape; xf=x.view(B*T,C); sp,dp=self.router(xf,add_gumbel=self.training)
        routed=0.0
        for e in range(self.num_experts):
            h=self.expert_proj[e](F.gelu(self.expert_fc[e](xf))); routed+=sp[:,e].unsqueeze(-1)*h
        shared=self.shared_proj(F.gelu(self.shared_fc(xf)))
        y=self.drop((shared+routed).view(B,T,C))
        aux=(-self.entw*entropy_mean(dp)) if self.routing_mode=="specialize" else torch.tensor(0.0,device=x.device)
        stats={"expert_selection_counts":(sp>0).float().sum(0),"mean_routing_probs":dp.mean(0)}
        return y,aux,stats

@dataclass
class GPTCfg:
    block_size:int=256; vocab_size:int=256; n_layer:int=6; n_head:int=6; n_embd:int=384
    dropout:float=0.1; bias:bool=True; attention_backend:str="sdpa"
    use_moe:bool=False; num_experts:int=3; k:int=1; router_hidden:int=128; blw:float=0.02
    entropy_penalty:float=0.001; routing_mode:str="specialize"; router_temp:float=1.0

class GPTBlock(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout, bias, backend="sdpa", moe: Optional[MoEFFN]=None):
        super().__init__(); self.ln1=LayerNorm(n_embd,bias); self.attn=CausalSelfAttention(n_embd,n_head,block,dropout,bias,backend)
        self.ln2=LayerNorm(n_embd,bias); self.moe=moe
        if moe is None:
            hidden=4*n_embd
            self.ffn=nn.Sequential(nn.Linear(n_embd,hidden,bias=bias), nn.GELU(), nn.Linear(hidden,n_embd,bias=bias), nn.Dropout(dropout))
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        if self.moe is None:
            x=x+self.ffn(self.ln2(x)); aux=torch.tensor(0.0,device=x.device); stats={}
        else:
            y,aux,stats=self.moe(self.ln2(x)); x=x+y
        return x, aux, stats

class NanoGPT(nn.Module):
    def __init__(self, cfg:GPTCfg):
        super().__init__(); self.cfg=cfg
        self.wte=nn.Embedding(cfg.vocab_size,cfg.n_embd); self.wpe=nn.Embedding(cfg.block_size,cfg.n_embd)
        blocks=[]
        for _ in range(cfg.n_layer):
            moe=None
            if cfg.use_moe: moe=MoEFFN(cfg.n_embd,cfg.num_experts,cfg.k,cfg.router_hidden,cfg.dropout,cfg.blw,cfg.entropy_penalty,cfg.routing_mode,cfg.router_temp)
            blocks.append(GPTBlock(cfg.n_embd,cfg.n_head,cfg.block_size,cfg.dropout,cfg.bias,cfg.attention_backend,moe))
        self.blocks=nn.ModuleList(blocks); self.ln_f=LayerNorm(cfg.n_embd,cfg.bias); self.lm_head=nn.Linear(cfg.n_embd,cfg.vocab_size,bias=False)
        self.lm_head.weight=self.wte.weight
        self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward_hidden(self, idx):
        B,T=idx.shape; assert T<=self.cfg.block_size
        pos=torch.arange(0,T,device=idx.device).long(); x=self.wte(idx)+self.wpe(pos)[None,:,:]
        aux_total=torch.tensor(0.0,device=idx.device); last={}
        for blk in self.blocks:
            x,aux,stats=blk(x); aux_total=aux_total+aux; last=stats
        x=self.ln_f(x)
        return x, aux_total, last
    def forward(self, idx, targets=None):
        # For compatibility: returns logits like a standard LM (scores), but loss is computed outside for EBM.
        x, aux_total, last = self.forward_hidden(idx)
        logits = self.lm_head(x)  # unnormalized scores
        loss=None
        if targets is not None:
            # Fallback CE (not used in EBM training unless explicitly chosen)
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1)) + aux_total
        return logits, loss, last
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.cfg.block_size else idx[:,-self.cfg.block_size:]
            logits,_,_=self(cond); logits=logits[:,-1,:]/max(1e-6,temperature)
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); idx_next=torch.multinomial(probs,1); idx=torch.cat((idx,idx_next),dim=1)
        return idx

def count_parameters(m: nn.Module) -> Tuple[int,int]:
    total=sum(p.numel() for p in m.parameters())
    train=sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, train

# =============================================================================
#                      ENERGY-BASED LOSSES (NCE / Sampled-Softmax)
# =============================================================================

class NegativeSampler:
    def __init__(self, vocab_size:int, counts: Optional[List[int]]=None, kind:str="unigram", temperature:float=1.0):
        self.vocab_size=vocab_size; self.kind=kind; self.temperature=max(1e-6, float(temperature))
        if counts is None or sum(counts)==0 or kind=="uniform":
            probs=torch.ones(vocab_size)
        else:
            probs=torch.tensor(counts, dtype=torch.float)
        probs=probs.pow(1.0/self.temperature)
        probs=probs / probs.sum()
        self.register = probs
        self.device="cpu"
    def to(self, device:str):
        self.device=device; self.register=self.register.to(device); return self
    @torch.no_grad()
    def sample(self, num_samples:int) -> torch.Tensor:
        # Returns shape (num_samples,)
        return torch.multinomial(self.register, num_samples=num_samples, replacement=True)
    def prob(self, tokens: torch.Tensor) -> torch.Tensor:
        return self.register[tokens]


def build_unigram_counts(encoded_ids: List[int], vocab_size: int) -> List[int]:
    counts=[0]*vocab_size
    for t in encoded_ids:
        if 0 <= t < vocab_size: counts[t]+=1
    # avoid zeros
    return [c if c>0 else 1 for c in counts]


def nce_loss_sampled(logits: torch.Tensor, targets: torch.Tensor, sampler: NegativeSampler, k_neg: int, reduction:str="mean"):
    """
    Noise-Contrastive Estimation for next-token energies.
    logits: (B,T,V) unnormalized scores (higher is lower energy)
    targets: (B,T)
    For each position, draw k_neg negative tokens from proposal q, compute binary logistic classification between data (1) and noise (0),
    using the standard NCE correction with q.
    """
    B,T,V = logits.shape
    device = logits.device

    # Gather positive scores
    pos_scores = logits.gather(-1, targets.unsqueeze(-1)).squeeze(-1)  # (B,T)

    # Sample negatives for each (B,T)
    num_positions = B*T
    neg_flat = sampler.sample(num_positions * k_neg).to(device)  # (B*T*k)
    neg = neg_flat.view(B, T, k_neg)  # (B,T,k)
    neg_scores = logits.gather(-1, neg)  # (B,T,k)

    # Proposal probabilities q for correction
    with torch.no_grad():
        q_pos = sampler.prob(targets.view(-1)).view(B,T) + 1e-9
        q_neg = sampler.prob(neg_flat).view(B,T,k_neg) + 1e-9

    # NCE objective: log sigma(s_pos - log(k*q_pos)) + sum_i log sigma(-(s_neg_i - log(k*q_neg_i)))
    s_pos = pos_scores - torch.log(k_neg * q_pos)
    s_neg = neg_scores - torch.log(k_neg * q_neg)

    loss_pos = F.softplus(-s_pos)  # -log sigma(s_pos)
    loss_neg = F.softplus(s_neg).sum(dim=-1)  # sum log(1+exp(s_neg)) == -sum log sigma(-s_neg)
    loss = loss_pos + loss_neg

    if reduction=="mean":
        return loss.mean()
    elif reduction=="sum":
        return loss.sum()
    return loss  # (B,T)


def sampled_softmax_loss(logits: torch.Tensor, targets: torch.Tensor, sampler: NegativeSampler, k_neg:int, reduction:str="mean"):
    """
    Sampled Softmax / InfoNCE surrogate: cross-entropy over a small set {pos + k_neg negatives}.
    Not exactly NCE, but widely used and stable.
    """
    B,T,V = logits.shape
    device = logits.device

    # sample negatives per position
    num_positions=B*T
    neg_flat = sampler.sample(num_positions * k_neg).to(device)
    neg = neg_flat.view(B,T,k_neg)

    pos_scores = logits.gather(-1, targets.unsqueeze(-1))  # (B,T,1)
    neg_scores = logits.gather(-1, neg)                    # (B,T,k)

    # concatenate and do CE
    cat_scores = torch.cat([pos_scores, neg_scores], dim=-1)  # (B,T,1+k)

    labels = torch.zeros(B,T, dtype=torch.long, device=device)  # index 0 is the positive
    loss = F.cross_entropy(cat_scores.view(-1, 1+k_neg), labels.view(-1), reduction=reduction)
    return loss

# =============================================================================
#                            TRAIN / SAMPLE / SAVELOAD
# =============================================================================

def train_energy(
    model,
    train_ids,
    val_ids,
    block_size,
    epochs,
    steps,
    batch,
    lr,
    device,
    title,
    loss_kind:str,
    sampler: NegativeSampler,
    k_neg:int,
    clip:float=1.0,
    val_subset_steps:int=50,
):
    model=model.to(device); sampler.to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,betas=(0.9,0.95),weight_decay=0.1)

    def _eval_once():
        model.eval(); with torch.no_grad():
            tot=0.0; n=0
            for _ in range(val_subset_steps):
                xb,yb=get_batch_tokens(val_ids,block_size,batch,device)
                logits,_,_ = model(xb, None)
                if loss_kind=="nce":
                    loss = nce_loss_sampled(logits, yb, sampler, k_neg)
                else:
                    loss = sampled_softmax_loss(logits, yb, sampler, k_neg)
                tot += loss.item(); n += 1
            return tot/max(1,n)

    for ep in range(1,epochs+1):
        t0=time.time(); model.train(); losses=[]
        for _ in range(steps):
            xb,yb=get_batch_tokens(train_ids,block_size,batch,device)
            logits,_,_ = model(xb, None)  # unnormalized scores
            if loss_kind=="nce":
                loss = nce_loss_sampled(logits, yb, sampler, k_neg)
            else:
                loss = sampled_softmax_loss(logits, yb, sampler, k_neg)
            opt.zero_grad(); loss.backward();
            if clip>0: torch.nn.utils.clip_grad_norm_(model.parameters(),clip)
            opt.step(); losses.append(loss.item())
        dt=time.time()-t0
        vloss=_eval_once()
        info(f"Epoch {ep}/{epochs}  train={sum(losses)/len(losses):.3f}  val={vloss:.3f}  time={dt:.1f}s", title=title)

@torch.no_grad()
def sample_lm(model,tok,device,start="\n",tokens=200,title="Sample",temperature=0.8,top_k=200):
    model.eval().to(device)
    x=torch.tensor([tok.encode(start)],dtype=torch.long,device=device)
    y=model.generate(x,max_new_tokens=tokens,temperature=temperature,top_k=top_k)[0].tolist()
    text=tok.decode(y); info(text, title=title)


def save_ckpt(path,payload):
    os.makedirs(os.path.dirname(path),exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload,path); ok(f"Saved to {path}")


def load_ckpt(path):
    p=torch.load(path,map_location="cpu"); ok(f"Loaded {path}"); return p

# =============================================================================
#                          INFERENCE STRATEGIES (unchanged)
# =============================================================================
@torch.no_grad()
def _last_token_logprobs(model, input_ids: torch.Tensor):
    logits, _, _ = model(input_ids)
    return F.log_softmax(logits[:, -1, :], dim=-1)


def test_time_adapt_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    adapt_steps: int = 2, adapt_lr: float = 5e-4, adapt_layers: str = "lm_head,ln_f",
    context_reuse: int = 64, temperature: float = 0.8, top_k: int = 200,
):
    """Few gradient steps on a small parameter subset over recent context before each token."""
    model = model.to(device)
    model.eval()
    # select params
    adapt_names = {n.strip() for n in adapt_layers.split(",")} if adapt_layers else set()
    for p in model.parameters(): p.requires_grad=False
    to_adapt=[]
    if "lm_head" in adapt_names:
        for p in model.lm_head.parameters(): p.requires_grad=True; to_adapt.append(p)
    if "ln_f" in adapt_names:
        for p in model.ln_f.parameters(): p.requires_grad=True; to_adapt.append(p)
    if "blocks[-1]" in adapt_names and hasattr(model,"blocks"):
        for p in model.blocks[-1].parameters(): p.requires_grad=True; to_adapt.append(p)
    opt = torch.optim.Adam([p for p in to_adapt if p.requires_grad], lr=adapt_lr) if to_adapt else None

    ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
    for _ in range(max_new_tokens):
        if opt is not None and ids.size(1)>1:
            model.train()
            for _step in range(adapt_steps):
                ctx=ids[:,-min(context_reuse, ids.size(1)-1):]
                inp, tgt = ctx[:,:-1], ctx[:,1:]
                opt.zero_grad(set_to_none=True)
                # Use CE for quick adaptation (stable)
                logits, _, _ = model(inp, None)
                ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
                ce.backward();
                torch.nn.utils.clip_grad_norm_(to_adapt, 1.0)
                opt.step()
        model.eval()
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        logits,_,_=model(cond); logits=logits[:,-1,:]/temperature
        if top_k is not None:
            v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
        probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1)
        ids=torch.cat([ids,nxt],dim=1)
    return tok.decode(ids[0].tolist())


def _rollout_logprob(model, start_ids: torch.Tensor, horizon: int, temperature: float, top_k: Optional[int]):
    model.eval(); ids=start_ids.clone()
    total_logprob=0.0; total_entropy=0.0
    for _ in range(horizon):
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        logits,_,_=model(cond); logits=logits[:,-1,:]/temperature
        if top_k is not None:
            v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
        logprobs=F.log_softmax(logits,dim=-1); probs=logprobs.exp()
        entropy=-(probs*logprobs).sum(dim=-1).mean()
        nxt=torch.multinomial(probs,1)
        total_logprob+=logprobs.gather(-1,nxt).mean().item()
        total_entropy+=entropy.item()
        ids=torch.cat([ids,nxt],dim=1)
    return total_logprob, total_entropy

@torch.no_grad()
def active_inference_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    particles: int = 8, horizon: int = 4, beta: float = 0.2, temperature: float = 0.9, top_k: Optional[int] = 200,
):
    """Minimize expected free energy proxy = -Σ log p + β·Σ entropy via short rollouts per candidate action."""
    model=model.to(device).eval()
    ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
    for _ in range(max_new_tokens):
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        base_lp=_last_token_logprobs(model,cond)  # (1,V)
        probs=base_lp.exp()
        cand=torch.multinomial(probs,num_samples=particles)  # (1,P)
        best_F=float("inf"); best_token=None
        for j in range(particles):
            a=cand[:,j:j+1]
            start=torch.cat([cond,a],dim=1)
            logp_sum, ent_sum=_rollout_logprob(model,start,horizon,temperature,top_k)
            F=-logp_sum + beta*ent_sum
            if F<best_F: best_F=F; best_token=a
        if best_token is None:
            best_token=torch.argmax(base_lp,dim=-1,keepdim=True)
        ids=torch.cat([ids,best_token],dim=1)
    return tok.decode(ids[0].tolist())

@torch.no_grad()
def system2_reasoning_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    branches: int = 8, temperature: float = 0.9, top_k: Optional[int] = 200, vote: str = "majority",
):
    """Self-consistency: sample multiple full continuations and pick majority or best average logprob."""
    model=model.to(device).eval()
    def _sample_once():
        ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
        for _ in range(max_new_tokens):
            cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
            logits,_,_=model(cond); logits=logits[:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1)
            ids=torch.cat([ids,nxt],dim=1)
        # score by avg logprob
        if ids.size(1)>1:
            inp, tgt = ids[:,:-1], ids[:,1:]
            lp = 0.0
            # compute logprob using full softmax for scoring only
            logits, _, _ = model(inp, None)
            logp = F.log_softmax(logits, dim=-1)
            lp = logp.gather(-1, tgt.unsqueeze(-1)).mean().item()
            avg_lp = lp
        else:
            avg_lp = 0.0
        return tok.decode(ids[0].tolist()), avg_lp

    samples=[_sample_once() for _ in range(branches)]
    if vote=="best_logprob":
        return max(samples, key=lambda x:x[1])[0]
    from collections import Counter
    cleaned=[s[0].strip() for s in samples]
    return Counter(cleaned).most_common(1)[0][0]

# =============================================================================
#                                     MAIN
# =============================================================================

def main(argv=None):
    ap=argparse.ArgumentParser(description="Energy-Based Transformer: BPE + MoE + NCE/SampledSoftmax + TTA/ActiveInference/System2")
    ap.add_argument("--mode",required=True,choices=["train_energy","infer","test_time","active_inference","system2_reasoning"])
    ap.add_argument("--device",default="auto",choices=["auto","cpu","cuda"])
    ap.add_argument("--epochs",type=int,default=2)
    ap.add_argument("--steps_per_epoch",type=int,default=200)
    ap.add_argument("--batch_size",type=int,default=64)
    ap.add_argument("--lr",type=float,default=3e-4)
    ap.add_argument("--block_size",type=int,default=256)
    ap.add_argument("--dropout",type=float,default=0.1)
    ap.add_argument("--n_layer",type=int,default=6)
    ap.add_argument("--n_head",type=int,default=6)
    ap.add_argument("--n_embd",type=int,default=384)
    ap.add_argument("--bias",action="store_true",default=True)
    ap.add_argument("--attention_backend",default="sdpa",choices=["sdpa","vanilla"])

    # MoE
    ap.add_argument("--use_moe",action="store_true")
    ap.add_argument("--num_experts",type=int,default=3)
    ap.add_argument("--k",type=int,default=1)
    ap.add_argument("--router_hidden",type=int,default=128)
    ap.add_argument("--blw",type=float,default=0.02)
    ap.add_argument("--entropy_penalty",type=float,default=0.001)
    ap.add_argument("--routing_mode",choices=["specialize","uniform"],default="specialize")
    ap.add_argument("--router_temp",type=float,default=1.0)

    # Tokenizer/data/checkpoints
    ap.add_argument("--tok_kind",default="basic",choices=["basic","regex"])
    ap.add_argument("--vocab_size",type=int,default=1024)
    ap.add_argument("--tok_prefix",default="artifacts/tokenizers/tiny_bpe_energy")
    ap.add_argument("--train_corpus",default="")
    ap.add_argument("--save_path",default="artifacts/gpt_moe_bpe_energy.pt")
    ap.add_argument("--load_path",default="artifacts/gpt_moe_bpe_energy.pt")
    ap.add_argument("--seed",type=int,default=1337)

    # common sampling
    ap.add_argument("--sample_start",default="\n")
    ap.add_argument("--sample_tokens",type=int,default=200)
    ap.add_argument("--temperature",type=float,default=0.8)
    ap.add_argument("--top_k",type=int,default=200)

    # EBM specifics
    ap.add_argument("--ebm_loss",choices=["nce","sampled_softmax"],default="nce")
    ap.add_argument("--negatives",type=int,default=64, help="k_neg per (B,T)")
    ap.add_argument("--neg_kind",choices=["unigram","uniform"],default="unigram")
    ap.add_argument("--neg_temp",type=float,default=1.0)

    # TTA
    ap.add_argument("--tta_steps",type=int,default=2)
    ap.add_argument("--tta_lr",type=float,default=5e-4)
    ap.add_argument("--tta_layers",default="lm_head,ln_f")
    ap.add_argument("--tta_context",type=int,default=64)

    # Active inference
    ap.add_argument("--ai_particles",type=int,default=8)
    ap.add_argument("--ai_horizon",type=int,default=4)
    ap.add_argument("--ai_beta",type=float,default=0.2)

    # System-2
    ap.add_argument("--s2_branches",type=int,default=8)
    ap.add_argument("--s2_vote",default="majority",choices=["majority","best_logprob"])

    args=ap.parse_args(argv)
    random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
    device=args.device if args.device!="auto" else ("cuda" if torch.cuda.is_available() else "cpu")

    # Load text
    if args.train_corpus and os.path.exists(args.train_corpus):
        full=open(args.train_corpus,"r",encoding="utf-8").read()
        split=int(0.9*len(full)); train_txt, val_txt = full[:split], full[split:]
    else:
        train_txt, val_txt = load_tiny_shakespeare("./data")

    # Tokenizer
    tok, tok_model_path, vocab_size_actual = train_or_load_tokenizer(args.tok_kind,args.vocab_size,train_txt+val_txt,args.tok_prefix,verbose=True)

    # Encode
    train_ids = tok.encode(train_txt)
    val_ids   = tok.encode(val_txt)

    if len(train_ids) <= args.block_size + 1:
        new_bs=max(16, min(args.block_size, len(train_ids)-2))
        warn(f"Auto-adjusting block_size {args.block_size} -> {new_bs}", title="Safety")
        args.block_size=new_bs

    # Negative sampler from corpus statistics
    unigram_counts = build_unigram_counts(train_ids, vocab_size_actual)
    sampler = NegativeSampler(vocab_size_actual, counts=unigram_counts, kind=args.neg_kind, temperature=args.neg_temp)

    def build_model():
        cfg=GPTCfg(
            block_size=args.block_size, vocab_size=vocab_size_actual, n_layer=args.n_layer, n_head=args.n_head,
            n_embd=args.n_embd, dropout=args.dropout, bias=args.bias, attention_backend=args.attention_backend,
            use_moe=args.use_moe, num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden,
            blw=args.blw, entropy_penalty=args.entropy_penalty, routing_mode=args.routing_mode, router_temp=args.router_temp
        )
        return cfg, NanoGPT(cfg)

    if args.mode=="train_energy":
        cfg, model = build_model()
        train_energy(
            model,train_ids,val_ids,cfg.block_size,args.epochs,args.steps_per_epoch,args.batch_size,args.lr,device,
            title=f"Energy-Transformer ({args.ebm_loss}, k={args.negatives}, {'MoE' if cfg.use_moe else 'no-MoE'})",
            loss_kind=args.ebm_loss, sampler=sampler, k_neg=args.negatives
        )
        save_ckpt(args.save_path, {"kind":"gpt_energy","cfg":asdict(cfg),"state_dict":model.state_dict(),
                                   "tokenizer":{"kind":args.tok_kind,"model_path":tok_model_path,"vocab_size":vocab_size_actual},
                                   "ebm":{"loss":args.ebm_loss, "negatives":args.negatives, "neg_kind":args.neg_kind, "neg_temp":args.neg_temp}})

    elif args.mode=="infer":
        payload=load_ckpt(args.load_path); assert payload["kind"] in ("gpt_energy","gpt_bpe")
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        sample_lm(model,tok2,device,start=args.sample_start,tokens=args.sample_tokens,title="Energy Model Sample",temperature=args.temperature,top_k=args.top_k)

    elif args.mode=="test_time":
        payload=load_ckpt(args.load_path); assert payload["kind"] in ("gpt_energy","gpt_bpe")
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=test_time_adapt_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.tta_steps,args.tta_lr,
                                     args.tta_layers,args.tta_context,args.temperature,args.top_k)
        info(out, title="Test-Time Adaptation Output")

    elif args.mode=="active_inference":
        payload=load_ckpt(args.load_path); assert payload["kind"] in ("gpt_energy","gpt_bpe")
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=active_inference_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.ai_particles,
                                      args.ai_horizon,args.ai_beta,args.temperature,args.top_k)
        info(out, title="Active Inference Output")

    elif args.mode=="system2_reasoning":
        payload=load_ckpt(args.load_path); assert payload["kind"] in ("gpt_energy","gpt_bpe")
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=system2_reasoning_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.s2_branches,
                                       args.temperature,args.top_k,args.s2_vote)
        info(out, title="System-2 Reasoning Output")

# ================================ Colab/CLI ================================
if __name__ == "__main__":
    import sys
    argv = sys.argv[1:]
    if not argv:
        # Default for Colab/no-args: quick EBM demo train+infer on tiny Shakespeare
        os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
        # If a checkpoint exists, just infer; else run a tiny train then infer
        ckpt_path = "artifacts/gpt_moe_bpe_energy.pt"
        if os.path.exists(ckpt_path):
            argv = [
                "--mode","infer",
                "--load_path", ckpt_path,
                "--sample_start","Explain why the sky appears blue in two sentences.",
                "--sample_tokens","64",
            ]
        else:
            argv = [
                "--mode","train_energy",
                "--epochs","1","--steps_per_epoch","80",
                "--batch_size","64","--lr","3e-4",
                "--tok_kind","regex","--vocab_size","1024",
                "--save_path", ckpt_path,
            ]
    main(argv)


## End-to-end: BPE tokenizer + NanoGPT (+ optional MoE) + ENERGY-BASED training (NCE/InfoNCE, Sampled-Softmax) + TTA / Active Inference / System-2 + Adaptive Computation (easy/hard tokens) with confidence metrics & refine strategies.

In [ ]:
#!/usr/bin/env python3
# unified_gpt_moe_bpe_energy_adaptive.py
# End-to-end: BPE tokenizer + NanoGPT (+ optional MoE) + ENERGY-BASED training (NCE/InfoNCE, Sampled-Softmax)
# + TTA / Active Inference / System-2 + Adaptive Computation (easy/hard tokens) with confidence metrics & refine strategies.
#
# Upgrades vs original:
# - forward() now returns {"logits", "energies", "probs"} simultaneously (compat preserved for CE).
# - generate() now supports adaptive computation (stop metrics: maxprob/entropy/margin/energy_gap;
#   refine strategies: anneal/tta/active/branches).
# - All original components (RegexTokenizer, MoE, NCE/sampled softmax, ckpt, rich console, CLI) remain.

import os, math, json, argparse, random, time, urllib.request
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, Any, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------- pretty console (optional; fallback if missing) -----------
try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.table import Table
    from rich import box
    console = Console()
    def info(msg, **kw): console.print(Panel(msg, **({"border_style":"cyan","title":"Info","box":box.ROUNDED} | kw)))
    def warn(msg, **kw): console.print(Panel(msg, **({"border_style":"yellow","title":"Warning","box":box.ROUNDED} | kw)))
    def ok(msg, **kw):   console.print(Panel(msg, **({"border_style":"green","title":"OK","box":box.ROUNDED} | kw)))
except Exception:
    class _Dummy:
        def print(self, *a, **k): print(*a)
        def rule(self, *a, **k): print("="*60, *(a or ()), "="*60)
    console = _Dummy()
    def info(msg, **kw): print("[INFO]", msg)
    def warn(msg, **kw): print("[WARN]", msg)
    def ok(msg, **kw):   print("[ OK ]", msg)

# =============================================================================
#                               TOKENIZER (minBPE)
# =============================================================================
import unicodedata
import regex as re

def _replace_ctl(s: str) -> str:
    return "".join(ch if unicodedata.category(ch)[0] != "C" else f"\\u{ord(ch):04x}" for ch in s)

def _render_tok(t: bytes) -> str:
    return _replace_ctl(t.decode("utf-8", errors="replace"))

def _get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for p in zip(ids, ids[1:]): counts[p] = counts.get(p, 0) + 1
    return counts

def _merge(ids, pair, idx):
    newids=[]; i=0
    while i < len(ids):
        if ids[i]==pair[0] and i < len(ids)-1 and ids[i+1]==pair[1]:
            newids.append(idx); i+=2
        else:
            newids.append(ids[i]); i+=1
    return newids

class Tokenizer:
    def __init__(self):
        self.merges={}; self.pattern=""; self.special_tokens={}
        self.vocab={i:bytes([i]) for i in range(256)}
    def _rebuild_vocab(self):
        vocab={i:bytes([i]) for i in range(256)}
        for (a,b),i in self.merges.items(): vocab[i]=vocab[a]+vocab[b]
        for s,i in self.special_tokens.items(): vocab[i]=s.encode("utf-8")
        self.vocab=vocab
    def save(self, prefix):
        os.makedirs(os.path.dirname(prefix), exist_ok=True) if os.path.dirname(prefix) else None
        with open(prefix+".model","w",encoding="utf-8") as f:
            f.write("minbpe v1\n"); f.write(f"{self.pattern}\n"); f.write(f"{len(self.special_tokens)}\n")
            for s,i in self.special_tokens.items(): f.write(f"{s} {i}\n")
            for (a,b),_i in self.merges.items(): f.write(f"{a} {b}\n")
        with open(prefix+".vocab","w",encoding="utf-8") as f:
            inv={idx:pair for pair,idx in self.merges.items()}
            for i,tok in self.vocab.items():
                s=_render_tok(tok)
                if i in inv:
                    a,b=inv[i]; f.write(f"[{_render_tok(self.vocab[a])}][{_render_tok(self.vocab[b])}] -> [{s}] {i}\n")
                else:
                    f.write(f"[{s}] {i}\n")
    def load(self, model_file):
        merges={}; specials={}
        idx=256
        with open(model_file,"r",encoding="utf-8") as f:
            version=f.readline().strip(); assert version=="minbpe v1"
            self.pattern=f.readline().strip()
            ns=int(f.readline().strip())
            for _ in range(ns):
                s,si=f.readline().strip().split(); specials[s]=int(si)
            for line in f:
                a,b=map(int,line.split()); merges[(a,b)]=idx; idx+=1
        self.merges=merges; self.special_tokens=specials; self._rebuild_vocab()

class BasicTokenizer(Tokenizer):
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size>=256; num_merges=vocab_size-256
        ids=list(text.encode("utf-8")); merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats=_get_stats(ids); pair=max(stats,key=stats.get)
            idx=256+i; ids=_merge(ids,pair,idx); merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
            if verbose and i<10: info(f"merge {i+1}/{num_merges}: {pair} -> {idx}")
        self.merges=merges; self.vocab=vocab
    def encode(self, text):
        ids=list(text.encode("utf-8"))
        while len(ids)>=2:
            stats=_get_stats(ids); pair=min(stats,key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=_merge(ids,pair,self.merges[pair])
        return ids
    def decode(self, ids): return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

GPT4_SPLIT = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexTokenizer(Tokenizer):
    def __init__(self): super().__init__(); self.pattern=GPT4_SPLIT; self.compiled=re.compile(self.pattern)
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size>=256; num_merges=vocab_size-256
        chunks=re.findall(self.compiled,text); ids=[list(c.encode("utf-8")) for c in chunks]
        merges={}; vocab={i:bytes([i]) for i in range(256)}
        for i in range(num_merges):
            stats={}; [ _get_stats(ci,stats) for ci in ids ]
            pair=max(stats,key=stats.get); idx=256+i; ids=[_merge(ci,pair,idx) for ci in ids]
            merges[pair]=idx; vocab[idx]=vocab[pair[0]]+vocab[pair[1]]
            if verbose and i<10: info(f"merge {i+1}/{num_merges}: {pair} -> {idx}")
        self.merges=merges; self.vocab=vocab
    def _encode_chunk(self, bs):
        ids=list(bs)
        while len(ids)>=2:
            stats=_get_stats(ids); pair=min(stats,key=lambda p:self.merges.get(p,float("inf")))
            if pair not in self.merges: break
            ids=_merge(ids,pair,self.merges[pair])
        return ids
    def encode(self, text):
        ids=[]; [ids.extend(self._encode_chunk(c.encode("utf-8"))) for c in re.findall(self.compiled,text)]
        return ids
    def decode(self, ids): return BasicTokenizer.decode(self, ids)

def _new_tok(kind): return BasicTokenizer() if kind=="basic" else RegexTokenizer()

def train_or_load_tokenizer(kind,vocab_size,text,prefix,verbose=True):
    os.makedirs(os.path.dirname(prefix),exist_ok=True) if os.path.dirname(prefix) else None
    model=prefix+".model"
    if os.path.exists(model):
        tok=_new_tok(kind); tok.load(model); vs=max(tok.vocab.keys())+1
        if verbose: ok(f"Loaded tokenizer {model} (size={vs})", title="Tokenizer")
        return tok,model,vs
    tok=_new_tok(kind); tok.train(text,max(256,vocab_size),verbose); tok.save(prefix); vs=max(tok.vocab.keys())+1
    if verbose: ok(f"Saved tokenizer {prefix}.model (size={vs})", title="Tokenizer")
    return tok,model,vs

# =============================================================================
#                              DATA (tiny shakespeare)
# =============================================================================

def _repeat_to_len(s: str, target_len: int) -> str:
    if not s: s=" \n"
    return (s * ((target_len // len(s))+1))[:target_len]

def load_tiny_shakespeare(data_dir="./data", target_len=100_000):
    os.makedirs(data_dir,exist_ok=True)
    path=os.path.join(data_dir,"tinyshakespeare_input.txt")
    if not os.path.exists(path):
        try:
            url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url,path)
        except Exception:
            warn("Could not download dataset; writing fallback sample.")
            sample=("From fairest creatures we desire increase,\n"
                    "That thereby beauty's rose might never die,\n"
                    "But as the riper should by time decease,\n"
                    "His tender heir might bear his memory:\n")
            with open(path,"w",encoding="utf-8") as f: f.write(_repeat_to_len(sample, target_len))
    text=open(path,"r",encoding="utf-8").read()
    split=int(0.9*len(text)); return text[:split], text[split:]

# utility: minibatches of token ids

def get_batch_tokens(ids, block_size, batch_size, device):
    if len(ids) <= block_size + 1:
        raise RuntimeError(f"Sequence too short ({len(ids)}) for block_size={block_size}.")
    ix=torch.randint(len(ids)-block_size-1,(batch_size,))
    x=torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y=torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)

# =============================================================================
#                               MODEL (GPT + MoE)
# =============================================================================
class LayerNorm(nn.Module):
    def __init__(self,n,bias): super().__init__(); self.weight=nn.Parameter(torch.ones(n)); self.bias=nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self,x): return F.layer_norm(x,self.weight.shape,self.weight,self.bias,1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self,n_embd,n_head,block,dropout,bias,backend="sdpa"):
        super().__init__(); assert n_embd % n_head == 0
        self.n_head=n_head; self.n_embd=n_embd; self.dropout=dropout; self.backend=backend
        self.c_attn=nn.Linear(n_embd,3*n_embd,bias=bias); self.c_proj=nn.Linear(n_embd,n_embd,bias=bias)
        if backend=="vanilla" or not hasattr(F,"scaled_dot_product_attention"):
            self.register_buffer("bias", torch.tril(torch.ones(block,block)).view(1,1,block,block))
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.c_attn(x).split(self.n_embd,dim=2)
        q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        if self.backend=="sdpa" and hasattr(F,"scaled_dot_product_attention"):
            y=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=self.dropout if self.training else 0.0)
        else:
            att=(q@k.transpose(-2,-1))/math.sqrt(k.size(-1))
            att=att.masked_fill(self.bias[:,:,:T,:T]==0,float("-inf"))
            att=F.softmax(att,dim=-1); y=att@v
        y=y.transpose(1,2).contiguous().view(B,T,C); return self.c_proj(y)

def entropy_mean(probs,eps=1e-9): return -(probs.clamp_min(eps)*probs.clamp_min(eps).log()).sum(-1).mean()

class TopKRouter(nn.Module):
    def __init__(self,in_dim,hidden_dim,num_experts,k=1,temp=1.0):
        super().__init__(); self.net=nn.Sequential(nn.Linear(in_dim,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,num_experts))
        self.k=k; self.temp=float(temp); self.register_buffer("logits_bias", torch.zeros(num_experts))
    @staticmethod
    def _gumbel(shape, device):
        u=torch.rand(shape,device=device).clamp_(1e-9,1-1e-9); return -torch.log(-torch.log(u))
    def forward(self,x,add_gumbel=False):
        logits=self.net(x)/self.temp + self.logits_bias
        if add_gumbel and self.training: logits = logits + self._gumbel(logits.shape, logits.device)
        probs=F.softmax(logits,dim=-1)
        if self.k>=probs.size(-1): return probs, probs
        topk_vals,topk_idx=torch.topk(probs,self.k,dim=-1)
        mask=torch.zeros_like(probs); mask.scatter_(dim=-1,index=topk_idx,src=torch.ones_like(topk_vals))
        sp=probs*mask; sp=sp/(sp.sum(dim=-1,keepdim=True)+1e-9); return sp,probs

class MoEFFN(nn.Module):
    def __init__(self,in_dim,num_experts=3,k=1,router_hidden=128,dropout=0.1,blw=0.02,entropy_penalty=0.001,
                 routing_mode="specialize", router_temp=1.0):
        super().__init__(); hidden=4*in_dim
        self.expert_fc=nn.ModuleList([nn.Linear(in_dim,hidden) for _ in range(num_experts)])
        self.expert_proj=nn.ModuleList([nn.Linear(hidden,in_dim) for _ in range(num_experts)])
        self.shared_fc=nn.Linear(in_dim,hidden); self.shared_proj=nn.Linear(hidden,in_dim)
        self.router=TopKRouter(in_dim,router_hidden,num_experts,k,temp=router_temp)
        self.drop=nn.Dropout(dropout); self.num_experts=num_experts
        self.blw=blw; self.entw=entropy_penalty; self.routing_mode=routing_mode
    def forward(self,x):
        B,T,C=x.shape; xf=x.view(B*T,C); sp,dp=self.router(xf,add_gumbel=self.training)
        routed=0.0
        for e in range(self.num_experts):
            h=self.expert_proj[e](F.gelu(self.expert_fc[e](xf))); routed+=sp[:,e].unsqueeze(-1)*h
        shared=self.shared_proj(F.gelu(self.shared_fc(xf)))
        y=self.drop((shared+routed).view(B,T,C))
        aux=(-self.entw*entropy_mean(dp)) if self.routing_mode=="specialize" else torch.tensor(0.0,device=x.device)
        stats={"expert_selection_counts":(sp>0).float().sum(0),"mean_routing_probs":dp.mean(0)}
        return y,aux,stats

@dataclass
class GPTCfg:
    block_size:int=256; vocab_size:int=256; n_layer:int=6; n_head:int=6; n_embd:int=384
    dropout:float=0.1; bias:bool=True; attention_backend:str="sdpa"
    use_moe:bool=False; num_experts:int=3; k:int=1; router_hidden:int=128; blw:float=0.02
    entropy_penalty:float=0.001; routing_mode:str="specialize"; router_temp:float=1.0

class GPTBlock(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout, bias, backend="sdpa", moe: Optional[MoEFFN]=None):
        super().__init__(); self.ln1=LayerNorm(n_embd,bias); self.attn=CausalSelfAttention(n_embd,n_head,block,dropout,bias,backend)
        self.ln2=LayerNorm(n_embd,bias); self.moe=moe
        if moe is None:
            hidden=4*n_embd
            self.ffn=nn.Sequential(nn.Linear(n_embd,hidden,bias=bias), nn.GELU(), nn.Linear(hidden,n_embd,bias=bias), nn.Dropout(dropout))
    def forward(self,x):
        x=x+self.attn(self.ln1(x))
        if self.moe is None:
            x=x+self.ffn(self.ln2(x)); aux=torch.tensor(0.0,device=x.device); stats={}
        else:
            y,aux,stats=self.moe(self.ln2(x)); x=x+y
        return x, aux, stats

class NanoGPT(nn.Module):
    def __init__(self, cfg:GPTCfg):
        super().__init__(); self.cfg=cfg
        self.wte=nn.Embedding(cfg.vocab_size,cfg.n_embd); self.wpe=nn.Embedding(cfg.block_size,cfg.n_embd)
        blocks=[]
        for _ in range(cfg.n_layer):
            moe=None
            if cfg.use_moe: moe=MoEFFN(cfg.n_embd,cfg.num_experts,cfg.k,cfg.router_hidden,cfg.dropout,cfg.blw,cfg.entropy_penalty,cfg.routing_mode,cfg.router_temp)
            blocks.append(GPTBlock(cfg.n_embd,cfg.n_head,cfg.block_size,cfg.dropout,cfg.bias,cfg.attention_backend,moe))
        self.blocks=nn.ModuleList(blocks); self.ln_f=LayerNorm(cfg.n_embd,cfg.bias); self.lm_head=nn.Linear(cfg.n_embd,cfg.vocab_size,bias=False)
        self.lm_head.weight=self.wte.weight
        self.apply(self._init)
    def _init(self,m):
        if isinstance(m,nn.Linear): nn.init.normal_(m.weight,0.0,0.02)
        if isinstance(m,nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m,nn.Embedding): nn.init.normal_(m.weight,0.0,0.02)
    def forward_hidden(self, idx):
        B,T=idx.shape; assert T<=self.cfg.block_size
        pos=torch.arange(0,T,device=idx.device).long(); x=self.wte(idx)+self.wpe(pos)[None,:,:]
        aux_total=torch.tensor(0.0,device=idx.device); last={}
        for blk in self.blocks:
            x,aux,stats=blk(x); aux_total=aux_total+aux; last=stats
        x=self.ln_f(x)
        return x, aux_total, last

    # >>> UPGRADED: return logits, energies, probs (dict)
    def forward(self, idx, targets=None):
        x, aux_total, last = self.forward_hidden(idx)
        logits = self.lm_head(x)  # unnormalized scores (higher = better)
        outputs = {
            "logits": logits,
            "energies": -logits,                # canonical EBM energy (lower = better)
            "probs": F.softmax(logits, dim=-1)  # normalized for convenience
        }
        loss=None
        if targets is not None:
            # CE kept for compatibility (TTA, quick scoring); EBM training uses external losses
            loss=F.cross_entropy(logits.view(-1,logits.size(-1)), targets.view(-1)) + aux_total
        return outputs, loss, last

    # >>> UPGRADED: adaptive computation in generate()
    @torch.no_grad()
    def generate(self,
                 idx,
                 max_new_tokens,
                 temperature=1.0,
                 top_k=None,
                 # adaptive compute
                 adapt_conf: bool = False,
                 stop_metric: str = "maxprob",   # 'maxprob' | 'entropy' | 'margin' | 'energy_gap'
                 conf_threshold: float = 0.8,
                 max_refine: int = 3,
                 refine_strategy: str = "anneal",# 'anneal' | 'tta' | 'active' | 'branches'
                 # refine hyperparams
                 tta_steps: int = 2,
                 tta_lr: float = 5e-4,
                 tta_layers: str = "lm_head,ln_f",
                 ai_particles: int = 6,
                 ai_horizon: int = 3,
                 ai_beta: float = 0.2,
                 branch_count: int = 6):
        """
        Adaptive autoregressive generation (variable per-token compute).
        Returns (B, T + max_new_tokens) token ids.
        """
        device = idx.device

        def _apply_topk(scores):
            if top_k is not None:
                v,_=torch.topk(scores, min(top_k, scores.size(-1)))
                scores[scores < v[:,[-1]]] = -float("Inf")
            return scores

        def _confidence(probs: torch.Tensor, energies: torch.Tensor) -> torch.Tensor:
            # returns (B,) score where higher => more confident/easy
            if stop_metric == "maxprob":
                return probs.max(dim=-1).values
            elif stop_metric == "entropy":
                logp = (probs+1e-9).log()
                ent = -(probs * logp).sum(dim=-1)  # lower entropy => more confident
                return -ent
            elif stop_metric == "margin":
                top2 = torch.topk(probs, k=2, dim=-1).values
                return top2[:,0] - top2[:,1]
            elif stop_metric == "energy_gap":
                # difference between best and second-best in -E (i.e., logits)
                top2 = torch.topk(-energies, k=2, dim=-1).values
                return top2[:,0] - top2[:,1]
            else:
                return probs.max(dim=-1).values

        def _refine_anneal(scores: torch.Tensor, step: int) -> torch.Tensor:
            factor = 0.9 ** step
            return scores / max(1e-6, (temperature * factor))

        def _refine_tta(single_ids: torch.Tensor):
            # Few CE steps on selected lightweight params, then recompute last-step logits
            self.train()
            for p in self.parameters(): p.requires_grad=False
            adapt_names = {n.strip() for n in tta_layers.split(",")} if tta_layers else set()
            to_adapt=[]
            if "lm_head" in adapt_names:
                for p in self.lm_head.parameters(): p.requires_grad=True; to_adapt.append(p)
            if "ln_f" in adapt_names:
                for p in self.ln_f.parameters(): p.requires_grad=True; to_adapt.append(p)
            if "blocks[-1]" in adapt_names and hasattr(self,"blocks"):
                for p in self.blocks[-1].parameters(): p.requires_grad=True; to_adapt.append(p)
            if to_adapt:
                opt=torch.optim.Adam([p for p in to_adapt if p.requires_grad], lr=tta_lr)
                ctx = single_ids[:, -min(self.cfg.block_size, single_ids.size(1)):]
                if ctx.size(1) >= 2:
                    inp, tgt = ctx[:, :-1], ctx[:, 1:]
                    for _ in range(tta_steps):
                        opt.zero_grad(set_to_none=True)
                        out, _, _ = self(inp, None)
                        logits = out["logits"]
                        ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
                        ce.backward()
                        torch.nn.utils.clip_grad_norm_(to_adapt, 1.0)
                        opt.step()
            self.eval()
            cond = single_ids if single_ids.size(1) <= self.cfg.block_size else single_ids[:, -self.cfg.block_size:]
            out, _, _ = self(cond, None)
            return out["logits"][:, -1, :]

        def _refine_active(base_ids: torch.Tensor):
            # choose next token via short rollouts minimizing expected free energy proxy
            cond = base_ids if base_ids.size(1) <= self.cfg.block_size else base_ids[:, -self.cfg.block_size:]
            o, _, _ = self(cond, None)
            base_lp = F.log_softmax(o["logits"][:, -1, :], dim=-1)
            probs = base_lp.exp()
            cand = torch.multinomial(probs, num_samples=ai_particles)
            best_F = torch.full((cand.size(0),), float("inf"), device=base_ids.device)
            best_tok = torch.zeros((cand.size(0), 1), dtype=torch.long, device=base_ids.device)
            for j in range(ai_particles):
                a = cand[:, j:j+1]
                ids = torch.cat([cond, a], dim=1)
                logp_sum = torch.zeros((ids.size(0),), device=ids.device)
                ent_sum  = torch.zeros((ids.size(0),), device=ids.device)
                for _ in range(ai_horizon):
                    c = ids if ids.size(1) <= self.cfg.block_size else ids[:, -self.cfg.block_size:]
                    o2, _, _ = self(c, None)
                    l = o2["logits"][:, -1, :] / max(1e-6, temperature)
                    if top_k is not None:
                        v,_ = torch.topk(l, min(top_k, l.size(-1)))
                        l[l < v[:,[-1]]] = -float("Inf")
                    lp = F.log_softmax(l, dim=-1)
                    p  = lp.exp()
                    nxt = torch.multinomial(p, 1)
                    logp_sum += lp.gather(-1, nxt).squeeze(-1)
                    ent_sum  += (-(p * lp).sum(dim=-1))
                    ids = torch.cat([ids, nxt], dim=1)
                Fscore = -logp_sum + ai_beta * ent_sum
                mask = Fscore < best_F
                best_F = torch.where(mask, Fscore, best_F)
                best_tok = torch.where(mask.unsqueeze(-1), a, best_tok)
            # Create a sparse score preferring best_tok
            new_scores = torch.full_like(o["logits"][:, -1, :], -float("Inf"))
            new_scores.scatter_(1, best_tok, 0.0)
            return new_scores

        def _refine_branches(base_ids: torch.Tensor):
            # sample multiple one-token branches; pick by short continuation avg logprob
            cond = base_ids if base_ids.size(1) <= self.cfg.block_size else base_ids[:, -self.cfg.block_size:]
            o, _, _ = self(cond, None)
            l0 = o["logits"][:, -1, :]
            p0 = F.softmax(_apply_topk(l0 / max(1e-6, temperature)), dim=-1)
            cand = torch.multinomial(p0, num_samples=branch_count)  # (B,K)
            scores_like = torch.full_like(l0, -float("Inf"))
            steps = max(1, ai_horizon // 2)
            for j in range(branch_count):
                a = cand[:, j:j+1]
                ids = torch.cat([cond, a], dim=1)
                logp_sum = torch.zeros((ids.size(0),), device=ids.device)
                for _ in range(steps):
                    c = ids if ids.size(1) <= self.cfg.block_size else ids[:, -self.cfg.block_size:]
                    o2, _, _ = self(c, None)
                    l = o2["logits"][:, -1, :] / max(1e-6, temperature)
                    if top_k is not None:
                        v,_ = torch.topk(l, min(top_k, l.size(-1)))
                        l[l < v[:,[-1]]] = -float("Inf")
                    lp = F.log_softmax(l, dim=-1)
                    p  = lp.exp()
                    nxt = torch.multinomial(p, 1)
                    logp_sum += lp.gather(-1, nxt).squeeze(-1)
                    ids = torch.cat([ids, nxt], dim=1)
                scores_like.scatter_(1, a, logp_sum.unsqueeze(-1))
            return scores_like

        for _ in range(max_new_tokens):
            cond=idx if idx.size(1)<=self.cfg.block_size else idx[:,-self.cfg.block_size:]
            out,_,_=self(cond, None)
            logits_last = out["logits"][:,-1,:]/max(1e-6,temperature)
            logits_last = _apply_topk(logits_last)
            probs_last  = F.softmax(logits_last, dim=-1)
            energies_last = -logits_last

            if adapt_conf:
                conf = _confidence(probs_last, energies_last)  # (B,)
                need_refine = conf < conf_threshold
            else:
                need_refine = torch.zeros((idx.size(0),), dtype=torch.bool, device=device)

            refined_logits = logits_last.clone()
            if need_refine.any():
                # Batch-wise refinement for simplicity/perf; per-sample loop is also feasible if needed
                if refine_strategy == "anneal":
                    for step in range(1, max_refine+1):
                        refined_logits = _apply_topk(_refine_anneal(refined_logits, step))
                        p = F.softmax(refined_logits, dim=-1)
                        conf = _confidence(p, -refined_logits)
                        if (conf >= conf_threshold).all(): break
                elif refine_strategy == "tta":
                    refined_logits = _apply_topk(_refine_tta(idx))
                elif refine_strategy == "active":
                    refined_logits = _apply_topk(_refine_active(idx))
                elif refine_strategy == "branches":
                    refined_logits = _apply_topk(_refine_branches(idx))

            final_probs = F.softmax(refined_logits, dim=-1)
            idx_next=torch.multinomial(final_probs,1)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx

def count_parameters(m: nn.Module) -> Tuple[int,int]:
    total=sum(p.numel() for p in m.parameters())
    train=sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, train

# =============================================================================
#                      ENERGY-BASED LOSSES (NCE / Sampled-Softmax)
# =============================================================================

class NegativeSampler:
    def __init__(self, vocab_size:int, counts: Optional[List[int]]=None, kind:str="unigram", temperature:float=1.0):
        self.vocab_size=vocab_size; self.kind=kind; self.temperature=max(1e-6, float(temperature))
        if counts is None or sum(counts)==0 or kind=="uniform":
            probs=torch.ones(vocab_size)
        else:
            probs=torch.tensor(counts, dtype=torch.float)
        probs=probs.pow(1.0/self.temperature)
        probs=probs / probs.sum()
        self.register = probs
        self.device="cpu"
    def to(self, device:str):
        self.device=device; self.register=self.register.to(device); return self
    @torch.no_grad()
    def sample(self, num_samples:int) -> torch.Tensor:
        # Returns shape (num_samples,)
        return torch.multinomial(self.register, num_samples=num_samples, replacement=True)
    def prob(self, tokens: torch.Tensor) -> torch.Tensor:
        return self.register[tokens]

def build_unigram_counts(encoded_ids: List[int], vocab_size: int) -> List[int]:
    counts=[0]*vocab_size
    for t in encoded_ids:
        if 0 <= t < vocab_size: counts[t]+=1
    return [c if c>0 else 1 for c in counts]

def nce_loss_sampled(logits: torch.Tensor, targets: torch.Tensor, sampler: NegativeSampler, k_neg: int, reduction:str="mean"):
    """
    Noise-Contrastive Estimation for next-token energies (vector EBMs over vocab).
    logits: (B,T,V) unnormalized scores (higher is better, energy = -score)
    """
    B,T,V = logits.shape
    device = logits.device
    pos_scores = logits.gather(-1, targets.unsqueeze(-1)).squeeze(-1)  # (B,T)
    num_positions = B*T
    neg_flat = sampler.sample(num_positions * k_neg).to(device)       # (B*T*k)
    neg = neg_flat.view(B, T, k_neg)
    neg_scores = logits.gather(-1, neg)                                # (B,T,k)
    with torch.no_grad():
        q_pos = sampler.prob(targets.view(-1)).view(B,T) + 1e-9
        q_neg = sampler.prob(neg_flat).view(B,T,k_neg) + 1e-9
    s_pos = pos_scores - torch.log(k_neg * q_pos)
    s_neg = neg_scores - torch.log(k_neg * q_neg)
    loss_pos = F.softplus(-s_pos)
    loss_neg = F.softplus(s_neg).sum(dim=-1)
    loss = loss_pos + loss_neg
    if reduction=="mean": return loss.mean()
    if reduction=="sum":  return loss.sum()
    return loss

def sampled_softmax_loss(logits: torch.Tensor, targets: torch.Tensor, sampler: NegativeSampler, k_neg:int, reduction:str="mean"):
    """
    Sampled Softmax / InfoNCE surrogate: cross-entropy over {pos + k_neg negatives}.
    """
    B,T,V = logits.shape
    device = logits.device
    num_positions=B*T
    neg_flat = sampler.sample(num_positions * k_neg).to(device)
    neg = neg_flat.view(B,T,k_neg)
    pos_scores = logits.gather(-1, targets.unsqueeze(-1))  # (B,T,1)
    neg_scores = logits.gather(-1, neg)                    # (B,T,k)
    cat_scores = torch.cat([pos_scores, neg_scores], dim=-1)  # (B,T,1+k)
    labels = torch.zeros(B,T, dtype=torch.long, device=device)
    return F.cross_entropy(cat_scores.view(-1, 1+k_neg), labels.view(-1), reduction=reduction)

# =============================================================================
#                            TRAIN / SAMPLE / SAVELOAD
# =============================================================================

def train_energy(
    model,
    train_ids,
    val_ids,
    block_size,
    epochs,
    steps,
    batch,
    lr,
    device,
    title,
    loss_kind:str,
    sampler: NegativeSampler,
    k_neg:int,
    clip:float=1.0,
    val_subset_steps:int=50,
):
    model=model.to(device); sampler.to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,betas=(0.9,0.95),weight_decay=0.1)

    def _eval_once():
        model.eval();
        with torch.no_grad():
            tot=0.0; n=0
            for _ in range(val_subset_steps):
                xb,yb=get_batch_tokens(val_ids,block_size,batch,device)
                outputs,_,_ = model(xb, None)
                logits = outputs["logits"]
                if loss_kind=="nce":
                    loss = nce_loss_sampled(logits, yb, sampler, k_neg)
                else:
                    loss = sampled_softmax_loss(logits, yb, sampler, k_neg)
                tot += loss.item(); n += 1
            return tot/max(1,n)

    for ep in range(1,epochs+1):
        t0=time.time(); model.train(); losses=[]
        for _ in range(steps):
            xb,yb=get_batch_tokens(train_ids,block_size,batch,device)
            outputs,_,_ = model(xb, None)  # unnormalized scores
            logits = outputs["logits"]
            if loss_kind=="nce":
                loss = nce_loss_sampled(logits, yb, sampler, k_neg)
            else:
                loss = sampled_softmax_loss(logits, yb, sampler, k_neg)
            opt.zero_grad(); loss.backward();
            if clip>0: torch.nn.utils.clip_grad_norm_(model.parameters(),clip)
            opt.step(); losses.append(loss.item())
        dt=time.time()-t0
        vloss=_eval_once()
        info(f"Epoch {ep}/{epochs}  train={sum(losses)/len(losses):.3f}  val={vloss:.3f}  time={dt:.1f}s", title=title)

@torch.no_grad()
def sample_lm(model,tok,device,start="\n",tokens=200,title="Sample",temperature=0.8,top_k=200,
              adapt_conf=False, stop_metric="maxprob", conf_threshold=0.8, max_refine=3,
              refine_strategy="anneal", tta_steps=2, tta_lr=5e-4, tta_layers="lm_head,ln_f",
              ai_particles=8, ai_horizon=4, ai_beta=0.2, branch_count=6):
    model.eval().to(device)
    x=torch.tensor([tok.encode(start)],dtype=torch.long,device=device)
    y=model.generate(
        x, max_new_tokens=tokens, temperature=temperature, top_k=top_k,
        adapt_conf=adapt_conf, stop_metric=stop_metric, conf_threshold=conf_threshold,
        max_refine=max_refine, refine_strategy=refine_strategy,
        tta_steps=tta_steps, tta_lr=tta_lr, tta_layers=tta_layers,
        ai_particles=ai_particles, ai_horizon=ai_horizon, ai_beta=ai_beta,
        branch_count=branch_count
    )[0].tolist()
    text=tok.decode(y); info(text, title=title)

def save_ckpt(path,payload):
    os.makedirs(os.path.dirname(path),exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload,path); ok(f"Saved to {path}")

def load_ckpt(path):
    p=torch.load(path,map_location="cpu"); ok(f"Loaded {path}"); return p

# =============================================================================
#                          INFERENCE STRATEGIES (unchanged APIs)
# =============================================================================
@torch.no_grad()
def _last_token_logprobs(model, input_ids: torch.Tensor):
    outputs, _, _ = model(input_ids)
    logits = outputs["logits"]
    return F.log_softmax(logits[:, -1, :], dim=-1)

def test_time_adapt_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    adapt_steps: int = 2, adapt_lr: float = 5e-4, adapt_layers: str = "lm_head,ln_f",
    context_reuse: int = 64, temperature: float = 0.8, top_k: int = 200,
):
    """Few gradient steps on a small parameter subset over recent context before each token."""
    model = model.to(device)
    model.eval()
    adapt_names = {n.strip() for n in adapt_layers.split(",")} if adapt_layers else set()
    for p in model.parameters(): p.requires_grad=False
    to_adapt=[]
    if "lm_head" in adapt_names:
        for p in model.lm_head.parameters(): p.requires_grad=True; to_adapt.append(p)
    if "ln_f" in adapt_names:
        for p in model.ln_f.parameters(): p.requires_grad=True; to_adapt.append(p)
    if "blocks[-1]" in adapt_names and hasattr(model,"blocks"):
        for p in model.blocks[-1].parameters(): p.requires_grad=True; to_adapt.append(p)
    opt = torch.optim.Adam([p for p in to_adapt if p.requires_grad], lr=adapt_lr) if to_adapt else None

    ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
    for _ in range(max_new_tokens):
        if opt is not None and ids.size(1)>1:
            model.train()
            for _step in range(adapt_steps):
                ctx=ids[:,-min(context_reuse, ids.size(1)-1):]
                inp, tgt = ctx[:,:-1], ctx[:,1:]
                opt.zero_grad(set_to_none=True)
                outputs, _, _ = model(inp, None)
                logits = outputs["logits"]
                ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
                ce.backward();
                torch.nn.utils.clip_grad_norm_(to_adapt, 1.0)
                opt.step()
        model.eval()
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        outputs,_,_=model(cond); logits=outputs["logits"][:,-1,:]/temperature
        if top_k is not None:
            v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
        probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1)
        ids=torch.cat([ids,nxt],dim=1)
    return tok.decode(ids[0].tolist())

def _rollout_logprob(model, start_ids: torch.Tensor, horizon: int, temperature: float, top_k: Optional[int]):
    model.eval(); ids=start_ids.clone()
    total_logprob=0.0; total_entropy=0.0
    for _ in range(horizon):
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        outputs,_,_=model(cond); logits=outputs["logits"][:,-1,:]/temperature
        if top_k is not None:
            v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
        logprobs=F.log_softmax(logits,dim=-1); probs=logprobs.exp()
        entropy=-(probs*logprobs).sum(dim=-1).mean()
        nxt=torch.multinomial(probs,1)
        total_logprob+=logprobs.gather(-1,nxt).mean().item()
        total_entropy+=entropy.item()
        ids=torch.cat([ids,nxt],dim=1)
    return total_logprob, total_entropy

@torch.no_grad()
def active_inference_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    particles: int = 8, horizon: int = 4, beta: float = 0.2, temperature: float = 0.9, top_k: Optional[int] = 200,
):
    """Minimize expected free energy proxy = -Σ log p + β·Σ entropy via short rollouts per candidate action."""
    model=model.to(device).eval()
    ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
    for _ in range(max_new_tokens):
        cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
        base_lp=_last_token_logprobs(model,cond)  # (1,V)
        probs=base_lp.exp()
        cand=torch.multinomial(probs,num_samples=particles)  # (1,P)
        best_F=float("inf"); best_token=None
        for j in range(particles):
            a=cand[:,j:j+1]
            start=torch.cat([cond,a],dim=1)
            logp_sum, ent_sum=_rollout_logprob(model,start,horizon,temperature,top_k)
            F=-logp_sum + beta*ent_sum
            if F<best_F: best_F=F; best_token=a
        if best_token is None:
            best_token=torch.argmax(base_lp,dim=-1,keepdim=True)
        ids=torch.cat([ids,best_token],dim=1)
    return tok.decode(ids[0].tolist())

@torch.no_grad()
def system2_reasoning_generate(
    model: nn.Module, tok, device: str, prompt: str, max_new_tokens: int = 128,
    branches: int = 8, temperature: float = 0.9, top_k: Optional[int] = 200, vote: str = "majority",
):
    """Self-consistency: sample multiple full continuations and pick majority or best average logprob."""
    model=model.to(device).eval()
    def _sample_once():
        ids=torch.tensor([tok.encode(prompt)],dtype=torch.long,device=device)
        for _ in range(max_new_tokens):
            cond=ids if ids.size(1)<=model.cfg.block_size else ids[:,-model.cfg.block_size:]
            outputs,_,_=model(cond); logits=outputs["logits"][:,-1,:]/temperature
            if top_k is not None:
                v,_=torch.topk(logits, min(top_k, logits.size(-1))); logits[logits<v[:,[-1]]]=-float("Inf")
            probs=F.softmax(logits,dim=-1); nxt=torch.multinomial(probs,1)
            ids=torch.cat([ids,nxt],dim=1)
        # score by avg logprob
        if ids.size(1)>1:
            inp, tgt = ids[:,:-1], ids[:,1:]
            outputs, _, _ = model(inp, None)
            logp = F.log_softmax(outputs["logits"], dim=-1)
            avg_lp = logp.gather(-1, tgt.unsqueeze(-1)).mean().item()
        else:
            avg_lp = 0.0
        return tok.decode(ids[0].tolist()), avg_lp

    samples=[_sample_once() for _ in range(branches)]
    if vote=="best_logprob":
        return max(samples, key=lambda x:x[1])[0]
    from collections import Counter
    cleaned=[s[0].strip() for s in samples]
    return Counter(cleaned).most_common(1)[0][0]

# =============================================================================
#                                     MAIN
# =============================================================================

def main(argv=None):
    ap=argparse.ArgumentParser(description="Energy-Based Transformer: BPE + MoE + NCE/SampledSoftmax + TTA/ActiveInference/System2 + Adaptive Compute")
    ap.add_argument("--mode",required=True,choices=["train_energy","infer","test_time","active_inference","system2_reasoning"])
    ap.add_argument("--device",default="auto",choices=["auto","cpu","cuda"])
    ap.add_argument("--epochs",type=int,default=2)
    ap.add_argument("--steps_per_epoch",type=int,default=200)
    ap.add_argument("--batch_size",type=int,default=64)
    ap.add_argument("--lr",type=float,default=3e-4)
    ap.add_argument("--block_size",type=int,default=256)
    ap.add_argument("--dropout",type=float,default=0.1)
    ap.add_argument("--n_layer",type=int,default=6)
    ap.add_argument("--n_head",type=int,default=6)
    ap.add_argument("--n_embd",type=int,default=384)
    ap.add_argument("--bias",action="store_true",default=True)
    ap.add_argument("--attention_backend",default="sdpa",choices=["sdpa","vanilla"])

    # MoE
    ap.add_argument("--use_moe",action="store_true")
    ap.add_argument("--num_experts",type=int,default=3)
    ap.add_argument("--k",type=int,default=1)
    ap.add_argument("--router_hidden",type=int,default=128)
    ap.add_argument("--blw",type=float,default=0.02)
    ap.add_argument("--entropy_penalty",type=float,default=0.001)
    ap.add_argument("--routing_mode",choices=["specialize","uniform"],default="specialize")
    ap.add_argument("--router_temp",type=float,default=1.0)

    # Tokenizer/data/checkpoints
    ap.add_argument("--tok_kind",default="basic",choices=["basic","regex"])
    ap.add_argument("--vocab_size",type=int,default=1024)
    ap.add_argument("--tok_prefix",default="artifacts/tokenizers/tiny_bpe_energy")
    ap.add_argument("--train_corpus",default="")
    ap.add_argument("--save_path",default="artifacts/gpt_moe_bpe_energy.pt")
    ap.add_argument("--load_path",default="artifacts/gpt_moe_bpe_energy.pt")
    ap.add_argument("--seed",type=int,default=1337)

    # common sampling
    ap.add_argument("--sample_start",default="\n")
    ap.add_argument("--sample_tokens",type=int,default=200)
    ap.add_argument("--temperature",type=float,default=0.8)
    ap.add_argument("--top_k",type=int,default=200)

    # EBM specifics
    ap.add_argument("--ebm_loss",choices=["nce","sampled_softmax"],default="nce")
    ap.add_argument("--negatives",type=int,default=64, help="k_neg per (B,T)")
    ap.add_argument("--neg_kind",choices=["unigram","uniform"],default="unigram")
    ap.add_argument("--neg_temp",type=float,default=1.0)

    # TTA
    ap.add_argument("--tta_steps",type=int,default=2)
    ap.add_argument("--tta_lr",type=float,default=5e-4)
    ap.add_argument("--tta_layers",default="lm_head,ln_f")
    ap.add_argument("--tta_context",type=int,default=64)

    # Active inference
    ap.add_argument("--ai_particles",type=int,default=8)
    ap.add_argument("--ai_horizon",type=int,default=4)
    ap.add_argument("--ai_beta",type=float,default=0.2)

    # System-2
    ap.add_argument("--s2_branches",type=int,default=8)
    ap.add_argument("--s2_vote",default="majority",choices=["majority","best_logprob"])

    # Adaptive computation flags
    ap.add_argument("--adapt_conf", action="store_true", help="Enable adaptive per-token compute")
    ap.add_argument("--stop_metric", default="maxprob", choices=["maxprob","entropy","margin","energy_gap"])
    ap.add_argument("--conf_threshold", type=float, default=0.8)
    ap.add_argument("--max_refine", type=int, default=3)
    ap.add_argument("--refine_strategy", default="anneal", choices=["anneal","tta","active","branches"])
    ap.add_argument("--branch_count", type=int, default=6)

    args=ap.parse_args(argv)
    random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
    device=args.device if args.device!="auto" else ("cuda" if torch.cuda.is_available() else "cpu")

    # Load text
    if args.train_corpus and os.path.exists(args.train_corpus):
        full=open(args.train_corpus,"r",encoding="utf-8").read()
        split=int(0.9*len(full)); train_txt, val_txt = full[:split], full[split:]
    else:
        train_txt, val_txt = load_tiny_shakespeare("./data")

    # Tokenizer
    tok, tok_model_path, vocab_size_actual = train_or_load_tokenizer(args.tok_kind,args.vocab_size,train_txt+val_txt,args.tok_prefix,verbose=True)

    # Encode
    train_ids = tok.encode(train_txt)
    val_ids   = tok.encode(val_txt)

    if len(train_ids) <= args.block_size + 1:
        new_bs=max(16, min(args.block_size, len(train_ids)-2))
        warn(f"Auto-adjusting block_size {args.block_size} -> {new_bs}", title="Safety")
        args.block_size=new_bs

    # Negative sampler from corpus statistics
    unigram_counts = build_unigram_counts(train_ids, vocab_size_actual)
    sampler = NegativeSampler(vocab_size_actual, counts=unigram_counts, kind=args.neg_kind, temperature=args.neg_temp)

    def build_model():
        cfg=GPTCfg(
            block_size=args.block_size, vocab_size=vocab_size_actual, n_layer=args.n_layer, n_head=args.n_head,
            n_embd=args.n_embd, dropout=args.dropout, bias=args.bias, attention_backend=args.attention_backend,
            use_moe=args.use_moe, num_experts=args.num_experts, k=args.k, router_hidden=args.router_hidden,
            blw=args.blw, entropy_penalty=args.entropy_penalty, routing_mode=args.routing_mode, router_temp=args.router_temp
        )
        return cfg, NanoGPT(cfg)

    if args.mode=="train_energy":
        cfg, model = build_model()
        total_params, trainable_params = count_parameters(model)
        info(f"Params: total={total_params:,} trainable={trainable_params:,}", title="Model")
        train_energy(
            model,train_ids,val_ids,cfg.block_size,args.epochs,args.steps_per_epoch,args.batch_size,args.lr,device,
            title=f"Energy-Transformer ({args.ebm_loss}, k={args.negatives}, {'MoE' if cfg.use_moe else 'no-MoE'})",
            loss_kind=args.ebm_loss, sampler=sampler, k_neg=args.negatives
        )
        save_ckpt(args.save_path, {"kind":"gpt_energy","cfg":asdict(cfg),"state_dict":model.state_dict(),
                                   "tokenizer":{"kind":args.tok_kind,"model_path":tok_model_path,"vocab_size":vocab_size_actual},
                                   "ebm":{"loss":args.ebm_loss, "negatives":args.negatives, "neg_kind":args.neg_kind, "neg_temp":args.neg_temp}})

    elif args.mode=="infer":
        payload=load_ckpt(args.load_path); assert payload["kind"] in ("gpt_energy","gpt_bpe")
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        sample_lm(
            model,tok2,device,start=args.sample_start,tokens=args.sample_tokens,
            title="Energy Model Sample",temperature=args.temperature,top_k=args.top_k,
            adapt_conf=args.adapt_conf, stop_metric=args.stop_metric, conf_threshold=args.conf_threshold,
            max_refine=args.max_refine, refine_strategy=args.refine_strategy,
            tta_steps=args.tta_steps, tta_lr=args.tta_lr, tta_layers=args.tta_layers,
            ai_particles=args.ai_particles, ai_horizon=args.ai_horizon, ai_beta=args.ai_beta,
            branch_count=args.branch_count
        )

    elif args.mode=="test_time":
        payload=load_ckpt(args.load_path); assert payload["kind"] in ("gpt_energy","gpt_bpe")
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=test_time_adapt_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.tta_steps,args.tta_lr,
                                     args.tta_layers,args.tta_context,args.temperature,args.top_k)
        info(out, title="Test-Time Adaptation Output")

    elif args.mode=="active_inference":
        payload=load_ckpt(args.load_path); assert payload["kind"] in ("gpt_energy","gpt_bpe")
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=active_inference_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.ai_particles,
                                      args.ai_horizon,args.ai_beta,args.temperature,args.top_k)
        info(out, title="Active Inference Output")

    elif args.mode=="system2_reasoning":
        payload=load_ckpt(args.load_path); assert payload["kind"] in ("gpt_energy","gpt_bpe")
        tinfo=payload["tokenizer"]; tok2=_new_tok(tinfo["kind"]); tok2.load(tinfo["model_path"])
        cfg=GPTCfg(**payload["cfg"]); model=NanoGPT(cfg); model.load_state_dict(payload["state_dict"])
        out=system2_reasoning_generate(model,tok2,device,args.sample_start,args.sample_tokens,args.s2_branches,
                                       args.temperature,args.top_k,args.s2_vote)
        info(out, title="System-2 Reasoning Output")

# ================================ Colab/CLI ================================
if __name__ == "__main__":
    import sys
    argv = sys.argv[1:]
    if not argv:
        # Default for Colab/no-args: quick EBM demo train+infer on tiny Shakespeare
        os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
        ckpt_path = "artifacts/gpt_moe_bpe_energy.pt"
        if os.path.exists(ckpt_path):
            argv = [
                "--mode","infer",
                "--load_path", ckpt_path,
                "--sample_start","Explain why the sky appears blue in two sentences.",
                "--sample_tokens","64",
                "--adapt_conf",                   # turn on adaptive compute for fun
                "--stop_metric","margin",
                "--refine_strategy","anneal",
                "--conf_threshold","0.85",
                "--max_refine","2",
            ]
        else:
            argv = [
                "--mode","train_energy",
                "--epochs","1","--steps_per_epoch","80",
                "--batch_size","64","--lr","3e-4",
                "--tok_kind","regex","--vocab_size","1024",
                "--save_path", ckpt_path,
            ]
    main(argv)


## End-to-End: MinBPE Tokenizer + NanoGPT (SDPA/Flash Attention) + Sparse MoE Router (Top-k, Gumbel, Rescue Bias) + LoRA PEFT + KV-Cache Compression (fp16/int8) + Distillation + Pruning + Quantization + Mixed Precision & Grad Checkpointing + Reward Model & GRPO Alignment + TTA (Adaptive Compute) / Active Inference / System-2 Reasoning / Speculative Decoding + Retrieval-Augmented Compression (optional) + Sharded Training & CPU Offloading (stubs)


In [ ]:
# ===== Cell 1: Utils, Data, Tokenizer (MinBPE) =====
import os, math, json, time, random, urllib.request, argparse, shlex, unicodedata
from dataclasses import dataclass, asdict
from typing import List, Tuple, Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as _ckpt

# pretty logs (optional)
try:
    from rich.console import Console
    from rich.panel import Panel
    from rich import box
    console = Console()
    def info(msg, title=None, style="cyan"):
        console.print(Panel(msg, title=title or "info", border_style=style, box=box.ROUNDED))
except Exception:
    console = None
    def info(msg, title=None, style=None): print(f"[{title or 'info'}] {msg}")

# ------------------ Repro ------------------
def set_seed(seed=1337):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# ------------------ Data -------------------
def _repeat_to_len(s: str, target_len: int) -> str:
    if not s: s = " \n"
    return (s * ((target_len // len(s)) + 1))[:target_len]

# def load_tiny_shakespeare(data_dir="./data", target_len=100_000) -> Tuple[str, str]:
#     os.makedirs(data_dir, exist_ok=True)
#     path = os.path.join(data_dir, "tinyshakespeare_input.txt")
#     if not os.path.exists(path):
#         try:
#             url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
#             urllib.request.urlretrieve(url, path)
#         except Exception:
#             sample = ("From fairest creatures we desire increase,\n"
#                       "That thereby beauty's rose might never die,\n"
#                       "But as the riper should by time decease,\n"
#                       "His tender heir might bear his memory:\n")
#             open(path, "w", encoding="utf-8").write(_repeat_to_len(sample, target_len))
#     text = open(path, "r", encoding="utf-8").read()
#     split = int(0.9 * len(text))
#     return text[:split], text[split:]

import os
import urllib.request
import pandas as pd
from typing import Tuple

def _repeat_to_len(sample: str, target_len: int) -> str:
    """Repeat a string until reaching the target length."""
    return (sample * (target_len // len(sample) + 1))[:target_len]

def load_dataset(
    data_dir: str = "./data",
    target_len: int = 100_000,
    use_tinystories: bool = False,
    train_csv: str = "tinystories_train.csv",
    val_csv: str = "tinystories_val.csv"
) -> Tuple[str, str]:
    """
    Load either TinyShakespeare (default) or TinyStories if `use_tinystories=True`.

    Args:
        data_dir: Folder where data is stored.
        target_len: Length of fallback Shakespeare text if download fails.
        use_tinystories: If True, load TinyStories CSVs instead of TinyShakespeare.
        train_csv: Filename for TinyStories training CSV.
        val_csv: Filename for TinyStories validation CSV.

    Returns:
        (train_text, val_text)
    """
    os.makedirs(data_dir, exist_ok=True)

    if use_tinystories:
        # --- Load TinyStories from two CSV files ---
        train_path = os.path.join(data_dir, train_csv)
        val_path   = os.path.join(data_dir, val_csv)

        if not os.path.exists(train_path) or not os.path.exists(val_path):
            raise FileNotFoundError(
                f"Missing TinyStories files. Expected:\n - {train_path}\n - {val_path}"
            )

        train_df = pd.read_csv(train_path)
        val_df   = pd.read_csv(val_path)

        # Assume each row has a column 'text' with the story
        if "text" not in train_df.columns or "text" not in val_df.columns:
            raise ValueError("CSVs must contain a 'text' column with story strings.")

        train_text = " ".join(train_df["text"].dropna().astype(str).tolist())
        val_text   = " ".join(val_df["text"].dropna().astype(str).tolist())
        return train_text, val_text

    else:
        # --- Load TinyShakespeare ---
        path = os.path.join(data_dir, "tinyshakespeare_input.txt")
        if not os.path.exists(path):
            try:
                url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
                urllib.request.urlretrieve(url, path)
            except Exception:
                sample = (
                    "From fairest creatures we desire increase,\n"
                    "That thereby beauty's rose might never die,\n"
                    "But as the riper should by time decease,\n"
                    "His tender heir might bear his memory:\n"
                )
                open(path, "w", encoding="utf-8").write(_repeat_to_len(sample, target_len))

        text = open(path, "r", encoding="utf-8").read()
        split = int(0.9 * len(text))
        return text[:split], text[split:]

# ------------------------- Tokenizer Helpers -------------------------
def encode_text(tok, text: str):
    ids = tok.encode(text)
    return torch.tensor(ids, dtype=torch.long)


def decode_text(tok, ids: list[int]):
    """Convert list of token IDs back to string using the tokenizer."""
    if hasattr(tok, "decode"):
        return tok.decode(ids)
    raise AttributeError("Tokenizer object has no .decode method")

# ------------------ MinBPE -----------------
import regex as re

def _get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for p in zip(ids, ids[1:]): counts[p] = counts.get(p, 0) + 1
    return counts

def _merge(ids, pair, new_idx):
    out, i = [], 0
    while i < len(ids):
        if ids[i] == pair[0] and i < len(ids)-1 and ids[i+1] == pair[1]:
            out.append(new_idx); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

def _replace_ctl(s: str) -> str:
    return "".join(ch if unicodedata.category(ch)[0] != "C" else f"\\u{ord(ch):04x}" for ch in s)

def _render_tok(t: bytes) -> str:
    return _replace_ctl(t.decode("utf-8", errors="replace"))

class _BaseTok:
    def __init__(self):
        self.merges: Dict[Tuple[int,int], int] = {}
        self.pattern: str = ""
        self.special: Dict[str, int] = {}
        self.vocab: Dict[int, bytes] = self._build_vocab()
    def _build_vocab(self):
        v = {i: bytes([i]) for i in range(256)}
        for (a,b), i in self.merges.items(): v[i] = v[a] + v[b]
        for s, i in self.special.items(): v[i] = s.encode("utf-8")
        return v
    def save(self, prefix: str):
        os.makedirs(os.path.dirname(prefix), exist_ok=True) if os.path.dirname(prefix) else None
        with open(prefix + ".model", "w", encoding="utf-8") as f:
            f.write("minbpe v1\n"); f.write(self.pattern + "\n"); f.write(str(len(self.special)) + "\n")
            for s, i in self.special.items(): f.write(f"{s} {i}\n")
            for (a,b), _ in self.merges.items(): f.write(f"{a} {b}\n")
        with open(prefix + ".vocab", "w", encoding="utf-8") as f:
            inv = {i:p for p,i in self.merges.items()}
            for i, tok in self.vocab.items():
                s = _render_tok(tok)
                if i in inv:
                    a,b = inv[i]; sa=_render_tok(self.vocab[a]); sb=_render_tok(self.vocab[b])
                    f.write(f"[{sa}][{sb}] -> [{s}] {i}\n")
                else: f.write(f"[{s}] {i}\n")
    def load(self, model_file: str):
        assert model_file.endswith(".model")
        merges, special = {}, {}
        idx = 256
        with open(model_file, "r", encoding="utf-8") as f:
            assert f.readline().strip() == "minbpe v1"
            self.pattern = f.readline().strip()
            n = int(f.readline().strip())
            for _ in range(n):
                s, i = f.readline().strip().split(); special[s] = int(i)
            for line in f:
                a, b = map(int, line.split())
                merges[(a,b)] = idx; idx += 1
        self.merges, self.special = merges, special
        self.vocab = self._build_vocab()
    def decode(self, ids: List[int]) -> str:
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

class BasicTokenizer(_BaseTok):
    def train(self, text: str, vocab_size: int, verbose=False):
        assert vocab_size >= 256; merges = {}
        ids = list(text.encode("utf-8"))
        vocab = {i: bytes([i]) for i in range(256)}
        for i in range(vocab_size - 256):
            stats = _get_stats(ids)
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = _merge(ids, pair, idx); merges[pair] = idx; vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose: print(f"merge {i+1}: {pair} -> {idx}")
        self.merges, self.vocab = merges, vocab
    def encode(self, text: str) -> List[int]:
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = _get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges: break
            ids = _merge(ids, pair, self.merges[pair])
        return ids

GPT4_SPLIT = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

class RegexTokenizer(_BaseTok):
    def __init__(self): super().__init__(); self.pattern = GPT4_SPLIT; self.compiled = re.compile(self.pattern)
    def train(self, text: str, vocab_size: int, verbose=False):
        assert vocab_size >= 256; merges = {}; vocab = {i: bytes([i]) for i in range(256)}
        chunks = re.findall(self.compiled, text); ids_list = [list(c.encode("utf-8")) for c in chunks]
        for i in range(vocab_size - 256):
            stats = {}; [_get_stats(c, stats) for c in ids_list]
            pair = max(stats, key=stats.get); idx = 256 + i
            ids_list = [_merge(c, pair, idx) for c in ids_list]
            merges[pair] = idx; vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
            if verbose: print(f"merge {i+1}: {pair} -> {idx}")
        self.merges, self.vocab = merges, vocab
    def encode(self, text: str) -> List[int]:
        out = []
        for chunk in re.findall(self.compiled, text):
            ids = list(chunk.encode("utf-8"))
            while len(ids) >= 2:
                stats = _get_stats(ids)
                pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
                if pair not in self.merges: break
                ids = _merge(ids, pair, self.merges[pair])
            out.extend(ids)
        return out

def _new_tok(kind: str):
    return BasicTokenizer() if kind == "basic" else RegexTokenizer()

def train_or_load_tokenizer(kind, vocab_size, text, out_prefix, verbose=True):
    os.makedirs(os.path.dirname(out_prefix), exist_ok=True) if os.path.dirname(out_prefix) else None
    model_path = out_prefix + ".model"
    if os.path.exists(model_path):
        tok = _new_tok(kind); tok.load(model_path)
        if verbose: info(f"Loaded {model_path} (size={max(tok.vocab.keys())+1})", "Tokenizer", "magenta")
        return tok, model_path, max(tok.vocab.keys()) + 1
    tok = _new_tok(kind); vocab_size = max(256, vocab_size)
    if verbose: info(f"Training {kind} BPE (vocab={vocab_size})", "Tokenizer")
    tok.train(text, vocab_size, verbose=False); tok.save(out_prefix)
    if verbose: info(f"Saved tokenizer to {out_prefix}.model", "Tokenizer", "green")
    return tok, model_path, max(tok.vocab.keys()) + 1

# ------------------ batching ------------------
def get_batch_tokens(ids: List[int], block_size: int, batch_size: int, device: str):
    if len(ids) <= block_size + 1:
        raise RuntimeError(f"Token length {len(ids)} too small for block_size {block_size}")
    ix = torch.randint(len(ids) - block_size - 1, (batch_size,))
    x = torch.stack([torch.tensor(ids[i:i+block_size]) for i in ix]).long()
    y = torch.stack([torch.tensor(ids[i+1:i+1+block_size]) for i in ix]).long()
    return x.to(device), y.to(device)


In [ ]:
# ===== Cell 2: Model (GPT/MoE/LoRA), KV-cache, pruning/quant, ckpt =====
import types

# ------------------ Blocks ------------------
class LayerNorm(nn.Module):
    def __init__(self, n, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n))
        self.bias = nn.Parameter(torch.zeros(n)) if bias else None
    def forward(self, x): return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block, dropout=0.1, bias=True, backend="sdpa"):
        super().__init__(); assert n_embd % n_head == 0
        self.n_head=n_head; self.n_embd=n_embd; self.backend=backend; self.drop=dropout
        self.c_attn = nn.Linear(n_embd, 3*n_embd, bias=bias)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=bias)
        self.resid_dropout = nn.Dropout(dropout)
        if backend=="vanilla" or not hasattr(F, "scaled_dot_product_attention"):
            self.register_buffer("mask", torch.tril(torch.ones(block,block)).view(1,1,block,block))
    def forward(self, x, kv_cache=None):
        B,T,C = x.size()
        q,k,v = self.c_attn(x).split(C, dim=2)
        q = q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        k = k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
        v = v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)

        # KV cache hook (optional)
        if kv_cache is not None:
            k, v = kv_cache.append_and_get(k, v)  # [B,H,t,d]
        if self.backend == "sdpa" and hasattr(F, "scaled_dot_product_attention"):
            y = F.scaled_dot_product_attention(q, k, v, is_causal=(kv_cache is None), dropout_p=self.drop if self.training else 0.0)
        else:
            att = (q @ k.transpose(-2,-1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.mask[:,:,:T,:k.size(-2)].to(att.device) == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            y = att @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.c_proj(y))

# ------------------ MoE ------------------
class TopKRouter(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=1, temperature=1.0, add_gumbel=True):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, num_experts))
        self.k = k; self.temperature = float(temperature); self.add_gumbel = add_gumbel
        self.register_buffer("logits_bias", torch.zeros(num_experts))
    @staticmethod
    def _gumbel(shape, device): u = torch.rand(shape, device=device).clamp_(1e-9,1-1e-9); return -torch.log(-torch.log(u))
    def forward(self, x):
        logits = self.net(x) + self.logits_bias
        if self.add_gumbel and self.training: logits = logits + self._gumbel(logits.shape, logits.device)
        probs = F.softmax(logits / self.temperature, dim=-1)
        if self.k >= probs.size(-1): return probs, probs
        topk_vals, topk_idx = torch.topk(probs, self.k, dim=-1)
        mask = torch.zeros_like(probs); mask.scatter_(dim=-1, index=topk_idx, src=torch.ones_like(topk_vals))
        sp = probs * mask; sp = sp / (sp.sum(dim=-1, keepdim=True) + 1e-9)
        return sp, probs

def _entropy_mean(p, eps=1e-9): return -(p.clamp_min(eps) * p.clamp_min(eps).log()).sum(-1).mean()
def _kl_uniform(p, eps=1e-9):
    E = p.size(-1); m = p.mean(0); return torch.sum(m * (m.add(eps).log() - math.log(1.0/E)))

class MoEFFN(nn.Module):
    def __init__(self, in_dim, expansion=4, num_experts=3, k=1, router_hidden=128, dropout=0.1,
                 balance_w=0.02, entropy_w=0.001, add_gumbel=True, temperature=1.0,
                 rescue_lambda=0.0, usage_momentum=0.99):
        super().__init__()
        hidden = expansion * in_dim
        self.shared_fc = nn.Linear(in_dim, hidden); self.shared_proj = nn.Linear(hidden, in_dim)
        self.expert_fc = nn.ModuleList([nn.Linear(in_dim, hidden) for _ in range(num_experts)])
        self.expert_proj = nn.ModuleList([nn.Linear(hidden, in_dim) for _ in range(num_experts)])
        self.router = TopKRouter(in_dim, router_hidden, num_experts, k, temperature, add_gumbel)
        self.act = nn.GELU(); self.drop = nn.Dropout(dropout)
        self.balance_w = balance_w; self.entropy_w = entropy_w
        self.k = k; self.num_experts = num_experts
        self.register_buffer("ema_usage", torch.zeros(num_experts))
        self.rescue_lambda = rescue_lambda; self.usage_momentum = usage_momentum
    def forward(self, x):  # [B,T,C]
        B,T,C = x.shape; xf = x.reshape(B*T, C)
        shared = self.shared_proj(self.act(self.shared_fc(xf)))
        sp, dp = self.router(xf)
        routed = 0.0
        for e in range(self.num_experts):
            h = self.expert_proj[e](self.act(self.expert_fc[e](xf)))
            routed = routed + sp[:, e].unsqueeze(-1) * h
        y = (shared + routed).view(B,T,C); y = self.drop(y)
        # losses & stats
        with torch.no_grad():
            counts = (sp > 0).float().sum(0)
            mean_dp = dp.mean(0)
            m = self.usage_momentum; self.ema_usage.mul_(m).add_((1-m) * mean_dp)
            if self.rescue_lambda > 0 and self.training:
                target = torch.full_like(self.ema_usage, 1.0/self.num_experts)
                usage = self.ema_usage / (self.ema_usage.sum() + 1e-9)
                bias = self.rescue_lambda * (target - usage); bias = bias - bias.mean()
                self.router.logits_bias.copy_(bias)
        aux = self.balance_w * _kl_uniform(dp) - self.entropy_w * _entropy_mean(dp)
        stats = {"expert_selection_counts": counts, "mean_routing_probs": dp.mean(0)}
        return y, aux, stats

# ------------------ GPT ------------------
@dataclass
class GPTCfg:
    block_size: int = 256
    vocab_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.1
    bias: bool = True
    attention_backend: str = "sdpa"

    # MoE knobs
    use_moe: bool = False
    num_experts: int = 3
    k: int = 1
    router_hidden: int = 128
    balance_w: float = 0.02
    entropy_w: float = 0.001
    router_temp: float = 1.0
    add_gumbel: bool = False
    rescue_lambda: float = 0.0
    usage_momentum: float = 0.99

    # Training/utilities
    grad_checkpoint: bool = False   # <-- add this
    amp: bool = False               # <-- and this
    lora_r: int = 0                 # <-- if LoRA is enabled
    lora_alpha: int = 0
    lora_dropout: float = 0.0


class GPTBlock(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__()
        self.cfg = cfg
        self.ln1 = LayerNorm(cfg.n_embd, cfg.bias)
        self.attn = CausalSelfAttention(cfg.n_embd, cfg.n_head, cfg.block_size, cfg.dropout, cfg.bias, cfg.attention_backend)
        self.ln2 = LayerNorm(cfg.n_embd, cfg.bias)
        self.moe = MoEFFN(cfg.n_embd, 4, cfg.num_experts, cfg.k, cfg.router_hidden, cfg.dropout,
                          cfg.balance_w, cfg.entropy_w, cfg.add_gumbel, cfg.router_temp,
                          cfg.rescue_lambda, cfg.usage_momentum) if cfg.use_moe else None
        if self.moe is None:
            hidden = 4 * cfg.n_embd
            self.ffn = nn.Sequential(nn.Linear(cfg.n_embd, hidden, bias=cfg.bias),
                                     nn.GELU(), nn.Linear(hidden, cfg.n_embd, bias=cfg.bias),
                                     nn.Dropout(cfg.dropout))
    def forward_impl(self, x, kv_cache=None):
        x = x + self.attn(self.ln1(x), kv_cache=kv_cache)
        if self.moe is None:
            x = x + self.ffn(self.ln2(x)); aux = torch.tensor(0.0, device=x.device); stats = {}
        else:
            y, aux, stats = self.moe(self.ln2(x)); x = x + y
        return x, aux, stats
    def forward(self, x, kv_cache=None):
        if self.cfg.grad_checkpoint and self.training:
            return _ckpt(lambda _x: self.forward_impl(_x, kv_cache)[0], x), torch.tensor(0.0, device=x.device), {}
        return self.forward_impl(x, kv_cache)

class NanoGPT(nn.Module):
    def __init__(self, cfg: GPTCfg):
        super().__init__(); self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([GPTBlock(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = LayerNorm(cfg.n_embd, cfg.bias)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # weight sharing
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear): nn.init.normal_(m.weight, 0.0, 0.02);
        if isinstance(m, nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)
        if isinstance(m, nn.Embedding): nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None, kv_cache=None):
        B,T = idx.shape; assert T <= self.cfg.block_size
        pos = torch.arange(0, T, device=idx.device)
        x = self.wte(idx) + self.wpe(pos)[None, :, :]
        aux_total = torch.tensor(0.0, device=idx.device)
        for blk in self.blocks:
            x, aux, _ = blk(x, kv_cache)
            aux_total = aux_total + (aux if aux is not None else 0.0)
        x = self.ln_f(x); logits = self.lm_head(x); loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            loss = loss + aux_total
        return logits, loss

# ------------------------- Training loop -------------------------
def train_lm(model, train_ids, val_ids, block_size, epochs, steps_per_epoch,
             batch_size, lr, device, title="Train"):
    model.to(device)
    model.train()

    # Optimizer
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler("cuda" if device.startswith("cuda") else "cpu")

    def get_batch(split="train"):
        data = train_ids if split=="train" else val_ids
        ix = torch.randint(len(data) - block_size, (batch_size,))
        x = torch.stack([data[i:i+block_size] for i in ix])
        y = torch.stack([data[i+1:i+1+block_size] for i in ix])
        return x.to(device), y.to(device)

    for epoch in range(1, epochs+1):
        losses = []
        for step in range(steps_per_epoch):
            x, y = get_batch("train")
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda" if device.startswith("cuda") else "cpu"):
                logits, loss = model(x, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())

        # Eval on validation
        with torch.no_grad():
            x, y = get_batch("val")
            _, val_loss = model(x, y)

        print(f"[{title}] epoch {epoch}/{epochs} "
              f"train={sum(losses)/len(losses):.3f} val={val_loss.item():.3f}")

# ===== Utilities (add to Cell 2) =====
def sample_lm(model, tok, device, start="\n", tokens=100, title="Sample"):
    model.eval()

    # 🔧 force x onto the model’s device instead of CLI’s device arg
    model_device = next(model.parameters()).device
    x = torch.tensor([tok.encode(start)], dtype=torch.long, device=model_device)

    out = [start]

    with torch.no_grad():
        for _ in range(tokens):
            logits, _ = model(x[:, -model.cfg.block_size:], None)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx = torch.multinomial(probs, num_samples=1)
            out.append(tok.decode(idx[0].tolist()))
            x = torch.cat([x, idx], dim=1)

    txt = "".join(out)
    print(f"\n╭{'─'*99}╮")
    print(f"│ {title.center(97)} │")
    print(f"╰{'─'*99}╯")
    print(txt)
    return txt


# ------------------ LoRA (very small PEFT) ------------------
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16.0, dropout=0.0):
        super().__init__()
        self.base = base; self.r = r; self.alpha = alpha
        self.A = nn.Parameter(torch.zeros(base.in_features, r))
        self.B = nn.Parameter(torch.zeros(r, base.out_features))
        self.scaling = alpha / max(1, r)
        self.drop = nn.Dropout(dropout)
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
        for p in self.base.parameters(): p.requires_grad = False
    def forward(self, x):
        return self.base(x) + self.drop(x) @ self.A @ self.B * self.scaling

def inject_lora(model: NanoGPT, cfg: GPTCfg):
    if cfg.lora_r <= 0: return model
    for m in model.modules():
        if isinstance(m, CausalSelfAttention):
            m.c_attn = LoRALinear(m.c_attn, cfg.lora_r, cfg.lora_alpha, cfg.lora_dropout)  # pack QKV
            m.c_proj = LoRALinear(m.c_proj, cfg.lora_r, cfg.lora_alpha, cfg.lora_dropout)
        if isinstance(m, GPTBlock) and m.moe is None:
            lin1, lin2 = m.ffn[0], m.ffn[2]
            m.ffn[0] = LoRALinear(lin1, cfg.lora_r, cfg.lora_alpha, cfg.lora_dropout)
            m.ffn[2] = LoRALinear(lin2, cfg.lora_r, cfg.lora_alpha, cfg.lora_dropout)
    return model

# ------------------ KV Cache w/ compression ------------------
class KVCache:
    """Simple KV cache with dtype=fp16 or int8 quantized storage."""
    def __init__(self, mode="fp16"):
        self.mode = mode; self.k = None; self.v = None; self.scale = None
    def append_and_get(self, k, v):  # k,v: [B,H,T,d]
        if self.k is None:
            if self.mode == "int8":
                s = k.abs().amax(dim=(-2,-1), keepdim=True).clamp_min(1e-6)
                self.k = (k / s).to(torch.int8); self.v = (v / s).to(torch.int8); self.scale = s
            else:
                self.k, self.v = k.half(), v.half()
        else:
            if self.mode == "int8":
                s = torch.maximum(self.scale, k.abs().amax(dim=(-2,-1), keepdim=True))
                self.k = torch.cat([self.k, (k / s).to(torch.int8)], dim=-2)
                self.v = torch.cat([self.v, (v / s).to(torch.int8)], dim=-2)
                self.scale = s
            else:
                self.k = torch.cat([self.k, k.half()], dim=-2); self.v = torch.cat([self.v, v.half()], dim=-2)
        # dequant for attention
        if self.mode == "int8":
            kf = self.k.float() * self.scale; vf = self.v.float() * self.scale
        else:
            kf, vf = self.k.float(), self.v.float()
        return kf, vf

# ------------------ Pruning & Quantization ------------------
def global_magnitude_prune_(model: nn.Module, keep_ratio=0.9):
    params = torch.cat([p.detach().abs().flatten() for p in model.parameters() if p.requires_grad])
    k = int(keep_ratio * params.numel())
    threshold = params.kthvalue(max(1, min(params.numel()-1, params.numel()-k))).values.item()
    with torch.no_grad():
        for p in model.parameters():
            if p.requires_grad:
                mask = p.abs() >= threshold
                p.mul_(mask)
    info(f"Pruned to keep_ratio={keep_ratio}", "Pruning")

def dynamic_quantize_for_inference(model: nn.Module):
    try:
        model.eval()
        q = torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
        info("Applied dynamic int8 quantization to Linear layers.", "Quantization", "green")
        return q
    except Exception as e:
        info(f"quantization skipped: {e}", "Quantization", "yellow"); return model

# ------------------ Checkpoint I/O ------------------
def save_ckpt(path: str, payload: dict):
    os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None
    torch.save(payload, path); info(f"Saved to {path}", "Checkpoint", "magenta")

def load_ckpt(path: str) -> dict:
    pay = torch.load(path, map_location="cpu"); info(f"Loaded {path}", "Checkpoint", "magenta"); return pay


In [ ]:
# ===== Cell 3: Training & Inference Utilities (with pruning + quantization) =====
import argparse, json, shlex, os
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

# -------------- Core Train --------------
def train_xent(model, train_ids, val_ids, block_size, epochs, steps, batch, lr, device,
               amp=True, grad_clip=1.0, title="Train"):
    model = model.to(device)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, betas=(0.9,0.95), weight_decay=0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=amp and device.startswith("cuda"))
    for ep in range(1, epochs+1):
        model.train(); losses=[]; t0=time.time()
        for _ in range(steps):
            xb,yb = get_batch_tokens(train_ids, block_size, batch, device)
            with torch.cuda.amp.autocast(enabled=amp and device.startswith("cuda")):
                _, loss = model(xb, yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if grad_clip: scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(opt); scaler.update()
            losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            xb,yb = get_batch_tokens(val_ids, block_size, batch, device)
            _, vloss = model(xb, yb)
        dt = time.time()-t0
        info(f"epoch {ep}/{epochs} train={sum(losses)/len(losses):.3f} val={vloss.item():.3f} time={dt:.1f}s", title)

# -------------- Distillation --------------
def distill_step(student: NanoGPT, teacher: NanoGPT, xb, yb, temperature=1.0, kd_alpha=0.5):
    with torch.no_grad():
        t_logits, _ = teacher(xb, None); t_logits = t_logits / temperature
        t_probs = F.softmax(t_logits, dim=-1)
    s_logits, ce = student(xb, yb)
    kd = F.kl_div(F.log_softmax(s_logits/temperature, dim=-1), t_probs, reduction="batchmean") * (temperature**2)
    return kd_alpha * kd + (1 - kd_alpha) * ce

def train_distill(student, teacher, train_ids, val_ids, block_size, epochs, steps, batch, lr, device):
    student = student.to(device); teacher = teacher.to(device).eval()
    opt = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad], lr=lr)
    for ep in range(1,epochs+1):
        student.train(); losses=[]
        for _ in range(steps):
            xb,yb = get_batch_tokens(train_ids, block_size, batch, device)
            loss = distill_step(student, teacher, xb, yb)
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        xb,yb = get_batch_tokens(val_ids, block_size, batch, device); student.eval()
        with torch.no_grad():
            _, v = student(xb,yb)
        info(f"[Distill] epoch {ep}/{epochs} train={sum(losses)/len(losses):.3f} val={v.item():.3f}", "KD")

# -------------- Reward Model & GRPO --------------
class RewardModel(nn.Module):
    def __init__(self, base: NanoGPT): super().__init__(); self.base = base; self.head = nn.Linear(base.cfg.n_embd, 1)
    def forward(self, idx):  # mean pooled hidden (reuse logits path)
        with torch.no_grad():
            B,T = idx.shape
            pos = torch.arange(0,T,device=idx.device); x = self.base.wte(idx) + self.base.wpe(pos)[None,:,:]
            for blk in self.base.blocks: x,_,_ = blk.forward_impl(x) if hasattr(blk,'forward_impl') else blk(x)
            h = self.base.ln_f(x)
        return self.head(h.mean(1))

def load_pairs(path): return [json.loads(l) for l in open(path, "r", encoding="utf-8")]

def train_reward_model(rm, tok, pairs_path, block_size, epochs, steps, batch, lr, device):
    pairs = load_pairs(pairs_path)
    opt = torch.optim.AdamW(rm.parameters(), lr=lr)
    for ep in range(1,epochs+1):
        random.shuffle(pairs); losses=[]
        for _ in range(steps):
            b = random.sample(pairs, min(batch, len(pairs)))
            cl = [torch.tensor(tok.encode(p["chosen"])[:block_size]) for p in b]
            rl = [torch.tensor(tok.encode(p["rejected"])[:block_size]) for p in b]
            cl = nn.utils.rnn.pad_sequence(cl, batch_first=True).to(device)
            rl = nn.utils.rnn.pad_sequence(rl, batch_first=True).to(device)
            rc, rr = rm(cl), rm(rl)
            loss = -F.logsigmoid(rc - rr).mean()
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        info(f"[RM] epoch {ep}/{epochs} loss={sum(losses)/len(losses):.3f}", "RewardModel", "green")

def grpo_step(model, encode_fn, prompts, gen_tokens, opt, device):
    model.train(); rewards=[]; logps=[]
    for s in prompts:
        x = torch.tensor([encode_fn(s)], device=device); y = model.generate(x, gen_tokens)[0]
        inp, tgt = y[:-1].unsqueeze(0), y[1:].unsqueeze(0)
        logits, _ = model(inp, tgt)
        logp = F.log_softmax(logits, dim=-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1).mean()
        rewards.append(logp); logps.append(logp)
    rewards = torch.stack(rewards); adv = rewards - rewards.mean()
    loss = -(adv.detach() * torch.stack(logps)).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    return float(loss.item()), float(rewards.mean().item())

# -------------- Inference (TTC, Active, S2, Speculative) --------------
@torch.no_grad()
def generate_standard(model: NanoGPT, start_ids: List[int], max_new=64, temperature=1.0, top_k=50, top_p=1.0,
                      kv_mode=None, device="cpu"):
    model.eval().to(device)
    idx = torch.tensor([start_ids], device=device)
    cache = KVCache(kv_mode) if kv_mode else None
    for _ in range(max_new):
        cond = idx[:, -model.cfg.block_size:]
        logits, _ = model(cond, None, kv_cache=cache)
        logits = logits[:, -1, :] / max(1e-4, temperature)
        if top_k:
            v,_ = torch.topk(logits, min(top_k, logits.size(-1))); logits[logits < v[:,[-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p and top_p < 1.0:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum = torch.cumsum(sorted_probs, dim=-1)
            mask = cum > top_p
            sorted_probs[mask] = 0; probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            idx_next = torch.multinomial(probs, 1)
            idx_next = sorted_idx.gather(-1, idx_next)
        else:
            idx_next = torch.multinomial(probs, 1)
        idx = torch.cat([idx, idx_next], dim=1)
    return idx[0].tolist()

@torch.no_grad()
def generate_adaptive(model, start_ids, max_new=64, retry_budget=2, entropy_th=2.2, device="cpu"):
    """Adaptive test-time compute: retry tokens whose entropy is high."""
    out = start_ids[:]
    for _ in range(max_new):
        ids = torch.tensor([out[-model.cfg.block_size:]], device=device)
        logits,_ = model(ids, None)
        last = logits[:, -1, :]
        probs = F.softmax(last, dim=-1); H = -(probs * probs.clamp_min(1e-9).log()).sum(-1).item()
        if H > entropy_th and retry_budget > 0:
            # resample 3 times and pick the most confident
            cands = []; confs=[]
            for _ in range(3):
                nxt = torch.multinomial(probs, 1).item()
                conf = probs[0, nxt].item(); cands.append(nxt); confs.append(conf)
            out.append(cands[int(torch.tensor(confs).argmax())]); retry_budget -= 1
        else:
            out.append(torch.multinomial(probs, 1).item())
    return out

@torch.no_grad()
def generate_active_inference(model, start_ids, horizon=16, particles=4, device="cpu"):
    """Roll out multiple particles; pick token minimizing (NLL - entropy) ~ expected free energy."""
    ids = torch.tensor([start_ids[-model.cfg.block_size:]], device=device)
    logits,_ = model(ids, None); last = logits[:, -1, :]
    probs = F.softmax(last, dim=-1)
    topv, topi = torch.topk(probs, k=min(64, probs.size(-1)), dim=-1)
    cand_tokens = topi[0].tolist()
    scores = []
    for t in cand_tokens[:particles*4]:
        seq = start_ids + [t]
        # short rollout:
        seq_t = torch.tensor([seq[-model.cfg.block_size:]], device=device)
        l,_ = model(seq_t, None)
        p = F.softmax(l[:, -1, :], dim=-1); H = -(p * p.clamp_min(1e-9).log()).sum(-1).item()
        nll = -torch.log(probs[0, t].clamp_min(1e-9)).item()
        scores.append((nll - 0.1*H, t))
    scores.sort()
    return start_ids + [scores[0][1]]

@torch.no_grad()
def system2_reasoning(model, tok, prompt, passes=3, device="cpu"):
    """First-pass; then self-critique & refine; pick best by average logprob."""
    def score(ids):
        x = torch.tensor([ids[:-1]], device=device); y = torch.tensor([ids[1:]], device=device)
        logits,_ = model(x, y); lp = F.log_softmax(logits, dim=-1).gather(-1, y.unsqueeze(-1)).mean().item()
        return lp
    base = generate_standard(model, tok.encode(prompt), max_new=64, device=device)
    candidates = [base]
    for _ in range(passes-1):
        critique = " Please check for correctness and refine details: "
        ids = tok.encode(prompt + tok.decode(base[len(tok.encode(prompt)):]) + critique)
        refined = generate_standard(model, ids, max_new=48, device=device)
        candidates.append(refined)
    best = max(candidates, key=score)
    return best

@torch.no_grad()
def speculative_decoding(verifier: NanoGPT, draft: NanoGPT, start_ids, max_new=64, device="cpu"):
    """Small draft proposes tokens; verifier accepts if logprob agrees."""
    out = torch.tensor([start_ids], device=device)
    for _ in range(max_new):
        # draft proposes 4
        d_logits,_ = draft(out[:, -draft.cfg.block_size:], None)
        d_next = torch.argmax(d_logits[:, -1, :], dim=-1, keepdim=True)
        # verifier check
        v_logits,_ = verifier(out[:, -verifier.cfg.block_size:], None)
        v_probs = F.softmax(v_logits[:, -1, :], dim=-1)
        accept = v_probs.gather(-1, d_next).squeeze(-1) > 0.05
        if accept.item():
            out = torch.cat([out, d_next], dim=1)
        else:
            v_next = torch.multinomial(v_probs, 1)
            out = torch.cat([out, v_next], dim=1)
    return out[0].tolist()

# (reuse console, train_lm, sample_lm, save_ckpt, load_ckpt, GPTCfg, NanoGPT, etc. from previous cells)

# ------------------------- Pruning Utility -------------------------
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

def apply_pruning(model: nn.Module, prune_pct: float,
                  path_with_masks="artifacts/gpt_moe_pruned_with_masks.pt",
                  path_clean="artifacts/gpt_moe_pruned.pt",
                  extra_payload=None):
    """
    Apply pruning and save two versions:
      1) With pruning masks (weight_orig + weight_mask) -> path_with_masks
      2) Cleaned (pruning made permanent, only weight) -> path_clean
    """
    if prune_pct > 0:
        parameters_to_prune = []
        for _, module in model.named_modules():
            if isinstance(module, nn.Linear):
                parameters_to_prune.append((module, "weight"))
        prune.global_unstructured(
            parameters_to_prune,
            pruning_method=prune.L1Unstructured,
            amount=prune_pct,
        )
        print(f"[Pruning] Applied global {prune_pct*100:.1f}% sparsity.")

    # --- Save with masks ---
    payload = {
        "kind": "gpt_bpe",
        "cfg": model.cfg.__dict__,
        "state_dict": model.state_dict(),
    }
    if extra_payload:
        payload.update(extra_payload)

    torch.save(payload, path_with_masks)
    print(f"[Checkpoint] Saved pruned model with masks -> {path_with_masks}")

    # --- Remove pruning wrappers (make pruning permanent) ---
    for _, module in model.named_modules():
        if isinstance(module, nn.Linear) and hasattr(module, "weight_orig"):
            prune.remove(module, "weight")

    # --- Save cleaned version ---
    payload_clean = {
        "kind": "gpt_bpe",
        "cfg": model.cfg.__dict__,
        "state_dict": model.state_dict(),
    }
    if extra_payload:
        payload_clean.update(extra_payload)

    torch.save(payload_clean, path_clean)
    print(f"[Checkpoint] Saved cleaned pruned model -> {path_clean}")

    return model


# ------------------------- Quantization Utility -------------------------
def apply_quantization(model: nn.Module, quantize: str):
    if quantize == "int8_dynamic":
        model = torch.quantization.quantize_dynamic(
            model, {nn.Linear}, dtype=torch.qint8
        )
        print("[Quantization] Applied int8 dynamic quantization.")
    elif quantize == "fp16":
        model = model.half()
        print("[Quantization] Converted model to fp16.")
    return model

# ------------------------- Main CLI -------------------------
def main(argv=None):
    ap = argparse.ArgumentParser(description="BPE-based NanoGPT with MoE, pruning, quantization, and LoRA")
    ap.add_argument("--mode", required=True,
                   choices=["train_gpt_bpe","infer_gpt_bpe","train_rm","train_grpo","distill"])
    ap.add_argument("--device", default="auto", choices=["auto","cpu","cuda"])

    # Model size
    ap.add_argument("--block_size", type=int, default=256)
    ap.add_argument("--n_layer", type=int, default=12)
    ap.add_argument("--n_head", type=int, default=12)
    ap.add_argument("--n_embd", type=int, default=384)
    ap.add_argument("--dropout", type=float, default=0.1)
    ap.add_argument("--bias", action="store_true", default=True)
    ap.add_argument("--attention_backend", default="sdpa", choices=["sdpa","vanilla"])

    # MoE knobs
    ap.add_argument("--use_moe", action="store_true")
    ap.add_argument("--num_experts", type=int, default=5)
    ap.add_argument("--k", type=int, default=2)
    ap.add_argument("--router_hidden", type=int, default=128)
    ap.add_argument("--balance_w", type=float, default=0.02)
    ap.add_argument("--entropy_w", type=float, default=0.001)
    ap.add_argument("--router_temp", type=float, default=1.0)
    ap.add_argument("--add_gumbel", action="store_true")
    ap.add_argument("--rescue_lambda", type=float, default=0.0)
    ap.add_argument("--usage_momentum", type=float, default=0.99)

    # Training knobs
    ap.add_argument("--epochs", type=int, default=2)
    ap.add_argument("--steps_per_epoch", type=int, default=200)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--amp", action="store_true")
    ap.add_argument("--grad_checkpoint", action="store_true")
    ap.add_argument("--seed", type=int, default=1337)

    # LoRA knobs
    ap.add_argument("--lora_r", type=int, default=0, help="LoRA rank (0 disables)")
    ap.add_argument("--lora_alpha", type=int, default=16, help="LoRA alpha scaling")
    ap.add_argument("--lora_dropout", type=float, default=0.0, help="LoRA dropout")

    # Tokenizer
    ap.add_argument("--tok_kind", default="basic", choices=["basic","regex"])
    ap.add_argument("--vocab_size", type=int, default=512)
    ap.add_argument("--tok_prefix", default="artifacts/tokenizers/tiny_bpe")

    # Paths
    ap.add_argument("--save_path", default="artifacts/gpt_moe.pt")
    ap.add_argument("--load_path", default="artifacts/gpt_moe.pt")

    # Inference
    ap.add_argument("--sample_start", default="\n")
    ap.add_argument("--sample_tokens", type=int, default=100)
    ap.add_argument("--kv_cache", default="none", choices=["none","fp16","int8"])
    ap.add_argument("--ttc", action="store_true")        # adaptive test-time compute
    ap.add_argument("--active", action="store_true")     # active inference
    ap.add_argument("--system2", action="store_true")    # multi-pass refine
    ap.add_argument("--speculative", action="store_true")# speculative decoding

    # Advanced training
    ap.add_argument("--rm_pairs", default="")
    ap.add_argument("--teacher_path", default="")

    # NEW: pruning & quantization
    ap.add_argument("--prune_pct", type=float, default=0.0,
                    help="Fraction of weights to prune globally after training")
    ap.add_argument("--quantize", default="none", choices=["none","int8_dynamic","fp16"],
                    help="Quantization method for inference")

    args = ap.parse_args(argv)
    set_seed(args.seed)
    device = args.device if args.device!="auto" else ("cuda" if torch.cuda.is_available() else "cpu")

    # ------------------------- Train -------------------------
    if args.mode == "train_gpt_bpe":

        ## HEEEEEEEREEEEEE

        # train_txt, val_txt = load_tiny_shakespeare("./data")

        # # Load TinyShakespeare (default)
        # train, val = load_dataset()

        # Load TinyStories (requires tinystories_train.csv and tinystories_val.csv in ./data)
        train, val = load_dataset(use_tinystories=True,
                                  train_csv="tinystories_train.csv",
                                  val_csv="tinystories_val.csv")

        tok, tok_model_path, vocab_size_actual = train_or_load_tokenizer(
            args.tok_kind, args.vocab_size, train_txt + val_txt, args.tok_prefix, verbose=True
        )
        train_ids = encode_text(tok, train_txt)
        val_ids   = encode_text(tok, val_txt)

        cfg = GPTCfg(
            block_size=args.block_size, vocab_size=vocab_size_actual,
            n_layer=args.n_layer, n_head=args.n_head, n_embd=args.n_embd,
            dropout=args.dropout, bias=args.bias, attention_backend=args.attention_backend,
            use_moe=args.use_moe, num_experts=args.num_experts, k=args.k,
            router_hidden=args.router_hidden,
            balance_w=args.balance_w, entropy_w=args.entropy_w,   # matches new fields
            router_temp=args.router_temp, add_gumbel=args.add_gumbel,
            rescue_lambda=args.rescue_lambda, usage_momentum=args.usage_momentum
        )

        model = NanoGPT(cfg)

        # 🔧 Apply LoRA if requested
        if args.lora_r > 0:
            from peft import LoraConfig, get_peft_model
            lora_cfg = LoraConfig(
                r=args.lora_r, lora_alpha=args.lora_alpha,
                target_modules=["c_attn","c_proj","lm_head"],
                lora_dropout=args.lora_dropout, bias="none", task_type="CAUSAL_LM"
            )
            model = get_peft_model(model, lora_cfg)

        train_lm(model, train_ids, val_ids, args.block_size,
                 args.epochs, args.steps_per_epoch,
                 args.batch_size, args.lr, device, "GPT Train")

        # 🔧 Apply pruning before saving
        apply_pruning(
              model, args.prune_pct,
              path_with_masks="artifacts/gpt_moe_with_masks.pt",
              path_clean="artifacts/gpt_moe_clean.pt",
              extra_payload={
                  "tokenizer": {
                      "kind": args.tok_kind,
                      "model_path": tok_model_path,
                      "vocab_size": vocab_size_actual,
                  }
              }
          )


        save_ckpt(args.save_path, {
            "kind":"gpt_bpe",
            "cfg":cfg.__dict__,
            "state_dict":model.state_dict(),
            "tokenizer": {
                "kind": args.tok_kind,
                "model_path": tok_model_path,
                "vocab_size": vocab_size_actual,
            }
        })

    # ------------------------- Inference -------------------------
    elif args.mode == "infer_gpt_bpe":
        payload = load_ckpt(args.load_path); assert payload["kind"]=="gpt_bpe"
        tok_info = payload["tokenizer"]; tok = _new_tok(tok_info["kind"]); tok.load(tok_info["model_path"])
        cfg = GPTCfg(**payload["cfg"]); model = NanoGPT(cfg); model.load_state_dict(payload["state_dict"])

        # 🔧 Apply quantization for inference
        model = apply_quantization(model, args.quantize)

        sample_lm(model, tok, device, start=args.sample_start, tokens=args.sample_tokens, title="Sample")

    else:
        raise NotImplementedError(f"Mode {args.mode} not yet implemented in this demo")

# Run from notebook
def run_cli(cmd: str):
    argv = shlex.split(cmd)
    main(argv)




In [ ]:
# # ===== Cell 4: Example execution (safe/fast) =====

# # 1) Train a tiny GPT-BPE with MoE + AMP + LoRA (small so it runs quick)
# run_cli(
#     "--mode train_gpt_bpe --tok_kind basic --vocab_size 512 "
#     "--epochs 1 --steps_per_epoch 25 --batch_size 64 --lr 3e-4 "
#     "--use_moe --num_experts 3 --k 1 --router_hidden 128 "
#     "--amp --lora_r 8 --lora_alpha 16 --lora_dropout 0.0 "
#     "--save_path artifacts/gpt_moe.pt --tok_prefix artifacts/tokenizers/tiny_bpe"
# )

# # 2) Inference: standard + TTC/Active/System-2/Speculative examples
# print("\n--- Standard decoding ---")
# run_cli(
#     "--mode infer_gpt_bpe --load_path artifacts/gpt_moe.pt "
#     "--sample_start 'To be, or not to be,' --sample_tokens 60 --kv_cache fp16"
# )

# print("\n--- Adaptive Test-Time Compute (entropy-aware) ---")
# run_cli(
#     "--mode infer_gpt_bpe --load_path artifacts/gpt_moe.pt "
#     "--sample_start 'Once upon a time,' --sample_tokens 60 --ttc"
# )

# print("\n--- Active Inference (particles) ---")
# run_cli(
#     "--mode infer_gpt_bpe --load_path artifacts/gpt_moe.pt "
#     "--sample_start 'The quick brown fox' --active"
# )

# print("\n--- System-2 Reasoning (multi-pass refine) ---")
# run_cli(
#     "--mode infer_gpt_bpe --load_path artifacts/gpt_moe.pt "
#     "--sample_start 'Explain gravity simply.' --system2"
# )

# print("\n--- Speculative Decoding (draft vs verifier) ---")
# run_cli(
#     "--mode infer_gpt_bpe --load_path artifacts/gpt_moe.pt "
#     "--sample_start 'Hello world,' --sample_tokens 40 --speculative"
# )


In [ ]:
# ===== Cell 5: Long Training + Compression + Inference =====

# 1) Train a longer GPT-BPE with MoE + AMP + LoRA + Pruning
run_cli(
  "--mode train_gpt_bpe --tok_kind basic --vocab_size 512 "
  "--epochs 20 --steps_per_epoch 100 --batch_size 64 "
  "--use_moe --num_experts 3 --k 1 --router_hidden 128 "
  "--prune_pct 0.2 --save_path artifacts/gpt_moe_pruned.pt "
  "--tok_prefix artifacts/tokenizers/tiny_bpe"
)

# 2) Quantized inference (int8 dynamic) - Standard decoding
print("\n--- Standard decoding (quantized) ---")
run_cli(
    "--mode infer_gpt_bpe --load_path artifacts/gpt_moe_pruned.pt "
    "--sample_start 'In the beginning,' --sample_tokens 80 "
    "--quant int8_dynamic --kv_cache fp16"
)

# 3) Adaptive Test-Time Compute (entropy-aware)
print("\n--- Adaptive Test-Time Compute (entropy-aware) ---")
run_cli(
    "--mode infer_gpt_bpe --load_path artifacts/gpt_moe_pruned.pt "
    "--sample_start 'In the beginning,' --sample_tokens 80 --quantize int8_dynamic"
)

# 4) Active Inference (particles)
print("\n--- Active Inference (particles) ---")
run_cli(
    "--mode infer_gpt_bpe --load_path artifacts/gpt_moe_pruned.pt "
    "--sample_start 'The quick brown fox' --active"
)

# 5) System-2 Reasoning (multi-pass refine)
print("\n--- System-2 Reasoning (multi-pass refine) ---")
run_cli(
    "--mode infer_gpt_bpe --load_path artifacts/gpt_moe_pruned.pt "
    "--sample_start 'Explain quantum entanglement simply.' --system2"
)

# 6) Speculative Decoding (draft vs verifier)
print("\n--- Speculative Decoding (draft vs verifier) ---")
run_cli(
    "--mode infer_gpt_bpe --load_path artifacts/gpt_moe_pruned.pt "
    "--sample_start 'Hello world,' --sample_tokens 60 --speculative"
)


╭─────────────────────────────────────────────────── Tokenizer ───────────────────────────────────────────────────╮
│ Loaded artifacts/tokenizers/tiny_bpe.model (size=512)                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[GPT Train] epoch 1/20 train=5.168 val=4.397
[GPT Train] epoch 2/20 train=3.936 val=3.941
[GPT Train] epoch 3/20 train=3.643 val=3.792
[GPT Train] epoch 4/20 train=3.490 val=3.704
[GPT Train] epoch 5/20 train=3.291 val=3.539
[GPT Train] epoch 6/20 train=3.046 val=3.415
[GPT Train] epoch 7/20 train=2.806 val=3.261
[GPT Train] epoch 8/20 train=2.620 val=3.072
[GPT Train] epoch 9/20 train=2.459 val=3.194
[GPT Train] epoch 10/20 train=2.299 val=3.171
[GPT Train] epoch 11/20 train=2.139 val=3.172
[GPT Train] epoch 12/20 train=1.976 val=3.166
[GPT Train] epoch 13/20 train=1.794 val=3.342
[GPT Train] epoch 14/20 train=1.594 val=3.505
[GPT Train] epoch 15/20 train=1.397 val=3.640
[GPT Train] epoch 16/20 train=1.196 val=3.820
[GPT Train] epoch 17/20 train=1.014 val=3.910
[GPT Train] epoch 18/20 train=0.849 val=4.150
[GPT Train] epoch 19/20 train=0.717 val=4.349
[GPT Train] epoch 20/20 train=0.606 val=4.531
[Pruning] Applied global 20.0% sparsity.
[Checkpoint] Saved pruned model with masks -> ar

╭────────────────────────────────────────────────── Checkpoint ───────────────────────────────────────────────────╮
│ Saved to artifacts/gpt_moe_pruned.pt                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Standard decoding (quantized) ---


╭────────────────────────────────────────────────── Checkpoint ───────────────────────────────────────────────────╮
│ Loaded artifacts/gpt_moe_pruned.pt                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...


╭───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               Sample                                              │
╰───────────────────────────────────────────────────────────────────────────────────────────────────╯
The quick brown fox issue
As bringing forth our large together, in days,
My tongue could not prove a true-mous waste.
And this shall haply by our father's joy dog!
Farewell, my loving lord, what news?

DUCHESS 

--- System-2 Reasoning (multi-pass refine) ---


╭────────────────────────────────────────────────── Checkpoint ───────────────────────────────────────────────────╮
│ Loaded artifacts/gpt_moe_pruned.pt                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


╭───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               Sample                                              │
╰───────────────────────────────────────────────────────────────────────────────────────────────────╯
Explain quantum entanglement simply.':
If I must finds the grace with her shrunks of me.

QUEEN ELIZABETH:
Brother Clarence, be thou bAnd wrong'd he from you?

RIVERS:
So hath he tempt him in his lealth.

GLOUCESTER:
How far

--- Speculative Decoding (draft vs verifier) ---


╭────────────────────────────────────────────────── Checkpoint ───────────────────────────────────────────────────╮
│ Loaded artifacts/gpt_moe_pruned.pt                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


╭───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               Sample                                              │
╰───────────────────────────────────────────────────────────────────────────────────────────────────╯
Hello world,--

SICINIUS:

BRUTUS:
The very worthy choice is very well to discatch: he is gone
In him prattable, he shall not be se
